# Threshold-Aware Initialization (TAI) — Neurocomputing Experimental Harness

**Single-A100, multi-session, self-monitoring research artifact.** Colab Pro+, hard GPU-only, `Run all`. Stack pinned to PyTorch 2.6+, CUDA 12.4+, Transformers 4.49+.

---

## 1. Problem, method, and the gap this paper closes

In absmean ternary training a weight survives quantization to {−1,+1} only when |w| ≥ γ/2 (γ = mean|W|). For any **zero-mean** initialization the initial *active fraction* is ρ₀ ≈ 0.69 **independent of the std σ** (verified empirically in Cell 13b and derived analytically). Consequently *every* standard initializer — Gaussian, Xavier, Kaiming, LSUV — collapses to the **same** locked ternary operating point (~31% sparsity) and offers **no control** over where ternary training begins.

**Threshold-Aware Initialization (TAI)** makes ρ₀ an explicit, tunable hyperparameter, and an optional **γ-warmup** anneals the threshold during early training. The contribution is *control over the ternary operating point*, with consequences for (i) convergence speed, (ii) final quality at convergence, and (iii) the achievable quality–sparsity trade-off.

## 2. Pre-registered hypotheses (frozen before the full grid)

- **H1 — Convergence.** At an equal token budget, TAI reaches a calibrated target loss in fewer optimizer steps than the strongest ternary baseline.
- **H2 — Quality at the token budget.** At the 20M-token budget, TAI is *at least equivalent* to the pre-registered ternary baseline (BitNet b1.58). Tested with a paired t-test **and** a TOST equivalence test (margin derived from the ternary-vs-fp16 gap, not a magic constant), so "no worse" is itself falsifiable. **Whether this is a claim about *convergence* is decided by data, not by assumption:** Cell 11c re-runs both arms at 2× tokens and compares. If doubling the budget moves val loss by less than the equivalence margin, H2 is reported as *quality at convergence*; otherwise it is reported as *quality at a fixed 20M-token budget* and the manifest records `h2_wording_supported: false`. The stronger wording is never assumed.
- **H3 — Causal sparsity control (primary contribution).** Sweeping ρ₀ monotonically and causally shifts the *final learned* sparsity (per-seed Spearman over a 6-point grid, aggregated across 15 seeds). **Two arms are swept, not one:** `tai` (shape the init, keep γ=mean|W|) and `gamma_offset` (keep a standard init, scale γ by an analytically-derived constant to hit the same ρ₀). Without the second arm TAI would be the only method in the design capable of moving ρ₀ at all, and H3 could not fail for any reason other than broken machinery. With it, H3 is a real test *and* the sweep answers the attribution question directly: is the contribution the **operating point**, or TAI's **initializer**?
- **H4 — Stability.** TAI does not inflate variance (Levene). Variance is measured over the pooled **seed × fold** matrix — every run is scored on all K held-out folds while the model is still in memory — so the claim covers data-sampling spread and not only init/order seed spread.
- **H5 — Quality–sparsity Pareto.** There exists ρ₀* placing TAI on or above the baseline's quality-vs-sparsity frontier; the frontier is reported, not a single point.
- **H6 — Scale consistency.** The *direction* of the TAI effect is consistent across **six** model scales spanning ~1.4M → ~117M non-embed params, analyzed with a mixed-effects model (random intercept per seed) plus a per-scale sign-consistency test, not a 3-point regression.
- **H7 — Architecture transfer (revision R1).** The sign of the TAI effect is preserved on a **second block family** (GQA attention + SwiGLU FFN, `decoder_v2`) at two representative scales.
- **H8 — Operating-point optimality (revision R2).** Feeding the sweep-optimal ρ₀\* back into the grid (`tai_star`) at the top scale does not degrade — and is tested against — the fixed-ρ₀ result.

**Pre-registered falsification & confound controls (added).** (i) *H3 placebo* — a label-shuffled init (`tai_negctrl`) whose realised threshold split is fixed while the ρ₀ label varies; the causal sparsity effect must vanish here (per-seed sign test over 15 seeds). (ii) *Depth confound* — a width-only scale ladder (fixed `n_layers`) refits the H6 slope with depth held constant. (iii) *Init vs threshold* — a {fp16,ternary}×{standard,TAI} 2×2 isolates whether TAI's gain is a generic initializer effect or ternary operating-point control. (iv) *Leakage-free split* — a per-side guard band plus K interleaved val folds removes window-overlap leakage and lets H4 variance include data-sampling spread.

## 3. Design (frozen)

**Main grid:** a 9-point scale ladder (~1.4M → ≥50M non-embed) split into a **confirmatory set** (4 scales at the full method set × 15 seeds, token-equal at **20M tokens**, loss-plateau verified) and an **exploratory set** (remaining ladder points at reduced seeds, for the H6 scaling trend only), so every claim states which set it rests on. Methods are deliberately **non-degenerate**: `fp16_control` (upper bound), `standard` and `lsuv` (**one** family of the locked-ρ₀ baseline, not two — see §3b; the other locked inits are shown in Cell 13b to coincide and are not re-run as separate grid points), `bitnet158` (an **independent** published ternary recipe), `sparse_sched` (a **gradual-pruning / threshold-annealing baseline** so TAI is compared against an explicit sparsity-scheduling method and not only against fixed-threshold inits), **`gamma_offset` (the mechanism-matched control: standard init + a constant threshold scale calibrated to TAI's ρ₀ — the arm that lets the init-vs-threshold question be answered rather than assumed)**, `tai`, `tai_warmup`, and `tai_star` (ρ₀\* fed back from the sweep). **Plus:** a 6×15 ρ₀ sweep (H3/H5), a γ-warmup ablation, an LR-sensitivity ablation, and an efficiency table that is both **measured** (latency, tokens/sec, peak-memory, sparsity) and **theoretical** (effective 1.58-bit model size and compression vs fp16).

**Evaluation:** real GPT-2 BPE corpus (FineWeb-EDU), held-out lm-eval (ARC-Easy + HellaSwag, loglikelihood mode, reporting both **acc** and **acc_norm**), plus next-token top-1 as an internal transfer proxy. The synthetic copy-task proxy from the prior version is **dropped** (it mechanically tracks active-weight density and is not method-discriminating).

**Statistics:** paired t-test (primary) + Wilcoxon (support) + Cohen's d with nonparametric bootstrap CI + Holm-Bonferroni over the pre-registered family + Levene (H4) + **TOST** (H2) + **mixed-effects** scale model (H6). Post-hoc power is intentionally not reported (circular); CIs crossing zero are flagged as unresolved sign.

## 3b. Corrections made to this harness (and why they matter)

This notebook was audited and revised. Ten of the findings were not stylistic — they were defects that would have
invalidated results had the grid been run as written:

| # | Defect | Consequence had it shipped |
|---|---|---|
| **R0** | `n_layers`, `n_heads`, `ffn_hidden` were swallowed by an end-of-line comment and were never `Cfg` fields | Every `Cfg(...)` call raised `TypeError`; **the notebook could not execute at all**. A second bug (`n_kv_heads = n_heads//2`, not a divisor for odd head counts) would have malformed the entire `decoder_v2` ladder behind H7. |
| **R1** | `init_tai`'s advertised "8-step fixed-point search" was **dead code** — output is bit-identical with the loop deleted | The method section would have described a search that does nothing. The realised ρ₀ comes from `mask = u < ρ₀` by construction; the loop never changed an outcome. |
| **R3** | The plateau gate compared `mean(tail)` to `final` on the **single-batch training loss** | The gate measured batch noise, not convergence: perfectly flat synthetic traces passed only 63–100% of the time depending on the noise σ. A level test is also satisfied *mechanically* by cosine LR decay. **H2 rested entirely on this gate.** |
| **R6** | TAI was the **only arm able to move ρ₀ at all** | H3 was close to unfalsifiable-by-construction. It could not fail except through broken machinery. |
| **R13** | `init_lsuv` was orthogonal-init + a static rescale — **not LSUV**, whose defining half is the data-driven variance pass. The grid sold `{standard, lsuv}` as "two distinct locked-ρ₀ families" and took the per-seed **best of the two** as a baseline | Measured: ρ₀ = 0.6900±0.0011 vs 0.6906±0.0013 — one family, two labels. "Best of two" was max-selection over one family, biasing the baseline optimistically while advertising breadth the design lacked. Implementing LSUV properly revealed a deeper fact: BitLinear pre-normalises its input with RMSNorm, so output variance at init is already 1.0004 and **LSUV is mathematically degenerate in this architecture** — it can never be a second family here. Now measured (`LSUV_DEGENERATE`), reported, and the cross-architecture baseline (H7) rebuilt on `bitnet158` + `gamma_offset`. |
| **R14** | `train_run` set the global `QUANT_MODE`/`QUANT_RECIPE` and restored them ~64 lines later **with no `try/finally`** | `SystemExit` at the session boundary is this notebook's *designed* control flow. An `fp16_control` run hitting the budget leaked `full_precision=True`: the next session's `tai` and `bitnet158` runs would train in **full precision**, produce plausible losses, save under a valid `run_id`, and be skipped forever by `have_run()` — a ternary paper written from full-precision numbers, with `final_sparsity==0` as the only trace. Fixed with a context manager, an entry assertion, and a post-hoc sparsity guard. |
| **R15** | The split promised *interleaved* folds but built **contiguous slices**, and only fold 0's TRAIN was ever trained on | R5 scored folds 1–4 from **inside the training data** — measured **100% overlap**. H4's "data-sampling variance" would have been four train-loss readings presented as an honesty improvement. Rebuilt as one genuinely interleaved held-out region split into K unseen sub-folds. |
| **R16** | `BitLinear.sparsity()` ignored `QUANT_MODE` and `GAMMA_SCALE`, measuring a counterfactual instead of the forward pass that ran | `gamma_offset` — whose whole mechanism *is* a shifted γ — measured a flat ~31% at every ρ₀. H3's mechanism control, H5's second frontier and fig10 would have "proved" it cannot control sparsity **when only the ruler was wrong** (31.0% → 25.1% after the fix, matching TAI's 24.9%). `fp16_control` also reported 31.2% despite quantising nothing, and the function could never return 0 — silently disarming R14's guard. |
| **R17** | `train_run` routed init by `method.startswith("tai")` — which also matches **`tai_negctrl`**, H3's pre-registered *falsification* arm | The placebo silently received the **real** TAI init, so it tracked ρ₀ as strongly as the true arm (Spearman ≈ +1) against a frozen prediction of "FLAT (~0)" — and the manifest would have reported this under `h3_strong_falsification`. The paper's most rigorous instrument would have declared the primary contribution an artefact **even if the real effect were sound**. A broken placebo doesn't weaken a claim; it inverts it. Verified after the fix: placebo flat at 30.9% across labels 0.4–0.9 while the real arm moves 59.7% → 9.8%. |
| **R18** | Cell 18 — the **primary H1–H4 statistics cell** — referenced two never-defined names (`cp`, `cpm`), remnants of an unimplemented "copy accuracy" metric | `NameError` on the first method of the first scale: **the entire confirmatory analysis crashed before printing a single result**. Found by executing the cell on synthetic results; both removed. Invisible to static reading — the names looked like plausible locals. |

R15, R16, R17 and R18 were found only by **running the code** on CPU at toy scale — static reading had already passed over both several times. Each is documented in-place at the fix site and recorded in the manifest under `revisions`. The remaining revisions
(R2 TAI hyper-parameter validity region, R5 seed×fold variance for H4, R7 efficiency framing) tighten claims that were
weaker than they appeared.

**What this means for reading the results.** ρ₀ realisation is now *verified* per init call rather than asserted;
"plateau-verified" now means a slope CI on a held-out val trace; H2's wording is decided by a 2× budget check rather
than by assumption; and H3 now has a competitor that could beat it. Several of these changes make the paper's claims
*harder* to support, which is the point — a claim that cannot fail is not evidence.

## 4. Honesty boundary

Every number is measured at the scale that fits one A100. The top scale (≥50M non-embed) is a *mid-scale confirmation point*, not a 2B/4T claim; all conclusions are restricted to the tested regime in the Limitations, and the manifest records the exact boundary. Downstream ability is read via the loglikelihood protocol (above-chance, method-discriminating at this scale), never free generation. We report whatever the data show, including null/mixed H1/H2. **Three limits are structural and are stated rather than worked around:** (i) no ternary kernel is benchmarked — the deployment numbers in Cell 13e are an arithmetic model, and the measured table is the cost of *simulating* ternary in bf16, not of running it; (ii) the scaling-law extrapolation is labelled PREDICTION-not-evidence and H6's primary readout is per-scale sign-consistency, not the slope; (iii) if the 2× budget check fails, "convergence" leaves the paper's vocabulary and H2 becomes a token-equal claim.

## 5. Cell map
1 GPU guard · 2 env+determinism · 3 Drive+config · 4 notifier+hooks · 5 quant primitives · 6 BitLinear(+FP16 + BitNet-b1.58 switch) · 7 model (9-scale ladder) · 8 TAI + init strategies · 9 Drive-cached data (real GPT-2 BPE, 20M+ tokens) · 10 run-store · 11b lm-eval (loglik) · **11 main experiment (plateau-verified, budget+email+resume)** · 12 **ρ₀ sweep + Pareto** · 13 formal statistics (+TOST +mixed-effects) · 13b theory verification · 13c γ-warmup ablation · 12b **ρ₀\* feedback (R2)** · 12c **cross-architecture decoder_v2 (R1)** · 13 statistics · 13b theory · 13c γ-warmup ablation · 13d LR ablation · **13e efficiency (measured)** · **13f data-sampling variance for H4 (#5)** · **13g fp16+TAI-init 2×2 ablation (#3)** · 14 figures+LaTeX (**+fig7 H3 placebo/SNR, +fig8 init×precision 2×2, +fig9 two-slope, +results\_controls.tex**) · 15 manifest+final email.

## ▶ How to Run (two-stage, Run-All-safe)

**Stage 1 — Preflight (automatic on first Run All).** With `RUN_CFG["preflight_only"]=True` (default), Run All builds the real GPT-2 BPE corpus, loads lm-eval tasks, runs one short smoke run on the smallest scale, prints a **PREFLIGHT REPORT** with a wall-clock projection across the declared scale ladder (9 points; 4 confirmatory + 5 exploratory), then **stops** (`SystemExit`) before the full grid. *A stop here is expected.*

**Stage 2 — Full experiment.** If `used_synthetic: false` and the projected hours/sessions are acceptable, set `RUN_CFG["preflight_only"]=False` in **Cell 3** and re-run. The grid checkpoints per run to Drive and exits cleanly at the session budget; re-run each session until it reports complete. Because the top scale (≥50M) is heavy, expect **several** sessions — resume is automatic.

**Read in the preflight report:** `used_synthetic:false` (real corpus), `lmeval_available:true`, `plateau_token_budget` (≥20M), and `projection.sessions_needed`.

---

### 🛡️ Run-All safety (what to expect)

This notebook is **safe to `Runtime → Run all`** at any time:

- **Setup cells** (1–10) always run: GPU guard, pinned stack, Drive mount, notifier, data build/cache, lm-eval task load.
- **Preflight stop is expected.** With `preflight_only=True` the run intentionally stops after the projection (`SystemExit`). Every heavy phase below is wrapped in a `_require(...)` guard, so if the run stopped (or a session-budget boundary exited mid-grid), those cells **print a one-line skip notice instead of crashing**. No `NameError` cascade.
- **To run the real experiment:** set `RUN_CFG["preflight_only"]=False` in Cell 3, then Run All again. Finished runs are skipped via the per-run checkpoint store; the grid resumes automatically across sessions until the manifest reports complete.
- **Optional tools degrade gracefully:** missing email secrets → console+log only; missing `statsmodels` → descriptive scale fallback; no network for FineWeb-EDU → flagged synthetic corpus (downstream lm-eval auto-disabled, never faked).


In [ ]:
# === CELL 1: HARD GPU-ONLY GUARD ===========================================
# Timing/throughput require A100; abort on CPU so misconfig can't corrupt results.
import sys
def _abort(m):
    print("\nFATAL:",m); print("Set Runtime -> A100 GPU, then Run all."); raise SystemExit(1)
try: import torch
except ImportError: _abort("Run after Cell 2 installs the stack.")
if not torch.cuda.is_available(): _abort("No CUDA device.")
cc=torch.cuda.get_device_capability(0)
print("GPU:",torch.cuda.get_device_name(0),"| sm_%d%d"%cc,
      "| VRAM %.1f GB"%(torch.cuda.get_device_properties(0).total_memory/1024**3))
print("A100 confirmed." if cc==(8,0) else "WARNING: not A100; timing assumes A100.")


In [ ]:
# === CELL 2: PINNED 2026+ STACK + GLOBAL DETERMINISM =======================
import os,sys,subprocess,random
def _pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a],check=False)
_pip("--upgrade","transformers>=4.49.0","datasets>=3.2.0","accelerate>=1.3.0",
     "matplotlib>=3.9.0","scipy>=1.14.0","pynvml>=12.0.0","tokenizers>=0.20.0",
     "statsmodels>=0.14.0","pandas>=2.2.0")
import torch,numpy as np
print("torch",torch.__version__,"| cuda",torch.version.cuda)
GLOBAL_SEED=1337
os.environ["PYTHONHASHSEED"]=str(GLOBAL_SEED); os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
def seed_everything(s=GLOBAL_SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()
# ============================================================================
# (R22) FAST_MODE — THROUGHPUT vs BIT-DETERMINISM, MADE AN EXPLICIT CHOICE.
# ----------------------------------------------------------------------------
# The original harness bought BIT-LEVEL reproducibility at a large throughput
# cost: SDPA pinned to the MATH backend (no flash kernel; the full B*H*T*T score
# matrix is materialised and kept for backward), deterministic algorithms on,
# cudnn autotuning off. Measured consequence: the declared grid needs ~250 A100-
# hours, most of it paid for a guarantee NO CLAIM IN THIS PAPER ACTUALLY USES.
#
# What the claims DO need is that every method sees the SAME init and the SAME
# data order for a given seed. That is provided by torch.manual_seed + the
# per-run generators in train_run, and is UNAFFECTED by anything below: flash
# attention changes only the floating-point reduction order inside one kernel,
# not which batch a seed draws. H1-H8 are all seed-aggregated statistics over
# n=15; they are invariant to last-bit reduction order by construction.
#
# What is LOST: rerunning one run twice will not give bit-identical loss. The
# manifest therefore reports "statistically reproducible", not "bit-reproducible"
# whenever FAST_MODE is on -- stated, not hidden.
#
# Set fast=False to restore the original bit-deterministic behaviour exactly.
FAST_CFG={
    "fast":True,          # master switch for everything in this block
    "flash_sdpa":True,    # allow flash / mem-efficient attention kernels
    "tf32":True,          # tf32 matmul (already on in the original)
    "ce_chunks":4,        # chunked cross-entropy (see Cell 7) -- lower peak VRAM
    "pin_batches":True,   # pinned staging buffer + async H2D in get_batch
    # torch.compile is deliberately DEFAULT-OFF. It is not a free win here:
    # BitLinear.forward reads GAMMA_SCALE["val"] as a PYTHON FLOAT, and dynamo
    # specialises on python-float values -- so the gamma-warmup and sparse_sched
    # arms, which change it EVERY STEP, would trigger a recompile per step and
    # run slower than eager. Enabling it safely needs GAMMA_SCALE turned into a
    # device tensor first. Left as a documented follow-up, not a silent trap.
    # (R25) ON, on measured evidence. tai_bench2 on the target A100 gave 2.03x
    # at dim512 (1.71x at dim384) with an eager-vs-compiled loss delta of 0.051
    # against a same-code run-to-run NOISE FLOOR of 0.029 -- i.e. the difference
    # is not distinguishable from the non-determinism flash attention already
    # introduces. The preflight A/B re-checks this on the REAL corpus before the
    # grid starts, so the decision is verified twice on different data.
    # Compile also LOWERS peak VRAM (10.09 -> 8.88 GB at dim384), which widens
    # dim1536's margin on a 40 GB card.
    "compile":True,
}
# (R25) Persist the inductor cache. Without this every one of the ~2800 runs
# re-pays kernel compilation; with it, only the first run of each (scale, arch)
# shape does. Session-local is enough -- a session runs hundreds of runs.
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR","/content/_inductor_cache")
_FAST=bool(FAST_CFG["fast"])
torch.backends.cudnn.deterministic=(not _FAST)
torch.backends.cudnn.benchmark=_FAST          # autotune conv/attn algos
torch.backends.cuda.matmul.allow_tf32=bool(FAST_CFG["tf32"]); torch.backends.cudnn.allow_tf32=True
try:
    from torch.nn.attention import sdpa_kernel,SDPBackend
    if _FAST and FAST_CFG["flash_sdpa"]:
        # prefer flash, fall back to mem-efficient, then math -- torch picks the
        # first that supports the given shapes/dtype.
        _SDPA_CTX=lambda: sdpa_kernel([SDPBackend.FLASH_ATTENTION,
                                       SDPBackend.EFFICIENT_ATTENTION,
                                       SDPBackend.MATH])
    else:
        _SDPA_CTX=lambda: sdpa_kernel(SDPBackend.MATH)
except Exception:
    from contextlib import nullcontext; _SDPA_CTX=nullcontext
try: torch.use_deterministic_algorithms(not _FAST,warn_only=True)
except Exception as e: print("det warn:",e)
DEVICE=torch.device("cuda"); DTYPE=torch.bfloat16
if _FAST:
    print("FAST_MODE ON  -> flash SDPA, cudnn autotune, non-deterministic kernels.")
    print("  seed-level comparability preserved; BIT-level reproducibility is not.")
    print("  set FAST_CFG['fast']=False to restore the original guarantee.")
else:
    print("Determinism locked (bit-reproducible). seed",GLOBAL_SEED)


In [ ]:
# === CELL 3: DRIVE MOUNT + PROJECT TREE + RUN CONFIG =======================
# All persistent artifacts live under /MyDrive/TAI_neuro/ so the multi-session
# sweep survives session resets and never re-downloads data. RUN_CFG centralizes
# every tunable. NEUROCOMPUTING REVISION: a 9-point scale ladder (~1.4M -> >=50M
# non-embed params) is DECLARED, but to stay honest under a fixed single-A100
# budget we separate (i) a CONFIRMATORY scale set that runs the full
# method x seed grid at token-equal 20M, from (ii) the remaining EXPLORATORY
# scales used only for the scaling-law trend (fewer seeds allowed). This removes
# the earlier '6 vs 9 scales' inconsistency: every claim states WHICH set it
# rests on. 20M-token plateau-verified budget, non-degenerate method set, 6-pt rho0.
import os,time,json
from pathlib import Path
IN_COLAB="google.colab" in sys.modules or os.path.exists("/content")
def mount_drive():
    if not IN_COLAB:
        print("Not in Colab -> local ./TAI_neuro"); return "./TAI_neuro"
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        return "/content/drive/MyDrive/TAI_neuro"
    except Exception as e:
        print("Drive mount failed -> local:",str(e)[:80]); return "/content/TAI_neuro"
PROJECT_ROOT=mount_drive()
PATHS={k:os.path.join(PROJECT_ROOT,k) for k in ["data","runs","figures","results","logs"]}
for p in PATHS.values(): Path(p).mkdir(parents=True,exist_ok=True)
print("Project root:",PROJECT_ROOT)
# ============================================================================
# (R31) CONSOLE TEE — the H1-H8 verdicts must survive the session.
# ----------------------------------------------------------------------------
# Cell 13 prints every hypothesis decision (supported / not, p, Cohen's d, CI,
# Holm-corrected) to stdout and nowhere else. The grid spans 3-4 sessions, so
# those lines live in a browser scrollback that a disconnect or a Runtime->
# Restart erases -- and they are the single most important artifact for the
# paper. Mirroring stdout to Drive costs nothing and duplicates no logic: the
# statistics cell stays the one source of truth, we just keep what it says.
# Appends, so re-running a session adds to the same day's log rather than
# clobbering it.
import sys,time as _time
class _Tee:
    def __init__(s,path):
        s.f=open(path,"a",buffering=1); s.out=sys.__stdout__
    def write(s,d):
        s.out.write(d)
        try: s.f.write(d)
        except Exception: pass
    def flush(s):
        s.out.flush()
        try: s.f.flush()
        except Exception: pass
    def isatty(s): return False
CONSOLE_LOG=os.path.join(PATHS["logs"],f"console_{_time.strftime('%Y%m%d')}.log")
try:
    if not isinstance(sys.stdout,_Tee):
        sys.stdout=_Tee(CONSOLE_LOG)
        print(f"[tee] console mirrored to {CONSOLE_LOG}")
except Exception as _e:
    print("console tee unavailable:",_e)
RUN_CFG={# (R30) SESSION BUDGET DISABLED — None means "run until the platform stops
         # us". R22 had set a self-imposed 22 h cutoff inside a 24 h Colab Pro+
         # session, which threw away ~2 h per session (~8 h, roughly 10% of the
         # whole experiment) to avoid a cost the checkpoint layer already covers:
         # save_run writes a temp file and os.replace()s it, which is atomic, so a
         # session killed mid-write CANNOT corrupt the store. What an abrupt kill
         # actually loses is the single run in flight -- a couple of minutes at the
         # small scales, ~7 minutes at dim1536 -- and have_run() re-runs exactly
         # that one on the next Run All. Paying 2 h to protect a 7-minute loss is
         # a bad trade. Set to a number (e.g. 22.0) to re-enable the soft cutoff.
         "session_budget_hours":None,
         # --- (R35) TOKEN-EQUAL BUDGET: 20M tokens. --------------------------
  #     PRE-REGISTRATION SAID "plateau-verified"; THE MEASUREMENT SAYS OTHERWISE.
  #     Across the first 502 main-grid runs, plateau_ok was true on ZERO of them:
  #     at dim384 the val trace still falls ~0.21 nats over the final 40% of
  #     probes, uniformly across scales (-0.20 to -0.27). The gate is working;
  #     the budget simply does not reach convergence.
  #     Consequence, recorded BEFORE the results were analysed: the protocol is
  #     TOKEN-EQUAL AT A FIXED BUDGET, which is legitimate and applies identically
  #     to every arm, but H1/H2 may NOT be worded as claims about converged
  #     quality. H2's TOST is defined over plateau-verified pairs and will
  #     therefore be reported as SKIPPED rather than computed over an empty set.
  #     The manifest field h2_operative_wording carries the licensed phrasing.
  #     The budget itself is UNCHANGED -- it is pre-registered, and moving it now
  #     to chase a plateau would be exactly the post-hoc adjustment this harness
  #     exists to prevent.
         # At bs*seq = 32*512 = 16384 tok/step this is ~1221 steps/run. Cell 11
         # gates each run with the (R3) slope test on the held-out val trace and
         # records the verdict in the manifest. (The comment previously described
         # the old level-test on train loss, which no longer exists.)
         "token_budget":20_000_000,"batch_size":32,"seq_len":512,
         "warmup_steps":120,"peak_lr":2e-3,"min_lr":1e-4,"weight_decay":0.1,"grad_clip":1.0,
         # --- (R3) PLATEAU GATE, REBUILT. The old gate compared mean(tail) to the
         #     final value of the SINGLE-BATCH TRAIN loss. That is dominated by
         #     batch noise (sigma ~0.05-0.15 nats vs a ~0.04 nat band), and a
         #     LEVEL comparison is satisfied mechanically by cosine LR decay. The
         #     gate now fits a SLOPE to the held-out VAL trace over the last
         #     plateau_frac of probes and requires the trend to be indistinguish-
         #     able from zero in BOTH directions (still-descending => undertrained;
         #     ascending => unstable). plateau_frac is raised 0.15 -> 0.40 so the
         #     tail holds >= plateau_min_tail probes and the slope CI has power.
         "plateau_tol":0.01,"plateau_frac":0.40,
         "plateau_min_tail":5,          # min val probes in tail; below -> cannot certify
         "eval_every_frac":0.08,        # val probe every 8% of steps (~12 probes/run)
         "eval_batches_trace":12,       # eval batches per probe (variance vs cost)
         # --- (R3) BUDGET-SUFFICIENCY CHECK. "Plateau at 20M tokens" is an
         #     assumption H2 rests on entirely. Cell 11c re-runs the primary scale
         #     at 2x tokens on budget_check_seeds and compares val_loss at 20M vs
         #     40M against the TOST margin. If the gap exceeds the margin, H2 is
         #     re-stated as "quality at a 20M-token budget", not "at convergence".
         "budget_check":True,
         "budget_check_mult":2.0,
         "budget_check_seeds":[1337,2026,7],
         # n=15 seeds for adequate power; effect sizes + bootstrap CIs reported.
         "seeds":[1337,2026,7,42,2718,9001,13,271,99,555,8,31415,777,2024,11],
         # 6-point rho0 grid (was 4): supports a Spearman trend AND a Pareto curve.
         "rho0_grid":[0.4,0.5,0.6,0.7,0.8,0.9],
         # --- (a) LEAKAGE-FREE CALIBRATION: calib seeds are DISJOINT from the
         #     15 reporting seeds above, so per-method LR* (and H1 target loss)
         #     are never selected on a seed whose final metric is reported.
         "calib_seeds":[100001,100002,100003],
         # --- (#1) PER-METHOD LR FAIRNESS: each (scale,method) gets its own LR*
         #     chosen on calib_seeds over lr_grid; the main grid then runs every
         #     method at ITS best LR, so no method is penalised by a shared LR.
         "per_method_lr":True,
         # --- (#2) WALL-CLOCK-EQUAL H1: in addition to the token-equal budget,
         #     report loss reached under a fixed wall-clock budget (incl. the
         #     TAI init-calibration + gamma-warmup overhead, timed honestly).
         # (R22) OFF by default: this phase re-runs the ENTIRE confirmatory grid
         # (4 scales x 8 methods x 15 seeds = 480 runs, ~10% of all compute) to
         # produce a SECONDARY reading of H1. Token-equal H1 is the pre-registered
         # primary. Turn back on only if a reviewer asks for the compute-equal
         # variant, or if wall-clock allows after the confirmatory grid lands.
         "wallclock_equal_h1":False,
         # --- (#4) CAUSAL INTERVENTION: rho0 sweep shares one base-noise tensor
         #     per seed; ONLY the threshold changes across rho0 (ceteris paribus).
         "causal_intervention":True,
         # --- (#6) PLATEAU ROBUSTNESS grid (recompute-only, no new runs).
         "plateau_tol_grid":[0.005,0.01,0.02],
         "plateau_frac_grid":[0.10,0.15,0.25],
         "lmeval_n":300,
         # ---- (R22) lm-eval COST CONTROL -------------------------------------
         # lm-eval was the single largest non-training cost: 2 tasks x 300 items
         # x ~4 options = ~2400 forward passes AT BATCH SIZE 1, per run, over
         # ~2800 runs. Two independent fixes:
         #  (i)  lmeval_batch: score many (item,option) sequences in ONE padded
         #       forward instead of one at a time (Cell 11b). Right-padding is
         #       safe under causal masking: a pad token can only affect positions
         #       AFTER it, and every scored position precedes the padding.
         #  (ii) lmeval_max_seeds: downstream accuracy is a coarse, high-variance
         #       readout used for a secondary paired test; it does not need all 15
         #       seeds. We score the FIRST k seeds of each (scale,method) cell --
         #       the same k for every method, so the comparison stays paired. The
         #       statistics cell already NaN-masks missing values, so unscored
         #       seeds drop out of the pairing cleanly rather than corrupting it.
         "lmeval_batch":16,
         "lmeval_max_seeds":3,
         # ---- (R22) LR CALIBRATION LENGTH ------------------------------------
         # The LR sweep was the LARGEST single compute item in the whole harness
         # (~42% of all training FLOPs): 11 scales x 8 methods x 3 LRs x 3 calib
         # seeds, each running the FULL 20M-token budget. But its only output is
         # an ARGMIN over 3 LRs -- the ranking separates long before convergence.
         # 300 steps (~5M tokens) is ample and cuts ~32% of total compute. The
         # chosen LR* is still applied to full-length runs, and the full curve is
         # still stored, so nothing downstream changes except the calibration cost.
         "lr_cal_steps":300,
         # (R26) divergence bound as a multiple of log(vocab). A model that has
         # learned nothing still predicts uniform and scores log(50257)=10.83;
         # anything materially above that is diverging, not merely undertrained.
         "divergence_factor":1.25,
         # (R23) eager-vs-compiled A/B inside the preflight. NOTE: _preflight()
         # runs on EVERY Run All -- preflight_only only controls whether the grid
         # follows it -- so this costs ~2 min per session, ~5 min for the whole
         # preflight. Kept True deliberately: at the start of each session it
         # re-verifies the corpus, the environment and the measured cost model on
         # that day's machine, which is worth minutes out of a 24 h session. Set
         # False if you would rather skip straight into the grid.
         "preflight_bench_compile":True,
         # ============================================================
         # RUN-ALL SAFETY.  A naive "Run All" must NOT silently launch a
         # multi-day training grid. Two gates:
         #   ARM_FULL_RUN=False -> every training phase is skipped; only
         #                         cheap setup + (optionally) preflight run.
         #   preflight_only=True -> even when armed, the experiment cell does
         #                         the smoke-test + budget projection and STOPS
         #                         before the grid. Flip BOTH to actually train.
         # To run the real experiment: set ARM_FULL_RUN=True AND
         # preflight_only=False, then Run All (resume is automatic across
         # sessions; safe to re-run).
         # ============================================================
         "ARM_FULL_RUN":True,     # ARMED for PHASE 0. Training is now permitted; the
                         # preflight_only switch below still stops the run before
                         # the full grid. Set False again to return to safe mode.
         "preflight_only":False,  # LAUNCHED. Phase 0 passed on the target A100:
                         # corpus real (FineWeb-EDU, not synthetic), lm-eval live,
                         # init loss 10.155 against a uniform baseline of 10.83,
                         # compile verified 1.29x at dim192 / 2.0-2.5x at dim512
                         # and dim1024 with a loss delta of 0.013, and a measured
                         # cost model of 55-75 GPU-hours over 3-4 sessions.
                         # Set back to True to re-run the preflight only.
         "bonferroni_alpha":0.0125,
         # --- (FIX-4) TOST margin is NO LONGER a magic constant. It is DERIVED
         #     from the data as a fraction of the fp16_control -> strongest-ternary
         #     gap at each scale: margin = tost_margin_frac * (loss_tern* - loss_fp16).
         #     Rationale: 'equivalent' must mean 'small relative to how much ternary
         #     already costs vs full precision', not an arbitrary nat value. The
         #     fixed value below is only a fallback when fp16_control is absent.
         "tost_margin":0.02,        # fallback only (nats)
         "tost_margin_frac":0.10,   # H2 margin = 10% of the ternary-vs-fp16 gap
         # --- (FIX-3) PRE-REGISTERED PRIMARY BASELINE. The grid no longer lets the
         #     statistics cell pick whichever ternary baseline LOOKS weakest per
         #     scale (a hidden researcher degree of freedom / winner's curse). The
         #     PRIMARY comparison is frozen to an INDEPENDENT published recipe
         #     (BitNet b1.58). The data-chosen 'min over baselines' contrast is
         #     still reported, but only as a SECONDARY sensitivity analysis.
         "primary_baseline":"bitnet158",
         # (R6) gamma_offset joins the secondary baselines: it is the ONLY arm
         #      besides TAI that can move rho0, so it is the mechanism-matched
         #      control for H3/H5. Kept SECONDARY (not the pre-registered primary)
         #      because the primary contrast was frozen to bitnet158 before the
         #      grid; promoting a late-added arm to primary would be exactly the
         #      researcher degree of freedom FIX-3 exists to prevent.
         "secondary_baselines":["standard","lsuv","sparse_sched","gamma_offset"],
         "sweep_arms_note":"rho0 sweep runs tai + gamma_offset (see Cell 12)",
         # --- (FIX-5) FROZEN STATISTICAL-TEST FAMILY + HIERARCHY. Holm-Bonferroni
         #     is applied WITHIN the pre-registered PRIMARY family only (one test
         #     per hypothesis at the confirmatory scale). Scale-sweep / secondary
         #     baseline / ablation tests are EXPLORATORY and reported without
         #     family-wise correction (and labelled as such), so the confirmatory
         #     claims are not diluted by exploratory multiplicity.
         "confirmatory_scale":"dim384",   # pre-registered primary-analysis scale
         "primary_family":["H1","H2","H3","H4"],
         # gamma-warmup ablation grid
         "warmup_start_grid":[0.3,0.4,0.5],
         "warmup_tau_grid":[120,240,480],
         # LR sensitivity grid
         # (R33) LR GRID WIDENED AND MADE ADAPTIVE.
         # Session-1 calibration put the optimum ON A GRID BOUNDARY in 23 of the
         # 32 confirmatory (scale, method) cells: every arm at dim256 and dim384
         # chose the top point 3e-3 with the curve still descending, and every arm
         # at dim1280/dim1536 chose the bottom point 1e-3. A boundary optimum on
         # its own would be tolerable; what is not is that the boundary SLOPES
         # differ by arm. At dim384 the 2e-3 -> 3e-3 gain is +0.026 for
         # gamma_offset and +0.046 for tai -- a 1.8x difference, and the sign of
         # the asymmetry flips at dim256. gamma_offset is the mechanism-matched
         # control for H3: it is precisely the comparison that decides whether
         # TAI's contribution is the init distribution or merely threshold
         # control. If that ranking moves when the LR constraint is relaxed, the
         # paper's central claim moves with it, and no post-hoc argument can
         # recover it.
         #
         # The base grid is widened at BOTH ends. Cost is far lower than it looks:
         # the run store keys calibration runs on (scale, method, seed, lr), so
         # every one of the 756 runs already completed is reused untouched and
         # only the NEW lr points cost anything. See lr_grid_adaptive below, which
         # cuts that further.
         "lr_grid":[5e-4,1e-3,2e-3,3e-3,5e-3],
         # (R33) Adaptive refinement: after evaluating the base grid, if the argmin
         # sits on an end point, extend once in that direction (x2 up, /2 down) and
         # re-evaluate, up to lr_grid_max_extend times. This turns "the optimum is
         # outside the grid" from an untestable limitation into a measured fact:
         # either the curve turns over -- and we can state that LR* is interior --
         # or it keeps descending, which is itself a reportable finding. Set
         # lr_grid_adaptive=False to use the fixed grid only.
         "lr_grid_adaptive":True,
         "lr_grid_max_extend":2,
         # (R33) VALIDATE THE SHORT-CALIBRATION PROXY.
         # R22 shortened calibration to 300 steps to save ~32% of total compute.
         # That is a proxy: warmup alone is 120 steps, so LR* is chosen from only
         # ~180 post-warmup steps and then applied to 1221-step runs. High LRs
         # systematically flatter themselves early, which is the same direction as
         # the upper-boundary problem above -- the two confounds compound. Before
         # trusting the proxy across the whole grid we test it once, directly: at
         # one scale, re-run the calibration curve at FULL length on one seed and
         # check the argmin agrees. Three runs, a few minutes. If it agrees the
         # proxy is defensible and the paper can say so with evidence; if it does
         # not, lr_cal_steps must go up before the main grid is trusted.
         # (R34) compile is skipped for runs shorter than this. Session-1 timings
         # show the per-run compile cost (~190 s at dim1280) exceeds the training
         # time of a 300-step calibration run, so compiling them is a net loss.
         # Full 1221-step runs stay compiled.
         "compile_min_steps":600,
         "lr_proxy_check_scale":"dim384",
         "lr_proxy_check_seed":100001,
         "ablation_seeds":[1337,2026,7,42,2718],
         # --- (FIX-1) EXPLICIT CONFIRMATORY vs EXPLORATORY SCALE SETS. The full
         #     method x 15-seed grid runs ONLY on confirmatory_scales (token-equal,
         #     plateau-verified): these carry H1/H2/H4 and the primary effect.
         #     exploratory_scales add ladder points for the H6 scaling-law trend
         #     with a reduced seed count (exploratory_seeds) to fit one A100. No
         #     cell may silently iterate 'all SIZES' for a confirmatory claim; it
         #     must reference one of these lists. Keep them DISJOINT in role even
         #     though confirmatory points are reused inside the trend fit.
         "confirmatory_scales":["dim256","dim384","dim512","dim768"],
         "exploratory_scales":["dim192","dim320","dim640","dim896","dim1024",
                               # (R19) large-scale points for external validity; H6
                               # sign-consistency only, not compute-optimal loss.
                               "dim1280","dim1536"],   # dim2048 (783M) opt-in below
         "scaling_optin_scales":["dim2048"],   # enable only if wall-clock allows
         # (R19) At a fixed single-A100 budget the largest models cannot see a
         # constant token/param ratio. Rather than pretend they converge, we hold the
         # TOKEN BUDGET fixed across scales (token-equal, not compute-optimal) and say
         # so: H6's readout is the SIGN of the TAI effect per scale, which a fixed
         # budget can measure honestly, not the absolute loss gap.
         "scaling_budget_mode":"token_equal",
         "exploratory_seeds":[1337,2026,7,42,2718],
         # --- (#5 DATA-VARIANCE) K interleaved val folds (Cell 9). Fold 0 is the
         #     primary split; the rest feed a recompute-only data-variance check
         #     (Cell 13f) that re-scores trained checkpoints on held-out folds so
         #     H4's variance claim includes data-sampling variance, not just seed.
         # (R15) The old split built K CONTIGUOUS slices and derived TRAIN per
         #     fold, while only fold 0's TRAIN was ever trained on -- so R5's
         #     per-fold scoring read folds 1..K-1 from INSIDE the training data
         #     (measured: 100% overlap). Rebuilt: ONE interleaved held-out region,
         #     removed from training once, then split into K genuinely-unseen
         #     sub-folds. val_chunk_tokens sets the interleaving granularity;
         #     holdout_frac sets how much of the corpus is held out in total.
         "val_folds":5,
         "val_chunk_tokens":50_000,
         "holdout_frac":0.06,
         # --- (#2 SCALE-VS-DEPTH CONFOUND) The 9-scale ladder co-varies width AND
         #     depth, so an H6 "effect grows with scale" could be a depth effect.
         #     width_ctrl_scales hold n_layers FIXED and vary only width (dim/heads/
         #     ffn), giving a depth-controlled scaling axis. Run at reduced seeds;
         #     analysed in Cell 13a3 as a second, confound-controlled slope.
         "width_ctrl_layers":6,
         "width_ctrl_scales":["wc384","wc512","wc768","wc1024"],
         "width_ctrl_seeds":[1337,2026,7,42,2718],
         # --- (#4 H3 NEGATIVE CONTROL) A pre-registered FALSIFICATION test for the
         #     causal sparsity claim: a knob that should NOT move final sparsity.
         #     We re-run the rho0 sweep machinery but feed a SHUFFLED rho0->noise
         #     mapping (rho0 relabelled, base noise fixed) so the threshold split
         #     no longer corresponds to its rho0 label. If final sparsity still
         #     tracks the (fake) label, the H3 effect is an artefact. Expected:
         #     FLAT (Spearman ~0). Reported alongside the real sweep.
         "h3_negative_control":True,
         # --- (#4 POWER) rho0 sweep gets the SAME 15 seeds as the main grid (was
         #     implicitly the full seed set already, but pinned here) so per-seed
         #     Spearman over the 6-pt grid is aggregated across 15 independent
         #     replications -> a sign test on 15 per-seed rho's, far higher power
         #     than one pooled n=6 correlation.
         "rho0_sweep_seeds_full":True,
         # --- (#3 fp16+TAI-init ABLATION) isolate whether TAI's gain is the init
         #     DISTRIBUTION (helps even in full precision) or the ternary OPERATING-
         #     POINT control (only meaningful under quantization). Cell 13g runs
         #     fp16_control under BOTH standard-init and TAI-init at the primary
         #     scale; if TAI-init ~ standard-init in fp16 but TAI > baseline in
         #     ternary, the gain is attributable to threshold control, not init.
         "fp16_init_ablation":True,
         "equiv_margin":0.05}
RUN_START_WALL=time.time()
def budget_exceeded():
    # (R30) None => never exceeded. Every call site uses this to decide whether to
    # stop cleanly BETWEEN runs; returning False lets the loop continue until Colab
    # terminates the session, which the atomic run store makes safe.
    _b=RUN_CFG.get("session_budget_hours")
    if _b is None: return False
    return (time.time()-RUN_START_WALL)>float(_b)*3600
print("RUN_CFG:",json.dumps(RUN_CFG,indent=2))

# ---- Run-All safety helpers -----------------------------------------------
# Every downstream PHASE cell calls _require(...) first. If an upstream artifact
# is missing (e.g. because preflight stopped the run, or a budget boundary
# exited mid-grid this session), the phase prints ONE explanatory line and skips
# itself instead of raising NameError. This makes `Run All` safe end-to-end:
# setup cells always run; heavy phases run only when their inputs exist.
def _exists(name):
    g=globals()
    return (name in g) and (g[name] is not None) and (g[name]!=[] ) and (g[name]!={})
def _require(*names):
    missing=[n for n in names if not _exists(n)]
    if missing:
        print(f"[skip] phase prerequisites not ready: {', '.join(missing)}.")
        print("       This is expected if preflight stopped the run or a session")
        print("       budget boundary exited mid-grid. Re-run after the grid completes.")
        return False
    return True
print("Run-All safety helpers ready (_require / _exists).")

# ---- VISIBLE RUN-MODE BANNER (printed on every Run All) ---------------------
# One place that states exactly what THIS Run All does, and the precise flag
# change to advance safe -> preflight -> full. Pure print; no side effects.
def _run_mode_banner():
    armed=RUN_CFG.get("ARM_FULL_RUN",False); pre=RUN_CFG.get("preflight_only",True)
    L="="*72
    if not armed:
        mode="SAFE MODE (no training, no GPU-hours consumed)"
        whatruns="Setup + definitions only; every training/analysis phase prints a skip and exits cleanly."
        nextstep="To smoke-test + project budget: set RUN_CFG['ARM_FULL_RUN']=True (keep preflight_only=True), then Run All again."
    elif pre:
        mode="PREFLIGHT MODE (one 30-step smoke run + budget projection, then STOP)"
        whatruns="Builds/loads real corpus + lm-eval, trains ONE smoke run on the smallest scale, prints PREFLIGHT REPORT, then stops."
        nextstep="If used_synthetic=false and projected hours OK: set RUN_CFG['preflight_only']=False, then Run All to launch the grid."
    else:
        mode="FULL EXPERIMENT (multi-session grid; checkpoint + resume to Drive)"
        whatruns="Runs the full token-equal grid + sweeps + ablations; finished runs are skipped; exits at the session time budget."
        nextstep="Re-run each session until the manifest reports complete. Idempotent: finished runs are never recomputed."
    print("\n"+L)
    print("RUN MODE:",mode)
    print(L)
    print("THIS Run All will:",whatruns)
    print("NEXT STEP:",nextstep)
    print("session_budget_hours=%s | ARM_FULL_RUN=%s | preflight_only=%s"
          % (RUN_CFG.get("session_budget_hours"),armed,pre))
    print(L)
_run_mode_banner()

In [ ]:
# === CELL 4: GMAIL SMTP NOTIFIER + GLOBAL EVENT HOOKS ======================
# Full tracking. Credentials ONLY from Colab Secrets (userdata):
#   GMAIL_USER (sender) , GMAIL_APP_PASSWORD (16-char App Password), GMAIL_TO
#   (recipient). Older names GMAIL_ADDRESS / NOTIFY_TO are also accepted.
# Per-run mails are tagged [run] (low priority); block/phase/boundary/error/done
# mails are untagged so they stand out. If secrets absent -> console+log only.
import smtplib,ssl,traceback,datetime
from email.mime.text import MIMEText
def _secret(n):
    try:
        from google.colab import userdata; return userdata.get(n)
    except Exception: return os.environ.get(n)
# Accept either naming scheme: user's (GMAIL_USER/GMAIL_TO) or the older
# (GMAIL_ADDRESS/NOTIFY_TO). Whichever is set in Colab Secrets is used.
GMAIL_ADDR=_secret("GMAIL_USER") or _secret("GMAIL_ADDRESS")
GMAIL_PW=_secret("GMAIL_APP_PASSWORD")
NOTIFY_TO=_secret("GMAIL_TO") or _secret("NOTIFY_TO") or GMAIL_ADDR
EMAIL_ENABLED=bool(GMAIL_ADDR and GMAIL_PW)
LOG_PATH=os.path.join(PATHS["logs"],"events.log")
def _log(line):
    with open(LOG_PATH,"a") as f:
        f.write(f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {line}\n")
# Email policy: send mail only for MAJOR events (phase/block/session-boundary/
# completion) and ALL errors (critical=True). Per-run events (tag=="run") are
# logged to console+Drive log but NOT emailed, to avoid hundreds of mails.
EMAIL_PER_RUN=False
def notify(subject,body="",tag=None,critical=False):
    pre=f"[TAI]{('['+tag+']') if tag else ''}"
    el=(time.time()-RUN_START_WALL)/3600.0
    body=f"{body}\n\nElapsed this session: {el:.2f} h"
    if tag!="run":
        print(f">>> {subject}")
    _log(f"{subject} | {body.splitlines()[0]}")
    # suppress per-run emails unless explicitly enabled; always allow errors
    email_this = critical or (tag!="run") or (tag=="run" and EMAIL_PER_RUN)
    if not EMAIL_ENABLED or not email_this: return
    try:
        msg=MIMEText(body); msg["Subject"]=f"{pre} {subject}"; msg["From"]=GMAIL_ADDR; msg["To"]=NOTIFY_TO
        with smtplib.SMTP_SSL("smtp.gmail.com",465,context=ssl.create_default_context(),timeout=30) as s:
            s.login(GMAIL_ADDR,GMAIL_PW); s.sendmail(GMAIL_ADDR,[NOTIFY_TO],msg.as_string())
    except Exception as e:
        print("   email failed (continuing):",str(e)[:100])
def _excepthook(et,ev,tb):
    if et is KeyboardInterrupt:
        notify("INTERRUPTED (manual/session reclaim)","Re-run Run-All to resume from unfinished runs.",critical=True)
    else:
        notify("EXCEPTION — halted",f"{et.__name__}: {ev}\n\n"+"".join(traceback.format_exception(et,ev,tb))[-1500:],critical=True)
    sys.__excepthook__(et,ev,tb)
sys.excepthook=_excepthook
try:
    _ip=get_ipython()
    def _h(self,et,ev,tb,tb_offset=None):
        notify("EXCEPTION (cell) — halted",f"{et.__name__}: {ev}\n\n"+"".join(traceback.format_exception(et,ev,tb))[-1500:],critical=True)
        return self.showtraceback((et,ev,tb),tb_offset=tb_offset)
    _ip.set_custom_exc((Exception,),_h)
except Exception: pass
print("Email:","ENABLED" if EMAIL_ENABLED else "DISABLED (console+log only)")
notify("RUN STARTED",f"Project {PROJECT_ROOT}\nBudget {RUN_CFG['session_budget_hours']} h")


In [ ]:
# === CELL 5: QUANTIZATION PRIMITIVES (absmean ternary + INT8 act + STE) =====
# Two ternary weight quantizers are provided so we can run an INDEPENDENT,
# published recipe (BitNet b1.58) alongside ours and test transportability:
#   weight_quant       : absmean ternary used throughout TAI (gamma = mean|W|).
#   weight_quant_bitnet : the BitNet b1.58 formulation (round-to-ternary with a
#                         beta = 1/mean|W| scale, clamp to {-1,0,1}). Same {-1,0,1}
#                         codebook, different scaling/threshold convention; lets a
#                         reviewer see TAI's effect under a recipe we did not design.
import torch
def activation_quant(x):
    s=127.0/x.abs().amax(dim=-1,keepdim=True).clamp_(min=1e-5); return (x*s).round().clamp_(-128,127)/s
def weight_quant(w,gamma_scale=1.0):
    g=(w.abs().mean()*gamma_scale).clamp_(min=1e-5); return (w/g).round().clamp_(-1,1)*g
def weight_quant_bitnet(w,gamma_scale=1.0):
    # BitNet b1.58: scale by beta=1/mean|w|, round to nearest integer, clamp ternary.
    beta=1.0/(w.abs().mean()*gamma_scale).clamp_(min=1e-5)
    return (w*beta).round().clamp_(-1,1)/beta
def weight_quant_sched(w,gamma_scale=1.0):
    # gradual-pruning baseline: identical to absmean but gamma_scale is driven
    # by a rising schedule (set in the training loop), raising the effective
    # threshold over time toward a target sparsity. TAI-independent control.
    g=(w.abs().mean()*gamma_scale).clamp_(min=1e-5); return (w/g).round().clamp_(-1,1)*g
def ste(x,fn): return x+(fn(x)-x).detach()
# (R41) named _tq, not _tq -- see the note in Cell 11: `import time as _tq` is a
# module-level alias and notebook cells share one namespace.
_w=torch.randn(256,256); _g=_w.abs().mean(); _tq=(_w/_g).round().clamp(-1,1)
assert set(torch.unique(_tq).tolist()).issubset({-1.,0.,1.})
_tb=weight_quant_bitnet(_w); assert set(torch.unique((_tb/ _tb.abs().amax().clamp(min=1e-9)).round()).tolist()).issubset({-1.,0.,1.})
print("Quant self-test OK (absmean + bitnet-b1.58):",sorted(torch.unique(_tq).tolist()))

In [ ]:
# === CELL 6: BitLinear WITH FP16-CONTROL + BitNet-b1.58 SWITCH + GAMMA-WARMUP =
import torch,torch.nn as nn
class _RMS(nn.Module):
    def __init__(s,d,eps=1e-5): super().__init__(); s.weight=nn.Parameter(torch.ones(d)); s.eps=eps
    def forward(s,x): v=x.pow(2).mean(-1,keepdim=True); return x*torch.rsqrt(v+s.eps)*s.weight
def _mk(d,eps): return nn.RMSNorm(d,eps=eps) if hasattr(nn,"RMSNorm") else _RMS(d,eps)
GAMMA_SCALE={"val":1.0}; QUANT_MODE={"full_precision":False}
# ============================================================================
# (R23) DEVICE-TENSOR MIRROR OF GAMMA_SCALE — the prerequisite for torch.compile.
# ----------------------------------------------------------------------------
# GAMMA_SCALE["val"] is read inside BitLinear.forward as a PYTHON FLOAT. Dynamo
# specialises graphs on python scalar VALUES, so every distinct gamma produces a
# new graph. Two of the eight arms -- tai_warmup and sparse_sched -- move gamma
# EVERY STEP by design, which would mean a recompilation per step: compiled mode
# would run slower than eager, and only for the two arms whose whole point is the
# gamma schedule. That is why compile was left off in R22.
#
# Fix: keep the float (every guard, reset, log and assertion below still reads
# it, untouched) and mirror it into a persistent 0-dim tensor that is mutated
# IN PLACE with fill_(). Dynamo guards tensors on shape/dtype, not value, so the
# graph survives a changing gamma. One tensor is cached per device so CPU-side
# construction and CUDA-side training both work.
#
# The float remains the single source of truth; the tensor is a shadow. Every
# write goes through set_gamma(), and assert_quant_state_clean() cross-checks the
# two at every run boundary, so a missed write site fails loudly at the next run
# rather than silently training at the wrong threshold.
_GAMMA_T={}
def _gamma_t(ref):
    d=ref.device; t=_GAMMA_T.get(d)
    if t is None:
        t=torch.tensor(float(GAMMA_SCALE["val"]),dtype=torch.float32,device=d); _GAMMA_T[d]=t
    return t
def set_gamma(v):
    """The ONLY sanctioned way to move the ternarisation threshold."""
    v=float(v); GAMMA_SCALE["val"]=v
    for t in _GAMMA_T.values(): t.fill_(v)
def gamma_mirror_ok(rtol=1e-6):
    # RELATIVE tolerance, and it must be loose enough for fp32 round-trip. The
    # float is a python double; the mirror is fp32, so a value like 1.6 -- the
    # sparse_sched target, i.e. a value this harness uses on every sparse_sched
    # step -- comes back as 1.6000000238. An absolute 1e-9 check would therefore
    # have reported "mirror out of sync" on a perfectly healthy run and aborted
    # the arm. Caught by the equivalence test, not by reading the code.
    v=float(GAMMA_SCALE["val"]); scale=max(1.0,abs(v))
    return all(abs(float(t.item())-v)<=rtol*scale for t in _GAMMA_T.values())
# QUANT_RECIPE selects the ternary weight quantizer for the WHOLE model:
#   "absmean" -> weight_quant (TAI's recipe)   "bitnet" -> weight_quant_bitnet
QUANT_RECIPE={"name":"absmean"}
def _wq(w):
    g=_gamma_t(w)          # (R23) 0-dim tensor, not a python float -> compile-safe
    if QUANT_RECIPE["name"]=="bitnet": return weight_quant_bitnet(w,g)
    if QUANT_RECIPE["name"]=="sched":  return weight_quant_sched(w,g)
    return weight_quant(w,g)
class BitLinear(nn.Linear):
    def __init__(s,i,o,bias=False,eps=1e-5): super().__init__(i,o,bias=bias); s.eps=eps; s.norm=_mk(i,eps)
    def forward(s,x):
        x=s.norm(x)
        if QUANT_MODE["full_precision"]: return nn.functional.linear(x,s.weight,s.bias)
        wq=ste(s.weight,_wq); return nn.functional.linear(ste(x,activation_quant),wq,s.bias)
    @torch.no_grad()
    def sparsity(s):
        # ---------------------------------------------------------------
        # (R16 FIX) This measured a quantity the model never used. It hard-coded
        # gamma = mean|W| (i.e. gamma_scale = 1.0) and ignored both QUANT_MODE and
        # GAMMA_SCALE, so it reported "the sparsity you would get IF you ternarised
        # with the default threshold" rather than the sparsity of the forward pass
        # that actually ran. Two consequences, both caught by a CPU smoke-run:
        #
        #  (1) gamma_offset (R6) BROKEN AT THE MEASUREMENT. That arm's entire
        #      mechanism is running with gamma_scale=c (e.g. 0.7987 for rho0=0.75).
        #      Measured at c=1 it returns ~31% -- the locked-baseline value -- for
        #      EVERY rho0. H3's mechanism control, H5's second frontier and fig10
        #      would all have shown a flat line and "proved" that gamma_offset
        #      cannot control sparsity, when in fact only the ruler was wrong.
        #      Smoke-run evidence: tai -> 24.9%, gamma_offset -> 31.0% at the same
        #      rho0=0.75, where both should land near 25%.
        #  (2) fp16_control reported final_sparsity=31.2% despite never quantising
        #      anything -- a meaningless number that flows into the manifest, fig2
        #      and the H5 Pareto axis. It also silently disarmed the R14 guard
        #      ("a ternary run with sparsity==0 means a full-precision leak"),
        #      because this function can never return 0.
        #
        # Now it mirrors BitLinear.forward exactly: full precision -> no weight is
        # zeroed, so sparsity is 0 by definition; otherwise ternarise through the
        # SAME quantizer and the SAME live GAMMA_SCALE the forward pass uses.
        if QUANT_MODE["full_precision"]:
            return 0.0                      # nothing is quantised -> nothing is zeroed
        return (_wq(s.weight)==0).float().mean().item()
    @torch.no_grad()
    def sparsity_at_default_gamma(s):
        # The old semantics, kept explicitly for the ONE place they are correct:
        # Cell 13b measures the INITIAL active ratio of each initializer under the
        # canonical absmean threshold (gamma_scale=1), which is exactly the
        # sigma-independence claim. Naming it separately stops the two meanings
        # from being confused again.
        g=s.weight.abs().mean().clamp_(min=1e-5)
        return ((s.weight/g).round().clamp_(-1,1)==0).float().mean().item()
    @torch.no_grad()
    def active_ratio(s):
        # Cell 13b / theory verification: always the canonical-threshold reading,
        # independent of whatever GAMMA_SCALE a training loop happens to have set.
        return 1.0-s.sparsity_at_default_gamma()
print("BitLinear ready (FP16 + BitNet-b1.58 switch + gamma-warmup hook).")

In [ ]:
# === CELL 7: SIZE-PARAMETRIC DECODER (9-scale ladder: ~1.4M -> >=50M non-embed for H6) =====
import math,torch,torch.nn as nn
from dataclasses import dataclass,asdict,replace
@dataclass
class Cfg:
    # (R0 FIX) Previously `n_layers`, `n_heads` and `ffn_hidden` were swallowed by an
    # end-of-line comment on the `vocab_size` line, so they were NEVER dataclass
    # fields. Every `Cfg(dim=..., n_layers=...)` call in SIZES therefore raised
    # TypeError (unexpected keyword argument) -- the notebook could not run at all.
    # Each field is now on its own line and an assertion below guards the schema.
    vocab_size:int=8192          # PLACEHOLDER; rebound to the real GPT-2 BPE vocab at end of Cell 9
    dim:int=384
    n_layers:int=6
    n_heads:int=6
    ffn_hidden:int=1024
    max_seq_len:int=256
    eps:float=1e-5
    rope_theta:float=10000.0
    arch:str="decoder_v1"
    n_kv_heads:int=0             # n_kv_heads>0 enables GQA (decoder_v2)
def _rope(seq,hd,th,dev,dt):
    inv=1.0/(th**(torch.arange(0,hd,2,device=dev).float()/hd)); fr=torch.outer(torch.arange(seq,device=dev).float(),inv)
    return torch.cos(fr).to(dt),torch.sin(fr).to(dt)
def _ar(x,cos,sin):
    x1,x2=x[...,0::2],x[...,1::2]; cos=cos[None,None,:x1.shape[-2],:]; sin=sin[None,None,:x1.shape[-2],:]
    o=torch.empty_like(x); o[...,0::2]=x1*cos-x2*sin; o[...,1::2]=x1*sin+x2*cos; return o
class Att(nn.Module):
    def __init__(s,c):
        super().__init__(); s.h,s.dh=c.n_heads,c.dim//c.n_heads
        s.q=BitLinear(c.dim,c.dim);s.k=BitLinear(c.dim,c.dim);s.v=BitLinear(c.dim,c.dim);s.o=BitLinear(c.dim,c.dim)
    def forward(s,x,cos,sin):
        B,T,D=x.shape
        q=s.q(x).view(B,T,s.h,s.dh).transpose(1,2);k=s.k(x).view(B,T,s.h,s.dh).transpose(1,2);v=s.v(x).view(B,T,s.h,s.dh).transpose(1,2)
        q,k=_ar(q,cos,sin),_ar(k,cos,sin)
        with _SDPA_CTX(): y=nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True)
        return s.o(y.transpose(1,2).contiguous().view(B,T,D))
class FFN(nn.Module):
    def __init__(s,c): super().__init__(); s.up=BitLinear(c.dim,c.ffn_hidden); s.down=BitLinear(c.ffn_hidden,c.dim)
    def forward(s,x): return s.down(torch.relu(s.up(x)).square())
class Blk(nn.Module):
    def __init__(s,c): super().__init__(); s.attn=Att(c); s.ffn=FFN(c)
    def forward(s,x,cos,sin): x=x+s.attn(x,cos,sin); return x+s.ffn(x)

# ---- Architecture variant B (decoder_v2): GQA attention + SwiGLU FFN --------
# Reviewer request R1: show TAI's effect is not specific to one block design.
# Same BitLinear projections (so the quantization story is identical); only the
# attention head-sharing and the FFN nonlinearity/gating differ.
class Att2(nn.Module):
    def __init__(s,c):
        super().__init__(); s.h=c.n_heads; s.dh=c.dim//c.n_heads
        s.kv=c.n_kv_heads if c.n_kv_heads>0 else c.n_heads; s.rep=s.h//s.kv
        s.q=BitLinear(c.dim,c.dim); s.k=BitLinear(c.dim,s.kv*s.dh)
        s.v=BitLinear(c.dim,s.kv*s.dh); s.o=BitLinear(c.dim,c.dim)
    def forward(s,x,cos,sin):
        B,T,D=x.shape
        q=s.q(x).view(B,T,s.h,s.dh).transpose(1,2)
        k=s.k(x).view(B,T,s.kv,s.dh).transpose(1,2)
        v=s.v(x).view(B,T,s.kv,s.dh).transpose(1,2)
        q=_ar(q,cos,sin); k=_ar(k,cos,sin)
        if s.rep>1:  # expand kv heads to match q heads (grouped-query)
            k=k.repeat_interleave(s.rep,dim=1); v=v.repeat_interleave(s.rep,dim=1)
        with _SDPA_CTX(): y=nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True)
        return s.o(y.transpose(1,2).contiguous().view(B,T,D))
class FFN2(nn.Module):  # SwiGLU
    def __init__(s,c):
        super().__init__(); s.gate=BitLinear(c.dim,c.ffn_hidden)
        s.up=BitLinear(c.dim,c.ffn_hidden); s.down=BitLinear(c.ffn_hidden,c.dim)
    def forward(s,x): return s.down(nn.functional.silu(s.gate(x))*s.up(x))
class Blk2(nn.Module):
    def __init__(s,c): super().__init__(); s.attn=Att2(c); s.ffn=FFN2(c)
    def forward(s,x,cos,sin): x=x+s.attn(x,cos,sin); return x+s.ffn(x)

# ============================================================================
# (R22) CHUNKED CROSS-ENTROPY — same arithmetic, far lower peak VRAM.
# ----------------------------------------------------------------------------
# The original line was:
#     cross_entropy(lg.view(-1,V).float(), targets.view(-1))
# With bs=32, seq=512, V=50257 the .float() call materialises a
# [16384 x 50257] fp32 tensor = 3.29 GB, plus the log_softmax it feeds and the
# gradient flowing back through it. Peak is ~10-12 GB -- and CRUCIALLY this cost
# does NOT scale with `dim`: the smallest model on the ladder pays exactly the
# same as the largest. Combined with the score matrices the MATH SDPA backend
# used to keep, dim1536 (352M) was at real risk of OOM on a 40 GB A100, which
# would have killed the R19 scale extension the whole external-validity argument
# rests on.
#
# Splitting the flattened batch into `ce_chunks` pieces and summing the per-chunk
# SUM losses before dividing by n is ALGEBRAICALLY IDENTICAL to the mean over the
# whole batch (no approximation, no reweighting -- every token still contributes
# exactly once with weight 1/n). What changes is that only one chunk's fp32
# expansion is live at a time, cutting the transient peak by ~ce_chunks.
def _ce_loss(lg,targets,chunks=None):
    V=lg.size(-1); flat=lg.reshape(-1,V); tgt=targets.reshape(-1); n=tgt.numel()
    try: k=int(FAST_CFG.get("ce_chunks",1)) if "FAST_CFG" in globals() else 1
    except Exception: k=1
    if k<=1 or n<=k:
        return nn.functional.cross_entropy(flat.float(),tgt)
    step=(n+k-1)//k; tot=None
    for a in range(0,n,step):
        b=min(n,a+step)
        part=nn.functional.cross_entropy(flat[a:b].float(),tgt[a:b],reduction="sum")
        tot=part if tot is None else tot+part
    return tot/n

class LM(nn.Module):
    def __init__(s,c):
        super().__init__(); s.cfg=c; s.tok=nn.Embedding(c.vocab_size,c.dim)
        _BLK=Blk2 if c.arch=="decoder_v2" else Blk
        s.blocks=nn.ModuleList(_BLK(c) for _ in range(c.n_layers)); s.norm=_mk(c.dim,c.eps)
        s.head=nn.Linear(c.dim,c.vocab_size,bias=False); s.head.weight=s.tok.weight
        # ====================================================================
        # (R27) THE TIED EMBEDDING / OUTPUT HEAD WAS NEVER INITIALISED.
        # --------------------------------------------------------------------
        # nn.Embedding defaults to normal_(0, 1), and this weight is TIED to the
        # output head, so at step 0 every logit is a sum of `dim` products of two
        # unit-variance terms: logit std ~ sqrt(dim). Measured initial loss was
        # 121.6 (dim192), 211.4 (dim384), 351.9 (dim768) against a uniform-
        # prediction baseline of log(50257) = 10.83.
        #
        # apply_init() in Cell 8 walks BitLinear layers only; nothing in the
        # notebook ever touched s.tok, so all eight arms inherited PyTorch's
        # default. Three consequences, all bad for this specific experiment:
        #   (1) A large slice of the fixed 20M-token budget is spent merely
        #       shrinking the embedding norm before any language modelling
        #       happens -- and the budget was chosen to reach a plateau.
        #   (2) The waste is SCALE-DEPENDENT (it grows as sqrt(dim)), so larger
        #       models start proportionally further from sanity. That biases H6
        #       (sign consistency across scale) and the scaling-law fit in a
        #       direction that has nothing to do with rho0.
        #   (3) This is a paper ABOUT INITIALISATION. A large init-driven
        #       transient shared by every arm dilutes the very effect being
        #       measured, and could equally mask a real TAI advantage or
        #       manufacture a spurious one.
        # Fixed with the standard GPT-2 value (std 0.02), which puts initial loss
        # at 10.88 / 10.93 / 10.94 across the ladder -- i.e. at the uniform
        # baseline and, importantly, SCALE-INDEPENDENT. This is a bug fix, not a
        # tuning choice: the previous state was "uninitialised". It is identical
        # for every arm and does not touch any BitLinear, so the mechanism under
        # test (rho0 of the ternary layers) is unaffected.
        nn.init.normal_(s.tok.weight,mean=0.0,std=float(getattr(c,"emb_init_std",0.02)))
        cos,sin=_rope(c.max_seq_len,c.dim//c.n_heads,c.rope_theta,DEVICE,DTYPE)
        s.register_buffer("rope_cos",cos,persistent=False); s.register_buffer("rope_sin",sin,persistent=False)
    def forward(s,idx,targets=None):
        x=s.tok(idx)
        for b in s.blocks: x=b(x,s.rope_cos,s.rope_sin)
        x=s.norm(x); lg=s.head(x); loss=None
        if targets is not None: loss=_ce_loss(lg,targets)
        return lg,loss
    def bitlinears(s): return [m for m in s.modules() if isinstance(m,BitLinear)]
# (#2) DENSER SCALE LADDER: 9 scales (was 6). The 3 added points (dim320, dim640,
#      dim896) fill gaps so the scaling-law fit in Cell 13 has more leverage --
#      especially on the upper arm, which is what an "effect persists with scale"
#      claim must rest on. More points => tighter slope CI in H6 and a more
#      credible extrapolation band. ColabPro+ A100 makes the extra runs affordable.
SIZES={"dim192":Cfg(dim=192,n_layers=4,n_heads=4,ffn_hidden=512),
       "dim256":Cfg(dim=256,n_layers=4,n_heads=4,ffn_hidden=688),
       "dim320":Cfg(dim=320,n_layers=5,n_heads=5,ffn_hidden=853),
       "dim384":Cfg(dim=384,n_layers=6,n_heads=6,ffn_hidden=1024),
       "dim512":Cfg(dim=512,n_layers=8,n_heads=8,ffn_hidden=1365),
       "dim640":Cfg(dim=640,n_layers=9,n_heads=10,ffn_hidden=1706),
       "dim768":Cfg(dim=768,n_layers=10,n_heads=12,ffn_hidden=2048),
       "dim896":Cfg(dim=896,n_layers=11,n_heads=14,ffn_hidden=2389),
       "dim1024":Cfg(dim=1024,n_layers=12,n_heads=16,ffn_hidden=2730),  # ~117M non-embed
       # ---- (R19) EXTENDED SCALING LADDER for external validity -------------------
       # The single biggest weakness of the original design was scale: the whole
       # ladder topped out at 117M non-embed params, so any "TAI survives scaling"
       # claim rested on less than one order of magnitude and an n=9 slope fit. These
       # three points extend the ladder to 352M (and optionally 783M), taking the
       # confirmatory-to-largest span past 1.5 orders of magnitude. They are declared
       # EXPLORATORY, not confirmatory: at a single-A100 budget they cannot be trained
       # Chinchilla-optimally (352M wants ~7B tokens; the harness runs a fixed, much
       # smaller token budget), so their ROLE is H6 sign-consistency -- "does the sign
       # of the TAI effect persist as scale grows" -- NOT a compute-optimal loss
       # comparison. That is the honest thing a fixed budget can test, and it is
       # exactly what a reviewer asking "does this hold at scale?" needs to see.
       # dim2048 (783M) is included but OFF by default (see RUN_CFG scale lists);
       # enable it only if wall-clock allows, since a single run approaches the
       # session boundary at this size.
       "dim1280":Cfg(dim=1280,n_layers=14,n_heads=16,ffn_hidden=3413),  # ~214M non-embed
       "dim1536":Cfg(dim=1536,n_layers=16,n_heads=16,ffn_hidden=4096),  # ~352M non-embed
       "dim2048":Cfg(dim=2048,n_layers=20,n_heads=16,ffn_hidden=5461)}  # ~783M non-embed (opt-in)
# (#3) FULL-LADDER decoder_v2 (GQA + SwiGLU). Was 2 representative scales; now the
#      ENTIRE scale ladder is rebuilt as decoder_v2 so architecture transfer can be
#      tested AS A FUNCTION OF SCALE -- i.e. the cross-arch generalization claim and
#      the scaling-law claim are supported on two independent axes at once. Set
#      RUN_CFG["v2_scales"] to a subset to economise; defaults to all SIZES.
# (R0 FIX) n_kv_heads was computed as n_heads//2, which is NOT a divisor of
# n_heads whenever n_heads is odd (e.g. dim320 has n_heads=5 -> n_kv_heads=2, and
# 5 % 2 != 0). GQA requires n_heads % n_kv_heads == 0 to form equal-size query
# groups; the old expression would have produced a malformed decoder_v2 ladder
# for every odd-head scale, silently corrupting the H7 architecture-transfer test.
# We now pick the LARGEST proper divisor <= n_heads/2 (falling back to 1 for
# prime/small head counts, i.e. full MQA), which is well-defined for every scale.
def _kv_heads(nh):
    for d in range(nh//2, 0, -1):
        if nh % d == 0: return d
    return 1
SIZES_V2={k:replace(SIZES[k],arch="decoder_v2",n_kv_heads=_kv_heads(SIZES[k].n_heads))
          for k in SIZES}
# (#2 DEPTH-CONTROLLED LADDER) width_ctrl: FIX n_layers, vary only width so the
# H6 scaling axis is not confounded with depth. heads scale with width (head_dim
# held ~constant at 64); ffn ~ 8/3*dim (SwiGLU-style ratio rounded). Same
# BitLinear/quant path as the main ladder, decoder_v1 block. Analysed as a
# second, confound-controlled slope in the scaling-law cell.
_WC_L=6
def _wc_cfg(d):
    h=max(1,d//64)
    return Cfg(dim=d,n_layers=_WC_L,n_heads=h,ffn_hidden=int(round(d*8/3/64))*64)
SIZES_WC={"wc384":_wc_cfg(384),"wc512":_wc_cfg(512),
          "wc768":_wc_cfg(768),"wc1024":_wc_cfg(1024)}
def n_params(c):
    base=(4*c.dim*c.dim+2*c.dim*c.ffn_hidden)*c.n_layers
    return base + (c.dim*c.ffn_hidden*c.n_layers if c.arch=="decoder_v2" else 0)  # SwiGLU gate
for n,c in SIZES.items(): print(f"{n}: dim {c.dim} layers {c.n_layers} (~{n_params(c)/1e6:.1f}M non-embed)")

# ---- (R0 FIX) SCHEMA + GEOMETRY GUARDS -------------------------------------
# The dataclass bug above was silent-until-TypeError; these asserts make any
# future recurrence fail LOUDLY at definition time rather than mid-grid, and
# additionally catch head/width mismatches that would otherwise corrupt a run.
_REQ_FIELDS=("vocab_size","dim","n_layers","n_heads","ffn_hidden","max_seq_len",
             "eps","rope_theta","arch","n_kv_heads")
_probe=Cfg()
for _f in _REQ_FIELDS:
    assert hasattr(_probe,_f), f"Cfg schema broken: missing field {_f}"
assert set(Cfg.__dataclass_fields__.keys())==set(_REQ_FIELDS), \
    f"Cfg field set drifted: {sorted(Cfg.__dataclass_fields__)}"
for _name,_c in list(SIZES.items())+list(SIZES_V2.items())+list(SIZES_WC.items()):
    assert _c.dim % _c.n_heads==0, f"{_name}: dim {_c.dim} not divisible by n_heads {_c.n_heads}"
    assert (_c.dim//_c.n_heads)%2==0, f"{_name}: head_dim must be even for RoPE pairing"
    if _c.n_kv_heads: assert _c.n_heads % _c.n_kv_heads==0, f"{_name}: n_heads must be divisible by n_kv_heads"
    assert _c.n_layers>0 and _c.ffn_hidden>0, f"{_name}: degenerate config"
print("Cfg schema + geometry guards PASSED for",
      len(SIZES)+len(SIZES_V2)+len(SIZES_WC),"configs.")


In [ ]:
# === CELL 8: TAI INIT + GAMMA-WARMUP + STRATEGIES (THE METHOD) =============
import torch,math,numpy as np
# (R2) TAI's own hyper-parameters are now EXPLICIT and sweepable rather than
# hard-coded magic numbers buried in the function body. dm = dormant magnitude
# relative to the active magnitude (am is fixed at 1.0; only the RATIO matters
# because of the final std-normalisation), jitter = half-width of the uniform
# multiplicative jitter. Cell 13c-b sweeps both and reports sensitivity.
TAI_HP={"dm":0.15,"jitter":0.15,"last_realised_rho0":None}
# (R13) Records whether LSUV's data-driven pass actually did anything. In a
# pre-normalised (RMSNorm-in-BitLinear) architecture it provably cannot; we
# measure rather than assume, and the manifest reports the answer.
LSUV_DEGENERATE={"rescale_ops":None,"max_abs_var_dev_at_init":None,"is_noop":None}
def init_standard(w,fi): torch.nn.init.normal_(w,0.0,1.0/math.sqrt(fi))
def init_bitnet(w,fi): torch.nn.init.normal_(w,0.0,(1.0/math.sqrt(fi))*0.5)
def init_tai(w,fi,rho0=0.75,gen=None,dm=None,jitter=None,verify=True):
    # ---------------------------------------------------------------------
    # Threshold-Aware Init.  Target: after absmean ternarisation (gamma=mean|W|,
    # survive iff |w| >= gamma/2), the fraction of NON-ZERO weights equals rho0.
    #
    # (R1 FIX -- HONESTY + DEAD CODE).  The previous implementation advertised an
    # "8-step fixed-point search" over an active-magnitude `am`.  That loop was
    # verifiably DEAD: its output is bit-identical with the loop removed, because
    #   (a) `vals` is recomputed from scratch at the top of every iteration, so the
    #       final `am` update is never applied;
    #   (b) the realised active set is decided entirely by `mask = u_mask < rho0`,
    #       which already has expectation rho0 by construction -- while the dormant
    #       magnitude dm=0.15 sits far below the threshold for any am in the search
    #       range, so no iterate ever changes the outcome;
    #   (c) the final std-renormalisation makes the absolute scale of am/dm
    #       irrelevant anyway (only their RATIO matters).
    # Reporting a search that does nothing would misdescribe the method, so the
    # loop is REMOVED and replaced by the closed-form construction it was
    # approximating, plus an explicit post-hoc VERIFICATION of the realised rho0.
    #
    # Closed form.  With a two-point magnitude split (active am, dormant dm) and
    # multiplicative jitter J ~ U(1-a, 1+a):
    #     gamma = E|w| = rho0*am*E[J] + (1-rho0)*dm*E[J] = (rho0*am + (1-rho0)*dm)
    # (E[J]=1).  A weight survives iff |w| >= gamma/2.  Actives survive iff
    #     am*(1-a) >= gamma/2   and dormants die iff  dm*(1+a) < gamma/2.
    # Both hold with margin for the default (am=1, dm<=0.35, a=0.15) over the whole
    # rho0 grid -- we ASSERT this separation rather than assume it, so that a future
    # change to dm/jitter cannot silently break the guarantee (dm>=0.45 does break
    # it; see the dm sweep in Cell 13c-b).
    #
    # (#4) CAUSAL-INTERVENTION MODE: with a torch.Generator `gen`, all draws come
    # from it, so the sweep can hold base noise fixed and vary ONLY rho0.
    n=w.numel(); dev=w.device; target=1.0/math.sqrt(fi)
    am=1.0
    dm=float(TAI_HP["dm"] if dm is None else dm)
    a=float(TAI_HP["jitter"] if jitter is None else jitter)
    def _rand(k):
        if gen is None: return torch.rand(k,device=dev)
        return torch.rand(k,generator=gen).to(dev)   # CPU gen -> cross-device determinism
    u_mask=_rand(n); u_jit=_rand(n); u_sign=_rand(n)
    jit=(1.0-a)+(2.0*a)*u_jit            # U(1-a, 1+a), mean 1
    signs=torch.where(u_sign<.5,-1.,1.)
    mask=u_mask<rho0
    mag=torch.where(mask,torch.full((n,),am,device=dev),
                         torch.full((n,),dm,device=dev))*jit
    vals=mag*signs
    # --- separation guarantee (closed form, scale-invariant) -----------------
    gamma_pred=rho0*am+(1.0-rho0)*dm
    assert am*(1.0-a) >= 0.5*gamma_pred, (
        f"TAI: active magnitude can fall below threshold (rho0={rho0}, dm={dm}, jitter={a})")
    assert dm*(1.0+a) <  0.5*gamma_pred, (
        f"TAI: dormant magnitude can survive threshold (rho0={rho0}, dm={dm}, jitter={a}) "
        f"-- rho0 would NOT be realised; see dm sweep in Cell 13c-b")
    # std-normalise to the fan-in target; ternary active ratio is scale-invariant
    vals=vals*(target/vals.std().clamp(min=1e-9))
    # --- post-hoc verification: did we actually realise rho0? ----------------
    if verify:
        g=vals.abs().mean().clamp(min=1e-9)
        rho_real=((vals/g).round().clamp(-1,1)!=0).float().mean().item()
        TAI_HP["last_realised_rho0"]=rho_real
        # tolerance is the binomial sampling error of drawing n Bernoulli(rho0)
        tol=max(0.02, 5.0*math.sqrt(max(rho0*(1-rho0),1e-9)/max(n,1)))
        assert abs(rho_real-rho0)<=tol, (
            f"TAI failed to realise rho0: target {rho0:.3f} got {rho_real:.3f} (tol {tol:.4f})")
    with torch.no_grad(): w.copy_(vals.view_as(w))
def init_xavier(w,fi): torch.nn.init.xavier_normal_(w)
def init_kaiming(w,fi): torch.nn.init.kaiming_normal_(w,nonlinearity="relu")
def init_lsuv(w,fi):
    # (R13) PER-TENSOR PRE-STEP ONLY. This is the orthogonal pre-initialisation
    # half of LSUV (Mishkin & Matas, 2016); the DEFINING half -- the data-driven
    # forward-pass variance calibration -- cannot happen here because this function
    # sees one weight tensor and no data. `apply_init` runs lsuv_calibrate() after
    # this to complete the algorithm. Called alone, this is NOT LSUV.
    # (R32 FIX) orthogonal_ calls torch.linalg.qr, whose LAPACK/cuSOLVER backend
    # (geqrf) has NO bfloat16 kernel -- on either CPU or CUDA:
    #     NotImplementedError: "geqrf_cuda" not implemented for 'BFloat16'
    # Models are built with .to(DEVICE, DTYPE) where DTYPE is bfloat16, so this
    # raised the moment the grid reached the first `lsuv` arm, 9 minutes into the
    # run. Only lsuv is affected: every other init uses elementwise ops that
    # bfloat16 supports, which is why fp16_control and standard completed first
    # and the failure looked like it came out of nowhere.
    # Fix: do the QR in fp32 on a scratch tensor, then copy back. Verified: the
    # columns stay orthonormal through the bf16 round-trip (max |QtQ - I| ~ 1e-3,
    # i.e. bf16 rounding, not a broken factorisation) and the resulting std
    # matches the 1/sqrt(fan_in) target to five decimals. Nothing about the
    # algorithm changes -- only the dtype the factorisation is computed in.
    with torch.no_grad():
        _tmp=torch.empty(w.shape,dtype=torch.float32,device=w.device)
        torch.nn.init.orthogonal_(_tmp)
        _tmp.mul_(1.0/math.sqrt(fi)/_tmp.std().clamp(min=1e-9))
        w.copy_(_tmp)

@torch.no_grad()
def lsuv_calibrate(model,batch,tol=0.01,max_iter=8,target_var=1.0,verbose=False):
    # ---------------------------------------------------------------------
    # (R13) REAL LSUV -- and an honest report of what it does in THIS architecture.
    #
    # WHAT WAS WRONG BEFORE. `init_lsuv` was orthogonal-init + a static
    # 1/sqrt(fan_in) rescale, with no data pass. That is not LSUV: the algorithm's
    # entire content is layer-sequential unit-variance calibration against real
    # activations. Cell 11 nonetheless sold {standard, lsuv} as "two distinct
    # locked-rho0 FAMILIES" and Cell 12c takes the per-seed BEST of the two as its
    # baseline. Measured, the old pair was standard rho0=0.6900+/-0.0011 vs lsuv
    # rho0=0.6906+/-0.0013 -- a 0.0006 gap, far below seed noise. Two zero-mean
    # tensors normalised to the same sigma are ONE family with two labels, so
    # "best of two baselines" was "best of one, sampled twice": a max-selection
    # that mildly inflates the baseline while misrepresenting baseline coverage.
    #
    # WHAT WE FOUND WHEN WE IMPLEMENTED IT PROPERLY. Measured on this model, every
    # BitLinear's output variance at init is already 1.0004 +/- 0.0005, so LSUV
    # converges in ZERO iterations and rescales nothing. This is not a bug -- it is
    # forced by the architecture: BitLinear applies RMSNorm to its input, so the
    # input is unit-variance by construction, and an orthogonal weight matrix
    # scaled to 1/sqrt(fan_in) maps unit-variance input to unit-variance output
    # analytically. LSUV exists to fix variance drift through DEEP UNNORMALISED
    # stacks; a pre-normalised architecture has no drift to fix. LSUV is therefore
    # DEGENERATE here -- mathematically, not accidentally.
    #
    # CONSEQUENCE, STATED RATHER THAN HIDDEN. `lsuv` cannot be a second baseline
    # family in this architecture, no matter how it is implemented. We keep the
    # correct implementation (so the claim is tested rather than assumed), report
    # the no-op honestly via LSUV_DEGENERATE, and Cell 11's "two distinct families"
    # wording is corrected: the genuine second family is `gamma_offset` (R6), which
    # differs at the OPERATING POINT rather than merely in weight distribution.
    # tol is tightened 0.10 -> 0.01 so that if a future architecture change DOES
    # introduce variance drift, this function will actually engage instead of
    # silently breaking out on the first iterate (the R1 failure mode).
    was_training=model.training; model.eval()
    xb=batch[0] if isinstance(batch,(tuple,list)) else batch
    n_rescaled=0; max_dev=0.0
    for bl in model.bitlinears():
        for _ in range(max_iter):
            acts={}
            h=bl.register_forward_hook(lambda m,i,o,_a=acts: _a.__setitem__("o",o.detach()))
            try:
                with torch.autocast("cuda",dtype=DTYPE): model(xb)
            finally:
                h.remove()
            o=acts.get("o")
            if o is None: break
            v=o.float().var().item()
            if v<=0 or not math.isfinite(v): break
            max_dev=max(max_dev,abs(v-target_var))
            if abs(v-target_var)<tol: break
            bl.weight.mul_(1.0/math.sqrt(v/target_var)); n_rescaled+=1
    if was_training: model.train()
    # Record whether LSUV was a genuine intervention or a mathematical no-op, so
    # the manifest states which of the two it was in the run that actually ran.
    LSUV_DEGENERATE["rescale_ops"]=n_rescaled
    LSUV_DEGENERATE["max_abs_var_dev_at_init"]=float(max_dev)
    LSUV_DEGENERATE["is_noop"]=bool(n_rescaled==0)
    if verbose:
        print(f"  LSUV: {n_rescaled} rescale ops, max |var-1| at init = {max_dev:.5f}"
              + ("  -> NO-OP (pre-normalised architecture)" if n_rescaled==0 else ""))
    return model

# ---------------------------------------------------------------------------
# (R20 FIX) A DUPLICATED, SHADOWING DEFINITION BLOCK WAS REMOVED HERE.
# ---------------------------------------------------------------------------
# Cell 8 defined init_xavier, init_kaiming, init_lsuv and lsuv_calibrate TWICE.
# Python keeps the LAST binding, so the version that actually ran was the older,
# pre-R13 one -- silently undoing R13 inside R13's own cell. Three consequences,
# none of which raised:
#
#  (1) LSUV_DEGENERATE WAS NEVER WRITTEN. All three assignments live in the FIRST
#      lsuv_calibrate; the second has none. The dict therefore stayed at its
#      declared {None,None,None} for the whole grid, and Cell 15 wrote it to the
#      manifest as "lsuv_degeneracy" under prose claiming the value was MEASURED.
#      The LSUV-is-degenerate-in-a-pre-normalised-architecture finding -- one of
#      the few genuinely novel observations in this work -- was unmeasurable.
#  (2) The preflight checklist item `LSUV_DEGENERATE.is_noop == true` could never
#      pass, and would have read as an architecture surprise rather than a
#      missing measurement.
#  (3) tol reverted 0.01 -> 0.10, exactly the loosening R13 documented tightening
#      (its stated purpose: engage if a future architecture DOES drift).
#
# Found by grepping for duplicate top-level definitions -- invisible to linear
# reading because both blocks look correct in isolation.
INIT={"standard":init_standard,"bitnet":init_bitnet,"xavier":init_xavier,
      "kaiming":init_kaiming,"lsuv":init_lsuv,"tai":init_tai}
# (R13) Initializers that need a model+data stage after the per-tensor pass.
# apply_init only sees tensors, so it cannot complete these; train_run calls
# finish_init(model, batch, method) right after apply_init. Keeping the list
# explicit means a future data-driven init cannot be silently half-applied.
INIT_NEEDS_DATA={"lsuv"}
def finish_init(model,method,batch):
    base=("tai_negctrl" if method=="tai_negctrl"
          else "tai" if method.startswith("tai") else method)
    if base in INIT_NEEDS_DATA and batch is not None:
        return lsuv_calibrate(model,batch)
    return model
def apply_init(model,method,rho0=0.75,gen=None):
    # gen: optional torch.Generator. When provided AND method is TAI, every
    # BitLinear is initialised from this single generator so the whole-model
    # init is a deterministic function of (gen-state, rho0). The sweep uses this
    # to hold the init RNG fixed and vary ONLY rho0 (causal intervention #4).
    if method=="tai_negctrl": base="tai_negctrl"      # (#4) H3 falsification arm
    elif method.startswith("tai"): base="tai"
    else: base=method
    for bl in model.bitlinears():
        fi=bl.weight.shape[1]
        if base in ("tai","tai_negctrl"): INIT[base](bl.weight,fi,rho0=rho0,gen=gen)
        else: INIT[base](bl.weight,fi)
# ============================================================================
# (R6) MECHANISM-MATCHED BASELINE: `gamma_offset`
# ----------------------------------------------------------------------------
# THE PROBLEM THIS FIXES. As designed, TAI was the ONLY arm in the entire grid
# capable of moving rho0 off its locked ~0.69 value. H3 ("sweeping rho0 shifts
# final sparsity") was therefore close to untestable-by-construction: no other
# method could even attempt the intervention, so H3 could not fail for any reason
# other than the machinery being broken. A reviewer's first question is exactly
# the one the design could not answer: *is the INIT the right place to control
# rho0, or would simply moving the THRESHOLD do the same job?*
#
# gamma_offset answers it. It leaves the initializer completely untouched
# (standard zero-mean Gaussian) and instead FIXES the absmean threshold scale to
# a constant c chosen so that the initial active fraction equals the SAME rho0
# TAI targets. Both arms then start ternary training at an identical operating
# point, reached by two different mechanisms:
#     TAI          : shape the weight MAGNITUDE distribution, keep gamma = mean|W|
#     gamma_offset : keep the standard magnitude distribution, scale gamma by c
# This is the cleanest possible attribution test for the paper's central claim.
# Pre-registered readings:
#   * TAI ~ gamma_offset on quality  -> the contribution is "rho0 is controllable
#     and rho0 matters", NOT "TAI's initializer is special". The paper must then
#     be framed around the operating point, with gamma_offset as the simpler
#     recommended implementation. This is a REFRAMING, not a refutation.
#   * TAI > gamma_offset             -> the init distribution carries value beyond
#     the operating point (e.g. threshold-scaling distorts the effective LR /
#     gradient scale of every layer, while TAI does not). That is a genuinely
#     stronger result, and it is only credible BECAUSE this arm exists to lose.
#   * TAI < gamma_offset             -> report it; the simpler method wins.
# Note gamma_offset is NOT free: it permanently rescales gamma, which changes the
# STE gradient scale, whereas TAI's effect decays as weights move. That asymmetry
# is precisely the mechanism the comparison is designed to expose.
from math import erf as _erf, sqrt as _sqrt, pi as _pi
def _rho_of_c(c):
    # For zero-mean Gaussian W: gamma = c*E|w| = c*sigma*sqrt(2/pi); a weight
    # survives iff |w| >= gamma/2, i.e. |z| >= c*sqrt(2/pi)/2 (sigma cancels --
    # the same sigma-independence that locks every standard init at c=1).
    z=c*_sqrt(2/_pi)/2
    return 2*(1-0.5*(1+_erf(z/_sqrt(2))))
def gamma_scale_for_rho0(rho0,lo=1e-4,hi=20.0,iters=200):
    # Monotone decreasing rho(c) -> bisection is exact to machine precision.
    # rho0=0.69 recovers c=1.0 (the locked baseline), which is the sanity check.
    for _ in range(iters):
        mid=0.5*(lo+hi)
        if _rho_of_c(mid)>rho0: lo=mid
        else: hi=mid
    return 0.5*(lo+hi)
GAMMA_OFFSET_C={}   # rho0 -> c cache (filled by train_run)
_c69=gamma_scale_for_rho0(0.69)
assert abs(_c69-1.0)<0.02, f"gamma_scale_for_rho0 miscalibrated: rho0=0.69 -> c={_c69:.4f}, expected ~1.0"
print(f"(R6) gamma_offset calibration OK: rho0=0.69 -> c={_c69:.4f} (locked baseline recovers c=1)")

def gamma_warmup_value(step,warmup_steps,start=0.4):
    if warmup_steps<=0 or step>=warmup_steps: return 1.0
    return start+(1.0-start)*(step/warmup_steps)
# (#4 H3 NEGATIVE CONTROL) init_tai_shuffled: identical machinery to init_tai but
# the rho0 LABEL is decoupled from the threshold split it produces. We realise the
# split for a FIXED reference rho0 (0.69, the locked baseline operating point) so
# the actual init is the SAME regardless of the rho0 passed in; the caller still
# records the (fake) rho0 label. If the sweep's sparsity~rho0 relationship were an
# artefact of the analysis rather than a causal effect of the threshold split,
# this control would still show a trend. Expected result: FLAT (Spearman ~0).
def init_tai_shuffled(w,fi,rho0=0.75,gen=None):
    return init_tai(w,fi,rho0=0.69,gen=gen)   # label varies, realised split fixed
INIT["tai_negctrl"]=init_tai_shuffled
# init-function definitions above are always available; the GPU demo below only
# runs when armed, so a fresh Run-All in safe mode touches no GPU here.
if RUN_CFG.get("ARM_FULL_RUN",False):
    # (R41) named _mdl, not _m: Cell 13 binds `import math as _m` at module level
    # and notebook cells share one namespace, so a stray _m here would replace the
    # math module for every later cell. Same class as the nn and _t collisions.
    seed_everything(); _mdl=LM(SIZES["dim384"]).to(DEVICE,DTYPE)
    for r in [0.5,0.9]:
        apply_init(_mdl,"tai",rho0=r); print(f"TAI rho0={r} -> init active {np.mean([bl.active_ratio() for bl in _mdl.bitlinears()]):.3f}")
    del _mdl; torch.cuda.empty_cache()
else:
    print("TAI init strategies defined. [safe mode] init demo skipped (ARM_FULL_RUN off).")


In [ ]:
# === CELL 9: DRIVE-CACHED DATA (real GPT-2 BPE tokenizer, tokenize ONCE) =====
# REVISION (external validity): byte-level tokenization is replaced by a real
# GPT-2 BPE tokenizer so that (a) training distribution matches what standard
# LMs see, and (b) the held-out lm-eval-harness tasks in Cell 11b are scored
# on the SAME vocabulary the model was trained on. Vocab is the full GPT-2 BPE
# (50257) so harness option-strings tokenize correctly; corpus + val are cached
# to Drive as uint16 memmaps and never re-downloaded across sessions.
import os,json,numpy as np,torch
SEQ=RUN_CFG["seq_len"]; TARGET_TOKENS=48_000_000  # >= 2x(budget)/0.95 so 20M-token runs never exhaust the split
def _build_tokenizer():
    from transformers import AutoTokenizer
    tok=AutoTokenizer.from_pretrained("gpt2")
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    return tok
TOKENIZER=_build_tokenizer()
VOCAB=TOKENIZER.vocab_size          # 50257 (real BPE, not byte-mod-8192)
DATA_BIN=os.path.join(PATHS["data"],f"corpus_gpt2bpe_{TARGET_TOKENS}.npy")
DATA_META=DATA_BIN+".meta.json"
def build_and_cache():
    if os.path.exists(DATA_BIN) and os.path.exists(DATA_META):
        notify("DATA CACHE HIT",json.load(open(DATA_META))["source"],tag="data"); return
    notify("DATA BUILD START","download + real BPE tokenize once",tag="data")
    ids=[]; src="synthetic"; eot=TOKENIZER.eos_token_id
    try:
        from datasets import load_dataset
        ds=load_dataset("HuggingFaceFW/fineweb-edu",name="sample-10BT",
                        split="train",streaming=True); n=0
        for i,ex in enumerate(ds):
            t=ex.get("text","")
            if not t: continue
            enc=TOKENIZER.encode(t); enc.append(eot)
            ids.extend(enc); n+=len(enc)
            if n>=TARGET_TOKENS: src="FineWeb-EDU (GPT-2 BPE)"; break
            if i>200000: break
    except Exception as e: notify("DATA STREAM FALLBACK",str(e)[:100],tag="data")
    if len(ids)<TARGET_TOKENS:
        # deterministic synthetic fallback over the REAL vocab so downstream
        # tokenization stays consistent; flagged explicitly in the manifest.
        rng=np.random.default_rng(GLOBAL_SEED)
        base=rng.integers(0,VOCAB,8192,dtype=np.uint32)
        arr=np.tile(base,TARGET_TOKENS//len(base)+1)[:TARGET_TOKENS].copy()
        flip=rng.random(TARGET_TOKENS)<0.1; arr[flip]=rng.integers(0,VOCAB,int(flip.sum()),dtype=np.uint32)
        ids=arr.tolist(); src="synthetic (deterministic, real-vocab)"
    data=np.asarray(ids[:TARGET_TOKENS],dtype=np.uint16 if VOCAB<65536 else np.uint32)
    np.save(DATA_BIN,data)
    json.dump({"source":src,"tokens":int(len(data)),"vocab":int(VOCAB),
               "tokenizer":"gpt2"},open(DATA_META,"w"))
    notify("DATA BUILD DONE",f"{len(data):,} tokens {src} vocab {VOCAB}",tag="data")
build_and_cache()
DATA=torch.from_numpy(np.asarray(np.load(DATA_BIN,mmap_mode="r"),dtype=np.int64))
# G5 (no leakage) — HARDENED. Two reviewer-facing fixes:
#  (#1 LEAKAGE GUARD BAND) A single 95/5 cut is NOT enough: get_batch draws a
#    random window [i,i+seq); a train window starting in [_sp-seq,_sp) overlaps
#    val tokens. We therefore insert a GUARD BAND of `seq` tokens that belongs to
#    NEITHER split, and clamp the train range to [0,_sp-seq) so no train window
#    can ever reach into val. The guard band is dropped, not trained or scored.
#  (#5 VAL IS NOT A SINGLE TAIL BLOCK) Reporting val on only the last 5% makes
#    "seed variance" (H4) blind to data-sampling variance and risks a tail with a
#    different topic mix. We instead carve K=5 INTERLEAVED val folds spread across
#    the corpus (every K-th contiguous block), so val covers the whole document
#    distribution and a held-out-fold protocol is available. The primary VAL used
#    everywhere downstream is fold 0 (kept identical in spirit to before, so all
#    existing run records remain comparable); the other folds feed the new
#    data-variance ablation (Cell 13f) WITHOUT retraining.
# ============================================================================
# (R15) HELD-OUT SPLIT — REBUILT. Two defects, one of them a data leak.
# ----------------------------------------------------------------------------
# DEFECT 1 — the comment did not describe the code. It promised "K INTERLEAVED
# val folds spread across the corpus" and "every K-th contiguous block". The code
# built _val_ranges=[(f*blk,(f+1)*blk)] — five CONTIGUOUS slices, i.e. fold f is
# simply the f-th 20% of the corpus. Nothing was interleaved. The stated
# justification ("val covers the whole document distribution") therefore did not
# hold: with FineWeb-EDU streamed in order, each fold is a different slab of
# documents, so fold-to-fold differences are DISTRIBUTION SHIFT, not sampling noise.
#
# DEFECT 2 (the leak) — TRAIN was derived per-fold, but only fold 0's TRAIN is ever
# used for training, while R5 scores the model on EVERY fold's val. Since fold 0's
# TRAIN = everything outside fold 0, the val blocks of folds 1..K-1 are *inside*
# the training data. Measured on a 100k-token toy corpus: fold 0 overlap 0/20000,
# folds 1-4 overlap 20000/20000 — a 100% leak. H4's "data-sampling variance" would
# have been computed from four TRAIN-loss measurements mislabelled as held-out,
# biasing TAI's variance downward against a baseline measured the same wrong way,
# and reporting a leaked number as an honesty improvement.
#
# THE FIX — one global held-out region, carved out of training ONCE, then split
# into K sub-folds. Every sub-fold is genuinely unseen. To honour the original
# intent (val should span the corpus, not just a tail slab), the held-out region
# is INTERLEAVED for real: we take every K_STRIDE-th contiguous chunk across the
# whole corpus, so held-out data is spread over the full document distribution
# while remaining disjoint from train, with a guard band on each side of every
# chunk to kill window-overlap leakage.
_NFOLD=int(RUN_CFG.get("val_folds",5))
_GUARD=RUN_CFG["seq_len"]
_CHUNK=int(RUN_CFG.get("val_chunk_tokens",50_000))   # granularity of interleaving
_HOLD_FRAC=float(RUN_CFG.get("holdout_frac",0.06))   # share of corpus held out in total

# 1) choose interleaved chunk starts spanning the WHOLE corpus
_n_chunks_total=max(1,len(DATA)//_CHUNK)
_n_hold=max(_NFOLD,int(round(_n_chunks_total*_HOLD_FRAC)))
_n_hold=min(_n_hold,_n_chunks_total)
_stride=max(1,_n_chunks_total//_n_hold)
_hold_chunks=[i*_stride for i in range(_n_hold) if i*_stride<_n_chunks_total]
_hold_ranges=[(c*_CHUNK,min(len(DATA),(c+1)*_CHUNK)) for c in _hold_chunks]

# 2) TRAIN = corpus minus (held-out chunks + guard bands on both sides)
_excl=[(max(0,a-_GUARD),min(len(DATA),b+_GUARD)) for a,b in _hold_ranges]
_train_parts=[];_cur=0
for a,b in _excl:
    if a>_cur: _train_parts.append(DATA[_cur:a])
    _cur=max(_cur,b)
if _cur<len(DATA): _train_parts.append(DATA[_cur:])
TRAIN=torch.cat(_train_parts) if len(_train_parts)>1 else _train_parts[0]

# 3) the held-out region is split into K sub-folds; ALL are outside TRAIN
_hold_blocks=[DATA[a:b] for a,b in _hold_ranges]
VAL_FOLDS=[]
for f in range(_NFOLD):
    _mine=[blk for j,blk in enumerate(_hold_blocks) if j%_NFOLD==f]
    if not _mine: _mine=[_hold_blocks[f%len(_hold_blocks)]]
    _v=torch.cat(_mine) if len(_mine)>1 else _mine[0]
    VAL_FOLDS.append((TRAIN,_v))     # TRAIN is IDENTICAL for every fold (no leak)
VAL=VAL_FOLDS[0][1]                  # primary split = sub-fold 0

# 4) prove the properties we claim, rather than asserting them in a comment.
assert len(VAL)>RUN_CFG["seq_len"]*RUN_CFG["batch_size"]*5, "primary val fold too small"
_train_total=len(TRAIN); _hold_total=sum(len(b) for b in _hold_blocks)
assert _train_total+_hold_total<=len(DATA), "split accounting failed"
# no held-out token may live inside any training window: guaranteed by construction
# (train parts are built from the complement of held-out+guard), verified here on a
# sample for cheapness -- a full set-intersection on 48M tokens is wasteful.
_rng_chk=np.random.default_rng(0)
for _f,(_tr,_va) in enumerate(VAL_FOLDS):
    _idx=_rng_chk.integers(0,len(_va),size=min(2000,len(_va)))
    _probe=set(_va[_idx].tolist())
    _tr_s=set(_tr[_rng_chk.integers(0,len(_tr),size=min(20000,len(_tr)))].tolist())
    # token VALUES repeat across a corpus, so value-overlap is meaningless; the real
    # guarantee is positional and holds by construction. We assert the invariant that
    # actually matters: every fold shares ONE training set, so no fold can be "held
    # out for training" while being trained on -- the R15 leak is structurally absent.
    assert _tr is TRAIN, f"fold {_f} has a different TRAIN -> per-fold-train leak is back"
print(f"Split (R15): {len(_hold_ranges)} interleaved held-out chunks of {_CHUNK:,} tok "
      f"(stride {_stride}), guard={_GUARD} tok/side")
print(f"  TRAIN={_train_total:,} tok | held-out total={_hold_total:,} tok "
      f"({100*_hold_total/len(DATA):.1f}%) | {_NFOLD} sub-folds | primary VAL={len(VAL):,} tok")
print(f"  every sub-fold is disjoint from the SINGLE shared TRAIN -> R5's per-fold "
      f"scores are genuinely held out")
DATA_META_INFO=json.load(open(DATA_META))
USED_SYNTHETIC=DATA_META_INFO["source"].startswith("synthetic")
print("Corpus:",DATA_META_INFO["source"],"|",len(DATA),"tokens | vocab",VOCAB)
if USED_SYNTHETIC:
    print("WARNING: synthetic fallback active — downstream lm-eval numbers are not "
          "meaningful and will be flagged in the manifest.")
# ============================================================================
# (R22) VECTORISED get_batch — identical batches, no python loop, async copy.
# ----------------------------------------------------------------------------
# The original built the batch with a PYTHON LOOP over bs=32 slices plus two
# torch.stack calls, then issued a BLOCKING host-to-device copy -- twice, once
# for x and once for y, copying nearly the same 16k tokens both times. At small
# scales the model step is only a few tens of ms, so this host-side work was a
# non-trivial share of wall-clock, and the blocking copy serialised CPU and GPU.
#
# The replacement draws ix from the SAME generator with the SAME shape and dtype,
# so the sampled offsets -- and therefore the exact token windows every seed sees
# -- are BIT-IDENTICAL to before. Data order is unchanged; only how the bytes are
# gathered and moved changes. This matters: it means runs completed before this
# patch remain directly comparable to runs after it.
_STAGE={}
def _pinned(bs,seq,dtype):
    k=(bs,seq,dtype)
    if k not in _STAGE:
        try: _STAGE[k]=torch.empty((bs,seq),dtype=dtype).pin_memory()
        except Exception: _STAGE[k]=torch.empty((bs,seq),dtype=dtype)
    return _STAGE[k]
def get_batch(d,bs,seq,gen):
    ix=torch.randint(0,len(d)-seq-1,(bs,),generator=gen)   # unchanged draw
    win=d[ix[:,None]+torch.arange(seq+1)[None,:]]          # (bs, seq+1) one gather
    try:
        if "FAST_CFG" in globals() and FAST_CFG.get("pin_batches",False):
            buf=_pinned(bs,seq+1,win.dtype); buf.copy_(win)
            win=buf.to(DEVICE,non_blocking=True)           # async H2D from pinned
        else:
            win=win.to(DEVICE)
    except Exception:
        win=win.to(DEVICE)
    return win[:,:-1].contiguous(), win[:,1:].contiguous()

# rebind every size config to the REAL tokenizer vocab (Cell 7 used a placeholder)
from dataclasses import replace
# ============================================================================
# (R24 FIX) max_seq_len WAS NEVER REBOUND — EVERY RUN WOULD HAVE CRASHED.
# ----------------------------------------------------------------------------
# Cfg declares max_seq_len=256 as a placeholder, exactly like vocab_size=8192.
# This line rebound vocab_size and SILENTLY LEFT max_seq_len AT 256, while
# RUN_CFG["seq_len"]=512 drives every batch. LM.__init__ builds the RoPE cos/sin
# buffers from c.max_seq_len, so the tables covered 256 positions and the first
# training step of EVERY run raised:
#     RuntimeError: The size of tensor a (512) must match the size of tensor b
#                   (256) at non-singleton dimension 2
# Not a slowdown, not a subtle bias: total failure, on run #1, before a single
# gradient step. The existing schema/geometry guard passed all 28 configs because
# it validated Cfg fields against each other and never against RUN_CFG.
# Found by a standalone benchmark that instantiated the real model at the real
# sequence length -- the notebook's own preflight would have hit it too, but only
# after the 40-minute corpus download.
SIZES={k:replace(v,vocab_size=VOCAB,max_seq_len=max(v.max_seq_len,RUN_CFG["seq_len"])) for k,v in SIZES.items()}
SIZES_V2={k:replace(v,vocab_size=VOCAB,max_seq_len=max(v.max_seq_len,RUN_CFG["seq_len"])) for k,v in SIZES_V2.items()}
SIZES_WC={k:replace(v,vocab_size=VOCAB,max_seq_len=max(v.max_seq_len,RUN_CFG["seq_len"])) for k,v in SIZES_WC.items()}
# The guard the original was missing: configs must be validated against the
# TRAINING config, not only against themselves.
for _nm,_d in (("SIZES",SIZES),("SIZES_V2",SIZES_V2),("SIZES_WC",SIZES_WC)):
    for _k,_v in _d.items():
        assert _v.max_seq_len>=RUN_CFG["seq_len"], (
            f"{_nm}[{_k}].max_seq_len={_v.max_seq_len} < RUN_CFG['seq_len']="
            f"{RUN_CFG['seq_len']}: RoPE tables would be too short and every "
            "forward pass would raise.")
        assert _v.vocab_size==VOCAB, f"{_nm}[{_k}] vocab not rebound"
print("SIZES vocab rebound to",VOCAB,"->",{k:v.vocab_size for k,v in SIZES.items()})


In [ ]:
# === CELL 10: RUN-STORE (per-run checkpoint / resume to Drive) ==============
# Each completed run is one JSON in runs/. run_id encodes (phase,size,method,seed,
# rho0). On Run-All, finished run_ids are loaded and skipped; only the remainder
# execute. Atomic write prevents corruption if a session dies mid-save.
import os,json
def run_id(phase,size,method,seed,rho0=None):
    r=f"r{rho0}" if rho0 is not None else "r-"
    return f"{phase}__{size}__{method}__s{seed}__{r}"
def run_path(rid): return os.path.join(PATHS["runs"],rid+".json")
def have_run(rid): return os.path.exists(run_path(rid))
def load_run(rid): return json.load(open(run_path(rid)))
# (R42) SHARED JSON KEY SANITISER.
# json.dumps rejects any non-(str,int,float,bool,None) KEY outright, and `default=`
# does not help -- it is consulted for unknown VALUES only. Several analysis
# collections here are keyed by the tuple (size, method); the manifest write, which
# is the LAST artifact produced after the entire compute budget has been spent,
# died on exactly that. Keys are coerced to STRINGS rather than merely to legal
# types, because json.dumps(sort_keys=True) -- needed for a stable hash -- cannot
# order a dict that mixes int and str keys either.
def json_safe(o):
    if isinstance(o,dict):
        return {("/".join(map(str,k)) if isinstance(k,tuple) else str(k)):json_safe(v)
                for k,v in o.items()}
    if isinstance(o,(list,tuple)): return [json_safe(x) for x in o]
    return o

def save_run(rid,rec):
    tmp=run_path(rid)+".tmp"; json.dump(rec,open(tmp,"w")); os.replace(tmp,run_path(rid))
def all_runs(phase=None):
    out=[]
    for fn in os.listdir(PATHS["runs"]):
        if not fn.endswith(".json"): continue
        if phase and not fn.startswith(phase+"__"): continue
        try: out.append(json.load(open(os.path.join(PATHS["runs"],fn))))
        except Exception: pass
    return out
print("Run-store ready. Existing runs:",len([f for f in os.listdir(PATHS['runs']) if f.endswith('.json')]))


In [ ]:
# === CELL 11b: DOWNSTREAM lm-eval (loglikelihood mode, real BPE) ============
# REVISION (external validity). At this scale, *generation* benchmarks are at
# chance; the methodologically correct evaluation is the loglikelihood/multiple-
# choice protocol used by lm-eval-harness: for each question, score every answer
# option by log-likelihood and pick the argmax, reporting BOTH harness metrics:
#   acc      = token-length-normalized loglik (mean log p per continuation token)
#   acc_norm = byte-length-normalized loglik (sum log p / #UTF-8 bytes), the
#              standard HellaSwag metric that removes option-length bias.
# This yields above-chance, method-discriminating signal even for small models
# because it reads the model's calibrated preference rather than its (untrained)
# free-generation ability. We evaluate ARC-Easy and HellaSwag validation splits.
# Tasks are cached to Drive once; if download fails we skip (flagged), never fake.
import os,json,math,torch,numpy as np
LMEVAL_CACHE=os.path.join(PATHS["data"],"lmeval_tasks.json")
LMEVAL_N=int(RUN_CFG.get("lmeval_n",300))  # items per task (subset for speed)
def _load_tasks():
    if os.path.exists(LMEVAL_CACHE):
        return json.load(open(LMEVAL_CACHE))
    tasks={}
    try:
        from datasets import load_dataset
        # ARC-Easy: question + 3-5 options, one correct
        arc=load_dataset("allenai/ai2_arc","ARC-Easy",split="validation")
        ae=[]
        for ex in arc:
            opts=ex["choices"]["text"]; labs=ex["choices"]["label"]
            if ex["answerKey"] not in labs: continue
            gold=labs.index(ex["answerKey"])
            ae.append({"ctx":ex["question"].strip()+" Answer:",
                       "conts":[" "+o.strip() for o in opts],"gold":gold})
            if len(ae)>=LMEVAL_N: break
        tasks["arc_easy"]=ae
        # HellaSwag: context + 4 endings, one correct
        hs=load_dataset("Rowan/hellaswag",split="validation")
        he=[]
        for ex in hs:
            if ex["label"]=="" : continue
            ctx=(ex["ctx_a"]+" "+ex["ctx_b"]).strip()
            he.append({"ctx":ctx,"conts":[" "+e.strip() for e in ex["endings"]],
                       "gold":int(ex["label"])})
            if len(he)>=LMEVAL_N: break
        tasks["hellaswag"]=he
        json.dump(tasks,open(LMEVAL_CACHE,"w"))
        notify("LMEVAL TASKS CACHED",f"arc_easy {len(tasks['arc_easy'])}, hellaswag {len(tasks['hellaswag'])}",tag="data")
    except Exception as e:
        notify("LMEVAL TASKS UNAVAILABLE",str(e)[:120],tag="data"); return {}
    return tasks
LMEVAL_TASKS=_load_tasks()
LMEVAL_AVAILABLE=bool(LMEVAL_TASKS) and not USED_SYNTHETIC

# ============================================================================
# (R22) BATCHED SCORING — same protocol, one padded forward instead of ~2400.
# ----------------------------------------------------------------------------
# The original scored every (item, option) pair with its own batch-size-1 forward
# pass: 2 tasks x 300 items x ~4 options ~= 2400 model calls PER RUN, on a GPU
# that is almost entirely idle at batch 1. Across the declared grid that was tens
# of hours spent on launch overhead alone.
#
# The scoring RULE is untouched: for each option we still sum log p over exactly
# the continuation tokens, and still report both the token-normalised (acc) and
# byte-normalised (acc_norm) argmax, as lm-eval-harness does. Only the batching
# changes. RIGHT padding is safe here because attention is causal: a pad token at
# position j cannot influence any position < j, and every position we score
# precedes the padding of its own row. The mask makes pad positions contribute
# exactly zero to the sum.
@torch.no_grad()
def _score_padded(model,seqs,cont_lens,cont_bytes):
    """seqs: list of full token id lists (ctx+cont). Returns per-seq (sum_logp, ntok, nbytes)."""
    ML=model.cfg.max_seq_len
    seqs=[s[-ML:] if len(s)>ML else s for s in seqs]
    L=max(len(s) for s in seqs)
    if L<2: return [(-1e9,1,1)]*len(seqs)
    B=len(seqs)
    x=torch.zeros((B,L-1),dtype=torch.long); y=torch.zeros((B,L-1),dtype=torch.long)
    m=torch.zeros((B,L-1),dtype=torch.bool); ntok=[]
    for i,s in enumerate(seqs):
        n=len(s)
        if n<2: ntok.append(1); continue
        x[i,:n-1]=torch.tensor(s[:-1],dtype=torch.long)
        y[i,:n-1]=torch.tensor(s[1:],dtype=torch.long)
        c=max(1,min(cont_lens[i],n-1)); m[i,n-1-c:n-1]=True; ntok.append(c)
    x=x.to(DEVICE); y=y.to(DEVICE); m=m.to(DEVICE)
    with torch.autocast("cuda",dtype=DTYPE): lg,_=model(x)
    lp=torch.log_softmax(lg.float(),dim=-1)
    sel=lp.gather(2,y[:,:,None]).squeeze(2)*m
    sums=sel.sum(1).tolist()
    return [(float(sums[i]),int(max(ntok[i],1)),int(max(cont_bytes[i],1))) for i in range(B)]

@torch.no_grad()
def lmeval_accuracy(model,tasks=LMEVAL_TASKS):
    # returns {task: acc, task+"_norm": acc_norm} for each task.
    if not LMEVAL_AVAILABLE: return {}
    BS_LM=int(RUN_CFG.get("lmeval_batch",16))
    model.eval(); out={}
    for name,items in tasks.items():
        # flatten every (item, option) into one work list, then score in chunks
        seqs=[]; clen=[]; cby=[]; owner=[]
        for qi,it in enumerate(items):
            cctx=TOKENIZER.encode(it["ctx"])
            for c in it["conts"]:
                ct=TOKENIZER.encode(c)
                seqs.append(cctx+ct); clen.append(len(ct))
                cby.append(max(len(c.encode("utf-8")),1)); owner.append(qi)
        scores=[]
        for a in range(0,len(seqs),BS_LM):
            scores.extend(_score_padded(model,seqs[a:a+BS_LM],clen[a:a+BS_LM],cby[a:a+BS_LM]))
        per_q={}
        for (sm,nt,nb),qi in zip(scores,owner):
            per_q.setdefault(qi,[[],[]])
            per_q[qi][0].append(sm/max(nt,1))   # acc: per-token
            per_q[qi][1].append(sm/max(nb,1))   # acc_norm: per-byte
        c_acc=c_norm=0
        for qi,it in enumerate(items):
            tk,by=per_q.get(qi,[[],[]])
            if not tk: continue
            if int(np.argmax(tk))==it["gold"]:  c_acc+=1
            if int(np.argmax(by))==it["gold"]: c_norm+=1
        n=max(len(items),1)
        out[name]=c_acc/n; out[name+"_norm"]=c_norm/n
    model.train(); return out
print("lm-eval downstream:",
      "ENABLED (arc_easy+hellaswag, loglik mode)" if LMEVAL_AVAILABLE
      else "DISABLED (tasks unavailable or synthetic corpus)")


In [ ]:
# === CELL 11: MAIN EXPERIMENT (plateau-verified, budget + email + resume) ===
# Confirmatory scales x 7 methods x 15 seeds (+ reduced-seed exploratory ladder),
# token-equal at 20M tokens. Calibration runs
# (standard init, 3 seeds) are REUSED as standard-method results; H1 target loss
# = median over those trajectories. Each completed run persists to Drive. On
# session-budget exceed: persist, email SESSION BOUNDARY, exit (resume later).
#
# NEUROCOMPUTING REVISION:
#  * method set is non-degenerate: fp16_control, standard, lsuv, bitnet158
#    (INDEPENDENT recipe), sparse_sched, gamma_offset (R6), tai, tai_warmup.
#    (R13 CORRECTION) An earlier version of this comment called standard and lsuv
#    "two distinct locked-rho0 FAMILIES". That was false. Both are zero-mean and
#    normalised to the same sigma, and measured they sit at rho0=0.6900+/-0.0011
#    vs 0.6906+/-0.0013 -- one family, two labels. Worse, LSUV is DEGENERATE in
#    this architecture: BitLinear pre-normalises its input with RMSNorm, so output
#    variance at init is already 1.0004 and LSUV's calibration rescales nothing
#    (measured; see LSUV_DEGENERATE in the manifest). We keep `lsuv` as a grid
#    point because a locked baseline that arrives via orthogonal init is still a
#    fair baseline -- but it is NOT independent evidence of baseline breadth, and
#    any "best of {standard,lsuv}" contrast is max-selection over one family.
#    The genuine second family is `gamma_offset`, which differs at the OPERATING
#    POINT rather than in weight distribution.
#    The remaining locked inits (xavier/kaiming/bitnet-init) are shown in Cell
#    13b to coincide with `standard` at rho0~0.69 and are NOT re-run as grid
#    points -- spending 15 seeds on duplicates would misrepresent the baseline.
#  * each run is PLATEAU-CHECKED: mean loss over the last `plateau_frac` of steps
#    must be within `plateau_tol` of the final value, so H2 measures converged
#    quality rather than mid-training noise. The flag is stored per run.
#  * the synthetic copy-task proxy is DROPPED (it tracks active-weight density,
#    not method quality); next-token top-1 + real lm-eval remain.
import time,math,torch,torch.nn as nn,numpy as np
BS=RUN_CFG["batch_size"]; SEQ=RUN_CFG["seq_len"]; WARMUP=RUN_CFG["warmup_steps"]
PEAK_LR=RUN_CFG["peak_lr"]; MIN_LR=RUN_CFG["min_lr"]; WD=RUN_CFG["weight_decay"]; CLIP=RUN_CFG["grad_clip"]
SEEDS=RUN_CFG["seeds"]; TOKEN_BUDGET=RUN_CFG["token_budget"]
PLATEAU_TOL=RUN_CFG["plateau_tol"]; PLATEAU_FRAC=RUN_CFG["plateau_frac"]
# (R3) minimum number of val-trace points in the tail for the slope CI to have
# usable power; below this the gate returns False (cannot certify) rather than
# passing a run on the strength of 2-3 noisy probes.
PLATEAU_MIN_TAIL=int(RUN_CFG.get("plateau_min_tail",5))
# (method_name, opts). opts may carry: full_precision, recipe, rho0, gamma_warmup.
METHODS=[("fp16_control",{"full_precision":True}),
         ("standard",{}),
         ("lsuv",{}),
         ("bitnet158",{"recipe":"bitnet"}),              # independent published recipe
         ("sparse_sched",{"recipe":"sched","sched_target":1.6}),  # R3: gradual-pruning baseline (rising gamma -> ~48%+ forward sparsity)
         # (R6) MECHANISM-MATCHED CONTROL: standard init, but gamma is scaled by a
         # constant c chosen analytically so the INITIAL active fraction equals the
         # same rho0 TAI uses. Same operating point, different mechanism -> isolates
         # "control of rho0" from "TAI's particular initializer". See Cell 8.
         ("gamma_offset",{"rho0":0.75,"gamma_offset":True}),
         ("tai",{"rho0":0.75}),
         ("tai_warmup",{"rho0":0.75,"gamma_warmup":240})]
STEPS=math.ceil(TOKEN_BUDGET/(BS*SEQ))
# (FIX-1b) BIND the confirmatory/exploratory scale declaration to EXECUTION.
# The full method x 15-seed token-equal grid runs ONLY on confirmatory scales;
# exploratory ladder points run a reduced seed set (for the H6 trend only). This
# is what keeps the grid inside one A100's budget AND makes the manifest's scale
# claims match what actually ran. Any cell that trains MUST iterate SCALES_TO_RUN
# and size its seed loop with SEEDS_FOR(size) -- never raw SIZES / SEEDS.
_CONF_SET=[s for s in SIZES if s in RUN_CFG.get("confirmatory_scales",list(SIZES))]
_EXPL_SET=[s for s in SIZES if s in RUN_CFG.get("exploratory_scales",[])]
SCALES_TO_RUN=[s for s in SIZES if s in (set(_CONF_SET)|set(_EXPL_SET))] or list(SIZES)
def SEEDS_FOR(size):
    # confirmatory scales: full seed set; exploratory: reduced set for the trend.
    if size in _EXPL_SET and size not in _CONF_SET:
        return RUN_CFG.get("exploratory_seeds",RUN_CFG["seeds"][:5])
    return SEEDS
print(f"[grid] confirmatory={_CONF_SET} (x{len(SEEDS)} seeds) | "
      f"exploratory={_EXPL_SET} (x{len(RUN_CFG.get('exploratory_seeds',[]))} seeds)")
SCHED_TARGET={"val":1.0}  # rising gamma multiplier target for sparse_sched
def gamma_sched_value(t,steps,target):
    # linearly raise gamma multiplier 1.0 -> target over the first 70% of steps
    frac=min(1.0,t/max(1,int(0.7*steps))); return 1.0+(target-1.0)*frac
def lr_at(t):
    if t<WARMUP: return PEAK_LR*(t+1)/WARMUP
    p=(t-WARMUP)/max(1,STEPS-WARMUP); return MIN_LR+0.5*(PEAK_LR-MIN_LR)*(1+math.cos(math.pi*p))
@torch.no_grad()
def evaluate(model,n=20,split=None,seed_off=99):
    # (R5) `split` selects WHICH held-out fold to score. Default None -> VAL
    # (fold 0), preserving every existing call site. VAL_FOLDS[k] (built in Cell 9)
    # gives the K interleaved folds used by the data-variance decomposition.
    d=VAL if split is None else split
    model.eval(); g=torch.Generator().manual_seed(GLOBAL_SEED+seed_off); ls=[]
    for _ in range(n):
        xb,yb=get_batch(d,BS,SEQ,g)
        with torch.autocast("cuda",dtype=DTYPE): _,l=model(xb,yb)
        ls.append(l.item())
    model.train(); return float(np.mean(ls))
def smooth(xs,k=5):
    out=[];buf=[]
    for x in xs: buf.append(x); buf=buf[-k:]; out.append(sum(buf)/len(buf))
    return out
def steps_to_target(ts,tl,tgt):
    for s,l in zip(ts,smooth(tl)):
        if l<=tgt: return s
    return ts[-1]
def plateau_ok(ts,tl):
    # ---------------------------------------------------------------------
    # (R3 FIX) The previous test compared the MEAN of the last PLATEAU_FRAC of
    # recorded points to the FINAL point, both taken from the SINGLE-BATCH
    # TRAINING loss. Two independent defects made it uninformative:
    #
    #  (1) WRONG SIGNAL. Single-batch train loss has per-step sampling noise of
    #      order 0.05-0.15 nats, while the tolerance band is PLATEAU_TOL*|final|
    #      ~ 0.04 nats. The verdict was therefore dominated by batch noise, not by
    #      convergence: simulating PERFECTLY FLAT traces (zero trend by
    #      construction) passes only 63-100% of the time depending purely on the
    #      noise sigma. A gate whose output is a function of measurement noise
    #      cannot certify convergence -- and H2 rests entirely on this gate.
    #  (2) WRONG STATISTIC. |mean(tail) - final| is a LEVEL comparison, which a
    #      cosine-decayed LR makes trivially small near the end regardless of
    #      whether the model has converged: as lr -> min_lr the loss flattens
    #      MECHANICALLY. The quantity of interest is whether the loss is still
    #      DESCENDING, i.e. the SLOPE of the tail, not its flatness in level.
    #
    # Replacement: a regression-slope test on the low-variance HELD-OUT VAL trace
    # (`vl`, recorded every eval_every steps by train_run). We fit loss ~ a + b*step
    # over the last PLATEAU_FRAC of the val trace and require the 95% CI of the
    # slope b to be compatible with zero *and* the implied residual descent over
    # the tail to be below PLATEAU_TOL*|final|. This tests "no longer improving"
    # directly, is robust to noise (val loss is averaged over eval batches, and the
    # CI widens rather than the verdict flipping), and is not satisfied merely by
    # the LR schedule flattening the curve. Returns (ok, signed_tail_descent_nats).
    # Require enough tail points for the slope CI to have any power. With
    # eval_every_frac=0.08 a run yields ~12 val probes, so a 40% tail gives ~5
    # points (dof=3). Fewer than PLATEAU_MIN_TAIL points -> we cannot certify
    # convergence and must return False rather than pass by lack of evidence.
    if not tl or len(tl)<PLATEAU_MIN_TAIL+1: return False,float("nan")
    x=np.asarray(ts,float); y=np.asarray(tl,float)
    k=max(PLATEAU_MIN_TAIL,int(math.ceil(len(y)*PLATEAU_FRAC))); x=x[-k:]; y=y[-k:]
    if len(y)<PLATEAU_MIN_TAIL or np.ptp(x)<=0: return False,float("nan")
    n=len(y); xm=x.mean(); sxx=float(((x-xm)**2).sum())
    b=float(((x-xm)*(y-y.mean())).sum()/max(sxx,1e-12))      # slope (nats/step)
    a=float(y.mean()-b*xm)
    resid=y-(a+b*x); dof=max(n-2,1)
    se_b=math.sqrt(float((resid**2).sum())/dof/max(sxx,1e-12))
    ci=1.96*se_b
    descent=float(b*(x[-1]-x[0]))          # signed loss change across the tail
    final=float(y[-1]); band=PLATEAU_TOL*max(abs(final),1e-9)
    # Converged iff ALL THREE hold:
    #  (i)  no resolvable trend (slope CI covers 0) OR fitted descent negligible;
    #  (ii) not still DESCENDING by more than the tolerance band (under-training);
    #  (iii) not ASCENDING by more than the band either. (iii) is not pedantry:
    #       a diverging or destabilising run also has a "flat-ish" tail in level
    #       terms and would otherwise be certified as converged and admitted into
    #       H2, which would silently average instability into a quality claim.
    slope_flat=(abs(b)-ci)<=0 or abs(descent)<=band
    not_falling=descent>=-band
    not_rising =descent<= band
    return bool(slope_flat and not_falling and not_rising), descent
@torch.no_grad()
def downstream_eval(model,n_batches=12):
    """Internal transfer proxy: next-token top-1 on held-out val.
    (The prior synthetic copy-task proxy is intentionally removed; it tracked
    active-weight density rather than method quality and was not discriminating.)"""
    model.eval(); g=torch.Generator().manual_seed(GLOBAL_SEED+321)
    correct=tot=0
    for _ in range(n_batches):
        xb,yb=get_batch(VAL,BS,SEQ,g)
        with torch.autocast("cuda",dtype=DTYPE): lg,_=model(xb)
        correct+=(lg.argmax(-1)==yb).sum().item(); tot+=yb.numel()
    model.train(); return float(correct/max(tot,1))

def _train_run_core(size,cfg,method,seed,opts,_preflight_steps=None,lr=None,wall_budget=None):
    # (#2) WALL-CLOCK-EQUAL H1: _t0_run is started BEFORE model construction and
    #      init, so the TAI init-calibration (8-step search) and any gamma-warmup
    #      bookkeeping are charged to TAI's wall-clock -- no hidden free lunch.
    # (#1) PER-METHOD LR: `lr` overrides the peak LR (chosen per (scale,method)
    #      on disjoint calib seeds). Falls back to RUN_CFG peak_lr if None.
    # (#4) deterministic TAI init: a per-seed CPU generator is passed to apply_init
    #      so the TAI grid uses the shared-base-noise path (matches the sweep).
    # wall_budget (sec): if set, training stops once wall-clock exceeds it; the
    #      step it reached is recorded (used for the compute-equal H1 variant).
    import time as _t_run; _t0_run=_t_run.time()
    seed_everything(seed); gen=torch.Generator().manual_seed(seed)
    init_gen=torch.Generator().manual_seed(seed+777)   # CPU gen for TAI init draws
    peak_lr=float(lr) if lr is not None else PEAK_LR
    # (R14) Global quant state is now established by the train_run wrapper inside a
    # try/finally, so it is restored on every exit path (return, exception,
    # SystemExit at the session boundary, KeyboardInterrupt). The core only reads it.
    model=LM(cfg).to(DEVICE,DTYPE)
    # ---------------------------------------------------------------------
    # (R17 FIX) INIT ROUTING. The old chain tested `method.startswith("tai")`
    # FIRST, so "tai_negctrl" -- which also starts with "tai" -- was routed to the
    # REAL TAI init. The placebo arm therefore received the genuine, rho0-honouring
    # initializer, and `apply_init`'s own tai_negctrl branch was unreachable dead
    # code (verified: init_tai_shuffled returns a flat 0.6889 active ratio for
    # every label, but train_run never called it).
    #
    # Why this was the worst possible place for a routing bug. tai_negctrl is H3's
    # PRE-REGISTERED FALSIFICATION arm: the whole point is a knob that should NOT
    # move final sparsity, with the frozen prediction "Expected: FLAT (Spearman
    # ~0)". Wired to the real init, the placebo tracks rho0 just as strongly as the
    # true arm (Spearman ~ +1), and Cell 15 reports that under
    # `h3_strong_falsification`. The paper's most rigorous-looking instrument would
    # have concluded that H3 is an artefact -- killing the primary contribution
    # even if the real effect were perfectly sound. A broken placebo does not
    # weaken a claim; it inverts it.
    #
    # Routing is now EXPLICIT and order-independent, so no future method name that
    # happens to share a prefix can be silently swallowed by the wrong branch.
    if method=="tai_negctrl":         init_name="tai_negctrl"   # placebo: label-decoupled
    elif method.startswith("tai"):    init_name="tai"           # tai, tai_warmup, tai_star
    elif QUANT_MODE["full_precision"]: init_name="standard"
    elif method in ("bitnet158","sparse_sched","gamma_offset"):
        # (R6) gamma_offset deliberately keeps the STANDARD init: its whole point is
        # to reach TAI's operating point WITHOUT TAI's initializer.
        init_name="standard"
    else:                             init_name=method
    apply_init(model,init_name,rho0=opts.get("rho0",0.75),
               gen=(init_gen if init_name in ("tai","tai_negctrl") else None))
    # Prove the placebo really is label-decoupled, on every single negctrl run: its
    # realised rho0 must sit at the locked reference (0.69) regardless of the label
    # it was handed. If this ever fires, the falsification arm is live again.
    if method=="tai_negctrl" and TAI_HP.get("last_realised_rho0") is not None:
        _r=float(TAI_HP["last_realised_rho0"])
        assert abs(_r-0.69)<0.03, (
            f"tai_negctrl realised rho0={_r:.4f} for label {opts.get('rho0')}: the placebo is "
            "tracking its label, so it is no longer a negative control (R17 regression).")
    # (R13) Complete any data-driven initializer (currently: real LSUV). apply_init
    # only sees individual tensors, so LSUV's layer-sequential unit-variance pass
    # -- the half that makes it LSUV rather than orthogonal-init -- must run here,
    # with a real batch and the assembled model. The calibration batch is drawn
    # from a generator SEPARATE from the training-order generator (seed+555), so
    # calibration consumes no training randomness and the data order seen by every
    # method stays identical: LSUV must not get a different batch sequence than its
    # baselines as a side effect of being calibrated.
    if init_name in INIT_NEEDS_DATA:
        _cal_gen=torch.Generator().manual_seed(seed+555)
        _cal_x,_=get_batch(TRAIN,BS,SEQ,_cal_gen)
        finish_init(model,init_name,_cal_x)
        del _cal_x
    gw=opts.get("gamma_warmup",0)
    # (R6) gamma_offset: resolve the constant threshold scale c that puts a
    # STANDARD init at the requested rho0. Cached per rho0; held fixed for the
    # whole run (that permanence is the mechanism under test, see Cell 8).
    _gamma_off_c=None
    if opts.get("gamma_offset") and not opts.get("full_precision",False):
        _r0=float(opts.get("rho0",0.75))
        if _r0 not in GAMMA_OFFSET_C: GAMMA_OFFSET_C[_r0]=gamma_scale_for_rho0(_r0)
        _gamma_off_c=float(GAMMA_OFFSET_C[_r0])
    # (R23) fused AdamW: one kernel for the whole param list instead of one per
    # tensor. These models have many small parameters, so the optimizer step was
    # launch-bound. Mathematically identical; falls back if unsupported.
    def _mk_adamw(params,lr):
        try: return torch.optim.AdamW(params,lr=lr,weight_decay=WD,betas=(0.9,0.95),fused=True)
        except Exception: return torch.optim.AdamW(params,lr=lr,weight_decay=WD,betas=(0.9,0.95))
    opt=_mk_adamw(model.parameters(),peak_lr)
    # (R23) COMPILE THE TRAINING STEP ONLY, never the module we introspect.
    # model stays the raw nn.Module, so model.bitlinears(), .sparsity() and every
    # eval path behave exactly as before and cannot be perturbed by dynamo. Only
    # the hot loop goes through the compiled callable, and any failure -- unsupported
    # op, inductor error, missing triton -- silently falls back to eager rather
    # than taking down a multi-session grid.
    _step_fn=model
    # ========================================================================
    # (R34) COMPILE ONLY WHEN THE RUN IS LONG ENOUGH TO AMORTISE IT.
    # ------------------------------------------------------------------------
    # Session 1 measured the real cost: at dim1280 a 300-step calibration run
    # took 285 s, of which ~71 s was training and ~190 s was model construction
    # plus compilation. Decisive evidence that the compile cost is paid PER RUN
    # rather than once: within a (scale, method) cell the nine runs took 285,
    # 285, 285... to the second, with no warm-up effect at all. Every run builds
    # a fresh LM, so dynamo's guards miss and inductor re-lowers the graph; the
    # on-disk cache does not rescue a fresh module instance.
    #
    # Net effect on a SHORT run: compile is a large loss (190 s overhead against
    # 71 s of training). On a FULL 1221-step run at the same scale it is roughly
    # break-even to modestly positive (~290 s of training against the same 190 s).
    # The controlled benchmark's 2.0-2.5x was real but measured the steady-state
    # step, which is exactly the part that a 300-step run barely reaches.
    #
    # So: compile only when steps exceed compile_min_steps. Nothing about the
    # numerics changes -- the earlier noise-floor test showed eager and compiled
    # differ by less than run-to-run non-determinism -- so this is purely a
    # throughput decision, and it is now made per run instead of globally.
    # NOTE: _STEPS is assigned further down in this function, so the run length
    # is recomputed here from the same expression rather than referenced early --
    # referencing _STEPS at this point would raise UnboundLocalError on the very
    # first run of every session.
    _cmin=int(RUN_CFG.get("compile_min_steps",600))
    _run_steps=_preflight_steps if _preflight_steps else STEPS
    _want_compile=bool((globals().get("FAST_CFG") or {}).get("compile")) and _run_steps>=_cmin
    if _want_compile:
        try:
            _step_fn=torch.compile(model,dynamic=False)
        except Exception as _e:
            print("compile unavailable, using eager:",_e); _step_fn=model
    ts,tl=[],[]; model.train()
    # (R3) VAL TRACE: the plateau gate must run on a low-variance held-out signal,
    # not on single-batch train loss. We evaluate on val fold 0 every `eval_every`
    # steps and keep (vs, vl). Cost is ~2% of run time (20 eval batches per probe,
    # ~12 probes per run) and it is what makes H2's convergence claim checkable.
    vs,vl=[],[]
    _STEPS=_preflight_steps if _preflight_steps else STEPS
    _EVAL_EVERY=max(1,int(_STEPS*float(RUN_CFG.get("eval_every_frac",0.08))))
    def _lr_at(t):  # local scheduler honouring this run's peak_lr (#1)
        if t<WARMUP: return peak_lr*(t+1)/WARMUP
        p=(t-WARMUP)/max(1,_STEPS-WARMUP); return MIN_LR+0.5*(peak_lr-MIN_LR)*(1+math.cos(math.pi*p))
    _stopped_step=_STEPS
    for t in range(_STEPS):
        if wall_budget is not None and (_t_run.time()-_t0_run)>wall_budget:
            _stopped_step=t; break   # (#2) compute-equal variant: stop at budget
        for g in opt.param_groups: g["lr"]=_lr_at(t)
        if opts.get("recipe")=="sched" and not QUANT_MODE["full_precision"]:
            set_gamma(gamma_sched_value(t,_STEPS,opts.get("sched_target",1.6)))
        elif _gamma_off_c is not None and not QUANT_MODE["full_precision"]:
            set_gamma(_gamma_off_c)              # (R6) constant threshold offset
        elif gw and not QUANT_MODE["full_precision"]:
            set_gamma(gamma_warmup_value(t,gw))
        else:
            set_gamma(1.0)
        xb,yb=get_batch(TRAIN,BS,SEQ,gen)
        with torch.autocast("cuda",dtype=DTYPE): _,loss=_step_fn(xb,yb)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),CLIP); opt.step()
        if t%20==0 or t==_STEPS-1: ts.append(t); tl.append(loss.item())
        if (t%_EVAL_EVERY==0 and t>0) or t==_STEPS-1:
            vs.append(t); vl.append(evaluate(model,n=RUN_CFG.get("eval_batches_trace",12)))
    # (R16) MEASURE BEFORE RESETTING. sparsity() now mirrors the live forward path,
    # so it must be read while this run's own quant state is still in effect --
    # resetting GAMMA_SCALE to 1.0 first would measure gamma_offset (c=0.7987) and
    # any gamma-warmup/sched arm at the WRONG threshold, which is precisely the bug
    # R16 fixes. Order: read sparsity under the run's state, then evaluate, then let
    # the wrapper's finally restore the globals.
    sp=float(np.mean([bl.sparsity() for bl in model.bitlinears()]))
    # For the loss readout we deliberately use the run's own quantization state too:
    # val_loss must describe the model AS TRAINED, not a differently-thresholded
    # variant of it. (fp16_control keeps full_precision -> its val_loss is the fp16
    # upper bound, and its sparsity is 0.0 by definition.)
    val=evaluate(model)
    # (R5 FIX) DATA-SAMPLING VARIANCE FOR H4, THE CHEAP WAY. Cell 13f previously
    # reported that H4's variance claim could not include data-sampling spread
    # "because we did not persist weights". Persisting weights was never the only
    # option: the model is STILL IN MEMORY right here, so we simply score it on
    # every held-out fold now and store K numbers per run (a few floats in JSON,
    # zero extra Drive cost, ~1% extra compute). H4's Levene test can then run on
    # the pooled (seed x fold) losses, so "TAI does not inflate variance" covers
    # BOTH init/order variance AND data-sampling variance, as the hypothesis says.
    val_folds=[]
    if "VAL_FOLDS" in globals():
        # VAL_FOLDS[k] is a (train_k, val_k) pair built in Cell 9; we score val_k.
        for _k,(_tr_k,_va_k) in enumerate(VAL_FOLDS):
            val_folds.append(evaluate(model,n=RUN_CFG.get("eval_batches_trace",12),
                                      split=_va_k,seed_off=99+_k))
    nt_acc=downstream_eval(model)
    pk,gap=plateau_ok(vs,vl)     # (R3) gate now runs on the held-out val trace
    # ========================================================================
    # (R26) DIVERGENCE DETECTION — the gap the plateau gate cannot cover.
    # ------------------------------------------------------------------------
    # plateau_ok() asks whether the val trace has stopped DESCENDING. A run that
    # blew up has also stopped descending, so it passes the same gate -- the
    # comment above the plateau helper already noted this and no guard was ever
    # added. val_ppl compounds the problem: math.exp(min(val,20)) caps a diverged
    # run at a finite, innocuous-looking perplexity, so nothing downstream looks
    # wrong either.
    #
    # Why this matters more here than in an ordinary training script: every claim
    # is a SEED-PAIRED comparison between methods. If one arm blows up on 3 of 15
    # seeds and another does not, the paired test measures optimisation stability,
    # not the rho0 mechanism under test -- and it will report a large, significant,
    # entirely spurious effect. A single silent divergence can therefore manufacture
    # a positive result for H1/H2/H5.
    #
    # The bound is principled, not a magic number: a model that has learned nothing
    # at all still predicts the uniform distribution, giving log(vocab) ~= 10.83.
    # Anything materially above that is not "trained badly", it is diverging --
    # confidently wrong predictions. We flag rather than raise, because aborting a
    # session on one bad seed is worse than recording it and excluding it: the
    # record keeps have_run() from re-running it forever, and the manifest reports
    # how many were dropped. A high divergence count is itself a finding -- it
    # would mean lr_grid=[1e-3,2e-3,3e-3] is too hot for the larger scales.
    _uniform=math.log(max(VOCAB,2))
    _diverged=bool((not math.isfinite(val)) or val>_uniform*float(RUN_CFG.get("divergence_factor",1.25)))
    if (not _diverged) and len(vl)>=4:
        _tail=vl[-max(2,len(vl)//3):]
        _diverged=bool(_tail[-1]>_tail[0]*1.5 and _tail[-1]>_uniform)   # blowing up late
    if _diverged:
        print(f"    !! DIVERGED: {method}/{size}/s{seed} val_loss={val:.3f} "
              f"(uniform baseline {_uniform:.2f}) — recorded but EXCLUDED from analysis.")
    # (R22) lm-eval is scored on the FIRST lmeval_max_seeds seeds of every
    # (scale,method) cell rather than all 15. The same seeds are used for every
    # method, so the downstream comparison stays PAIRED; the statistics cell
    # already NaN-masks missing entries. Set lmeval_max_seeds to a large number
    # to restore full coverage.
    _lm_k=int(RUN_CFG.get("lmeval_max_seeds",10**6))
    _lm_seeds=set(RUN_CFG["seeds"][:_lm_k])|set(RUN_CFG.get("calib_seeds",[])[:1])
    _do_lm=LMEVAL_AVAILABLE and (seed in _lm_seeds) and (_preflight_steps is None)
    lm=lmeval_accuracy(model) if _do_lm else {}
    del model,opt; torch.cuda.empty_cache()
    rec={"size":size,"method":method,"seed":seed,"steps":ts,"loss":tl,
         "val_steps":vs,"val_trace":vl,          # (R3) held-out convergence trace
         "val_loss":val,"val_ppl":math.exp(min(val,20)),"final_sparsity":sp,
         "diverged":_diverged,                  # (R26) excluded downstream if True
         "uniform_baseline":_uniform,
         "val_loss_folds":val_folds,          # (R5) per-fold held-out losses

         "nexttoken_acc":nt_acc,"plateau_ok":pk,"plateau_tail_descent":gap,
         "plateau_gap":gap,                      # legacy alias (Cell 13a2/15 read this)
         "recipe":opts.get("recipe","absmean"),
         "gamma_offset_c":_gamma_off_c}        # (R6) None unless gamma_offset arm
    rec["arc_easy_acc"]=lm.get("arc_easy",float("nan"))
    rec["arc_easy_acc_norm"]=lm.get("arc_easy_norm",float("nan"))
    rec["hellaswag_acc"]=lm.get("hellaswag",float("nan"))
    rec["hellaswag_acc_norm"]=lm.get("hellaswag_norm",float("nan"))
    rec["wall_sec"]=round(_t_run.time()-_t0_run,2)   # per-run wall-clock (incl init)
    rec["peak_lr"]=peak_lr                            # (#1) LR* actually used
    rec["stopped_step"]=int(_stopped_step)            # (#2) <_STEPS iff wall-capped
    rec["wall_capped"]=bool(_stopped_step<_STEPS)
    return rec


# ============================================================================
# (R14) GLOBAL QUANT-STATE HYGIENE — prevents silent cross-run corruption.
# ----------------------------------------------------------------------------
# QUANT_MODE / QUANT_RECIPE / GAMMA_SCALE are process-global dicts read inside
# BitLinear.forward. The core used to SET them at the top and RESTORE them ~64
# lines later, past the whole training loop, with NO try/finally in between. Any
# early exit from that span leaks the state into the NEXT run.
#
# This is not hypothetical. `raise SystemExit(0)` at the session-budget boundary
# is the DESIGNED control flow here, and OOM is the expected failure at the top
# scales. Traced end-to-end:
#   fp16_control hits the budget boundary -> full_precision=True leaks -> next
#   session's `tai` / `bitnet158` runs execute in FULL PRECISION. No quantization
#   happens at all. Losses look plausible, nothing raises, and the record is saved
#   as "main__dim384__tai__s1337" -> have_run() reports it FINISHED -> resume never
#   re-runs it. The grid then compares TAI-in-fp16 to baselines-in-fp16 and reports
#   it as a ternary result. The only visible trace would be final_sparsity==0.
# Same class of leak for QUANT_RECIPE: a crashed bitnet158 run leaves
# recipe="bitnet", so the next `tai` run silently uses BitNet's quantizer and
# TAI's own absmean recipe is never tested.
#
# Two-part fix: (i) a context manager that restores on EVERY exit path; (ii) an
# entry assertion, so a dirty state fails LOUDLY on the first affected run instead
# of quietly yielding a full grid of wrong numbers. A third guard (post-hoc
# sparsity sanity) catches the same corruption from the data side.
from contextlib import contextmanager
_QUANT_STATE_DEFAULTS={"full_precision":False,"recipe":"absmean","gamma":1.0}

def assert_quant_state_clean(where=""):
    bad=[]
    if bool(QUANT_MODE.get("full_precision")) is not _QUANT_STATE_DEFAULTS["full_precision"]:
        bad.append(f'QUANT_MODE["full_precision"]={QUANT_MODE.get("full_precision")}')
    if QUANT_RECIPE.get("name")!=_QUANT_STATE_DEFAULTS["recipe"]:
        bad.append(f'QUANT_RECIPE["name"]={QUANT_RECIPE.get("name")!r}')
    if abs(float(GAMMA_SCALE.get("val",1.0))-_QUANT_STATE_DEFAULTS["gamma"])>1e-9:
        bad.append(f'GAMMA_SCALE["val"]={GAMMA_SCALE.get("val")}')
    # (R23) the device mirror must agree with the float at every run boundary. If
    # a future edit adds a gamma write that bypasses set_gamma(), the forward pass
    # would quietly use a stale threshold while every log and assertion reported
    # the intended one -- the exact shape of R16. This catches it at the next run.
    if "gamma_mirror_ok" in globals() and not gamma_mirror_ok():
        bad.append("GAMMA_SCALE float/tensor mirror out of sync (a write bypassed set_gamma)")
    if bad:
        raise RuntimeError(
            "Quantization global state was left dirty by a previous run"
            + (f" (at {where})" if where else "") + ": " + "; ".join(bad)
            + ". Refusing to start a new run: continuing would train under the wrong "
              "precision/recipe and save it as a valid result. Restart the runtime; "
              "state is in-process only, completed runs on Drive are unaffected.")

@contextmanager
def quant_state(full_precision=False,recipe="absmean"):
    assert_quant_state_clean("run start")     # never inherit a dirty predecessor state
    QUANT_MODE["full_precision"]=bool(full_precision)
    QUANT_RECIPE["name"]=str(recipe)
    set_gamma(1.0)
    try:
        yield
    finally:
        set_gamma(_QUANT_STATE_DEFAULTS["gamma"])
        QUANT_MODE["full_precision"]=_QUANT_STATE_DEFAULTS["full_precision"]
        QUANT_RECIPE["name"]=_QUANT_STATE_DEFAULTS["recipe"]

def train_run(size,cfg,method,seed,opts,_preflight_steps=None,lr=None,wall_budget=None):
    # (R14) Thin wrapper: owns the global quant state for the whole run and hands a
    # clean, correct state to the core. Every call site keeps its signature.
    with quant_state(full_precision=opts.get("full_precision",False),
                     recipe=opts.get("recipe","absmean")):
        rec=_train_run_core(size,cfg,method,seed,opts,
                            _preflight_steps=_preflight_steps,lr=lr,wall_budget=wall_budget)
    # ---- post-hoc corruption guard (data-side, independent of the state machine) --
    # A ternary run MUST produce non-zero weight sparsity: absmean ternarisation
    # zeroes every weight with |w| < gamma/2, which for any realistic init is a
    # large fraction. Exactly 0.0 means no quantization took place -- i.e. the run
    # silently executed in full precision. Catch it here rather than discovering it
    # in the stats cell (or not at all).
    if not opts.get("full_precision",False):
        _sp=rec.get("final_sparsity")
        if _sp is not None and float(_sp)<=1e-9:
            raise RuntimeError(
                f"Run {method}/{size}/s{seed} reports final_sparsity={_sp}, but it is a "
                "TERNARY arm: zero sparsity means quantization never ran (full-precision "
                "leak or a broken quantizer). Not saving this record; investigate before "
                "resuming — a saved record would be treated as finished by have_run().")
    return rec


# ---- RUN-ALL MASTER GATE ----------------------------------------------------
# If not armed, skip ALL training (preflight included). This makes a fresh
# "Run All" safe: setup/definition cells execute, analysis cells no-op via
# _require(), and nothing consumes GPU-hours until you explicitly arm the run.
if not RUN_CFG.get("ARM_FULL_RUN",False):
    print("="*70)
    print("RUN-ALL SAFE MODE: ARM_FULL_RUN=False -> no training executed.")
    print("Set RUN_CFG['ARM_FULL_RUN']=True (and preflight_only as desired),")
    print("then re-run this cell / Run All. Resume is automatic and idempotent.")
    print("="*70)
    RESULTS=[]; TARGET_LOSS_BY_SCALE={}; WALL_RESULTS=[]; DIVERGED_RUNS=[]
# ============================================================================
# (R35) PLATEAU REALITY CHECK — report it, do not let it pass silently.
# ----------------------------------------------------------------------------
# The pre-registration calls 20M tokens a "plateau-verified budget", and H2 is
# phrased in terms of converged quality. The first 502 main-grid runs came back
# with plateau_ok on ZERO of them: at dim384 the val trace still falls 0.21 nats
# across the final 40% of probes, and the picture is the same at every scale
# (-0.20 to -0.27). The gate is not broken -- it is correctly reporting that the
# models are still learning when the budget runs out.
#
# That does not invalidate anything: a token-equal comparison at a fixed budget
# is a legitimate and common protocol, and every arm is treated identically. What
# it invalidates is the WORDING. Claims must read "at a fixed 20M-token budget",
# never "at convergence", and H2's TOST -- which is defined over plateau-verified
# pairs -- will legitimately be skipped rather than silently computed over an
# empty set. The statistics cell already handles n<2 by skipping; this counter
# makes the situation visible in the log and in the manifest instead of leaving
# it to be discovered by reading 500 JSON files.
# (R39 FIX) DIVERGED_RUNS was declared ONLY inside the ARM_FULL_RUN=False branch
# above, i.e. only on the path where nothing trains. On a real run it never came
# into existence, so R26's collector and R38's invariant assert both referenced an
# undefined name and the cell died with NameError after the entire grid had been
# reloaded. Declared here at module level, unconditionally, alongside the other
# accumulators it belongs with. Guarded so that re-running the cell within a live
# session does not wipe a list the earlier pass already filled.
for _nm in ("RESULTS","WALL_RESULTS","DIVERGED_RUNS"):
    if _nm not in globals(): globals()[_nm]=[]
if "TARGET_LOSS_BY_SCALE" not in globals(): TARGET_LOSS_BY_SCALE={}

PLATEAU_STATS={"n":0,"n_ok":0,"gaps":[]}
def _note_plateau(rec):
    PLATEAU_STATS["n"]+=1
    if rec.get("plateau_ok"): PLATEAU_STATS["n_ok"]+=1
    g=rec.get("plateau_gap")
    if g is not None and g==g: PLATEAU_STATS["gaps"].append(float(g))
def plateau_summary():
    n,k=PLATEAU_STATS["n"],PLATEAU_STATS["n_ok"]
    gs=PLATEAU_STATS["gaps"]
    out={"n_runs":n,"n_plateau_ok":k,"frac":(k/n if n else None),
         "median_tail_gap":(float(np.median(gs)) if gs else None)}
    if n and k==0:
        print(f"\n  !! PLATEAU: 0/{n} runs reached plateau (median tail gap "
              f"{out['median_tail_gap']:.3f} nats). The 20M-token budget does NOT "
              f"reach convergence.\n     -> H2 must be read as TOKEN-EQUAL at a fixed "
              f"budget, not as converged quality. Recorded in the manifest.")
    return out
    raise SystemExit("Run-All safe mode: experiment not armed.")

# (R23) shared fused-AdamW factory for the ablation cells (22/24/25).
def _mk_adamw_g(params,lr):
    try: return torch.optim.AdamW(params,lr=lr,weight_decay=WD,betas=(0.9,0.95),fused=True)
    except Exception: return torch.optim.AdamW(params,lr=lr,weight_decay=WD,betas=(0.9,0.95))

# ---- PREFLIGHT: cheap smoke test + budget projection (runs before the grid) ----
PREFLIGHT_ONLY=RUN_CFG.get("preflight_only",True)
def _preflight():
    import time as _t
    rep={"tokenizer":"gpt2-bpe","vocab":int(VOCAB),
         "corpus_source":DATA_META_INFO["source"],"used_synthetic":bool(USED_SYNTHETIC),
         "lmeval_available":bool(LMEVAL_AVAILABLE),
         "plateau_token_budget":TOKEN_BUDGET,"steps_per_run":STEPS}
    assert VOCAB>1000, "vocab looks wrong (BPE expected ~50k)"
    assert len(TRAIN)>SEQ*BS*5, "train split too small for one batch window"
    if LMEVAL_AVAILABLE: rep["lmeval_tasks"]={k:len(v) for k,v in LMEVAL_TASKS.items()}
    else: print("PREFLIGHT WARN: lm-eval disabled (synthetic corpus or tasks "
                "unavailable); downstream columns will be NaN.")
    smallest=min(SIZES,key=lambda k:SIZES[k].dim)
    t0=_t.time()
    r=train_run(smallest,SIZES[smallest],"tai",seed=RUN_CFG["seeds"][0],
                opts={"rho0":0.75},_preflight_steps=30)
    dt=_t.time()-t0; sps=dt/30.0
    # ========================================================================
    # (R28) THE PROJECTION WAS MEASURING THE WRONG THING BY ~25x.
    # ------------------------------------------------------------------------
    # sps above is (total wall time of a 30-step run)/30, an interval that also
    # holds model construction, torch.compile warm-up (paid once) and -- worst --
    # the evaluation machinery. eval_every is a FRACTION of run length, so a
    # 30-step run probes every 2 steps: ~260 evaluation forwards bolted onto 30
    # training steps, all of which amortise away in a real 1221-step run.
    # Measured 1.2569 s/step where the true steady-state figure is 0.0648.
    #
    # (R29) AND FLOP-PROPORTIONAL SCALING IS ALSO WRONG.
    # ------------------------------------------------------------------------
    # Even with a correct sec/step at ONE scale, pricing the others in proportion
    # to FLOPs assumes constant throughput. It is not constant: measured MFU runs
    # from ~4% at dim192 to ~18% at dim1536, because the small models are
    # overhead-bound (fixed per-step launch, optimizer and loss costs that do not
    # shrink with dim). Actual dim1536/dim192 cost ratio is ~9x; the FLOP ratio
    # is ~39x. Scaling from the SMALLEST scale therefore over-prices the whole
    # ladder by roughly 2-3x -- 202 h projected against ~85 h measured.
    # Fix: time several scales and fit sec/step = a + b*FLOPs_per_token. The
    # intercept `a` IS the fixed overhead, so the affine form captures both
    # regimes instead of pretending only one exists.
    def _steady(_name,_warm=10,_timed=20):
        """sec/step in steady state: post-warm-up, no evaluation in the window."""
        _m=LM(SIZES[_name]).to(DEVICE,DTYPE)
        _o=torch.optim.AdamW(_m.parameters(),lr=1e-4,weight_decay=WD,betas=(0.9,0.95))
        _f=torch.compile(_m,dynamic=False) if (globals().get("FAST_CFG") or {}).get("compile") else _m
        _g=torch.Generator().manual_seed(0); _last=None
        with quant_state(False,"absmean"):
            for _i in range(_warm):
                _xb,_yb=get_batch(TRAIN,BS,SEQ,_g)
                with torch.autocast("cuda",dtype=DTYPE): _,_l=_f(_xb,_yb)
                _o.zero_grad(set_to_none=True); _l.backward(); _o.step()
            torch.cuda.synchronize(); _t2=_t.time()
            for _i in range(_timed):
                _xb,_yb=get_batch(TRAIN,BS,SEQ,_g)
                with torch.autocast("cuda",dtype=DTYPE): _,_l=_f(_xb,_yb)
                _o.zero_grad(set_to_none=True); _l.backward(); _o.step()
                _last=_l
            torch.cuda.synchronize(); _sps=(_t.time()-_t2)/_timed
        _pk=torch.cuda.max_memory_allocated()/1e9
        del _m,_o; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        return _sps,float(_last.item()),_pk
    _fpt=lambda c: 6.0*(n_params(c)+c.dim*c.vocab_size)+6.0*c.n_layers*SEQ*c.dim
    _probe=[x for x in (smallest,"dim512","dim1024") if x in SIZES]
    _obs={}
    for _nm in _probe:
        try:
            _sps,_ls,_pk=_steady(_nm); _obs[_nm]=_sps
            print(f"  steady-state {_nm:8s} {_sps*1000:7.1f} ms/step | peak {_pk:5.1f} GB | loss {_ls:.2f}")
            if _nm==smallest: sps=_sps; rep["init_loss_check"]=round(_ls,3)
        except Exception as _e:
            print(f"  steady-state {_nm}: failed ({str(_e)[:80]})")
    rep["sec_per_step_raw_smoke"]=round(dt/30.0,4)
    rep["steady_state_sec_per_step"]={k:round(v,4) for k,v in _obs.items()}
    _a,_b=None,None
    if len(_obs)>=2:
        _xs=[_fpt(SIZES[k]) for k in _obs]; _ys=[_obs[k] for k in _obs]
        _n=len(_xs); _mx=sum(_xs)/_n; _my=sum(_ys)/_n
        _b=sum((x-_mx)*(y-_my) for x,y in zip(_xs,_ys))/max(sum((x-_mx)**2 for x in _xs),1e-30)
        _a=_my-_b*_mx
        rep["cost_model"]={"fixed_overhead_sec":round(_a,4),"sec_per_flop_per_token":_b,
                           "probed_scales":list(_obs)}
        print(f"  cost model: sec/step = {_a:.4f} + {_b:.3e} * FLOPs/token"
              f"   (fixed overhead is {100*_a/max(_ys[0],1e-9):.0f}% of the smallest scale)")
    globals()["_COST_A"],globals()["_COST_B"],globals()["_COST_FPT"]=_a,_b,_fpt
    # ---------------------------------------------------------------------
    # (R23) MEASURE, DON'T GUESS. Everything above this line is a cost MODEL;
    # the numbers below are the only real evidence about where time goes. Two
    # readings, both cheap:
    #   (1) a kernel profile of a handful of steps -- tells you whether the loop
    #       is matmul-bound (little left to win), launch-bound (compile helps a
    #       lot) or dominated by the loss/eval path;
    #   (2) an eager-vs-compiled A/B at the smallest scale, reporting BOTH the
    #       speedup and whether the loss still matches. Compile stays off unless
    #       this says it is both faster and numerically equivalent.
    try:
        from torch.profiler import profile as _tprof, ProfilerActivity as _PA
        with _tprof(activities=[_PA.CPU,_PA.CUDA]) as _pr:
            train_run(smallest,SIZES[smallest],"tai",seed=RUN_CFG["seeds"][0],
                      opts={"rho0":0.75},_preflight_steps=8)
        _tbl=_pr.key_averages().table(sort_by="self_cuda_time_total",row_limit=12)
        print("\n--- kernel profile (smallest scale, 8 steps) ---\n"+_tbl)
        rep["profile_top"]=_tbl.splitlines()[:16]
    except Exception as _e:
        print("profiler unavailable:",_e)
    if RUN_CFG.get("preflight_bench_compile",True) and "FAST_CFG" in globals():
        _was=FAST_CFG.get("compile",False); _ab={}
        for _mode in (False,True):
            FAST_CFG["compile"]=_mode
            try:
                # (R29) steady-state, no eval, post-warm-up -- the previous version
                # timed a full 40-step train_run whose ~260 evaluation forwards and
                # one-off compile warm-up swamped the signal, reporting 1.06x where
                # a controlled benchmark measured 2.03x.
                _sps2,_ls2,_ = _steady(smallest,_warm=12,_timed=25)
                _ab["compiled" if _mode else "eager"]={
                    "sec_per_step":round(_sps2,4),"val_loss":round(_ls2,5)}
            except Exception as _e:
                _ab["compiled" if _mode else "eager"]={"error":str(_e)[:200]}
        FAST_CFG["compile"]=_was
        rep["compile_ab"]=_ab
        _e0,_e1=_ab.get("eager",{}),_ab.get("compiled",{})
        if "sec_per_step" in _e0 and "sec_per_step" in _e1:
            _spd=_e0["sec_per_step"]/max(_e1["sec_per_step"],1e-9)
            _dl=abs(_e0["val_loss"]-_e1["val_loss"])
            rep["compile_speedup"]=round(_spd,2); rep["compile_loss_delta"]=round(_dl,5)
            print(f"\ncompile A/B: {_spd:.2f}x speedup | val_loss delta {_dl:.5f}")
            print("  -> RECOMMEND FAST_CFG['compile']=True" if (_spd>1.15 and _dl<0.02)
                  else "  -> keep compile OFF (insufficient gain or loss mismatch)")
    # (R19) project per-scale cost ~ proportional to params, over the scales that
    # will ACTUALLY RUN (confirmatory + exploratory), NOT over every declared Cfg.
    # Without this, the opt-in 783M point (dim2048) and any other non-scheduled
    # config would inflate the wall-clock projection even though they never run,
    # making the preflight over-estimate and the user needlessly split sessions.
    _sched=[s for s in SIZES if s in (set(RUN_CFG.get("confirmatory_scales",[]))
                                      |set(RUN_CFG.get("exploratory_scales",[])))] or [smallest]
    # (R22 FIX) The relative-cost proxy was n_params(), which EXCLUDES the
    # embedding/output matrix. That matrix is dim x vocab (50257) and is the
    # dominant matmul at small scales: dim192 has 1.4M non-embed params but a
    # 9.7M-param tied head, so the smallest scale -- the one sec_per_step is
    # measured on -- is ~8x more expensive than its weight said. The proxy
    # therefore inflated the large/small cost ratio by ~7x (256x by params vs
    # ~39x by actual FLOPs) and the projection came out roughly an order of
    # magnitude too pessimistic. A user reading "270 sessions needed" would
    # reasonably abandon a grid that in fact fits in a handful.
    # Replaced with a proper per-token FLOP model: parameters (incl. head) plus
    # the quadratic attention term.
    def _flops_per_token(c):
        return 6.0*(n_params(c)+c.dim*c.vocab_size)+6.0*c.n_layers*RUN_CFG["seq_len"]*c.dim
    _f0=_flops_per_token(SIZES[smallest])
    wsum=sum(_flops_per_token(SIZES[s])/_f0 for s in _sched)
    # (R22) exploratory scales run a REDUCED seed set; the old formula charged
    # all 11 scales at 15 seeds. Count what SEEDS_FOR actually returns.
    n_runs=len(METHODS)*len(RUN_CFG["seeds"])
    _main_weighted=sum(_flops_per_token(SIZES[s])/_f0*len(METHODS)*len(SEEDS_FOR(s))
                       for s in _sched)
    # (R-c) ALSO budget the auxiliary phases added in revision so the projected
    # hours are not an under-estimate the user discovers only mid-grid:
    #   - per-method LR calibration: (scales x methods x lr_grid x calib_seeds)
    #   - rho0 sweep + H3 label-shuffled placebo: 2 x (rho0_grid x seeds) at dim384
    #   - fp16+TAI-init 2x2 ablation: ablation_seeds at the primary scale
    #   - width-controlled ladder: wc_scales x 3 methods x wc_seeds
    _nseed=len(RUN_CFG["seeds"])
    aux_runs=0
    # (R22) LR calibration now runs lr_cal_steps, not STEPS -> charge it at the
    # correct fraction, and over the SCHEDULED scales only.
    _cal_frac=float(RUN_CFG.get("lr_cal_steps") or STEPS)/max(STEPS,1)
    aux_runs+=int(len(_sched)*len(METHODS)*len(RUN_CFG.get("lr_grid",[1]))
                  *len(RUN_CFG.get("calib_seeds",[1]))*_cal_frac)  # LR cal (shortened)
    aux_runs+=2*len(RUN_CFG.get("rho0_grid",[0]))*_nseed          # sweep + placebo
    aux_runs+=len(RUN_CFG.get("ablation_seeds",[]))               # fp16 2x2 (only fp16_taiinit is new)
    aux_runs+=len(RUN_CFG.get("width_ctrl_scales",[]))*3*len(RUN_CFG.get("width_ctrl_seeds",[]))  # width-ctrl
    # main grid weighted by scale cost; aux runs are mostly at/below dim384, so we
    # charge them at the dim384 relative weight (conservative, not free).
    _w384=_flops_per_token(SIZES.get("dim384",SIZES[smallest]))/_f0
    if RUN_CFG.get("wallclock_equal_h1",False): _main_weighted*=2.0   # phase re-runs the grid
    # (R29) price every scale from the FITTED curve when we have one; fall back to
    # the old smallest-scale extrapolation only if the multi-scale probe failed.
    if _COST_A is not None and _COST_B is not None:
        _cost=lambda k: max(_COST_A+_COST_B*_COST_FPT(SIZES[k]),1e-6)
        _main=sum(_cost(s_)*STEPS*len(METHODS)*len(SEEDS_FOR(s_)) for s_ in _sched)
        if RUN_CFG.get("wallclock_equal_h1",False): _main*=2.0
        proj=(_main + aux_runs*_cost("dim384" if "dim384" in SIZES else smallest)*STEPS)*1.3
    else:
        proj=sps*STEPS*(_main_weighted + aux_runs*_w384)*1.3
    _bh=RUN_CFG.get("session_budget_hours")          # (R30) may be None
    bud=(float(_bh) if _bh is not None else 24.0)*3600   # report against the
                                                         # platform limit instead
    rep["smoke_run"]={"size":smallest,"sec_for_30_steps":round(dt,1),
                      "val_loss":round(r["val_loss"],3),
                      "arc":r.get("arc_easy_acc"),"hellaswag":r.get("hellaswag_acc")}
    rep["fast_mode"]=(FAST_CFG if "FAST_CFG" in globals() else None)
    rep["projection"]={"sec_per_step_smallest":round(sps,3),"scale_weight_sum":round(wsum,1),
        "main_runs":sum(len(METHODS)*len(SEEDS_FOR(s)) for s in _sched),
        "main_run_equivalents":round(_main_weighted,1),
        "aux_run_equivalents":int(aux_runs),"steps_per_run":STEPS,
        "projected_hours_all":round(proj/3600,2),"session_budget_hours":bud/3600,
        "fits_one_session":bool(proj<=bud),"sessions_needed":math.ceil(proj/bud)}
    print("\n"+"="*68+"\nPREFLIGHT REPORT\n"+"="*68)
    print(json.dumps(rep,indent=2,default=str))
    p=rep["projection"]
    print((f"\nOK: full grid (~{p['projected_hours_all']}h) fits one "
           f"{p['session_budget_hours']}h session.") if p["fits_one_session"]
          else (f"\nNOTE: ~{p['projected_hours_all']}h > {p['session_budget_hours']}h/"
                f"session -> ~{p['sessions_needed']} sessions; resume is automatic, "
                f"re-run this cell each session until complete."))
    notify("PREFLIGHT DONE",
           f"{rep['corpus_source']} | lm-eval={rep['lmeval_available']} | "
           f"budget {TOKEN_BUDGET/1e6:.0f}M tok | proj {p['projected_hours_all']}h / {p['sessions_needed']} sess",tag="preflight")
    return rep
PREFLIGHT_REPORT=_preflight()
if PREFLIGHT_ONLY:
    print("\nPREFLIGHT_ONLY=True -> stopping before the full grid. Review above, "
          "set RUN_CFG['preflight_only']=False, then re-run this cell.")
    raise SystemExit("Preflight complete; full grid intentionally skipped.")
# ---- end preflight ----

# ---- calibration (standard init, calib_seeds) -> reused as standard results ----
notify("PHASE START: main experiment",f"{len(SIZES)*len(METHODS)*len(SEEDS)} runs planned")

# =====================================================================
# (#1) PER-METHOD LR CALIBRATION  (leakage-free: calib_seeds are disjoint
#      from the reporting SEEDS).  For every (scale, method) we sweep
#      lr_grid on the calib seeds, pick the LR* minimising mean plateau
#      val_loss, and store the full sensitivity curve. The main grid then
#      runs each method at ITS LR*. Set RUN_CFG['per_method_lr']=False to
#      fall back to a single shared peak_lr (legacy behaviour).
# =====================================================================
LR_GRID=RUN_CFG.get("lr_grid",[5e-4,1e-3,2e-3,3e-3])
LRSTAR={}            # (size,method) -> chosen peak lr
LR_CURVE={}          # (size,method) -> {lr: mean_val_loss over calib seeds}
def _calibrate_lr(size,cfg,method,opts):
    key=(size,method)
    cid=run_id("lrcal",size,method,0)
    cached=load_run(cid) if have_run(cid) else None
    if cached and "curve" in cached:
        LR_CURVE[key]=cached["curve"]; LRSTAR[key]=cached["lrstar"]; return
    curve={}
    def _eval_lr(lr):
        """Mean calib val_loss at one lr; cached per (scale,method,seed,lr)."""
        k=f"{lr:.3e}"
        if k in curve: return curve[k]
        vs=[]
        for cs in RUN_CFG["calib_seeds"]:
            rid=run_id("lrcal",size,method,f"{cs}_{lr:.1e}")
            if have_run(rid): r=load_run(rid)
            else:
                if budget_exceeded():
                    notify("SESSION BOUNDARY (lr-cal)",f"{size}/{method} lr{lr:.1e}",critical=True); raise SystemExit(0)
                # (R22) calibration runs are SHORT: the only thing extracted from
                # them is an argmin, which separates long before the 20M-token
                # budget. Full-length calibration was ~42% of all training compute.
                # (R33) the proxy this rests on is now VALIDATED once at full
                # length -- see _validate_lr_proxy below.
                _cal_steps=RUN_CFG.get("lr_cal_steps")
                r=train_run(size,cfg,method,cs,opts,lr=lr,
                            _preflight_steps=(int(_cal_steps) if _cal_steps else None))
                save_run(rid,r)
            vs.append(r["val_loss"])
        curve[k]=float(np.mean(vs)); return curve[k]

    for lr in LR_GRID: _eval_lr(lr)
    # (R33) ADAPTIVE EXTENSION. If the argmin sits on an end point the true
    # optimum is outside the grid, and -- as session 1 showed -- different arms
    # sit on that boundary with different slopes, so the ranking itself may be an
    # artifact of where the grid happened to stop. Extend once in the offending
    # direction and re-check, up to lr_grid_max_extend times. Each extension is
    # 3 short runs; it is far cheaper than discovering afterwards that the main
    # grid was trained at a constrained LR.
    if RUN_CFG.get("lr_grid_adaptive",True):
        for _ in range(int(RUN_CFG.get("lr_grid_max_extend",2))):
            _lrs=sorted(float(k) for k in curve)
            _bst=min(_lrs,key=lambda x:curve[f"{x:.3e}"])
            if   _bst==max(_lrs): _new=_bst*2.0
            elif _bst==min(_lrs): _new=_bst/2.0
            else: break                     # interior optimum: grid is adequate
            if not (1e-5<=_new<=5e-2): break
            print(f"    [lr-ext] {size}/{method}: argmin at boundary {_bst:.1e} -> probing {_new:.1e}")
            _eval_lr(_new)
    _srt=sorted(curve.items(),key=lambda kv:kv[1])
    lrstar=float(_srt[0][0])
    _lrs=sorted(float(k) for k in curve)
    _interior=(min(_lrs)<lrstar<max(_lrs))
    if not _interior:
        print(f"    [lr-warn] {size}/{method}: LR*={lrstar:.1e} still on a grid "
              f"boundary after extension — record as a limitation for this cell.")
    LR_CURVE[key]=curve; LRSTAR[key]=lrstar
    save_run(cid,{"size":size,"method":method,"curve":curve,"lrstar":lrstar,
                  "lr_interior":bool(_interior),"lr_grid_probed":sorted(curve)})
    notify(f"LR* {size}/{method}",f"{lrstar} | curve {curve}",tag="lrcal")

def _validate_lr_proxy():
    """(R33) Test the 300-step calibration proxy against a full-length curve.

    LR* is picked from lr_cal_steps (300) but applied to STEPS (1221) runs. With
    120 warmup steps that is ~180 post-warmup steps of evidence, and high LRs
    look better early -- the same direction as the upper-boundary problem, so the
    two confounds compound. Rather than assume, measure: one scale, one seed,
    every lr in the base grid, at full length. Three or five runs.

    Outcome is recorded either way. Agreement makes the proxy defensible IN THE
    PAPER with evidence rather than assertion; disagreement means lr_cal_steps
    must rise before the main grid can be trusted, and it is far better to learn
    that now than after 60 GPU-hours."""
    sc=RUN_CFG.get("lr_proxy_check_scale")
    if not sc or sc not in SIZES: return None
    seed=int(RUN_CFG.get("lr_proxy_check_seed",RUN_CFG["calib_seeds"][0]))
    meth,opts="standard",{}
    for _m,_o in METHODS:
        if _m=="standard": opts=_o; break
    full={}
    for lr in LR_GRID:
        rid=run_id("lrproxy",sc,meth,f"{seed}_{lr:.1e}")
        if have_run(rid): r=load_run(rid)
        else:
            if budget_exceeded(): raise SystemExit(0)
            r=train_run(sc,SIZES[sc],meth,seed,opts,lr=lr)   # FULL length
            save_run(rid,r)
        full[f"{lr:.3e}"]=float(r["val_loss"])
    short=LR_CURVE.get((sc,meth),{})
    fs=float(sorted(full.items(),key=lambda kv:kv[1])[0][0])
    ss=float(sorted(short.items(),key=lambda kv:kv[1])[0][0]) if short else None
    agree=(ss is not None and abs(fs-ss)<1e-12)
    # rank correlation over the shared lrs is the more informative check: the
    # argmin can agree by luck while the ordering disagrees badly.
    _both=[k for k in full if k in short]
    rho=None
    if len(_both)>=3:
        try:
            from scipy.stats import spearmanr
            rho=float(spearmanr([short[k] for k in _both],[full[k] for k in _both]).correlation)
        except Exception: pass
    out={"scale":sc,"method":meth,"seed":seed,"short_curve":short,"full_curve":full,
         "lrstar_short":ss,"lrstar_full":fs,"argmin_agrees":bool(agree),"spearman":rho,
         "cal_steps":RUN_CFG.get("lr_cal_steps"),"full_steps":STEPS}
    print("\n[lr-proxy] short-vs-full calibration check at %s/%s:"%(sc,meth))
    print("   short(%d steps) LR*=%s | full(%d steps) LR*=%.1e | agree=%s | spearman=%s"
          %(RUN_CFG.get("lr_cal_steps") or STEPS,ss,STEPS,fs,agree,
            "n/a" if rho is None else f"{rho:.3f}"))
    if not agree:
        print("   !! PROXY DISAGREES — raise lr_cal_steps and re-derive LR* before "
              "trusting the main grid. Do NOT paper over this.")
    notify("LR proxy check",json.dumps({k:out[k] for k in
           ("scale","lrstar_short","lrstar_full","argmin_agrees","spearman")}),tag="lrproxy")
    return out

# (R33) INVALIDATE STALE LR* SUMMARIES, KEEP THE EXPENSIVE RUNS.
# _calibrate_lr short-circuits on a cached summary record, so widening LR_GRID
# would otherwise have no effect: the old three-point curve would be reloaded and
# the new points never probed. The summaries are cheap metadata; the 756 per-lr
# training runs behind them are not, and those are keyed on (scale, method, seed,
# lr) and stay valid. So we delete ONLY summaries whose probed grid does not
# cover the current LR_GRID. Runs already completed are reused untouched.
def _invalidate_stale_lr_summaries():
    n=0
    for size in SCALES_TO_RUN:
        for method,_ in METHODS:
            cid=run_id("lrcal",size,method,0)
            if not have_run(cid): continue
            try: rec=load_run(cid)
            except Exception: rec=None
            probed={float(k) for k in (rec or {}).get("curve",{})}
            if not probed or any(not any(abs(p-lr)<1e-12 for p in probed) for lr in LR_GRID):
                try: os.remove(run_path(cid)); n+=1
                except Exception: pass
    if n: print(f"[lr-grid] {n} stale LR* summary record(s) invalidated; "
                f"the {len(LR_GRID)}-point grid will be completed from cache + new runs only.")
    return n
_invalidate_stale_lr_summaries()

if RUN_CFG.get("per_method_lr",True):
    notify("PHASE START: per-method LR calibration",
           f"{len(SIZES)*len(METHODS)} (scale,method) cells x {len(LR_GRID)} lrs x {len(RUN_CFG['calib_seeds'])} calib seeds")
    for size in SCALES_TO_RUN:
        cfg=SIZES[size]
        for method,opts in METHODS:
            _calibrate_lr(size,cfg,method,opts)
    LR_PROXY_CHECK=_validate_lr_proxy()
    print("LR* table (peak lr chosen per scale x method):")
    for k in sorted(LRSTAR): print(f"  {k}: {LRSTAR[k]}")
    notify("PHASE DONE: per-method LR calibration",f"{len(LRSTAR)} LR* values")
else:
    for size in SCALES_TO_RUN:
        for method,_ in METHODS: LRSTAR[(size,method)]=PEAK_LR

def _lrstar(size,method): return LRSTAR.get((size,method),PEAK_LR)

# (#A) SCALE-AWARE H1 TARGET LOSS.  The token-equal H1 ("tokens to reach target
#      loss") is reported at every scale, so the target must be defined PER SCALE
#      -- a single dim384-derived target is not meaningful for a 50M model. For
#      each scale we calibrate the target from that scale's OWN standard-init
#      calib trajectories (disjoint calib seeds), as the 60th-percentile loss
#      (median over calib seeds). H1 at each scale is then "tokens for method M to
#      reach that scale's standard-init reference loss" -- i.e. a per-scale
#      relative speedup, which is the comparison a reviewer will accept across
#      scales. The dim384 standard calib runs double as reused standard results.
TARGET_LOSS_BY_SCALE={}
for size in SCALES_TO_RUN:
    cfg=SIZES[size]
    trajs=[]
    for seed in RUN_CFG["calib_seeds"]:
        rid=run_id("main",size,"standard",seed)
        if have_run(rid):
            rec=load_run(rid)
        else:
            if budget_exceeded():
                notify("SESSION BOUNDARY (calibration)",f"{size} standard","resume next session",critical=True); raise SystemExit(0)
            rec=train_run(size,cfg,"standard",seed,{},lr=_lrstar(size,"standard")); save_run(rid,rec)
            notify(f"run done: main {size} standard s{seed}",f"val {rec['val_loss']:.4f}",tag="run")
        trajs.append(smooth(rec["loss"]))
    TARGET_LOSS_BY_SCALE[size]=float(np.median([np.percentile(t,60) for t in trajs]))
# backward-compatible scalar (smallest scale) kept for any legacy reference
TARGET_LOSS=TARGET_LOSS_BY_SCALE[min(SIZES,key=lambda k:SIZES[k].dim)]
def _target(size): return TARGET_LOSS_BY_SCALE[size]
print(f"STEPS/run={STEPS} | H1 per-scale target loss (60pct, median/calib):")
# (R19) print only executed scales; TARGET_LOSS_BY_SCALE is keyed by scales that
# actually ran, so iterating over every declared config (incl. opt-in 783M) KeyErrors.
for s in SIZES:
    if s in TARGET_LOSS_BY_SCALE: print(f"    {s}: {TARGET_LOSS_BY_SCALE[s]:.3f}")

# ---- full grid (skips finished, including reused calibration) ----
for size in SCALES_TO_RUN:
    cfg=SIZES[size]
    for method,opts in METHODS:
        notify(f"BLOCK START: {size}/{method}","")
        for seed in SEEDS_FOR(size):
            rid=run_id("main",size,method,seed)
            if have_run(rid): continue
            if budget_exceeded():
                notify("SESSION BOUNDARY",f"stopped before {rid}. Re-run Run-All to resume.",critical=True); raise SystemExit(0)
            rec=train_run(size,cfg,method,seed,opts,lr=_lrstar(size,method))
            rec["steps_to_target"]=steps_to_target(rec["steps"],rec["loss"],_target(size))
            save_run(rid,rec)
            notify(f"run done: main {size} {method} s{seed}",
                   f"val {rec['val_loss']:.4f} | spars {rec['final_sparsity']*100:.1f}% | plateau {rec['plateau_ok']}",tag="run")

# backfill steps_to_target on reused calibration runs
# (R39) reset ALL collectors together. RESULTS was cleared here but DIVERGED_RUNS
# was not, so a second pass over the store in the same session would append
# duplicates to it and trip the R38 invariant with a confusing off-by-N.
RESULTS=[]; DIVERGED_RUNS=[]; PLATEAU_STATS={"n":0,"n_ok":0,"gaps":[]}
for size in SCALES_TO_RUN:
    cfg=SIZES[size]
    for method,_ in METHODS:
        for seed in SEEDS_FOR(size):
            rid=run_id("main",size,method,seed); rec=load_run(rid)
            if "steps_to_target" not in rec:
                rec["steps_to_target"]=steps_to_target(rec["steps"],rec["loss"],_target(size)); save_run(rid,rec)
            # (R38 FIX) INDENTATION: the R26 divergence filter was written one
            # level out, so it sat in the `method` loop rather than the `seed`
            # loop. Effect: after the seed loop finished, only the LAST seed's
            # record was examined -- exactly one run per (scale, method) entered
            # RESULTS instead of all 5 or 15. The grid has 793 completed runs and
            # RESULTS would have held 88. Every downstream test consumes RESULTS,
            # so H1/H2/H4 would have been computed on n=1 per cell: paired tests
            # would have collapsed or thrown, and any number that did survive
            # would have been meaningless. Silent, because 88 runs is a plausible
            # count and nothing asserts the expected size.
            # (R26) diverged runs stay on disk (so have_run() does not re-run them
            # forever) but never enter RESULTS, which is what every hypothesis
            # test consumes. Counted for the manifest: a nonzero count is a
            # finding about the LR grid, not a nuisance to be hidden.
            _note_plateau(rec)
            if rec.get("diverged"):
                DIVERGED_RUNS.append({k:rec.get(k) for k in ("size","method","seed","val_loss")})
            else:
                RESULTS.append(rec)
# (R38) Guard the invariant rather than trusting the indentation. RESULTS must
# hold one record per (scale, method, seed) minus divergences; anything else means
# the collection loop is broken again.
_expected=sum(len(METHODS)*len(SEEDS_FOR(_s)) for _s in SCALES_TO_RUN)
assert len(RESULTS)+len(DIVERGED_RUNS)==_expected, (
    f"RESULTS collection is wrong: got {len(RESULTS)}+{len(DIVERGED_RUNS)} records, "
    f"expected {_expected} (one per scale x method x seed). Do not trust any "
    f"downstream statistic until this is fixed.")
# report how many runs actually reached plateau (H2 validity gate)
_pl=[r for r in RESULTS if r.get("plateau_ok")]
print(f"Main experiment complete: {len(RESULTS)} runs ({len(_pl)} plateau-verified).")
notify("PHASE DONE: main experiment",f"{len(RESULTS)} runs ({len(_pl)} plateau-ok)")

# =====================================================================
# (#2) WALL-CLOCK-EQUAL H1.  The token-equal H1 above asks "how many
#      tokens to reach target loss". This pass asks the dual question:
#      "given the SAME wall-clock budget, what loss is reached?". For
#      each scale the budget = median wall_sec of REF_METHOD (strongest
#      non-TAI ternary baseline) at that scale, INCLUDING its init cost.
#      Every method trains under that identical budget; TAI's init-
#      calibration + gamma-warmup overhead is already inside its own
#      wall clock, so this is a strictly fair compute comparison.
#      fp16_control is expected to look strong (no quant overhead); that
#      asymmetry is reported, not hidden (see Limitations).
# =====================================================================
WALLCLOCK_H1=RUN_CFG.get("wallclock_equal_h1",True)
REF_METHOD=RUN_CFG.get("wall_ref_method","bitnet158")
WALL_RESULTS=[]
if WALLCLOCK_H1:
    notify("PHASE START: wall-clock-equal H1",f"ref={REF_METHOD}")
    wall_budget_by_scale={}
    # wall-clock-equal H1 is a CONFIRMATORY robustness check, not a ladder claim;
    # run it only on confirmatory scales to respect the single-A100 budget.
    _WALL_SCALES=[s for s in SCALES_TO_RUN if s in _CONF_SET] or SCALES_TO_RUN
    for size in _WALL_SCALES:
        ws=[r["wall_sec"] for r in RESULTS if r["size"]==size and r["method"]==REF_METHOD]
        if ws: wall_budget_by_scale[size]=float(np.median(ws))
    for size in _WALL_SCALES:
        cfg=SIZES[size]
        wb=wall_budget_by_scale.get(size)
        if wb is None: continue
        for method,opts in METHODS:
            for seed in SEEDS_FOR(size):
                rid=run_id("wallH1",size,method,seed)
                if have_run(rid): WALL_RESULTS.append(load_run(rid)); continue
                if budget_exceeded():
                    notify("SESSION BOUNDARY (wallH1)",f"before {rid}",critical=True); raise SystemExit(0)
                rec=train_run(size,cfg,method,seed,opts,lr=_lrstar(size,method),wall_budget=wb)
                rec["wall_budget_used"]=wb
                save_run(rid,rec); WALL_RESULTS.append(rec)
                notify(f"wallH1 {size} {method} s{seed}",
                       f"val {rec['val_loss']:.4f} @ {rec['stopped_step']}/{STEPS} steps",tag="wallH1")
    print(f"Wall-clock-equal H1 complete: {len(WALL_RESULTS)} runs.")
    notify("PHASE DONE: wall-clock-equal H1",f"{len(WALL_RESULTS)} runs")

In [ ]:
# === CELL 11c: (R3) BUDGET-SUFFICIENCY — IS 20M TOKENS ACTUALLY CONVERGENCE? =
# H2 claims "quality AT CONVERGENCE (>=20M tokens)". Every part of that claim
# rests on the assumption that 20M tokens is enough for the loss to have stopped
# improving. The plateau gate (Cell 11) now tests this per-run via the val-trace
# slope, but a slope test can only say "no trend RESOLVABLE at this noise level"
# -- it cannot rule out a slow, real descent that more tokens would reveal. The
# only way to settle it is to spend more tokens and look.
#
# DESIGN: at the confirmatory scale, re-run {primary baseline, tai_warmup} at
# budget_check_mult x the token budget on budget_check_seeds, holding everything
# else fixed. Compare each method's val_loss at 20M against the same run's val
# trace at 40M. Pre-registered decision rule:
#   * if |loss(2B) - loss(B)| <= TOST margin for BOTH arms -> 20M is a plateau in
#     the sense H2 needs; H2 keeps its "at convergence" wording.
#   * otherwise -> H2 is RE-STATED as "quality at a fixed 20M-token budget" and
#     the manifest records the shortfall. We do not quietly keep the stronger
#     wording; the claim degrades to what the data support.
# Cost: 2 methods x 3 seeds x 2x tokens ~= 12 run-equivalents at the primary scale.
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or (not RUN_CFG.get("budget_check",True)) \
   or not _require("RESULTS","STEPS"):
    print("[skip] (R3) budget-sufficiency check: ARM_FULL_RUN off, disabled, or prerequisites missing.")
    BUDGET_CHECK=None
else:
    import numpy as np, math
    notify("PHASE START: budget sufficiency (2x tokens)","H2 convergence assumption")
    _PS=RUN_CFG.get("confirmatory_scale","dim384"); _cfg=SIZES[_PS]
    _MULT=float(RUN_CFG.get("budget_check_mult",2.0))
    _LONG_STEPS=int(STEPS*_MULT)
    _ARMS=[(RUN_CFG.get("primary_baseline","bitnet158"),
            {"recipe":"bitnet"} if RUN_CFG.get("primary_baseline","bitnet158")=="bitnet158" else {}),
           ("tai_warmup",{"rho0":0.75,"gamma_warmup":240})]
    BUDGET_CHECK={"mult":_MULT,"scale":_PS,"steps_short":STEPS,"steps_long":_LONG_STEPS,"arms":{}}
    for method,opts in _ARMS:
        rows=[]
        for seed in RUN_CFG["budget_check_seeds"]:
            rid=run_id("budget",_PS,f"{method}_x{_MULT:g}",seed)
            if have_run(rid): rows.append(load_run(rid)); continue
            if budget_exceeded():
                notify("SESSION BOUNDARY (budget check)","resume next session",critical=True); raise SystemExit(0)
            # _preflight_steps is the generic "override step count" hook in train_run
            rec=train_run(_PS,_cfg,method,seed,opts,_preflight_steps=_LONG_STEPS,
                          lr=(LR_STAR.get((_PS,method)) if "LR_STAR" in dir() else None))
            rec["long_run"]=True; save_run(rid,rec); rows.append(rec)
            notify(f"run done: budget {method} s{seed} x{_MULT:g}",
                   f"val {rec['val_loss']:.4f}",tag="run")
        # val_loss at the SHORT budget is read off this long run's own val trace at
        # the step where the short run would have ended -> a within-run, paired
        # comparison (no cross-run seed noise contaminating the delta).
        deltas=[]
        for r in rows:
            vs=r.get("val_steps") or []; vl=r.get("val_trace") or []
            if not vs: continue
            j=min(range(len(vs)),key=lambda k:abs(vs[k]-STEPS))   # probe nearest 20M
            short=float(vl[j]); long=float(r["val_loss"])
            deltas.append(short-long)                             # >0 => still improved
        BUDGET_CHECK["arms"][method]={"n":len(deltas),
            "mean_improvement_nats":(float(np.mean(deltas)) if deltas else float("nan")),
            "sd":(float(np.std(deltas,ddof=1)) if len(deltas)>1 else float("nan")),
            "per_seed":[float(d) for d in deltas]}
    # decision against the SAME margin H2 uses (data-derived if available)
    _margin=float(TOST_MARGIN_BY_SCALE.get(_PS,RUN_CFG["tost_margin"])) \
            if "TOST_MARGIN_BY_SCALE" in dir() else float(RUN_CFG["tost_margin"])
    print("="*74)
    print(f"(R3) BUDGET SUFFICIENCY at {_PS}: val_loss({STEPS} steps) - val_loss({_LONG_STEPS} steps)")
    print(f"     positive = the model was STILL IMPROVING past the 20M-token budget")
    print(f"     H2 equivalence margin = {_margin:.4f} nats")
    print("="*74)
    _ok=True
    for m,d in BUDGET_CHECK["arms"].items():
        imp=d["mean_improvement_nats"]
        flag="within margin" if (not math.isnan(imp) and abs(imp)<=_margin) else "EXCEEDS MARGIN"
        if math.isnan(imp) or abs(imp)>_margin: _ok=False
        print(f"  {m:16s}: {imp:+.4f} nats (n={d['n']}, sd={d['sd']:.4f})  -> {flag}")
    BUDGET_CHECK["margin"]=_margin
    BUDGET_CHECK["h2_wording_supported"]=bool(_ok)
    print()
    if _ok:
        print("  VERDICT: doubling the budget moves val_loss by less than the equivalence")
        print("           margin for both arms => 20M tokens is a plateau in the sense H2")
        print("           requires. H2 keeps its 'quality at convergence' wording.")
    else:
        print("  VERDICT: at least one arm improved by MORE than the margin when given 2x")
        print("           tokens => 20M tokens is NOT convergence. H2 is hereby RE-STATED as")
        print("           'quality at a fixed 20M-token budget' (a token-equal claim, still")
        print("           valid and still falsifiable), and every downstream mention of")
        print("           'at convergence' must be read in that weaker sense. Recorded in")
        print("           the manifest as h2_wording_supported=false.")
    notify("PHASE DONE: budget sufficiency",
           f"h2_wording_supported={_ok}")


In [ ]:
# === CELL 12: RHO0 SWEEP (H3 causal control + H5 Pareto frontier) ==========
# ============================================================================
# (R37) H3 NARROWED — THE INTERVENTION IS REAL BUT ~93% OF IT IS ERASED.
# ----------------------------------------------------------------------------
# Recorded from the dim384 sweep (4 rho0 points x 15 seeds) BEFORE the dim768
# sweep, the placebo arm, or any Cell 13 statistic had been computed, so the
# revised wording is not a description of a finished result.
#
#   rho0   initial sparsity   FINAL sparsity   val_loss
#   0.40         0.60             0.3240        5.2519
#   0.50         0.50             0.3143        5.2171
#   0.60         0.40             0.3075        5.1955
#   0.70         0.30             0.3025        5.1901
#
# A THIRTY-point swing in initial sparsity leaves TWO points at the end:
#   final = 0.2800 + 0.0714 * initial      (R^2 = 0.978)
# The retention coefficient is 0.071 -- roughly 93% of the intervention is undone
# by training -- and the implied fixed point is 0.3015, essentially the value the
# `standard` arm converges to on its own (~0.313).
#
# H3 AS PRE-REGISTERED ("the initial operating point causally SHAPES final
# sparsity") IS NOT SUPPORTED IN ITS STRONG FORM. What the data support is
# narrower and, we think, more interesting: rho0 leaves a small, strictly ordered,
# highly reliable residue (rho0=0.40 vs 0.70: +0.0216, p=8.5e-15, d=44.6 -- the
# within-seed sparsity SD is 0.0004, so the ordering is far outside noise), on top
# of a strong attractor near 0.30 that training reaches from wherever it starts.
#
# The locked ~0.69 value is therefore NOT merely a default that every zero-mean
# initialiser happens to share; it is where the optimisation goes. That reframes
# the whole paper and explains H1: quality improves monotonically as rho0 moves
# BACK toward the locked point (5.2519 -> 5.1901), which is exactly why the `tai`
# arm loses to `standard` while `tai_warmup` -- which approaches the same point
# gradually -- wins.
#
# REPORTING CHANGE, NOT AN ANALYSIS CHANGE: the sweep, the placebo arm and every
# test are untouched. What changes is that RETENTION is now the headline quantity
# rather than "does sparsity move at all", because "does it move" was always going
# to be trivially yes and says nothing about durability. The placebo arm remains
# the falsification instrument: if tai_negctrl shows a comparable retention slope,
# even this narrowed effect is an artefact.
RHO0_RETENTION={}   # filled below; reported in the manifest
# Sweep rho0 over a grid at dim384.
# H3 (CAUSAL, not merely correlational): because train_run draws TAI init noise
#   from a per-seed generator (seed+777) that is INDEPENDENT of rho0, every rho0
#   at a given seed sees the SAME base noise and the SAME data order -- only the
#   absmean threshold split differs. So moving rho0 is a clean ceteris-paribus
#   intervention. We therefore test monotonicity WITHIN each seed (each seed is an
#   independent replication of the intervention) and aggregate the per-seed
#   Spearman rho's, rather than pooling all (rho0,seed) points (which can fake
#   monotonicity). Pooled Spearman is still reported for continuity.
# H5: trace the non-dominated quality(val_loss) vs sparsity frontier with
#   bootstrap uncertainty, and report rho0* (min val_loss s.t. sparsity>=target)
#   with a bootstrap CI over seeds -- the practical operating-point claim.
if not RUN_CFG.get("ARM_FULL_RUN",False):
    print("[skip] rho0 sweep: ARM_FULL_RUN=False (Run-All safe mode).")
elif not _require("RESULTS", "TARGET_LOSS_BY_SCALE"):
    pass  # prerequisites missing -> skip this phase (see message above)
else:
    import numpy as np,math,torch
    from scipy import stats
    from collections import defaultdict
    RHO0_GRID=RUN_CFG["rho0_grid"]
    SPARS_TARGET=RUN_CFG.get("pareto_sparsity_target",0.30)
    # (#B) CAUSAL CONTROL AT >1 SCALE.  H3's intervention is run at dim384 AND at
    #      the top scale, so we can test whether the rho0->sparsity causal effect
    #      is SCALE-INVARIANT (a far stronger claim than single-scale control).
    #      rho0_scales defaults to [dim384, top]; override via RUN_CFG.
    # second sweep scale = top CONFIRMATORY scale (not the largest declared scale,
    # which may be an exploratory point we never fully trained). Keeps H3's cross-
    # scale check inside budget. Primary sweep scale stays dim384 (full 15 seeds).
    _conf_sw=[s for s in SIZES if s in RUN_CFG.get("confirmatory_scales",list(SIZES))]
    _top=max(_conf_sw,key=lambda k:SIZES[k].dim) if _conf_sw else max(SIZES,key=lambda k:SIZES[k].dim)
    RHO0_SCALES=RUN_CFG.get("rho0_scales",["dim384",_top])
    RHO0_SCALES=[s for s in RHO0_SCALES if s in SIZES]
    def _sweep_seeds(_sz):
        # dim384 (H3 primary) keeps the full seed set; other sweep scales may use
        # the reduced exploratory set via SEEDS_FOR to respect the A100 budget.
        if _sz=="dim384": return SEEDS
        return SEEDS_FOR(_sz) if "SEEDS_FOR" in dir() else SEEDS
    notify("PHASE START: rho0 sweep",
           f"~{len(RHO0_GRID)*sum(len(_sweep_seeds(s)) for s in RHO0_SCALES)} runs over scales {RHO0_SCALES}")
    # (R6) THE SWEEP NOW RUNS TWO ARMS, NOT ONE.
    # Previously only `tai` was swept, which made H3 nearly unfalsifiable: TAI was
    # the sole method able to move rho0 at all, so "sweeping rho0 shifts sparsity"
    # could only fail if the machinery were broken. `gamma_offset` reaches the SAME
    # rho0 by scaling the threshold instead of shaping the init (see Cell 8), so
    # the sweep now contains a genuine competitor that could match or beat TAI.
    # H3 is tested on the TAI arm (pre-registered); the gamma_offset arm supplies
    # (i) a mechanism control for H3 -- if BOTH arms move sparsity with rho0, the
    # causal claim belongs to the OPERATING POINT rather than to TAI's initializer;
    # and (ii) a second Pareto frontier for H5, so the quality-sparsity claim is
    # made against a method that controls sparsity too, not only against fixed-
    # threshold baselines that cannot.
    SWEEP_ARMS=RUN_CFG.get("sweep_arms",[("tai",{}),("gamma_offset",{"gamma_offset":True})])
    SWEEP_BY_SCALE={}; SWEEP_ARM_ROWS={}
    for _sz in RHO0_SCALES:
        cfg=SIZES[_sz]; rows=[]
        for _arm,_aopts in SWEEP_ARMS:
            arm_rows=[]
            for rho0 in RHO0_GRID:
                for seed in _sweep_seeds(_sz):
                    rid=run_id("rho0",_sz,_arm,seed,rho0=rho0)
                    if have_run(rid):
                        _r=load_run(rid); arm_rows.append(_r)
                        if _arm=="tai": rows.append(_r)
                        continue
                    if budget_exceeded():
                        notify("SESSION BOUNDARY (rho0 sweep)",f"{_sz}/{_arm} rho0={rho0}",critical=True); raise SystemExit(0)
                    # train_run applies the deterministic shared-noise TAI init
                    # (gen=seed+777) internally; init_active is read back from the
                    # record, not re-derived under a different RNG path.
                    _o=dict(_aopts); _o["rho0"]=rho0
                    rec=train_run(_sz,cfg,_arm,seed,_o,lr=_lrstar(_sz,_arm))
                    rec["rho0"]=rho0; rec["scale"]=_sz; rec["arm"]=_arm
                    save_run(rid,rec); arm_rows.append(rec)
                    if _arm=="tai": rows.append(rec)
                    notify(f"run done: rho0 {_sz}/{_arm} {rho0} s{seed}",
                           f"spars {rec['final_sparsity']*100:.1f}%",tag="run")
            SWEEP_ARM_ROWS[(_sz,_arm)]=arm_rows
        SWEEP_BY_SCALE[_sz]=rows   # TAI arm only -> H3/H5 primary analysis unchanged
    # primary analysis scale = dim384 (kept as SWEEP for continuity with H5 below)
    SWEEP=SWEEP_BY_SCALE.get("dim384",SWEEP_BY_SCALE[RHO0_SCALES[0]])

    # ---------- (R37) RETENTION: how much of the intervention SURVIVES --------
    # The pre-registered H3 readout asked whether final sparsity tracks rho0 at
    # all. On the dim384 data it does -- but only barely, and "does it move" was
    # always going to be trivially yes. The quantity that decides whether the
    # initial operating point is a real design knob is the fraction of the
    # intervention that is still there at the end:
    #     final_sparsity = intercept + retention * initial_sparsity
    # retention ~ 1 means the starting point persists; retention ~ 0 means
    # training erases it and the system has an attractor. The intercept, and the
    # fixed point intercept/(1-retention), locate that attractor. Computed for
    # every swept (scale, arm) INCLUDING the placebo, because the placebo's
    # retention is the falsification test: a placebo slope comparable to the real
    # one means even this narrowed effect is measurement artefact.
    for (_sz,_arm),_rows in SWEEP_ARM_ROWS.items():
        _byr={}
        for r in _rows: _byr.setdefault(r["rho0"],[]).append(r["final_sparsity"])
        if len(_byr)<3: continue
        _x=np.array([1.0-k for k in sorted(_byr)])                  # initial sparsity
        _y=np.array([float(np.mean(_byr[k])) for k in sorted(_byr)])# final sparsity
        _sl,_ic=np.polyfit(_x,_y,1)
        _r2=float(np.corrcoef(_x,_y)[0,1]**2) if len(_x)>2 else float("nan")
        RHO0_RETENTION[f"{_sz}/{_arm}"]={
            "retention":float(_sl),"intercept":float(_ic),
            "fixed_point":float(_ic/(1.0-_sl)) if abs(1.0-_sl)>1e-9 else None,
            "r2":_r2,"rho0_points":sorted(_byr),
            "final_sparsity_by_rho0":{str(k):float(np.mean(_byr[k])) for k in sorted(_byr)},
            "n_per_point":{str(k):len(_byr[k]) for k in sorted(_byr)}}
        print(f"  [retention] {_sz}/{_arm}: final = {_ic:.4f} + {_sl:.4f}*initial "
              f"(R2={_r2:.3f}) -> {100*_sl:.1f}% of the intervention survives; "
              f"attractor at {_ic/(1.0-_sl):.4f}" if abs(1.0-_sl)>1e-9 else "")
    _real=RHO0_RETENTION.get("dim384/tai",{}).get("retention")
    _plac=RHO0_RETENTION.get("dim384/tai_negctrl",{}).get("retention")
    if _real is not None and _plac is not None:
        print(f"  [H3 falsification] real retention {_real:.4f} vs placebo {_plac:.4f}"
              f" -> {'PLACEBO MATCHES REAL: effect is artefact' if abs(_plac)>=0.5*abs(_real) else 'placebo flat: effect survives falsification'}")

    # ---------- H3: per-seed (within-intervention) monotonicity ----------
    by_seed=defaultdict(list)
    for s in SWEEP: by_seed[s["seed"]].append(s)
    per_seed_rho=[]
    for sd,rows in by_seed.items():
        rows=sorted(rows,key=lambda r:r["rho0"])
        xs=[r["rho0"] for r in rows]; ys=[r["final_sparsity"] for r in rows]
        if len(set(xs))>2:
            rr,_=stats.spearmanr(xs,ys)
            if not math.isnan(rr): per_seed_rho.append(rr)
    per_seed_rho=np.array(per_seed_rho)
    # pooled (continuity with prior version)
    xs=[s["rho0"] for s in SWEEP]; ys=[s["final_sparsity"] for s in SWEEP]
    rho_pool,p_pool=stats.spearmanr(xs,ys)
    # aggregate per-seed: one-sample test that per-seed rho's are centered > 0
    if len(per_seed_rho)>=2:
        # (R37) two-sided against 0; the SIGN is reported separately rather than
        # assumed, because the pre-registered code assumed the wrong one.
        t_ps,p_ps=stats.ttest_1samp(per_seed_rho,0.0)
        # bootstrap CI of the mean per-seed Spearman
        rng=np.random.default_rng(GLOBAL_SEED)
        bs=[rng.choice(per_seed_rho,len(per_seed_rho),replace=True).mean() for _ in range(5000)]
        ci=(float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5)))
    else:
        t_ps=p_ps=float("nan"); ci=(float("nan"),float("nan"))
    # ========================================================================
    # (R37) THE H3 TEST WAS LOOKING FOR THE WRONG SIGN.
    # ------------------------------------------------------------------------
    # rho0 is the initial ACTIVE ratio, so a HIGHER rho0 means FEWER zeros, i.e.
    # LOWER sparsity. The relationship is necessarily DECREASING. The original
    # code tested "per-seed Spearman centered > 0" and counted seeds with
    # "monotone NON-DECREASING sparsity in rho0" -- both encode the opposite
    # expectation. Measured on the completed dim384 sweep: per-seed Spearman is
    # -1.000 on all 15 seeds and the old frac_mono is 0.00.
    # This was not benign. The old t-test returns t=-inf, p=0.0 -- and a reader
    # scanning for "p<0.001" would have recorded H3 as strongly SUPPORTED while
    # the monotonicity counter said 0.00, i.e. two headline numbers in direct
    # contradiction, with the significant-looking one pointing the wrong way.
    # Fixed by testing the mechanism's actual prediction: rho0 and final sparsity
    # are NEGATIVELY related. Direction is stated explicitly so the sign can never
    # be read off an unsigned p-value again.
    def _mono_dec(rows):
        rows=sorted(rows,key=lambda r:r["rho0"]); v=[r["final_sparsity"] for r in rows]
        return all(v[i+1]<=v[i]+1e-6 for i in range(len(v)-1))
    frac_mono=float(np.mean([_mono_dec(r) for r in by_seed.values()]))
    # ------------------------------------------------------------------------
    # (R37) RETENTION: the quantity that actually decides H3.
    # A Spearman of -1 only says the ORDER is preserved; it says nothing about
    # MAGNITUDE, and magnitude is the whole claim. On dim384 a 30-point swing in
    # INITIAL sparsity (rho0 0.4->0.7 => 0.60->0.30) leaves a 2.15-point spread in
    # FINAL sparsity (0.3240->0.3025): about 7% of the intervention survives, and
    # all four operating points collapse toward ~0.31 -- the same value the
    # `standard` arm reaches unaided. So the honest reading is not "rho0 controls
    # final sparsity" but "rho0 leaves a small, ordered, highly reproducible
    # residual; training pulls every starting point back toward a strong ~0.31
    # attractor." Retention is reported as a first-class number so the paper
    # cannot lean on the correlation alone.
    _grid=sorted({r["rho0"] for r in SWEEP})
    _fin={g:float(np.mean([r["final_sparsity"] for r in SWEEP if r["rho0"]==g])) for g in _grid}
    _d_init=(max(_grid)-min(_grid))                      # = swing in initial sparsity
    _d_fin=(_fin[min(_grid)]-_fin[max(_grid)])           # positive if decreasing
    _retention=float(_d_fin/_d_init) if _d_init>0 else float("nan")
    _attractor=float(np.mean(list(_fin.values())))
    H3_RETENTION={"rho0_grid":_grid,"final_sparsity_by_rho0":_fin,
                  "initial_sparsity_swing":_d_init,"final_sparsity_swing":_d_fin,
                  "retention_fraction":_retention,"mean_final_sparsity":_attractor,
                  "interpretation":("fraction of the initial operating-point intervention "
                                    "that survives training; ~1.0 would mean rho0 sets final "
                                    "sparsity, ~0.0 means training erases it entirely")}
    print("H3 (does the initial operating point persist?):")
    print(f"    pooled Spearman(rho0,final_sparsity)={rho_pool:+.3f} (p={p_pool:.4g})"
          f"   [expected NEGATIVE: higher rho0 = fewer zeros]")
    print(f"    per-seed mean Spearman={np.nanmean(per_seed_rho):+.3f} "
          f"95% CI [{ci[0]:+.3f},{ci[1]:+.3f}] (n_seeds={len(per_seed_rho)})")
    print(f"    fraction of seeds monotone DECREASING in rho0 = {frac_mono:.2f}")
    print(f"    RETENTION = {_retention:.3f}  ({_d_fin*100:.2f} pt of final spread from "
          f"{_d_init*100:.0f} pt of initial swing)")
    print(f"    mean final sparsity across all rho0 = {_attractor:.4f}"
          f"   [the `standard` arm lands near 0.313 unaided]")
    if _retention<0.25:
        print(f"    -> H3 AS PRE-REGISTERED IS NOT SUPPORTED IN ITS STRONG FORM. The ordering "
              f"is perfectly reproducible, but only {_retention*100:.0f}% of the intervention "
              f"survives;\n       final sparsity is dominated by an attractor near "
              f"{_attractor:.3f}, not by rho0. Report as a bounded residual effect.")

    # ---------- H3 NEGATIVE CONTROL (placebo): RNG axis must NOT move sparsity ----
    # The causal claim is 'rho0 -> final_sparsity'. A skeptic asks: maybe sparsity
    # just tracks the init RNG draw, not rho0. Placebo test: at a FIXED rho0, vary
    # ONLY the init-noise seed across the SAME seed list and measure the spread of
    # final_sparsity. If the rho0 effect is real, the across-rho0 sparsity range
    # (signal) must dwarf the fixed-rho0 across-seed range (placebo/noise). We reuse
    # the existing sweep records (no new training): group by rho0 and report the
    # within-rho0 across-seed sparsity SD vs the across-rho0 mean-sparsity range.
    by_rho=defaultdict(list)
    for s in SWEEP: by_rho[s["rho0"]].append(s["final_sparsity"])
    within_sd=float(np.mean([np.std(v,ddof=1) for v in by_rho.values() if len(v)>1])) if by_rho else float("nan")
    across_range=float(np.ptp([np.mean(v) for v in by_rho.values()])) if by_rho else float("nan")
    snr=across_range/within_sd if (within_sd and within_sd>0) else float("nan")
    print("    [negative control] across-rho0 sparsity range (signal) ="
          f" {across_range:.4f} vs within-rho0 across-seed SD (placebo) = {within_sd:.4f}"
          f"  -> signal/noise = {snr:.1f}x")
    print("    (SNR >> 1 => sparsity is driven by the rho0 intervention, not init-RNG nuisance.)")
    H3_NEGCTRL={"across_rho0_range":across_range,"within_rho0_seed_sd":within_sd,"snr":snr}

    # ---------- H3 STRONG FALSIFICATION: label-shuffled placebo init ----------
    # (#4) A second, stronger negative control. `tai_negctrl` realises the SAME
    # init split for every rho0 label (init_tai_shuffled fixes the realised rho0
    # at 0.69 regardless of the label passed). We sweep the SAME rho0 grid/seeds
    # with this placebo method: if H3 is causal, the placebo's Spearman(label,
    # sparsity) must be ~0 while the real sweep is strongly positive. This is the
    # pre-registered falsification arm referenced in the manifest.
    if RUN_CFG.get("h3_negative_control",True):
        neg_rows=[]
        for rho0 in RHO0_GRID:
            for seed in _sweep_seeds("dim384"):
                rid=run_id("rho0neg","dim384","tai_negctrl",seed,rho0=rho0)
                if have_run(rid): neg_rows.append(load_run(rid)); continue
                if budget_exceeded():
                    notify("SESSION BOUNDARY (H3 negctrl)",f"rho0={rho0}",critical=True); raise SystemExit(0)
                rec=train_run("dim384",SIZES["dim384"],"tai_negctrl",seed,{"rho0":rho0},
                              lr=_lrstar("dim384","tai") if "_lrstar" in dir() else None)
                rec["rho0"]=rho0; rec["scale"]="dim384"; save_run(rid,rec); neg_rows.append(rec)
        # per-seed Spearman(label, sparsity) for the placebo
        nbys=defaultdict(list)
        for r in neg_rows: nbys[r["seed"]].append(r)
        neg_ps=[]
        for sd,rows in nbys.items():
            rows=sorted(rows,key=lambda r:r["rho0"]); xs=[r["rho0"] for r in rows]; ys=[r["final_sparsity"] for r in rows]
            if len(set(xs))>2:
                rr,_=stats.spearmanr(xs,ys)
                if not math.isnan(rr): neg_ps.append(rr)
        neg_ps=np.array(neg_ps)
        # (#4 POWER) per-seed SIGN TEST: in how many seeds is the REAL effect > the
        # placebo effect? A binomial sign test over 15 independent seeds is far
        # more powerful than one pooled n=6 Spearman.
        real_ps=per_seed_rho
        paired_seeds=[sd for sd in by_seed if sd in nbys]
        wins=0; npair=0
        rmap={sd:None for sd in by_seed}
        for sd,rows in by_seed.items():
            rows=sorted(rows,key=lambda r:r["rho0"])
            if len({r["rho0"] for r in rows})>2:
                rr,_=stats.spearmanr([r["rho0"] for r in rows],[r["final_sparsity"] for r in rows])
                rmap[sd]=rr
        for sd in paired_seeds:
            nrows=sorted(nbys[sd],key=lambda r:r["rho0"])
            if len({r["rho0"] for r in nrows})<=2 or rmap.get(sd) is None: continue
            nr,_=stats.spearmanr([r["rho0"] for r in nrows],[r["final_sparsity"] for r in nrows])
            if math.isnan(nr): continue
            npair+=1; wins+= int(rmap[sd] > nr)
        try:
            from scipy.stats import binomtest as _bt; sign_p=_bt(wins,npair,0.5).pvalue if npair else float("nan")
        except Exception:
            from scipy.stats import binom_test as _btb; sign_p=_btb(wins,npair,0.5) if npair else float("nan")
        print("H3 STRONG FALSIFICATION (label-shuffled placebo init):")
        print(f"    placebo mean per-seed Spearman = {np.nanmean(neg_ps):+.3f} (expected ~0)")
        print(f"    real > placebo in {wins}/{npair} seeds (sign-test p={sign_p:.4g})")
        H3_NEGCTRL.update({"placebo_mean_spearman":float(np.nanmean(neg_ps)) if len(neg_ps) else float("nan"),
                           "sign_test_wins":int(wins),"sign_test_n":int(npair),"sign_test_p":float(sign_p)})

    # ---------- H3 (#B): cross-scale consistency of the causal effect ----------
    def _per_seed_spearman(rows):
        bs=defaultdict(list)
        for r in rows: bs[r["seed"]].append(r)
        out=[]
        for sd,rr in bs.items():
            rr=sorted(rr,key=lambda r:r["rho0"]); xs=[r["rho0"] for r in rr]; ys=[r["final_sparsity"] for r in rr]
            if len(set(xs))>2:
                rho,_=stats.spearmanr(xs,ys)
                if not math.isnan(rho): out.append(rho)
        return np.array(out)
    H3_BY_SCALE={}
    for _sz in RHO0_SCALES:
        ps=_per_seed_spearman(SWEEP_BY_SCALE[_sz])
        H3_BY_SCALE[_sz]={"per_seed_mean":float(np.nanmean(ps)) if len(ps) else float("nan"),
                          "per_seed_sd":float(np.nanstd(ps,ddof=1)) if len(ps)>1 else float("nan"),
                          "n":int(len(ps)),"vals":ps.tolist()}
    print("H3 (#B) cross-scale causal effect (per-seed Spearman rho0->sparsity):")
    for _sz in RHO0_SCALES:
        h=H3_BY_SCALE[_sz]
        print(f"    {_sz}: mean={h['per_seed_mean']:+.3f} sd={h['per_seed_sd']:.3f} (n_seeds={h['n']})")
    if len(RHO0_SCALES)>=2:
        a=np.array(H3_BY_SCALE[RHO0_SCALES[0]]["vals"]); b=np.array(H3_BY_SCALE[RHO0_SCALES[-1]]["vals"])
        if len(a)>=2 and len(b)>=2:
            # do the two scales' causal effects differ? (Welch t); and is each > 0?
            t_d,p_d=stats.ttest_ind(a,b,equal_var=False)
            print(f"    scale-invariance test ({RHO0_SCALES[0]} vs {RHO0_SCALES[-1]}): "
                  f"Welch t={t_d:+.2f}, p={p_d:.3g}  "
                  f"(LARGE p => effect statistically indistinguishable across scale => "
                  f"control is scale-invariant)")
        H3_CROSS_SCALE={"scales":RHO0_SCALES,"by_scale":H3_BY_SCALE,
                        "invariance_p":float(p_d) if (len(a)>=2 and len(b)>=2) else float("nan")}
    else:
        H3_CROSS_SCALE={"scales":RHO0_SCALES,"by_scale":H3_BY_SCALE,"invariance_p":float("nan")}

    # ---------- H5: non-dominated Pareto frontier with bootstrap CI ----------
    byr=defaultdict(list)
    for s in SWEEP: byr[s["rho0"]].append(s)
    pts=[]
    for r in sorted(byr):
        g=byr[r]
        vl=np.array([x["val_loss"] for x in g]); sp=np.array([x["final_sparsity"] for x in g])
        pts.append({"rho0":r,"val_loss":float(vl.mean()),"val_loss_sd":float(vl.std(ddof=1)),
                    "sparsity":float(sp.mean()),"sparsity_sd":float(sp.std(ddof=1)),
                    "_vl":vl,"_sp":sp})
    # non-dominated set: a point is dominated if another has >= sparsity AND <= val_loss
    def _dominated(a,B):
        return any((b["sparsity"]>=a["sparsity"]-1e-9 and b["val_loss"]<=a["val_loss"]+1e-9
                    and (b["sparsity"]>a["sparsity"] or b["val_loss"]<a["val_loss"]))
                   for b in B if b is not a)
    frontier=[p for p in pts if not _dominated(p,pts)]
    print("H5 Pareto frontier (rho0 -> val_loss +/- sd, sparsity +/- sd); * = non-dominated:")
    for p in pts:
        star="*" if p in frontier else " "
        print(f"  {star} rho0 {p['rho0']:.2f} -> loss {p['val_loss']:.4f}+/-{p['val_loss_sd']:.4f}, "
              f"spars {p['sparsity']*100:.1f}+/-{p['sparsity_sd']*100:.1f}%")
    # rho0*: min mean val_loss among rho0 meeting the sparsity target; bootstrap CI
    elig=[p for p in pts if p["sparsity"]>=SPARS_TARGET] or pts
    RHO0_STAR=min(elig,key=lambda p:p["val_loss"])
    # bootstrap over seeds: how often is each rho0 the argmin-loss among eligible?
    rng=np.random.default_rng(GLOBAL_SEED+1)
    n_seed=len(SEEDS); win=defaultdict(int); B=4000
    rho_order=sorted(byr)
    for _ in range(B):
        idx=rng.integers(0,n_seed,n_seed)
        best=None;bestv=1e9
        for r in rho_order:
            g=byr[r]; vl=np.array([g[i]["val_loss"] for i in idx]).mean()
            sp=np.array([g[i]["final_sparsity"] for i in idx]).mean()
            if sp>=SPARS_TARGET and vl<bestv: bestv=vl;best=r
        if best is None:  # none meet target this resample -> global argmin
            for r in rho_order:
                g=byr[r]; vl=np.array([g[i]["val_loss"] for i in idx]).mean()
                if vl<bestv: bestv=vl;best=r
        win[best]+=1
    sel_stab={r:win[r]/B for r in rho_order}
    print(f"H5 rho0* (min loss, sparsity>={SPARS_TARGET:.0%}): rho0={RHO0_STAR['rho0']:.2f} "
          f"loss {RHO0_STAR['val_loss']:.4f} spars {RHO0_STAR['sparsity']*100:.1f}%")
    print("    selection stability (bootstrap P[rho0 is optimal]):",
          {f"{r:.2f}":round(sel_stab[r],3) for r in rho_order})
    # strip helper arrays before any serialization
    for p in pts: p.pop("_vl",None); p.pop("_sp",None)
    PARETO=pts; PARETO_FRONTIER=[p["rho0"] for p in frontier]
    H3_STATS={"cross_scale":H3_CROSS_SCALE,"pooled_spearman":float(rho_pool),"pooled_p":float(p_pool),
              "per_seed_mean":float(np.nanmean(per_seed_rho)) if len(per_seed_rho) else float("nan"),
              "per_seed_ci":ci,"per_seed_p":float(p_ps),"frac_monotone":frac_mono,
              "n_seeds":int(len(per_seed_rho))}
    H5_STATS={"rho0_star":RHO0_STAR["rho0"],"frontier":PARETO_FRONTIER,
              "selection_stability":sel_stab,"sparsity_target":SPARS_TARGET}
    # ---------- (R6) MECHANISM COMPARISON: is the INIT doing the work? ----------
    # The decisive attribution test. gamma_offset reaches the same rho0 by scaling
    # the threshold while keeping a standard init; TAI reaches it by shaping the
    # init while keeping gamma=mean|W|. Comparing them AT MATCHED rho0, paired by
    # seed, asks the question the rest of the sweep cannot: does TAI's initializer
    # contribute anything beyond putting the model at that operating point?
    #
    # Three pre-registered readouts, all reported regardless of which way they go:
    #  (1) CONTROL: does gamma_offset also move final sparsity with rho0? If yes,
    #      "rho0 is causally controllable" is a property of the THRESHOLD, and H3's
    #      contribution is the operating point, not TAI specifically.
    #  (2) QUALITY: paired val_loss difference at each rho0 (TAI - gamma_offset).
    #      Negative => TAI better.
    #  (3) The comparison is EXPLORATORY (not in the Holm family): it was added in
    #      revision, so it is reported with CIs and labelled as such rather than
    #      being smuggled into the confirmatory claims.
    MECH={}
    _go_rows=SWEEP_ARM_ROWS.get(("dim384","gamma_offset"),[])
    if _go_rows:
        # (1) control: per-seed Spearman(rho0, final_sparsity) for gamma_offset
        _go_by_seed=defaultdict(list)
        for r in _go_rows: _go_by_seed[r["seed"]].append(r)
        _go_rho=[]
        for sd,rows in _go_by_seed.items():
            rows=sorted(rows,key=lambda r:r["rho0"])
            xs=[r["rho0"] for r in rows]; ys=[r["final_sparsity"] for r in rows]
            if len(set(xs))>2:
                rr,_=stats.spearmanr(xs,ys)
                if not math.isnan(rr): _go_rho.append(rr)
        _go_rho=np.array(_go_rho,float)
        # (2) quality at matched rho0, paired by seed
        _tai_ix={(r["rho0"],r["seed"]):r for r in SWEEP}
        _go_ix ={(r["rho0"],r["seed"]):r for r in _go_rows}
        _keys=sorted(set(_tai_ix)&set(_go_ix))
        _d=np.array([_tai_ix[k]["val_loss"]-_go_ix[k]["val_loss"] for k in _keys],float)
        print("\n" + "-"*70)
        print("(R6) MECHANISM COMPARISON at dim384 — TAI vs gamma_offset (EXPLORATORY)")
        print("-"*70)
        if len(_go_rho):
            print(f"  gamma_offset sparsity control: per-seed mean Spearman(rho0,sparsity)"
                  f"={np.nanmean(_go_rho):+.3f} (n_seeds={len(_go_rho)})")
            print(f"  TAI          sparsity control: per-seed mean Spearman"
                  f"={np.nanmean(per_seed_rho):+.3f} (n_seeds={len(per_seed_rho)})")
            print("  => Both control rho0" if np.nanmean(_go_rho)>0.5 else
                  "  => Only TAI reliably controls rho0")
        if len(_d)>=2:
            # (R41) named _tstat, not _t: Cell 11 binds `import time as _t` at module
            # level and every cell shares one namespace, so `_t=` here would turn the
            # time module into a float for every later phase.
            _tstat,_p=stats.ttest_rel([_tai_ix[k]["val_loss"] for k in _keys],
                                      [_go_ix[k]["val_loss"] for k in _keys])
            rng2=np.random.default_rng(GLOBAL_SEED+7)
            _bs=[rng2.choice(_d,len(_d),replace=True).mean() for _ in range(5000)]
            _ci=(float(np.percentile(_bs,2.5)),float(np.percentile(_bs,97.5)))
            print(f"\n  val_loss difference (TAI - gamma_offset), paired over {len(_d)} (rho0,seed) cells:")
            print(f"    mean={_d.mean():+.4f} nats  95% CI [{_ci[0]:+.4f},{_ci[1]:+.4f}]  (paired t p={_p:.4g})")
            if _ci[0]<=0<=_ci[1]:
                print("    VERDICT: CI crosses zero -> the data CANNOT resolve which mechanism is")
                print("             better. The honest framing is that the OPERATING POINT is the")
                print("             contribution and gamma_offset is the simpler way to reach it;")
                print("             TAI's initializer is not shown to add value at this scale.")
            elif _d.mean()<0:
                print("    VERDICT: TAI better beyond the operating point -> the initializer itself")
                print("             carries value (threshold scaling perturbs the STE gradient scale")
                print("             permanently; TAI's effect decays as weights move). Report as the")
                print("             stronger result it is -- credible precisely because this arm could")
                print("             have won and did not.")
            else:
                print("    VERDICT: gamma_offset better -> the SIMPLER method wins. Report it; the")
                print("             paper's recommendation becomes threshold offset, not TAI init.")
            MECH={"n_cells":int(len(_d)),"mean_diff_tai_minus_gammaoffset":float(_d.mean()),
                  "ci":_ci,"paired_t_p":float(_p),
                  "gamma_offset_mean_spearman":(float(np.nanmean(_go_rho)) if len(_go_rho) else None),
                  "tai_mean_spearman":float(np.nanmean(per_seed_rho)) if len(per_seed_rho) else None,
                  "exploratory":True}
    else:
        print("\n[note] (R6) mechanism comparison skipped: no gamma_offset sweep rows "
              "(set RUN_CFG['sweep_arms'] to include it).")
    MECHANISM_COMPARISON=MECH

    notify("PHASE DONE: rho0 sweep",
           f"H3 per-seed rho={H3_STATS['per_seed_mean']:+.3f} mono={frac_mono:.2f} | rho0*={RHO0_STAR['rho0']}")


In [ ]:
# === CELL 12b: R2 — rho0* FEEDBACK INTO THE GRID (top scale, re-test H1/H2) ==
# Reviewer request R2: the main grid fixes rho0=0.75, but H5 may find a different
# optimum. Here we take RHO0_STAR from the sweep and run a `tai_star` method at
# the LARGEST scale (and dim384 for a paired mid-scale reference), then the stats
# cell compares tai_star vs the strongest baseline exactly like tai/tai_warmup.
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("RHO0_STAR", "RESULTS", "TARGET_LOSS"):
    print("[skip] rho0* feedback (R2): ARM_FULL_RUN off or prerequisites missing.")  # Run-All safe
else:
    import math,torch
    notify("PHASE START: rho0* feedback",f"rho0*={RHO0_STAR['rho0']}")
    STAR=float(RHO0_STAR["rho0"])
    # R2 re-tests the sweep-optimal rho0* at the TOP CONFIRMATORY scale (the
    # heaviest scale we actually run the full grid on) plus the primary scale,
    # so tai_star is paired against tai/tai_warmup on identical (scale,seed) cells.
    _conf=[s for s in SIZES if s in RUN_CFG.get("confirmatory_scales",list(SIZES))]
    _top_conf=max(_conf,key=lambda k:SIZES[k].dim) if _conf else max(SIZES,key=lambda k:SIZES[k].dim)
    _prim=RUN_CFG.get("confirmatory_scale","dim384")
    STAR_SIZES=list(dict.fromkeys([_top_conf,_prim]))
    for size in STAR_SIZES:
        cfg=SIZES[size]
        notify(f"BLOCK START: {size}/tai_star","")
        for seed in (SEEDS_FOR(size) if "SEEDS_FOR" in dir() else SEEDS):
            rid=run_id("main",size,"tai_star",seed)
            if have_run(rid): continue
            if budget_exceeded():
                notify("SESSION BOUNDARY (rho0* feedback)","resume next session",critical=True); raise SystemExit(0)
            rec=train_run(size,cfg,"tai_star",seed,{"rho0":STAR})  # tai_star starts with "tai" -> TAI init
            # (FIX) H1 must use THIS scale's reference loss, not the global
            # smallest-scale TARGET_LOSS (which would mis-scale large-model speed).
            _tgt=TARGET_LOSS_BY_SCALE[size] if ("TARGET_LOSS_BY_SCALE" in dir() and size in TARGET_LOSS_BY_SCALE) else TARGET_LOSS
            rec["steps_to_target"]=steps_to_target(rec["steps"],rec["loss"],_tgt)
            rec["rho0_used"]=STAR; save_run(rid,rec)
            notify(f"run done: main {size} tai_star s{seed}",
                   f"val {rec['val_loss']:.4f} | spars {rec['final_sparsity']*100:.1f}%",tag="run")
    # fold tai_star runs into RESULTS so the stats/figures cells see them
    for size in STAR_SIZES:
        for seed in (SEEDS_FOR(size) if "SEEDS_FOR" in dir() else SEEDS):
            rid=run_id("main",size,"tai_star",seed)
            if have_run(rid): RESULTS.append(load_run(rid))
    print(f"R2 rho0* feedback complete: tai_star (rho0={STAR}) on {STAR_SIZES}")
    notify("PHASE DONE: rho0* feedback",f"tai_star rho0={STAR} on {STAR_SIZES}")

In [ ]:
# === CELL 12c: R1 — CROSS-ARCHITECTURE GENERALIZATION (decoder_v2) ==========
# Reviewer request R1: is the TAI effect specific to one block design? We re-run
# the key contrast — strongest baseline (standard) vs tai_warmup — on a SECOND
# architecture family (GQA attention + SwiGLU FFN) at two representative scales.
# If the sign of (baseline - tai_warmup) matches decoder_v1, the effect transfers.
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("RESULTS", "SIZES_V2"):
    print("[skip] cross-architecture (R1): ARM_FULL_RUN off or prerequisites missing.")  # Run-All safe
else:
    import numpy as np,math
    from collections import defaultdict
    from scipy import stats
    notify("PHASE START: cross-architecture (decoder_v2)","")
    # (R13 CORRECTION) This used to run {standard, lsuv} and take the PER-SEED BEST
    # as "the strongest-of-set baseline", on the stated grounds that they are "two
    # distinct baseline families". They are not: both are zero-mean, normalised to
    # the same sigma, and land at rho0 0.6900 vs 0.6906 -- indistinguishable. Taking
    # a per-seed max over two draws of ONE family is not a stronger baseline, it is
    # a max-selection artefact that biases the baseline optimistically (E[max] >
    # E[single]) while advertising breadth the design does not have. Since H7 asks
    # only whether the SIGN of the TAI effect transfers, the honest comparator is
    # the pre-registered primary baseline used everywhere else, plus gamma_offset
    # -- the one arm that is genuinely a different family (it differs at the
    # operating point, not merely in weight distribution).
    _pb=RUN_CFG.get("primary_baseline","bitnet158")
    ARCH_METHODS=[(_pb,{"recipe":"bitnet"} if _pb=="bitnet158" else {}),
                  ("gamma_offset",{"rho0":0.75,"gamma_offset":True}),
                  ("tai_warmup",{"rho0":0.75,"gamma_warmup":240})]
    ARCH_RESULTS=[]
    # R1 tests TRANSFER, not a second full ladder: run two representative scales
    # (small + the confirmatory-primary) under the single-A100 budget, matching the
    # cell's stated design. Override via RUN_CFG["v2_scales"].
    _V2_DEFAULT=[s for s in ["dim256",RUN_CFG.get("confirmatory_scale","dim384")] if s in SIZES_V2]
    _V2_SCALES=[s for s in RUN_CFG.get("v2_scales",_V2_DEFAULT) if s in SIZES_V2]
    for size in _V2_SCALES:
        cfg=SIZES_V2[size]
        for method,opts in ARCH_METHODS:
            notify(f"BLOCK START: v2/{size}/{method}","")
            for seed in (SEEDS_FOR(size) if "SEEDS_FOR" in dir() else SEEDS):
                rid=run_id("arch2",size,method,seed)
                if have_run(rid): ARCH_RESULTS.append(load_run(rid)); continue
                if budget_exceeded():
                    notify("SESSION BOUNDARY (cross-arch)","resume next session",critical=True); raise SystemExit(0)
                rec=train_run(size,cfg,method,seed,opts); rec["arch"]="decoder_v2"
                save_run(rid,rec); ARCH_RESULTS.append(rec)
                notify(f"run done: arch2 {size} {method} s{seed}",f"val {rec['val_loss']:.4f}",tag="run")
    # direction-consistency test per scale: paired (standard - tai_warmup) on same seed
    print("R1 cross-architecture (decoder_v2) — TAI effect direction:")
    ARCH_SUMMARY={}
    for size in SIZES_V2:
        by=defaultdict(dict)
        for r in ARCH_RESULTS:
            if r["size"]==size: by[r["method"]][r["seed"]]=r["val_loss"]
        # (R13f) derive from what this cell actually ran, not from stale literals
        bcands=[k for k,_ in ARCH_METHODS if not k.startswith("tai") and k in by]
        _sf=(SEEDS_FOR(size) if "SEEDS_FOR" in dir() else SEEDS)
        seeds=[s for s in _sf if all(s in by.get(k,{}) for k in bcands) and s in by.get("tai_warmup",{})]
        if not seeds or not bcands: continue
        # R1 is a DIRECTION-TRANSFER test (exploratory): does sign(baseline - tai)
        # carry to a second block family? Compared against the locked-rho0 baseline
        # family (standard/lsuv, per-seed best); not the bitnet158 primary, so this
        # is reported as supportive evidence, not a confirmatory equivalence claim.
        b=np.array([min(by[k][s] for k in bcands) for s in seeds])  # per-seed best locked-rho0 baseline
        t=np.array([by["tai_warmup"][s] for s in seeds])
        diff=b-t  # >0 => TAI better, same convention as v1
        tp=stats.ttest_rel(t,b).pvalue if len(seeds)>=2 else float("nan")
        ARCH_SUMMARY[size]={"dLoss":float(diff.mean()),"t_p":float(tp),"n":len(seeds),
                            "sign":"TAI better" if diff.mean()>0 else "baseline better"}
        print(f"  {size} (v2): dLoss {diff.mean():+.4f} | t-p {tp:.4f} | {ARCH_SUMMARY[size]['sign']} (n={len(seeds)})")
    # cross-check vs v1 sign at the same nominal scales
    for size in SIZES_V2:
        _v1_names=tuple(k for k,_ in ARCH_METHODS)   # (R13f) same arms as v2
        v1=[r for r in RESULTS if r["size"]==size and r["method"] in _v1_names]
        if v1:
            _bn=[k for k,_ in ARCH_METHODS if not k.startswith("tai")]   # (R13f)
            b=min(np.mean([r["val_loss"] for r in v1 if r["method"]==k]) for k in _bn
                  if any(r["method"]==k for r in v1))
            t=np.mean([r["val_loss"] for r in v1 if r["method"]=="tai_warmup"])
            v1sign="TAI better" if (b-t)>0 else "baseline better"
            match = size in ARCH_SUMMARY and ARCH_SUMMARY[size]["sign"]==v1sign
            print(f"  {size}: v1 sign='{v1sign}' vs v2 sign='{ARCH_SUMMARY.get(size,{}).get('sign','NA')}' -> "
                  f"{'CONSISTENT' if match else 'DIFFERS'}")
    notify("PHASE DONE: cross-architecture",f"{len(ARCH_RESULTS)} v2 runs")

    # ---- EXTERNAL-VALIDITY SUMMARY: sign consistency over (architecture x scale) ----
    # Pool direction evidence across BOTH axes actually varied: the v1/v2 block
    # family and the scale ladder. Honest single-A100 external-validity statement:
    # "the effect points the same way in K of N independent (arch,scale) cells".
    # Recompute-only (no new training).
    cells_dir=[]
    for size in ARCH_SUMMARY:
        cells_dir.append((f"v2:{size}",ARCH_SUMMARY[size]["dLoss"]>0))
    from collections import defaultdict as _dd2
    _v1=_dd2(dict)
    for r in RESULTS:
        if r["method"] in tuple(k for k,_ in ARCH_METHODS)+("bitnet158",):   # (R13f)
            _v1[r["size"]].setdefault(r["method"],[]).append(r["val_loss"])
    for size,mm in _v1.items():
        bcand=[k for k in ([RUN_CFG.get("primary_baseline","bitnet158")]
                           +[k for k,_ in ARCH_METHODS if not k.startswith("tai")])
               if k in mm]
        bcand=list(dict.fromkeys(bcand))   # (R13f) de-dup, config-driven
        if "tai_warmup" in mm and bcand:
            b=min(np.mean(mm[k]) for k in bcand); t=np.mean(mm["tai_warmup"])
            cells_dir.append((f"v1:{size}",(b-t)>0))
    if cells_dir:
        # (R41 FIX) `nn` SHADOWED torch.nn AT MODULE LEVEL.
        # This cell binds the count of cells to the name `nn`, which is the global
        # alias for torch.nn established in Cell 6. Notebook cells share one
        # namespace, so from this line onward torch.nn WAS AN INTEGER: every later
        # phase that constructs a model died with
        #     AttributeError: 'int' object has no attribute 'Embedding'
        # in LM.__init__. The width-control ladder was the first phase to run
        # after this cell, so that is where it surfaced -- far from the cause, and
        # looking like a model bug rather than a name collision. Renamed to
        # _n_cells; nothing else about the test changes.
        kk=sum(1 for _,sg in cells_dir if sg); _n_cells=len(cells_dir)
        try:
            from scipy.stats import binomtest as _bt2; _p=_bt2(kk,_n_cells,0.5).pvalue
        except Exception:
            from scipy.stats import binom_test as _bt2b; _p=_bt2b(kk,_n_cells,0.5)
        EXT_VALIDITY={"cells":cells_dir,"n_consistent":kk,"n_total":_n_cells,"binom_p":float(_p)}
        print(f"\nEXTERNAL VALIDITY (direction): TAI better in {kk}/{_n_cells} independent "
              f"(architecture x scale) cells, binomial p={_p:.4f}")
        print("    cells:",", ".join(f"{lab}{'+' if sg else '-'}" for lab,sg in cells_dir))
    else:
        EXT_VALIDITY=None

In [ ]:
# === CELL 13: FORMAL STATISTICS (H1-H4) + BOOTSTRAP CI + HOLM + DOWNSTREAM + SCALE ==
# Full-rigor analysis. For each size and TAI variant vs the stronger baseline:
#  - paired t-test (primary), Wilcoxon (support), Cohen's d with bootstrap CI;
#  - effect size (Cohen's d) with a nonparametric bootstrap CI as the primary
#    honest summary of magnitude+uncertainty (post-hoc power is intentionally
#    NOT reported: computing power from the observed effect is circular and is
#    discouraged in the methodology literature; CIs convey the same information
#    without that fallacy);
#  - Holm-Bonferroni across the family of comparisons (less conservative than
#    plain Bonferroni); H4 via Levene. Then a SCALE-TREND regression of the TAI
#    effect vs log-params across the scale ladder (mixed-effects + per-scale sign test),
#    and downstream-accuracy comparisons (ARC-Easy + HellaSwag, loglik mode).
#    With n=15 we report effect sizes and bootstrap CIs rather than power, and
#    state plainly when a CI crosses zero (i.e. the data cannot resolve the sign).
if not _require("RESULTS"):
    pass  # prerequisites missing -> skip this phase (see message above)
else:
    import numpy as np,math
    from scipy import stats
    from collections import defaultdict
    def ci95(x):
        x=np.array(x,float); m=x.mean(); s=x.std(ddof=1) if len(x)>1 else 0.0
        return m,1.96*s/math.sqrt(len(x)),s
    def cohend(a,b):
        a,b=np.array(a,float),np.array(b,float)
        sp=math.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
        return (a.mean()-b.mean())/sp if sp>0 else 0.0
    def bootstrap_ci(diffs,n=5000,seed=0):
        r=np.random.default_rng(seed); diffs=np.asarray(diffs,float)
        bs=[r.choice(diffs,len(diffs),replace=True).mean() for _ in range(n)]
        return float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))

    def tost_equivalence(a,b,margin):
        # Two One-Sided Tests for paired data: is mean(a-b) within +/-margin?
        # Returns (equivalent_bool, p_tost). Equivalent if BOTH one-sided p<0.05.
        import numpy as _np
        d=_np.asarray(a,float)-_np.asarray(b,float); n=len(d)
        if n<2: return False,float("nan")
        m=d.mean(); s=d.std(ddof=1)/math.sqrt(n)
        if s==0: return (abs(m)<margin),0.0
        from scipy import stats as _st
        t_lo=(m+margin)/s; p_lo=1-_st.t.cdf(t_lo,n-1)   # H0: diff <= -margin
        t_hi=(m-margin)/s; p_hi=_st.t.cdf(t_hi,n-1)     # H0: diff >= +margin
        p=max(p_lo,p_hi); return (p<0.05),float(p)

    def holm(pairs,alpha=0.05):
        # pairs: list of (label,p); returns dict label->reject
        order=sorted(range(len(pairs)),key=lambda i:pairs[i][1]); m=len(pairs); out={}
        stop=False
        for rank,idx in enumerate(order):
            lab,p=pairs[idx]
            if not stop and p<alpha/(m-rank): out[lab]=True
            else: out[lab]=False; stop=True
        return out
    A=RUN_CFG["bonferroni_alpha"]; ANALYSIS={}
    _CONF=set(RUN_CFG.get("confirmatory_scales",list(SIZES)))
    _PRIMARY_SCALE=RUN_CFG.get("confirmatory_scale")
    # (R40 FIX) ITERATE OVER SCALES THAT WERE ACTUALLY RUN, NOT OVER SIZES.
    # SIZES is the full 12-point ladder INCLUDING dim2048, which is opt-in and is
    # not in SCALES_TO_RUN. For such a scale rows is empty, so summ stays empty,
    # and the primary-baseline fallback called min() on an empty sequence:
    #     ValueError: min() iterable argument is empty
    # -- after every completed scale had already been analysed and printed. Any
    # scale with no records cannot be analysed, so it is skipped explicitly and
    # the skip is announced rather than silently passed over.
    H2_TOKEN_EQUAL={}   # (R45) companion results, keyed "scale:method"
    _analysable=[s_ for s_ in SIZES if any(r["size"]==s_ for r in RESULTS)]
    _skipped=[s_ for s_ in SIZES if s_ not in _analysable]
    if _skipped:
        print(f"  [skip] no runs recorded for {', '.join(_skipped)} -- not analysed.")
    for size in _analysable:
        _tag=("PRIMARY" if size==_PRIMARY_SCALE else "confirmatory" if size in _CONF else "EXPLORATORY")
        rows=[r for r in RESULTS if r["size"]==size]; by=defaultdict(list)
        if rows: print(f"  [{_tag} scale]")
        for r in rows: by[r["method"]].append(r)
        print(f"\n===== {size} ====="); summ={}
        for m,rs in by.items():
            vl=[r["val_loss"] for r in rs]; sp=[r["final_sparsity"] for r in rs]
            # (FIX-6) seed-indexed loss + plateau flag, so H2 can restrict to the
            # PAIRED set of seeds that are plateau-verified in BOTH arms (H2 is a
            # converged-quality claim; mid-training runs must not enter it). See the R3 note
# below: the strength of the word "converged" is decided by Cell 11c, not assumed.
            _vl_by_seed={r["seed"]:r["val_loss"] for r in rs}
            _plat_by_seed={r["seed"]:bool(r.get("plateau_ok",False)) for r in rs}
            s2=[r.get("steps_to_target",float('nan')) for r in rs]
            nt=[r.get("nexttoken_acc",float('nan')) for r in rs]  # copytask proxy dropped (not method-discriminating)
            arc=[r.get("arc_easy_acc",float('nan')) for r in rs]; hsw=[r.get("hellaswag_acc",float('nan')) for r in rs]
            arcn=[r.get("arc_easy_acc_norm",float('nan')) for r in rs]; hswn=[r.get("hellaswag_acc_norm",float('nan')) for r in rs]
            mean,ci,std=ci95(vl); smean,_,_=ci95(sp); tmean,_,_=ci95(s2); ntm,_,_=ci95(nt)
            arcm=np.nanmean(arc) if np.any(~np.isnan(arc)) else float("nan")
            hswm=np.nanmean(hsw) if np.any(~np.isnan(hsw)) else float("nan")
            summ[m]={"val_mean":mean,"val_ci":ci,"val_std":std,"spars_mean":smean,"s2t_mean":tmean,
                     "nt_mean":ntm,"arc_mean":arcm,"hsw_mean":hswm,
                     "_vl":vl,"_s2t":s2,"_nt":nt,"_arc":arc,"_hsw":hsw,"_arcn":arcn,"_hswn":hswn,  # (R18) removed "_cp":cp -- cp was never defined; this NameError crashed the ENTIRE primary-stats cell (H1-H4)
                     "_vl_by_seed":_vl_by_seed,"_plat_by_seed":_plat_by_seed}
            print(f"  {m:13s} loss {mean:.4f}+/-{ci:.4f} std {std:.4f} spars {smean*100:.1f}% s2t {tmean:.0f} nt-acc {ntm:.3f}")   # (R18) removed 'copy {cpm}' -- cpm was never computed
        # (FIX-3) PRE-REGISTERED PRIMARY BASELINE. We do NOT pick the per-scale
        # weakest-looking ternary baseline (winner's curse). The confirmatory
        # comparison is frozen to an INDEPENDENT published recipe.
        PRIMARY=RUN_CFG["primary_baseline"]
        SECONDARY=[b for b in RUN_CFG["secondary_baselines"] if b in summ]
        if PRIMARY not in summ:
            print(f"  [warn] primary baseline {PRIMARY} absent at {size}; "
                  f"falling back to data-min (flagged EXPLORATORY).")
            present=[b for b in ([PRIMARY]+SECONDARY) if b in summ] or list(summ)
            if not present:
                # (R40) unreachable now that empty scales are skipped, but a
                # baseline cannot be chosen from nothing and this must never be a
                # crash that discards the analysis of every scale before it.
                print(f"  [skip] {size}: no methods with data; scale not analysed.")
                continue
            base=min(present,key=lambda k:summ[k]["val_mean"]); base_kind="EXPLORATORY(min)"
        else:
            base=PRIMARY; base_kind="PRE-REGISTERED(bitnet158)"
        # data-chosen strongest baseline kept ONLY as a secondary sensitivity label
        _all_tern=[b for b in ([PRIMARY]+SECONDARY) if b in summ]
        base_min=min(_all_tern,key=lambda k:summ[k]["val_mean"]) if _all_tern else base
        print(f"  primary baseline: {base} [{base_kind}] | secondary-min baseline: {base_min}")
        # (FIX-4) DATA-DEFINED TOST MARGIN = frac * (ternary_primary - fp16) gap.
        if "fp16_control" in summ:
            _gap=summ[base]["val_mean"]-summ["fp16_control"]["val_mean"]
            tost_margin=max(RUN_CFG["tost_margin_frac"]*abs(_gap),1e-4)
            _margin_src=f"{RUN_CFG['tost_margin_frac']:.0%} of {_gap:+.4f}-nat ternary-vs-fp16 gap"
        else:
            tost_margin=RUN_CFG["tost_margin"]; _margin_src="fallback constant (no fp16_control)"
        print(f"  H2 TOST margin = {tost_margin:.4f} nats  [{_margin_src}]")
        def _paired_plateau(mA,mB):
            # common seeds, plateau-verified in BOTH arms; returns aligned arrays
            sA,sB=summ[mA],summ[mB]
            seeds=[s for s in sA["_vl_by_seed"]
                   if s in sB["_vl_by_seed"] and sA["_plat_by_seed"].get(s) and sB["_plat_by_seed"].get(s)]
            a=np.array([sA["_vl_by_seed"][s] for s in seeds],float)
            b=np.array([sB["_vl_by_seed"][s] for s in seeds],float)
            return a,b,seeds
        pair_ps=[]
        for m in ["tai","tai_warmup","tai_star"]:  # +R2 rho0* variant
            if m in summ:
                # H1/H4 use all seeds (speed + variance are not converged-only claims);
                # H2 uses the PAIRED PLATEAU-VERIFIED subset. NOTE (R3): "plateau-verified" means
# the val-trace slope is indistinguishable from zero -- i.e. NOT RESOLVABLY still
# improving at this noise level. Whether that licenses the phrase "converged
# quality" is settled empirically by Cell 11c (2x token budget), NOT here; if
# BUDGET_CHECK["h2_wording_supported"] is False, every "converged" below must be
# read as "at the 20M-token budget". The manifest carries the operative wording.
                diffs=np.array(summ[base]["_vl"])-np.array(summ[m]["_vl"])  # >0 => TAI better
                tp=stats.ttest_rel(summ[m]["_vl"],summ[base]["_vl"]).pvalue
                wp=stats.wilcoxon(summ[m]["_vl"],summ[base]["_vl"]).pvalue if len(summ[m]["_vl"])>=6 else float('nan')
                d=cohend(summ[base]["_vl"],summ[m]["_vl"]); lo,hi=bootstrap_ci(diffs)
                pair_ps.append((f"{size}:{m}",tp))
                aH2,bH2,seedsH2=_paired_plateau(m,base)
                n_plat=len(seedsH2)
                if n_plat>=2:
                    eqv,ptost=tost_equivalence(aH2.tolist(),bH2.tolist(),tost_margin)
                    print(f"  H2-TOST {m} vs {base}: margin +/-{tost_margin:.4f} nats over {n_plat}/{len(summ[m]['_vl'])} "
                          f"plateau-paired seeds -> {'EQUIVALENT' if eqv else 'not shown equivalent'} (p_TOST {ptost:.4f})")
                else:
                    print(f"  H2-TOST {m} vs {base}: SKIPPED (only {n_plat} plateau-paired seeds; "
                          f"converged-quality claim not testable -> reported as unresolved)")
                    # ============================================================
                    # (R45) H2 COMPANION TEST — TOKEN-EQUAL, NOT CONVERGED.
                    # ------------------------------------------------------------
                    # The pre-registered H2 is an equivalence claim about CONVERGED
                    # quality, tested over plateau-verified paired seeds. With 0 of
                    # 760 runs reaching plateau it is untestable, and "untestable"
                    # is the honest verdict -- printed above, and NOT overwritten
                    # here. What follows is a SEPARATE, clearly-labelled test of the
                    # weaker claim the data can actually support: equivalence at a
                    # FIXED 20M-token budget, over all paired seeds.
                    #
                    # Why this is not a post-hoc rescue. The re-statement is forced
                    # by a MEASUREMENT (the budget does not reach plateau, confirmed
                    # independently by Cell 11c's 2x-budget check), not by an
                    # outcome, and it does not favour the method: on the completed
                    # grid it rejects equivalence for BOTH arms, reporting tai as
                    # WORSE and tai_warmup as BETTER by roughly ten times the
                    # margin. A substitution that makes the headline arm fail is not
                    # the shape p-hacking takes. It is recorded as a companion so a
                    # reader can see exactly which test was pre-registered and which
                    # was added, and the manifest carries both.
                    try:
                        _aTE=np.array(summ[base]["_vl"],float); _bTE=np.array(summ[m]["_vl"],float)
                        if len(_aTE)==len(_bTE) and len(_aTE)>=3:
                            _eq,_pte=tost_equivalence(_aTE.tolist(),_bTE.tolist(),tost_margin)
                            _dTE=_aTE-_bTE; _nTE=len(_dTE)
                            _seTE=_dTE.std(ddof=1)/max(np.sqrt(_nTE),1e-12)
                            _tc=stats.t.ppf(0.95,_nTE-1)
                            _loTE,_hiTE=_dTE.mean()-_tc*_seTE,_dTE.mean()+_tc*_seTE
                            _verd=("EQUIVALENT" if _eq else
                                   "BETTER than baseline" if _loTE>tost_margin else
                                   "WORSE than baseline" if _hiTE<-tost_margin else
                                   "inconclusive")
                            print(f"  H2-companion (TOKEN-EQUAL, not pre-registered) {m} vs {base}: "
                                  f"dLoss {_dTE.mean():+.4f} 90%CI[{_loTE:+.4f},{_hiTE:+.4f}] "
                                  f"margin +/-{tost_margin:.4f} -> {_verd}")
                            H2_TOKEN_EQUAL[f"{size}:{m}"]={
                                "baseline":base,"n_seeds":int(_nTE),
                                "d_loss_mean":float(_dTE.mean()),
                                "ci90":[float(_loTE),float(_hiTE)],
                                "margin":float(tost_margin),"tost_equivalent":bool(_eq),
                                "p_tost":float(_pte),"verdict":_verd,
                                "note":("companion to the pre-registered H2, which is untestable "
                                        "because no run reached plateau; this tests equivalence at "
                                        "a fixed token budget, not at convergence")}
                    except Exception as _e2:
                        print(f"  H2-companion {m}: could not compute ({str(_e2)[:60]})")
                sign_resolved = (lo>0) or (hi<0)
                print(f"  H2 {m} vs {base}: dLoss {diffs.mean():+.4f} bootCI[{lo:+.4f},{hi:+.4f}] {'(sign resolved)' if sign_resolved else '(CI crosses 0 -> sign unresolved)'} | t-p {tp:.4f} | wilcoxon(support) {wp:.4f} | d {d:+.2f}")
                # SECONDARY sensitivity: same contrast vs the data-min baseline (EXPLORATORY, no FWER)
                if base_min!=base:
                    d2=np.array(summ[base_min]["_vl"])-np.array(summ[m]["_vl"]); lo2,hi2=bootstrap_ci(d2)
                    print(f"    [exploratory] {m} vs data-min {base_min}: dLoss {d2.mean():+.4f} bootCI[{lo2:+.4f},{hi2:+.4f}]")
                tp1=stats.ttest_rel(summ[m]["_s2t"],summ[base]["_s2t"]).pvalue; d1=cohend(summ[base]["_s2t"],summ[m]["_s2t"])
                print(f"  H1 {m} vs {base}: dSteps {np.mean(summ[base]['_s2t'])-np.mean(summ[m]['_s2t']):+.0f} | t-p {tp1:.4f} | d {d1:+.2f}")
                lev=stats.levene(summ[m]["_vl"],summ[base]["_vl"]).pvalue
                print(f"  H4 {m}: Levene p {lev:.4f} (std {summ[m]['val_std']:.4f} vs {summ[base]['val_std']:.4f})")
                # downstream transfer: internal proxies + real lm-eval (loglik mode)
                ntp=stats.ttest_rel(summ[m]["_nt"],summ[base]["_nt"]).pvalue
                print(f"  downstream {m}: nt-acc {summ[m]['nt_mean']:.3f} vs {summ[base]['nt_mean']:.3f} (t-p {ntp:.4f})")
                def _paired_acc(tag,key,mk,bk):
                    a=np.array(summ[m][key],float); b=np.array(summ[base][key],float)
                    msk=~(np.isnan(a)|np.isnan(b))
                    if msk.sum()>=2 and (a[msk]-b[msk]).std(ddof=1)>0:
                        pp=stats.ttest_rel(a[msk],b[msk]).pvalue
                        lo,hi=bootstrap_ci((a[msk]-b[msk]).tolist())
                        print(f"  lm-eval {tag} {m}: {np.nanmean(a):.3f} vs {np.nanmean(b):.3f} | dAcc {(np.nanmean(a)-np.nanmean(b)):+.3f} bootCI[{lo:+.3f},{hi:+.3f}] | t-p {pp:.4f}")
                    elif msk.sum()>=1:
                        print(f"  lm-eval {tag} {m}: {np.nanmean(a):.3f} vs {np.nanmean(b):.3f} (insufficient spread for test)")
                    else:
                        print(f"  lm-eval {tag} {m}: unavailable (synthetic corpus or tasks not loaded)")
                _paired_acc("ARC-Easy(acc)","_arc",None,None)
                _paired_acc("ARC-Easy(acc_norm)","_arcn",None,None)
                _paired_acc("HellaSwag(acc)","_hsw",None,None)
                _paired_acc("HellaSwag(acc_norm)","_hswn",None,None)
        if pair_ps:
            # (FIX-5) Holm-Bonferroni over the PRE-REGISTERED primary val-loss
            # family only (TAI variants vs the frozen primary baseline). Secondary
            # / exploratory contrasts above are NOT in this family and carry no
            # family-wise correction (labelled [exploratory]).
            hh=holm(pair_ps,alpha=0.05); print("  Holm-Bonferroni (CONFIRMATORY val-loss family):",{k:('reject' if v else 'retain') for k,v in hh.items()})
        if "fp16_control" in summ:
            print(f"  FP16 gap: {base} is {summ[base]['val_mean']-summ['fp16_control']['val_mean']:+.4f} above FP16")
        # record TAI effect for the scale regression (effect = baseline - tai_warmup)
        ANALYSIS[size]=summ
    # H6 SCALE MODEL. The robust, single-A100-defensible claim is SIGN-CONSISTENCY
    # of the effect direction across scales -- NOT a significant slope (which a
    # mid-scale ladder often cannot resolve). We therefore report BOTH: (i) a
    # per-scale sign test (how many scales show effect>0, with a binomial p), as
    # the PRIMARY H6 readout; and (ii) the mixed-effects slope vs log-params as a
    # secondary 'does it narrow?' diagnostic. Effect is paired within seed using
    # the PRE-REGISTERED primary baseline (FIX-3), with the data-min baseline kept
    # only as a sensitivity check. Scales = confirmatory (full grid) + exploratory
    # (reduced-seed ladder points), each point labelled by its set.
    _PRIMARY=RUN_CFG["primary_baseline"]
    _CONF=set(RUN_CFG["confirmatory_scales"]); _EXPL=set(RUN_CFG["exploratory_scales"])
    _SCALE_ORDER=[s for s in SIZES if s in (_CONF|_EXPL)]
    SCALE_ROWS=[]  # (log_params, seed, effect, scale, set)
    PER_SCALE_EFF={}
    for size in _SCALE_ORDER:
        rows=[r for r in RESULTS if r["size"]==size]; bym=defaultdict(dict)
        for r in rows: bym[r["method"]][r["seed"]]=r["val_loss"]
        if "tai_warmup" not in bym: continue
        # PRIMARY baseline if present, else data-min over available ternary baselines
        if _PRIMARY in bym: base_keys=[_PRIMARY]; _bkind="primary"
        else:
            base_keys=[k for k in ("standard","lsuv","sparse_sched") if k in bym]; _bkind="datamin"
        if not base_keys: continue
        _set="conf" if size in _CONF else "expl"
        effs=[]
        for seed in RUN_CFG["seeds"]:
            bvals=[bym[k][seed] for k in base_keys if seed in bym[k]]
            if not bvals or seed not in bym["tai_warmup"]: continue
            eff=min(bvals)-bym["tai_warmup"][seed]  # >0 => TAI better
            SCALE_ROWS.append((math.log(n_params(SIZES[size])),seed,eff,size,_set))
            effs.append(eff)
        if effs: PER_SCALE_EFF[size]=(float(np.mean(effs)),_set,_bkind,len(effs))
    # (i) PRIMARY H6: per-scale sign consistency
    import numpy as _np
    _means=[(s,v[0],v[1]) for s,v in PER_SCALE_EFF.items()]
    n_pos=sum(1 for _,e,_ in _means if e>0); n_tot=len(_means)
    try:
        from scipy.stats import binomtest as _bt; _signp=_bt(n_pos,n_tot,0.5).pvalue if n_tot else float("nan")
    except Exception:
        from scipy.stats import binom_test as _bt0; _signp=_bt0(n_pos,n_tot,0.5) if n_tot else float("nan")
    print(f"\nH6 (PRIMARY) sign-consistency: {n_pos}/{n_tot} scales show TAI effect>0 "
          f"(binomial p={_signp:.4f}). Per scale:")
    for s,e,st in sorted(_means,key=lambda z:n_params(SIZES[z[0]])):
        print(f"    {s:8s}[{st}] {n_params(SIZES[s])/1e6:6.1f}M -> mean effect {e:+.4f} {'(TAI better)' if e>0 else '(baseline better)'}")
    print(f"\nH6 (SECONDARY) mixed-effects slope over {n_tot} scales, {len(SCALE_ROWS)} paired points:")
    # ========================================================================
    # (R43) H6 SLOPE — THE MIXED MODEL WAS SINGULAR, AND THE OBVIOUS REPAIR IS
    # THE WRONG ONE.
    # ------------------------------------------------------------------------
    # The pre-registered estimator was mixedlm(eff ~ logp, groups=seed). On the
    # completed grid it failed with "Random effects covariance is singular" and
    # fell through to a descriptive OLS that printed a slope WITH NO INTERVAL --
    # a point estimate with no uncertainty is not a result, and H6 was effectively
    # unreported. Cause: the seed-level random intercept has almost no variance to
    # estimate (between-seed SD of the effect is 0.017 against a between-scale SD
    # of 0.046), so the covariance collapses to the boundary.
    #
    # The tempting repair -- OLS with SEs clustered on seed -- is worse than the
    # failure, because it answers the wrong question. H6 asks whether the effect
    # VARIES WITH SCALE, and the unit of that question is the SCALE, of which there
    # are 11. Clustering on seed treats the 11 scales as independent replicates of
    # a single line and ignores scale-level residual structure, which here is as
    # large as the seed noise (scale-level residual SD 0.042 vs within-scale seed
    # SD 0.030). The consequence is not cosmetic:
    #     cluster on seed  : slope -0.0126  CI [-0.0181, -0.0071]  EXCLUDES zero
    #     cluster on scale : slope -0.0126  CI [-0.0236, -0.0016]  excludes zero
    #     scale-mean OLS   : slope -0.0127  CI [-0.0292, +0.0037]  SPANS zero
    # Same point estimate, opposite conclusions. Choosing the narrowest interval
    # here would manufacture a scale-dependence finding out of an analysis choice.
    #
    # We therefore report the CONSERVATIVE estimator as primary -- regression on
    # the 11 scale means, where n is the number of scales, which is what H6 is
    # actually about -- and carry the other two as sensitivity analyses, printed
    # side by side so any disagreement is visible rather than buried. All three
    # go into the manifest. If they disagree, the paper says the magnitude is not
    # resolved; it does not pick the one it likes.
    SCALE_SLOPE=None; SCALE_SLOPE_CI=None; SCALE_SLOPE_METHOD=None; SCALE_SLOPE_ALL={}
    def _fit_scale_slope(rows):
        out={}
        try:
            import pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
            df=pd.DataFrame(rows,columns=["logp","seed","eff","scale","set"])
            if df["logp"].nunique()<3: return out
            # PRIMARY: one point per scale; n = number of scales.
            g=df.groupby("scale").agg(logp=("logp","first"),eff=("eff","mean"))
            if len(g)>=3:
                m=sm.OLS(g["eff"],sm.add_constant(g["logp"])).fit()
                ci=m.conf_int().loc["logp"]
                out["scale_mean_ols"]={"slope":float(m.params["logp"]),
                    "ci":[float(ci[0]),float(ci[1])],"p":float(m.pvalues["logp"]),
                    "n_units":int(len(g)),"unit":"scale"}
            # SENSITIVITY: all points, SEs clustered on scale.
            try:
                m2=smf.ols("eff ~ logp",df).fit(cov_type="cluster",
                                                cov_kwds={"groups":df["scale"]})
                ci2=m2.conf_int().loc["logp"]
                out["cluster_by_scale"]={"slope":float(m2.params["logp"]),
                    "ci":[float(ci2[0]),float(ci2[1])],"p":float(m2.pvalues["logp"]),
                    "n_units":int(df["scale"].nunique()),"unit":"scale"}
            except Exception: pass
            # SENSITIVITY: all points, SEs clustered on seed (the anticonservative one).
            try:
                m3=smf.ols("eff ~ logp",df).fit(cov_type="cluster",
                                                cov_kwds={"groups":df["seed"]})
                ci3=m3.conf_int().loc["logp"]
                out["cluster_by_seed"]={"slope":float(m3.params["logp"]),
                    "ci":[float(ci3[0]),float(ci3[1])],"p":float(m3.pvalues["logp"]),
                    "n_units":int(df["seed"].nunique()),"unit":"seed (anticonservative)"}
            except Exception: pass
            # SENSITIVITY: the pre-registered mixed model, if it converges at all.
            try:
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    mm=smf.mixedlm("eff ~ logp",df,groups=df["seed"]).fit(reml=True,method="lbfgs")
                cim=mm.conf_int().loc["logp"]
                out["mixedlm_seed_prereg"]={"slope":float(mm.params["logp"]),
                    "ci":[float(cim[0]),float(cim[1])],"p":float(mm.pvalues["logp"]),
                    "n_units":int(df["seed"].nunique()),"unit":"seed"}
            except Exception as _me:
                out["mixedlm_seed_prereg"]={"failed":str(_me)[:80]}
        except Exception as _e:
            out["error"]=str(_e)[:120]
        return out

    SCALE_SLOPE_ALL=_fit_scale_slope(SCALE_ROWS)
    _prim=SCALE_SLOPE_ALL.get("scale_mean_ols")
    if _prim:
        SCALE_SLOPE=_prim["slope"]; SCALE_SLOPE_CI=tuple(_prim["ci"])
        SCALE_SLOPE_METHOD="scale_mean_ols"
        _spans=_prim["ci"][0]<=0<=_prim["ci"][1]
        print(f"    PRIMARY (regression on {_prim['n_units']} scale means): "
              f"slope {_prim['slope']:+.4f} 95% CI [{_prim['ci'][0]:+.4f},{_prim['ci'][1]:+.4f}] "
              f"p={_prim['p']:.4f}")
        print(f"      => {'CI SPANS ZERO: direction is scale-consistent but the effect is NOT'  if _spans else 'CI excludes zero: the effect IS'} "
              f"resolved as scale-varying.")
        print("    sensitivity (same slope, different unit of analysis):")
        for k in ("cluster_by_scale","cluster_by_seed","mixedlm_seed_prereg"):
            d=SCALE_SLOPE_ALL.get(k)
            if not d: continue
            if "failed" in d: print(f"      {k:22s} FAILED: {d['failed']}"); continue
            print(f"      {k:22s} {d['slope']:+.4f} CI[{d['ci'][0]:+.4f},{d['ci'][1]:+.4f}] "
                  f"n={d['n_units']} ({d['unit']})")
        _signs={k:(v['ci'][0]<=0<=v['ci'][1]) for k,v in SCALE_SLOPE_ALL.items() if 'ci' in v}
        if len(set(_signs.values()))>1:
            print("    !! ESTIMATORS DISAGREE on whether the CI covers zero. The primary "
                  "(scale-level) reading governs;\n       the paper must report the "
                  "magnitude as UNRESOLVED and state that the seed-clustered\n       "
                  "interval is anticonservative for a scale-level question.")
    else:
        print("    scale-slope could not be estimated:",SCALE_SLOPE_ALL.get("error","insufficient scales"))
    SCALE_REG={"points":SCALE_ROWS,"slope":SCALE_SLOPE,"slope_ci":SCALE_SLOPE_CI,
               "slope_estimator":SCALE_SLOPE_METHOD,"slope_all_estimators":SCALE_SLOPE_ALL,
               "note":("primary estimator regresses on scale MEANS because H6 asks about "
                       "variation across scales and the unit of that question is the scale; "
                       "seed-clustered intervals are anticonservative here (see R43)")}
    print("\nNOTE: n=15 per cell. We report effect sizes with bootstrap CIs and flag")
    print("when a CI crosses zero (sign unresolved). Claims are restricted to the")
    print("tested regime (<=50M+ non-embed params, 20M tokens, GPT-2 BPE); broader")
    print("scale is explicitly out of scope and listed under Limitations.")


In [ ]:
# === CELL 13a3: (#1) SCALING-LAW ANALYSIS OF THE TAI EFFECT (recompute-only) ===
# External-validity argument WITHOUT training a billion-param model: instead of
# treating the 9 scales as disconnected points, we model the TAI effect as a
# FUNCTION of scale and ask the question a reviewer actually cares about --
# "does the advantage survive as models grow?". Three outputs:
#   (i)  trend of effect (strongest-baseline_loss - tai_warmup_loss, paired per
#        seed) vs log(non-embed params), with bootstrap CI on the slope;
#   (ii) a NARROWING test: does the gap shrink toward zero with scale? (the
#        pessimistic null a skeptic assumes) -- reported as slope sign + CI;
#   (iii) an EXTRAPOLATION band one order of magnitude beyond the top scale,
#        explicitly labelled PREDICTION-not-evidence.
# Also fits effect vs scale for steps-to-target (speed) and sparsity-control.
# Nothing here trains; it reuses RESULTS / SWEEP from earlier cells.
if not _require("RESULTS"):
    pass  # prerequisites missing -> skip this phase (Run-All safe)
else:
    import numpy as np, math
    from collections import defaultdict
    from scipy import stats

    def _load_results():
        try: return RESULTS
        except NameError:
            return [load_run(run_id("main",s,m,sd))
                    for s in SIZES for m,_ in METHODS for sd in RUN_CFG["seeds"]
                    if have_run(run_id("main",s,m,sd))]
    _R=_load_results()
    # (R10) The ternary-baseline set was hard-coded here while every other cell
    # reads it from RUN_CFG. That silently excluded arms added later -- notably
    # `sparse_sched` and `gamma_offset` -- from the scaling-law contrast, so the
    # H6 trend would have been fitted against a WEAKER baseline set than the one
    # H1/H2 are tested against, quietly inflating the effect at every scale.
    # Now derived from the same pre-registered config the confirmatory cells use.
    TERN_BASE=[m for m in ([RUN_CFG.get("primary_baseline","bitnet158")]
                           + list(RUN_CFG.get("secondary_baselines",[])))
               if m not in ("fp16_control",)]
    TERN_BASE=list(dict.fromkeys(TERN_BASE))   # de-dup, preserve order

    def _effect_rows(metric, better="lower"):
        """Per (scale,seed): strongest baseline minus tai_warmup on that metric.
           Sign convention: positive => TAI better."""
        rows=[]
        for size in SIZES:
            g=[r for r in _R if r["size"]==size]; bym=defaultdict(dict)
            for r in g: bym[r["method"]][r["seed"]]=r.get(metric)
            if "tai_warmup" not in bym: continue
            tern=[k for k in TERN_BASE if k in bym]
            if not tern: continue
            for seed in RUN_CFG["seeds"]:
                bvals=[bym[k][seed] for k in tern if bym[k].get(seed) is not None]
                tv=bym["tai_warmup"].get(seed)
                if not bvals or tv is None: continue
                base=min(bvals) if better=="lower" else max(bvals)
                eff=(base-tv) if better=="lower" else (tv-base)
                rows.append((math.log(n_params(SIZES[size])), eff))
        return np.array(rows) if rows else np.empty((0,2))

    def _slope_with_ci(xy, nboot=5000):
        if len(xy)<3: return None
        x,y=xy[:,0],xy[:,1]
        b=np.polyfit(x,y,1)  # [slope,intercept]
        rng=np.random.default_rng(GLOBAL_SEED)
        bs=[]
        for _ in range(nboot):
            idx=rng.integers(0,len(x),len(x))
            bs.append(np.polyfit(x[idx],y[idx],1)[0])
        lo,hi=np.percentile(bs,[2.5,97.5])
        return {"slope":float(b[0]),"intercept":float(b[1]),
                "slope_ci":(float(lo),float(hi)),
                "mean_effect":float(y.mean()),
                "x_min":float(x.min()),"x_max":float(x.max())}

    print("="*70)
    print("(#1) SCALING-LAW ANALYSIS OF THE TAI EFFECT  (positive => TAI better)")
    print("="*70)
    for label,metric,better in [("val_loss gap (nats)","val_loss","lower"),
                                ("steps-to-target gap","steps_to_target","lower"),
                                ("sparsity-control gap","final_sparsity","higher")]:
        xy=_effect_rows(metric,better)
        fit=_slope_with_ci(xy)
        if fit is None:
            print(f"\n[{label}] insufficient data"); continue
        s,(lo,hi)=fit["slope"],fit["slope_ci"]
        # interpretation: effect vs log-params. slope>0 => effect GROWS with scale;
        # slope~0 => stable; slope<0 => shrinks. The skeptic's worry is slope<0 with
        # the band reaching 0 before relevant scales.
        if lo>0:   trend="GROWS with scale (CI>0)"
        elif hi<0: trend="SHRINKS with scale (CI<0)"
        else:      trend="scale-STABLE (CI spans 0)"
        print(f"\n[{label}]")
        print(f"   mean effect over scales = {fit['mean_effect']:+.4f}")
        print(f"   slope d(effect)/d(log P) = {s:+.4f}  95% CI [{lo:+.4f},{hi:+.4f}] -> {trend}")
        # extrapolation: project the effect at +1 and +2 decades of params (ln base e)
        for dec in (1,2):
            xp=fit["x_max"]+dec*math.log(10)
            pred=fit["slope"]*xp+fit["intercept"]
            print(f"   PREDICTION (NOT evidence) at +{dec} decade(s) params: effect ~ {pred:+.4f}")

    print("\nInterpretation guide for the paper:")
    print(" * A scale-STABLE or GROWING val_loss gap is the external-validity result:")
    print("   the advantage does not wash out across the measured 1.4M->~50M range,")
    print("   and the fitted trend does not predict it vanishing within an order of")
    print("   magnitude. This is OBSERVATIONAL support over the tested regime PLUS a")
    print("   clearly-labelled extrapolation -- never a claim of billion-scale proof.")
    print(" * Report the slope CI, not just the point, and carry the PREDICTION label")
    print("   into every figure caption that shows the extrapolation band.")

    # ---------------------------------------------------------------
    # (#2) DEPTH-CONFOUND-CONTROLLED SLOPE.  The main ladder co-varies width AND
    # depth; an "effect grows with scale" could be a depth artefact. We train a
    # SECOND ladder (SIZES_WC: fixed n_layers, width-only) at reduced seeds and
    # refit the val_loss-gap slope on it. If the sign/Ωsignificance of the slope
    # survives with depth held constant, the scaling claim is confound-controlled.
    # Trains only when armed; otherwise reports from whatever wc runs exist.
    SCALING_LAW_WIDTHCTRL=None
    if RUN_CFG.get("ARM_FULL_RUN",False) and "SIZES_WC" in dir():
        from collections import defaultdict as _dd
        _WCM=[("standard",{}),("bitnet158",{"recipe":"bitnet"}),
              ("tai_warmup",{"rho0":0.75,"gamma_warmup":240})]
        _wc_seeds=RUN_CFG.get("width_ctrl_seeds",RUN_CFG["seeds"][:5])
        for size in RUN_CFG.get("width_ctrl_scales",list(SIZES_WC)):
            if size not in SIZES_WC: continue
            cfg=SIZES_WC[size]
            for method,opts in _WCM:
                for seed in _wc_seeds:
                    rid=run_id("wc",size,method,seed)
                    if have_run(rid): continue
                    if budget_exceeded():
                        notify("SESSION BOUNDARY (width-ctrl)","resume next session",critical=True); raise SystemExit(0)
                    rec=train_run(size,cfg,method,seed,opts,
                                  lr=_lrstar(size,method) if "_lrstar" in dir() else None)
                    rec["ladder"]="width_ctrl"; save_run(rid,rec)
        # refit slope on width-ctrl points (positive => TAI better)
        wc_rows=[]
        for size in RUN_CFG.get("width_ctrl_scales",list(SIZES_WC)):
            if size not in SIZES_WC: continue
            recs=[load_run(run_id("wc",size,m,sd)) for m,_ in _WCM for sd in _wc_seeds
                  if have_run(run_id("wc",size,m,sd))]
            bym=_dd(dict)
            for r in recs: bym[r["method"]][r["seed"]]=r["val_loss"]
            if "tai_warmup" not in bym: continue
            tern=[k for k in ("standard","bitnet158") if k in bym]
            for seed in _wc_seeds:
                bvals=[bym[k][seed] for k in tern if bym[k].get(seed) is not None]
                tv=bym["tai_warmup"].get(seed)
                if bvals and tv is not None:
                    wc_rows.append((math.log(n_params(SIZES_WC[size])), min(bvals)-tv))
        if len(wc_rows)>=3:
            fit=_slope_with_ci(np.array(wc_rows))
            SCALING_LAW_WIDTHCTRL=fit
            s_,(lo_,hi_)=fit["slope"],fit["slope_ci"]
            trend=("GROWS (CI>0)" if lo_>0 else "SHRINKS (CI<0)" if hi_<0 else "STABLE (CI spans 0)")
            print("\n(#2) DEPTH-CONTROLLED scaling slope (width-only ladder, fixed depth):")
            print(f"   slope d(val_loss gap)/d(log P) = {s_:+.4f} 95% CI [{lo_:+.4f},{hi_:+.4f}] -> {trend}")
            print("   => if this matches the main-ladder sign, the H6 trend is not a depth artefact.")
    else:
        print("\n(#2) width-controlled slope: [skip] not armed or SIZES_WC missing.")

    # expose for the figures cell
    SCALING_LAW={}
    for label,metric,better in [("val_loss","val_loss","lower"),
                                ("steps","steps_to_target","lower"),
                                ("sparsity","final_sparsity","higher")]:
        fit=_slope_with_ci(_effect_rows(metric,better))
        if fit: SCALING_LAW[label]=fit
    if SCALING_LAW_WIDTHCTRL: SCALING_LAW["val_loss_widthctrl"]=SCALING_LAW_WIDTHCTRL


In [ ]:
# === CELL 13a2: (#6) PLATEAU ROBUSTNESS GRID (recompute-only, no new runs) ===
# H2's "converged-quality equivalence" must not hinge on one (tol,frac) choice.
# Here we recompute the plateau-verification RATE and the H2 equivalence verdict
# across a grid of tolerances x tail-fractions, reusing the stored trajectories
# only -- zero additional training. A flat table => the plateau gate (and hence
# H2) is robust to the operationalisation of "converged".
#
# (R3 FIX) This cell used to re-implement the OLD gate -- level-comparison on the
# noisy single-batch TRAIN loss -- which is now known to measure batch noise
# rather than convergence, and which Cell 11's gate no longer uses. A robustness
# grid over a broken statistic certifies nothing. It now recomputes the SAME
# slope-based gate as Cell 11 (`plateau_ok`) over the stored VAL trace, so the
# sensitivity analysis is a genuine perturbation of the real decision rule. Runs
# predating this revision have no val trace; they are counted and excluded rather
# than silently scored under the old rule.
if not _require("RESULTS"):
    pass  # prerequisites missing -> skip this phase (Run-All safe)
else:
    import numpy as np
    from itertools import product

    import math as _m
    def _plateau_slope(vs,vl,tol,frac,min_tail=None):
        # EXACT mirror of Cell 11's plateau_ok, with (tol,frac) as free parameters
        # so the grid perturbs the real rule rather than a legacy surrogate.
        min_tail=int(RUN_CFG.get("plateau_min_tail",5)) if min_tail is None else min_tail
        if not vl or len(vl)<min_tail+1: return False
        x=np.asarray(vs,float); y=np.asarray(vl,float)
        k=max(min_tail,int(_m.ceil(len(y)*frac))); x=x[-k:]; y=y[-k:]
        if len(y)<min_tail or np.ptp(x)<=0: return False
        n=len(y); xm=x.mean(); sxx=float(((x-xm)**2).sum())
        if sxx<=0: return False
        b=float(((x-xm)*(y-y.mean())).sum()/sxx); a=float(y.mean()-b*xm)
        resid=y-(a+b*x); dof=max(n-2,1)
        se_b=_m.sqrt(float((resid**2).sum())/dof/max(sxx,1e-12)); ci=1.96*se_b
        descent=float(b*(x[-1]-x[0])); band=tol*max(abs(float(y[-1])),1e-9)
        return bool(((abs(b)-ci)<=0 or abs(descent)<=band)
                    and descent>=-band and descent<=band)

    TOL_GRID=RUN_CFG.get("plateau_tol_grid",[0.005,0.01,0.02])
    # (R3) frac grid retargeted to the slope gate: fractions must leave enough val
    # probes in the tail for the slope CI to have power (see plateau_min_tail).
    FRAC_GRID=RUN_CFG.get("plateau_frac_grid_slope",[0.30,0.40,0.50])

    # RESULTS is the in-memory list from Cell 11; fall back to reloading from store.
    try:
        _R=RESULTS
    except NameError:
        _R=[load_run(run_id("main",s,m,sd))
            for s in SIZES for m,_ in METHODS for sd in RUN_CFG["seeds"]
            if have_run(run_id("main",s,m,sd))]

    _n_all=len(_R); _n_tr=len([r for r in _R if r.get("val_trace")])
    if _n_tr<_n_all:
        print(f"NOTE: {_n_all-_n_tr}/{_n_all} stored runs predate the val-trace revision "
              f"and are EXCLUDED from this grid (they cannot be scored under the current "
              f"gate). Re-run them to include; they are not scored under the old rule.")
    print("Plateau-verified fraction of runs over (tol x frac), slope gate on val trace:")
    hdr="tol\\frac  "+"  ".join(f"{f:>6.2f}" for f in FRAC_GRID); print(hdr)
    robust_tbl={}
    for tol in TOL_GRID:
        row=[]
        for frac in FRAC_GRID:
            _have=[r for r in _R if r.get("val_trace") and r.get("val_steps")]
            oks=[_plateau_slope(r["val_steps"],r["val_trace"],tol,frac) for r in _have]
            rate=float(np.mean(oks)) if oks else float("nan")
            robust_tbl[(tol,frac)]=rate; row.append(f"{rate:6.2f}")
        print(f"{tol:7.3f}  "+"  ".join(row))

    # H2 equivalence (TAI vs strongest baseline) recomputed under each (tol,frac):
    # only plateau-passing runs of BOTH arms enter the test, so the verdict can move
    # if the gate is fragile. We report the TOST equivalence p at each cell.
    def _tost_equiv(a,b,margin):
        a=np.asarray(a); b=np.asarray(b)
        if len(a)<2 or len(b)<2: return float("nan")
        from scipy import stats
        d=a.mean()-b.mean(); se=np.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b))
        if se==0: return float("nan")
        p_low=stats.norm.cdf((d+margin)/se); p_high=1-stats.norm.cdf((d-margin)/se)
        return float(max(p_low,p_high))   # max of the two one-sided p's

    BASE=RUN_CFG.get("wall_ref_method","bitnet158")
    MARGIN=RUN_CFG.get("equiv_margin",0.05)
    TOP=max(SIZES,key=lambda k:SIZES[k].dim)
    print(f"\nH2 TOST equivalence p (TAI vs {BASE}) at top scale {TOP}, "
          f"margin={MARGIN} nats, per (tol,frac):")
    print(hdr)
    for tol in TOL_GRID:
        row=[]
        for frac in FRAC_GRID:
            tai=[r["val_loss"] for r in _R if r["size"]==TOP and r["method"]=="tai"
                 and r.get("val_trace") and _plateau_slope(r["val_steps"],r["val_trace"],tol,frac)]
            bas=[r["val_loss"] for r in _R if r["size"]==TOP and r["method"]==BASE
                 and r.get("val_trace") and _plateau_slope(r["val_steps"],r["val_trace"],tol,frac)]
            p=_tost_equiv(tai,bas,MARGIN); row.append(f"{p:6.3f}")
        print(f"{tol:7.3f}  "+"  ".join(row))
    print("\n(If verdicts are stable across the grid, H2 is robust to the plateau "
          "definition; cells where either arm has <2 plateau runs show NaN.)")


In [ ]:
# === CELL 13b: THEORY VERIFICATION (sigma-independence) ====================
# Empirically confirms the core theoretical claim that motivates TAI: for ANY
# standard zero-mean init, the absmean-threshold leaves a sigma-INDEPENDENT
# initial active ratio (~0.69), so standard init cannot tune initial sparsity.
# This is the analytical backbone reviewers will scrutinize; we make it auditable.
import torch,numpy as np
from math import erf,sqrt,pi
print("sigma | empirical initial active ratio (absmean ternary)")
for s in [0.02,0.1,0.5,1.0,2.0]:
    w=torch.randn(1_000_000)*s; g=w.abs().mean()
    a=((w/g).round().clamp(-1,1)!=0).float().mean().item()
    print(f"  {s:5.2f} | {a:.4f}")
z=sqrt(2/pi)/2; rho=2*(1-0.5*(1+erf(z/sqrt(2))))
print(f"Analytical prediction (any sigma): {rho:.4f}")
# also show all common inits collapse to the same locked ratio
if RUN_CFG.get("ARM_FULL_RUN",False):
    # (R13) LSUV must be COMPLETED before it is measured here. This cell is the
    # evidentiary backbone of the paper's premise -- "no standard init controls
    # rho0" -- so measuring a HALF-APPLIED initializer would make the strongest
    # claim in the paper rest on the weakest evidence: showing that orthogonal-init
    # is locked proves nothing about LSUV. With the data-driven pass included, the
    # result becomes a real test that real LSUV could in principle fail.
    _cal_gen=torch.Generator().manual_seed(GLOBAL_SEED+555)
    _cal_x,_=get_batch(TRAIN,RUN_CFG["batch_size"],RUN_CFG["seq_len"],_cal_gen)
    for name in ["standard","xavier","kaiming","lsuv"]:
        seed_everything(); m=LM(SIZES["dim384"]).to(DEVICE,DTYPE); apply_init(m,name)
        if name in INIT_NEEDS_DATA: finish_init(m,name,_cal_x)   # complete real LSUV
        a=float(np.mean([bl.active_ratio() for bl in m.bitlinears()]))
        _tag="(locked)" if abs(a-0.69)<0.02 else "(NOT locked -- premise violated!)"
        del m; torch.cuda.empty_cache()
        print(f"  init {name:8s}: initial active {a:.3f} {_tag}")
    del _cal_x; torch.cuda.empty_cache()
else:
    print("  [safe mode] per-init model check skipped (ARM_FULL_RUN off); "
          "analytical sigma-independence above is GPU-free and already shown.")
print("=> No standard init controls initial sparsity; TAI does (Cell 8). Claim verified.")


In [ ]:
# (R23) These ablation cells run their OWN training loops. Their gamma writes
# must go through set_gamma() as well: a raw dict write would leave the device
# tensor stale, so the forward pass would ternarise at the PREVIOUS threshold
# while every log reported the intended one -- silent, and exactly the failure
# mode R16 documented. Rewritten mechanically; guarded by gamma_mirror_ok().
# === CELL 13c: GAMMA-WARMUP ABLATION (start x tau grid) =====================
# Isolates the gamma-warmup component: vary start in {0.3,0.4,0.5} and tau in
# {60,120,240} at fixed rho0=0.75 (dim384, ablation_seeds). Shows whether the
# warmup contributes beyond TAI-init alone, and how sensitive it is. Checkpointed
# and budget-aware like the main experiment.
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("RESULTS"):
    print("[skip] gamma-warmup ablation: ARM_FULL_RUN off or prerequisites missing.")  # Run-All safe
else:
    import numpy as np
    from collections import defaultdict
    notify("PHASE START: gamma-warmup ablation","")
    WARM_ABL=[]
    for st in RUN_CFG["warmup_start_grid"]:
        for tau in RUN_CFG["warmup_tau_grid"]:
            for seed in RUN_CFG["ablation_seeds"]:
                rid=run_id("warmup","dim384",f"st{st}_tau{tau}",seed)
                if have_run(rid): WARM_ABL.append(load_run(rid)); continue
                if budget_exceeded():
                    # (R14b) restore quant globals BEFORE the designed SystemExit, so a
                    # session-boundary exit cannot leak gamma/precision into the next
                    # session's runs (the same leak class fixed in train_run).
                    set_gamma(1.0); QUANT_MODE["full_precision"]=False
                    QUANT_RECIPE["name"]="absmean"
                    notify("SESSION BOUNDARY (warmup ablation)","resume next session",critical=True); raise SystemExit(0)
                # custom: tai with given warmup start/tau
                def _train_custom(st,tau,seed):
                    import math,torch,torch.nn as nn
                    seed_everything(seed); gen=torch.Generator().manual_seed(seed)
                    model=LM(SIZES["dim384"]).to(DEVICE,DTYPE); apply_init(model,"tai",rho0=0.75)
                    opt=_mk_adamw_g(model.parameters(),PEAK_LR); model.train()
                    for t in range(STEPS):
                        for g in opt.param_groups: g["lr"]=lr_at(t)
                        set_gamma(gamma_warmup_value(t,tau,start=st))
                        xb,yb=get_batch(TRAIN,BS,SEQ,gen)
                        with torch.autocast("cuda",dtype=DTYPE): _,loss=model(xb,yb)
                        opt.zero_grad(set_to_none=True); loss.backward()
                        nn.utils.clip_grad_norm_(model.parameters(),CLIP); opt.step()
                    set_gamma(1.0); v=evaluate(model)   # (R14b) see note below
                    del model,opt; torch.cuda.empty_cache(); return v
                v=_train_custom(st,tau,seed)
                rec={"start":st,"tau":tau,"seed":seed,"val_loss":v}; save_run(rid,rec); WARM_ABL.append(rec)
                notify(f"run done: warmup st{st} tau{tau} s{seed}",f"val {v:.4f}",tag="run")
    g=defaultdict(list)
    for r in WARM_ABL: g[(r["start"],r["tau"])].append(r["val_loss"])
    print("start tau  -> mean val_loss")
    for (st,tau),vs in sorted(g.items()): print(f"  {st:.1f}  {tau:3d} -> {np.mean(vs):.4f}")
    notify("PHASE DONE: gamma-warmup ablation",f"{len(WARM_ABL)} runs")
    WARMUP_ABLATION=WARM_ABL


In [ ]:
# === CELL 13c-b: (R2) TAI HYPER-PARAMETER ABLATION — dm x jitter ============
# The two numbers that define TAI's magnitude split (dormant magnitude `dm` and
# jitter half-width `a`) were previously hard-coded (0.15, 0.15) with no
# justification and no sensitivity analysis, even though they are genuine degrees
# of freedom of the method. A reviewer is entitled to ask: is rho0 really the
# knob, or is the result an artefact of dm=0.15?
#
# TWO-PART ANSWER, cheapest-first:
#  (A) REALISATION ablation (GPU-free, always runs): does the init still realise
#      the requested rho0 across the dm x jitter grid? This is a property of the
#      initializer alone, so it needs no training. It maps the VALID REGION and
#      shows the closed-form separation bound in Cell 8 is tight: dm below the
#      bound -> exact rho0; dm above -> the split collapses (dormants survive the
#      threshold and rho0 is silently NOT realised -- the old code's failure mode).
#  (B) QUALITY ablation (armed runs only): does final val_loss depend on dm?
#      Run at the confirmatory scale on ablation_seeds with rho0 fixed at 0.75.
#      Pre-registered expectation: FLAT inside the valid region (rho0 is the knob,
#      dm is not), which is what makes "TAI = control over rho0" an honest claim.
#      A significant dm effect would have to be reported as a second knob.
import numpy as np, math, torch

_DM_GRID   = RUN_CFG.get("tai_dm_grid",     [0.05,0.15,0.25,0.35,0.45])
_JIT_GRID  = RUN_CFG.get("tai_jitter_grid", [0.00,0.15,0.30])
_ABL_RHO0  = 0.75

# ---------- (A) realisation map: GPU-free, no training ----------------------
print("="*74)
print("(R2-A) TAI REALISATION MAP — does the init achieve the requested rho0?")
print("       entry = realised rho0 at target %.2f   ('BREAK' = separation bound violated)" % _ABL_RHO0)
print("="*74)
print("dm \\ jit  " + "  ".join(f"{j:>7.2f}" for j in _JIT_GRID))
_realise_tbl={}
for dm in _DM_GRID:
    row=[]
    for a in _JIT_GRID:
        w=torch.empty(512,512)
        g=torch.Generator().manual_seed(1337)
        try:
            # verify=True asserts realisation; separation asserts guard the bound
            init_tai(w,512,rho0=_ABL_RHO0,gen=g,dm=dm,jitter=a,verify=True)
            r=TAI_HP["last_realised_rho0"]; _realise_tbl[(dm,a)]=r; row.append(f"{r:7.4f}")
        except AssertionError:
            _realise_tbl[(dm,a)]=float("nan"); row.append("  BREAK")
    print(f"{dm:5.2f}    " + "  ".join(row))
_valid=[k for k,v in _realise_tbl.items() if not math.isnan(v)]
_maxerr=max((abs(_realise_tbl[k]-_ABL_RHO0) for k in _valid), default=float("nan"))
print(f"\nValid region: {len(_valid)}/{len(_realise_tbl)} cells | max |realised-target| "
      f"inside valid region = {_maxerr:.4f}")
print("=> Inside the valid region rho0 is realised exactly and is INDEPENDENT of (dm,jitter);")
print("   outside it the two-point split no longer separates and rho0 is not realised.")
print("   This is the pre-registered justification for the default dm=0.15, jitter=0.15.")
TAI_HP_REALISATION={"grid_dm":_DM_GRID,"grid_jitter":_JIT_GRID,"target_rho0":_ABL_RHO0,
                    "table":{f"dm{k[0]}_jit{k[1]}":v for k,v in _realise_tbl.items()},
                    "valid_cells":len(_valid),"total_cells":len(_realise_tbl),
                    "max_abs_err_valid":(None if math.isnan(_maxerr) else float(_maxerr))}

# ---------- (B) quality ablation: needs training ----------------------------
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("RESULTS"):
    print("\n[skip] (R2-B) dm quality ablation: ARM_FULL_RUN off or prerequisites missing.")
    TAI_HP_ABL=None
else:
    from collections import defaultdict
    from scipy import stats
    notify("PHASE START: TAI dm/jitter quality ablation","rho0 fixed at 0.75")
    _PS=RUN_CFG.get("confirmatory_scale","dim384"); _cfg=SIZES[_PS]
    # only sweep dm inside the VALID region (a broken split is not a competing
    # hyper-parameter setting, it is an invalid one -- and Cell 8 now refuses it)
    _DM_VALID=[dm for dm in _DM_GRID if not math.isnan(_realise_tbl.get((dm,0.15),float("nan")))]
    TAI_HP_ABL=[]
    for dm in _DM_VALID:
        for seed in RUN_CFG["ablation_seeds"]:
            rid=run_id("taihp",_PS,f"dm{dm}",seed)
            if have_run(rid): TAI_HP_ABL.append(load_run(rid)); continue
            if budget_exceeded():
                notify("SESSION BOUNDARY (TAI hp ablation)","resume next session",critical=True); raise SystemExit(0)
            TAI_HP["dm"]=dm     # train_run -> apply_init -> init_tai reads TAI_HP
            try:
                rec=train_run(_PS,_cfg,"tai",seed,{"rho0":_ABL_RHO0},
                              lr=(LR_STAR.get((_PS,"tai")) if "LR_STAR" in dir() else None))
            finally:
                TAI_HP["dm"]=0.15   # ALWAYS restore the default (global state hygiene)
            rec["tai_dm"]=dm; save_run(rid,rec); TAI_HP_ABL.append(rec)
            notify(f"run done: taihp dm{dm} s{seed}",f"val {rec['val_loss']:.4f}",tag="run")
    by=defaultdict(list)
    for r in TAI_HP_ABL: by[r["tai_dm"]].append(r["val_loss"])
    print("\n" + "="*74)
    print("(R2-B) FINAL val_loss vs dm (rho0=0.75 fixed, %s, %d seeds)" % (_PS,len(RUN_CFG["ablation_seeds"])))
    print("="*74)
    for dm in sorted(by):
        v=np.array(by[dm],float)
        print(f"  dm={dm:.2f}: {v.mean():.4f} +/- {v.std(ddof=1):.4f}")
    # one-way ANOVA across dm: pre-registered expectation is NON-significant
    if len(by)>=2 and all(len(v)>1 for v in by.values()):
        F,p=stats.f_oneway(*[np.array(by[d],float) for d in sorted(by)])
        _dflt=np.array(by.get(0.15,[]),float)
        _rng=max(np.mean(by[d]) for d in by)-min(np.mean(by[d]) for d in by)
        print(f"\n  one-way ANOVA over dm: F={F:.3f} p={p:.4f}  (EXPLORATORY, not in the Holm family)")
        print(f"  spread of dm-means = {_rng:.4f} nats")
        print("  VERDICT:", "dm does NOT materially affect quality -> rho0 is the operative knob (as claimed)."
              if p>0.05 else
              "dm DOES affect quality -> TAI has a SECOND knob; this must be reported, not hidden.")
        TAI_HP_ABL_STATS={"anova_F":float(F),"anova_p":float(p),"dm_mean_range":float(_rng),
                          "dm_grid_valid":_DM_VALID,"exploratory":True}
    else:
        TAI_HP_ABL_STATS=None
    notify("PHASE DONE: TAI dm/jitter ablation","")


In [ ]:
# === CELL 13d: LR SENSITIVITY ABLATION ======================================
# Tests robustness of the TAI conclusion to learning rate: re-runs tai_warmup vs
# the stronger baseline across lr in {1e-3,2e-3,3e-3} (dim384, ablation_seeds).
# If TAI's relative behavior is stable across LR, the conclusion is not an
# artifact of one optimizer setting (a common reviewer concern).
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("RESULTS"):
    print("[skip] LR ablation: ARM_FULL_RUN off or prerequisites missing.")  # Run-All safe
else:
    import numpy as np
    from collections import defaultdict
    notify("PHASE START: LR sensitivity","")
    LR_ABL=[]
    for lr in RUN_CFG["lr_grid"]:
        for method in ["standard","tai_warmup"]:
            for seed in RUN_CFG["ablation_seeds"]:
                rid=run_id("lr",f"lr{lr}","dim384_"+method,seed)
                if have_run(rid): LR_ABL.append(load_run(rid)); continue
                if budget_exceeded():
                    # (R14b) restore quant globals BEFORE the designed SystemExit, so a
                    # session-boundary exit cannot leak gamma/precision into the next
                    # session's runs (the same leak class fixed in train_run).
                    set_gamma(1.0); QUANT_MODE["full_precision"]=False
                    QUANT_RECIPE["name"]="absmean"
                    notify("SESSION BOUNDARY (LR ablation)","resume next session",critical=True); raise SystemExit(0)
                import math,torch,torch.nn as nn
                seed_everything(seed); gen=torch.Generator().manual_seed(seed)
                model=LM(SIZES["dim384"]).to(DEVICE,DTYPE)
                _im=("tai" if method.startswith("tai") else method)
                apply_init(model,_im,rho0=0.75)
                if _im in INIT_NEEDS_DATA:   # (R13) complete data-driven inits here too
                    _cg=torch.Generator().manual_seed(seed+555)
                    _cx,_=get_batch(TRAIN,BS,SEQ,_cg); finish_init(model,_im,_cx); del _cx
                gw=120 if method=="tai_warmup" else 0
                def lr_at_c(t):
                    if t<WARMUP: return lr*(t+1)/WARMUP
                    p=(t-WARMUP)/max(1,STEPS-WARMUP); return MIN_LR+0.5*(lr-MIN_LR)*(1+math.cos(math.pi*p))
                opt=_mk_adamw_g(model.parameters(),lr); model.train()
                for t in range(STEPS):
                    for g in opt.param_groups: g["lr"]=lr_at_c(t)
                    set_gamma(gamma_warmup_value(t,gw) if gw else 1.0)
                    xb,yb=get_batch(TRAIN,BS,SEQ,gen)
                    with torch.autocast("cuda",dtype=DTYPE): _,loss=model(xb,yb)
                    opt.zero_grad(set_to_none=True); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(),CLIP); opt.step()
                set_gamma(1.0); v=evaluate(model)
                del model,opt; torch.cuda.empty_cache()
                rec={"lr":lr,"method":method,"seed":seed,"val_loss":v}; save_run(rid,rec); LR_ABL.append(rec)
                notify(f"run done: lr{lr} {method} s{seed}",f"val {v:.4f}",tag="run")
    g=defaultdict(list)
    for r in LR_ABL: g[(r["lr"],r["method"])].append(r["val_loss"])
    print("lr     method        -> mean val_loss")
    for (lr,m),vs in sorted(g.items()): print(f"  {lr:.0e} {m:13s} -> {np.mean(vs):.4f}")
    notify("PHASE DONE: LR sensitivity",f"{len(LR_ABL)} runs")
    LR_ABLATION=LR_ABL


In [ ]:
# === CELL 13e: DEPLOYMENT MODEL (theoretical) + SIMULATION COST (measured) ===
# (R7 FIX -- FRAMING.) This cell used to be titled "EFFICIENCY" and led with a
# MEASURED latency/throughput table comparing the fp16 control against the
# "ternary (TAI)" build. That framing invites a specific misreading, and the
# numbers make it worse: with STE the ternary path keeps bf16 master weights and
# ADDS per-forward work (RMSNorm, activation quant, weight quant, rescale) on top
# of the same bf16 GEMM. It is therefore expected to be SLOWER and to use MORE
# memory than the fp16 control. Publishing that under the heading "efficiency"
# would either (a) read as evidence that ternary nets are slow -- which this
# harness cannot show, since it never runs a ternary kernel -- or (b) invite the
# reader to discount the table as a rigged comparison. Neither is honest.
#
# What is actually true, and what this cell now says:
#   (A) SIMULATION COST (measured, secondary). The overhead of SIMULATING ternary
#       arithmetic in bf16 during training/research. This is a property of the
#       HARNESS, not of ternary inference. It is reported because it explains the
#       wall-clock budget (and is charged honestly to TAI in the compute-equal H1),
#       NOT as a deployment claim. Slower here means nothing about deployment.
#   (B) DEPLOYMENT MODEL (theoretical, primary). What a real 1.58-bit kernel buys:
#       log2(3)=1.585 bits/weight information-theoretic, or 2 bits naively packed,
#       versus 16 bits for fp16. This is the number a hardware reader wants, and it
#       is a counting argument that does not depend on our GPU at all.
# Reporting order is inverted accordingly: the theoretical model leads, the
# simulation cost follows and is explicitly labelled as not-a-deployment-number.
# LIMITATION recorded in the manifest: no ternary kernel was benchmarked; the
# deployment model is arithmetic, not measurement, and is labelled as such.
# (R14b) also require quant_state: this cell mutates the global quant world and
# depends on Cell 11's hygiene helpers to put it back. Without them a failed bench
# would leave full_precision=True set for every later cell.
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("SIZES","quant_state"):
    print("[skip] efficiency bench: ARM_FULL_RUN off (no GPU work in safe mode).")  # Run-All safe
else:
    import time,math,torch,numpy as np,json,os
    BIG=max(SIZES,key=lambda k:SIZES[k].dim)
    def _bench(build_full_precision):
        # NOTE (R14b): _es.__enter__/__exit__ are used rather than a `with` block to
        # avoid re-indenting the measured section; the outer try/except in this cell
        # plus quant_state's own finally guarantee the globals are restored even if
        # the bench raises. assert_quant_state_clean() below verifies it.
        seed_everything(123); cfg=SIZES[BIG]
        # (R14b) Same leak class as train_run: this set the globals, then ran model
        # construction, init and 23 forward passes before restoring. An OOM here
        # (likely: this is the LARGEST scale, and the bench runs right after the
        # grid) would leave full_precision=True set for every later cell -- the
        # ablations in 13c/13d/13g would then silently run in full precision.
        # quant_state restores on every exit path.
        _es=quant_state(full_precision=build_full_precision,recipe="absmean")
        _es.__enter__()
        m=LM(cfg).to(DEVICE,DTYPE)
        if not build_full_precision: apply_init(m,"tai",rho0=0.75)
        g=torch.Generator().manual_seed(7); xb,_=get_batch(VAL,RUN_CFG["batch_size"],RUN_CFG["seq_len"],g)
        n_tok=RUN_CFG["batch_size"]*RUN_CFG["seq_len"]
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        with torch.no_grad():
            for _ in range(3):
                with torch.autocast("cuda",dtype=DTYPE): m(xb)
            torch.cuda.synchronize(); t0=time.time()
            for _ in range(20):
                with torch.autocast("cuda",dtype=DTYPE): m(xb)
            torch.cuda.synchronize(); dt=(time.time()-t0)/20.0
        peak=torch.cuda.max_memory_allocated()/1024**3
        sp=float(np.mean([bl.sparsity() for bl in m.bitlinears()])) if not build_full_precision else 0.0
        _es.__exit__(None,None,None)   # (R14b) restores fp/recipe/gamma defaults
        del m; torch.cuda.empty_cache()
        return {"latency_ms":round(dt*1000,2),"tokens_per_sec":round(n_tok/dt,0),
                "peak_mem_GB":round(peak,3),"sparsity_pct":round(sp*100,1)}
    def _theoretical(cfg):
        # quantized linears are the BitLinear weights; embedding/head stay fp16.
        nq=n_params(cfg)  # non-embed quantized params (matches the n_params accounting)
        LOG2_3=math.log2(3)            # 1.585 bits/weight (information-theoretic)
        NAIVE=2.0                      # bits/weight (simple 2-bit packing)
        fp16_bytes=nq*2.0
        info_bytes=nq*LOG2_3/8.0
        pack_bytes=nq*NAIVE/8.0
        return {"quantized_params_M":round(nq/1e6,2),
                "fp16_size_MB":round(fp16_bytes/1024**2,2),
                "ternary_info_size_MB":round(info_bytes/1024**2,2),
                "ternary_2bit_size_MB":round(pack_bytes/1024**2,2),
                "bits_per_weight_info":round(LOG2_3,3),"bits_per_weight_packed":NAIVE,
                "compression_vs_fp16_info":round(fp16_bytes/info_bytes,2),
                "compression_vs_fp16_2bit":round(fp16_bytes/pack_bytes,2)}
    try:
        _theo=_theoretical(SIZES[BIG])
        _fp=_bench(True); _tern=_bench(False)
        EFF={"deployment_model_theoretical":_theo,
             "simulation_cost_measured":{"fp16_control":_fp,"ternary_simulated_tai":_tern},
             "caveat":("measured numbers are the cost of SIMULATING ternary in bf16 (STE keeps "
                       "bf16 master weights and adds quant/norm work); they are NOT ternary "
                       "inference numbers and must not be read as such. No ternary kernel was "
                       "benchmarked in this work.")}
        print("="*74)
        print("(A) DEPLOYMENT MODEL — theoretical 1.58-bit, largest scale [PRIMARY]")
        print("="*74)
        for k,v in _theo.items(): print(f"  {k:28s} {v}")
        print("\n  This is a counting argument over the ternary codebook; it is independent")
        print("  of this GPU and of our STE implementation. It is a PREDICTION about a real")
        print("  1.58-bit kernel, not a measurement of one.")
        print()
        print("="*74)
        print("(B) SIMULATION COST — measured on this A100 [SECONDARY, not a deployment claim]")
        print("="*74)
        for k,v in EFF["simulation_cost_measured"].items(): print(f"  {k:24s} -> {v}")
        _ratio=(_tern["latency_ms"]/_fp["latency_ms"]) if _fp.get("latency_ms") else float("nan")
        print(f"\n  ternary-simulated / fp16 latency ratio = {_ratio:.2f}x")
        print("  EXPECTED to exceed 1.0: the simulated path runs the SAME bf16 GEMM plus")
        print("  RMSNorm + activation-quant + weight-quant + rescale. This measures the")
        print("  RESEARCH HARNESS overhead (and is charged to TAI in the compute-equal H1),")
        print("  NOT the speed of ternary inference. Drawing a deployment conclusion from")
        print("  this row -- in either direction -- would be unsupported.")
        assert_quant_state_clean("after efficiency bench")   # (R14b) prove no leak
        json.dump(json_safe(EFF) if "json_safe" in dir() else EFF,
                  open(os.path.join(PATHS["results"],"efficiency.json"),"w"),indent=2,default=str)
        notify("PHASE DONE: deployment model + simulation cost",
               json.dumps(EFF["deployment_model_theoretical"]))
    except Exception as e:
        # (R14b) force the globals back even if the bench died mid-measurement,
        # then re-verify: later cells must not inherit a full-precision world.
        set_gamma(1.0); QUANT_MODE["full_precision"]=False; QUANT_RECIPE["name"]="absmean"
        EFF={}; print("efficiency bench skipped:",str(e)[:120])
        assert_quant_state_clean("after failed efficiency bench")

In [ ]:
# === CELL 13f: (R5) DATA-SAMPLING VARIANCE FOR H4 (recompute-only) ==========
# H4 says "TAI does not inflate seed-to-seed variance". As originally measured,
# variance came from ONE val fold, so the estimate was conditional on a single
# draw of the evaluation data -- a reviewer can fairly object that the claim
# omits data-sampling variance entirely.
#
# The previous version of this cell acknowledged the gap but declared it
# unfixable without persisting weights ("we did not persist weights, so instead
# we provide the PROTOCOL"). That was a false constraint: the trained model is in
# memory at the end of train_run, so scoring it on all K held-out folds costs a
# few float ops and K numbers of JSON -- no checkpoints, no reloads, no retraining.
# train_run now stores `val_loss_folds` (R5), and this cell simply consumes it.
#
# DECOMPOSITION. For each method we have an n_seeds x K matrix L[s,f]. Then:
#   * seed variance   = Var_s( mean_f L[s,f] )      (init/order effects)
#   * fold variance   = Var_f( mean_s L[s,f] )      (data-sampling effects)
#   * H4 (pooled)     = Levene on the flattened L, TAI vs primary baseline
# Reporting all three separates "TAI is unstable" from "the val set is small",
# which the fold-0-only test could not distinguish.
if not _require("RESULTS"):
    pass
else:
    import numpy as np
    from scipy import stats
    _PS=RUN_CFG.get("confirmatory_scale","dim384")
    _BASE=RUN_CFG.get("primary_baseline","bitnet158")

    def _fold_matrix(meth):
        rows=[]
        for r in RESULTS:
            if r["size"]!=_PS or r["method"]!=meth: continue
            fl=r.get("val_loss_folds")
            if fl: rows.append([float(x) for x in fl])
        return np.array(rows,float) if rows else np.zeros((0,0))

    A=_fold_matrix("tai"); B=_fold_matrix(_BASE)
    print("="*72)
    print("(R5) DATA-SAMPLING VARIANCE FOR H4 — seed x fold decomposition")
    print("="*72)
    if A.size==0 or B.size==0 or A.ndim!=2 or A.shape[1]<2:
        # Runs from before the R5 revision carry no per-fold losses. Say so plainly
        # and fall back to the fold-0 test rather than fabricating a decomposition.
        print("No per-fold losses found in RESULTS (runs predate the R5 revision).")
        print("Falling back to the fold-0 test; re-run the grid to obtain the")
        print("full decomposition. H4 is reported at a SINGLE val fold below.")
        tai=[r["val_loss"] for r in RESULTS if r["size"]==_PS and r["method"]=="tai"]
        bas=[r["val_loss"] for r in RESULTS if r["size"]==_PS and r["method"]==_BASE]
        if len(tai)>=2 and len(bas)>=2:
            lev=stats.levene(tai,bas)
            print(f"  fold-0 H4: SD(tai)={np.std(tai,ddof=1):.4f} "
                  f"SD({_BASE})={np.std(bas,ddof=1):.4f} | Levene p={lev.pvalue:.4f}")
        DATA_VARIANCE_H4={"mode":"fold0_only",
                          "reason":"per-fold losses absent; grid predates R5"}
    else:
        K=A.shape[1]
        out={}
        for name,M in (("tai",A),(_BASE,B)):
            seed_means=M.mean(axis=1); fold_means=M.mean(axis=0)
            out[name]={
                "n_seeds":int(M.shape[0]),"n_folds":int(M.shape[1]),
                "grand_mean":float(M.mean()),
                "sd_across_seeds":float(seed_means.std(ddof=1)) if M.shape[0]>1 else float("nan"),
                "sd_across_folds":float(fold_means.std(ddof=1)) if M.shape[1]>1 else float("nan"),
                "sd_pooled":float(M.flatten().std(ddof=1)),
            }
            print(f"\n  {name}:  n_seeds={out[name]['n_seeds']}  K={out[name]['n_folds']}"
                  f"  mean={out[name]['grand_mean']:.4f}")
            print(f"    SD across SEEDS (data averaged out) : {out[name]['sd_across_seeds']:.4f}")
            print(f"    SD across FOLDS (seeds averaged out): {out[name]['sd_across_folds']:.4f}")
            print(f"    SD pooled (seed x fold)             : {out[name]['sd_pooled']:.4f}")
        # H4 proper: Levene on pooled (seed x fold) losses -> includes data variance
        lev_pool=stats.levene(A.flatten(),B.flatten())
        # and the seed-only test, for comparability with the original H4 statement
        lev_seed=stats.levene(A.mean(axis=1),B.mean(axis=1)) if min(A.shape[0],B.shape[0])>1 else None
        print("\n  H4 (Levene, TAI vs %s):" % _BASE)
        print(f"    pooled seed x fold : W={lev_pool.statistic:.3f}  p={lev_pool.pvalue:.4f}"
              f"   <- includes data-sampling variance (PRE-REGISTERED H4)")
        if lev_seed is not None:
            print(f"    seed-only          : W={lev_seed.statistic:.3f}  p={lev_seed.pvalue:.4f}"
                  f"   <- comparable to the original fold-0 statement")
        _alpha=RUN_CFG.get("bonferroni_alpha",0.0125)
        print("\n  VERDICT:", 
              f"no evidence that TAI inflates variance (p={lev_pool.pvalue:.3f} > alpha={_alpha})."
              if lev_pool.pvalue>_alpha else
              f"variance DIFFERS (p={lev_pool.pvalue:.3f} <= alpha={_alpha}); direction: "
              f"{'TAI higher' if A.flatten().std(ddof=1)>B.flatten().std(ddof=1) else 'TAI lower'}"
              " -- report as-is.")
        # Which source dominates? Useful context, not a hypothesis test.
        _sf=np.nanmean([out['tai']['sd_across_folds'],out[_BASE]['sd_across_folds']])
        _ss=np.nanmean([out['tai']['sd_across_seeds'],out[_BASE]['sd_across_seeds']])
        print(f"\n  Variance budget: mean SD across folds={_sf:.4f} vs across seeds={_ss:.4f}"
              f"  -> {'DATA sampling' if _sf>_ss else 'SEED'} dominates.")
        DATA_VARIANCE_H4={"mode":"seed_x_fold","per_method":out,
                          "levene_pooled_W":float(lev_pool.statistic),
                          "levene_pooled_p":float(lev_pool.pvalue),
                          "levene_seed_only_p":(float(lev_seed.pvalue) if lev_seed is not None else None),
                          "dominant_source":("fold" if _sf>_ss else "seed"),
                          "scale":_PS,"baseline":_BASE}


In [ ]:
# === CELL 13g: (#3) fp16 + TAI-INIT ABLATION — is the gain init or threshold? ==
# The crux a reviewer will press: does TAI help because (a) its weight-magnitude
# DISTRIBUTION is simply a better initializer (which would also help in full
# precision), or because (b) it controls the TERNARY OPERATING POINT (meaningful
# only under absmean quantization)? We disentangle them with a 2x2 at the primary
# scale: {fp16, ternary} x {standard-init, TAI-init}. Prediction if the claim is
# right: TAI-init ~ standard-init in fp16 (no threshold to control => no edge),
# but TAI > baseline in ternary. A significant fp16 TAI-init advantage would mean
# part of the effect is a generic init effect, which we would then report honestly.
if (not RUN_CFG.get("ARM_FULL_RUN",False)) or not _require("RESULTS","TARGET_LOSS_BY_SCALE"):
    print("[skip] fp16+TAI-init ablation (#3): ARM_FULL_RUN off or prerequisites missing.")
else:
    import numpy as np, math
    from scipy import stats
    _PS=RUN_CFG.get("confirmatory_scale","dim384"); cfg=SIZES[_PS]
    notify("PHASE START: fp16+TAI-init ablation","2x2 init x precision")
    # method tags encode (precision, init). We add two NEW cells to the 2x2:
    #   fp16_taiinit : full precision but initialised with TAI(rho0=0.75)
    #   (fp16_control [std init] and tai [ternary,TAI init] already exist in RESULTS)
    # plus ternary_stdinit is just `standard`. So we only need fp16_taiinit here.
    ABL=[]
    for seed in RUN_CFG["ablation_seeds"]:
        rid=run_id("fp16abl",_PS,"fp16_taiinit",seed)
        if have_run(rid): ABL.append(load_run(rid)); continue
        if budget_exceeded():
            notify("SESSION BOUNDARY (fp16abl)","resume next session",critical=True); raise SystemExit(0)
        # full_precision True + TAI init: reuse train_run with a method that starts
        # with "tai" (=> TAI init) AND full_precision opt (=> no quantization).
        rec=train_run(_PS,cfg,"tai",seed,{"rho0":0.75,"full_precision":True},
                      lr=_lrstar(_PS,"fp16_control") if "_lrstar" in dir() else None)
        rec["method"]="fp16_taiinit"; save_run(rid,rec); ABL.append(rec)
        notify(f"run done: fp16abl {_PS} fp16_taiinit s{seed}",f"val {rec['val_loss']:.4f}",tag="run")
    def _vl(meth,store=None):
        src=store if store is not None else RESULTS
        return [r["val_loss"] for r in src if r["size"]==_PS and r["method"]==meth
                and r["seed"] in RUN_CFG["ablation_seeds"]]
    fp16_std=_vl("fp16_control"); fp16_tai=[r["val_loss"] for r in ABL]
    tern_std=_vl("standard");     tern_tai=_vl("tai")
    print("="*70); print(f"(#3) 2x2 INIT x PRECISION ABLATION at {_PS} "
          f"(n={len(RUN_CFG['ablation_seeds'])} seeds)"); print("="*70)
    def _line(tag,a,b):
        if len(a)>=2 and len(b)>=2:
            d=np.mean(a)-np.mean(b); p=stats.ttest_rel(a[:min(len(a),len(b))],b[:min(len(a),len(b))]).pvalue
            print(f"  {tag}: dLoss(init effect)={d:+.4f}  paired t-p={p:.4f}")
            return {"dloss":float(d),"p":float(p)}
        print(f"  {tag}: insufficient data"); return None
    FP16_INIT_ABL={
      "fp16_init_effect":_line("FP16:  std-init - TAI-init",fp16_std,fp16_tai),
      "ternary_init_effect":_line("TERN:  std-init - TAI-init",tern_std,tern_tai)}
    print("\nReading: if the FP16 init effect ~0 (CI/​p) but the TERNARY init effect is")
    print("positive, TAI's benefit is attributable to ternary OPERATING-POINT control,")
    print("not to a generically better initializer. Report BOTH numbers in the paper.")
    notify("PHASE DONE: fp16+TAI-init ablation",str(FP16_INIT_ABL))


In [ ]:
# === CELL 14: PUBLICATION FIGURES (300 dpi) + LaTeX -> DRIVE ================
if not _require("RESULTS", "ANALYSIS", "SWEEP", "PARETO"):
    pass  # prerequisites missing -> skip this phase (see message above)
else:
    import os,numpy as np,matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from collections import defaultdict
    FIG=PATHS["figures"]; RES=PATHS["results"]
    # use the largest scale that ACTUALLY has runs (declared top scale may be an
    # exploratory/!run point under the single-A100 confirmatory/exploratory split).
    _have=set(r["size"] for r in RESULTS)
    BIG=max([s for s in SIZES if s in _have],key=lambda k:SIZES[k].dim) if _have else max(SIZES,key=lambda k:SIZES[k].dim)
    rows=[r for r in RESULTS if r["size"]==BIG]; by=defaultdict(list)
    for r in rows: by[r["method"]].append(r)
    plt.figure(figsize=(6,4))
    for m,rs in by.items():
        steps=rs[0]["steps"]; M=np.array([r["loss"] for r in rs]); plt.plot(steps,M.mean(0),label=m,lw=1.5)
    plt.xlabel("step"); plt.ylabel("training loss"); plt.title(f"Convergence by init ({BIG}, mean/15 seeds)")
    plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(f"{FIG}/fig1_convergence.png",dpi=300); plt.close()
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for ax,(size,summ) in zip(axes,ANALYSIS.items()):
        ms=list(summ.keys()); ax.bar(ms,[summ[m]["val_mean"] for m in ms],yerr=[summ[m]["val_ci"] for m in ms],capsize=4)
        ax.set_title(f"Final loss — {size}"); ax.set_ylabel("val loss"); ax.tick_params(axis="x",rotation=20)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig2_final_loss.png",dpi=300); plt.close()
    plt.figure(figsize=(6,4))
    grid=sorted(set(s["rho0"] for s in SWEEP))
    means=[np.mean([s["final_sparsity"] for s in SWEEP if s["rho0"]==g])*100 for g in grid]
    sds=[np.std([s["final_sparsity"] for s in SWEEP if s["rho0"]==g],ddof=1)*100 for g in grid]
    plt.errorbar(grid,means,yerr=sds,marker="o",capsize=4)
    plt.xlabel("target initial active ratio rho0"); plt.ylabel("final learned sparsity (%)")
    # (R37) The old title -- "initial operating point causally shapes final
    # sparsity" -- asserts exactly the strong claim the data does not support:
    # ~7% of the intervention survives and every operating point collapses toward
    # ~0.31. A figure title must describe what is plotted, not pre-judge it.
    _ret=(H3_RETENTION.get("retention_fraction") if "H3_RETENTION" in dir() else None)
    plt.title("Final sparsity vs initial active ratio"
              +(f"  (retention {_ret*100:.0f}%)" if _ret==_ret and _ret is not None else ""))
    plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(f"{FIG}/fig3_rho0_sweep.png",dpi=300); plt.close()
    def esc(s): return str(s).replace("_","\\_").replace("%","\\%")
    with open(f"{RES}/results_main.tex","w") as f:
        f.write("\\begin{tabular}{lcccc}\n\\hline\nMethod & Val loss & 95\\% CI & Sparsity & Steps-to-target \\\\\n\\hline\n")
        for m,d in ANALYSIS[BIG].items():
            f.write(f"{esc(m)} & {d['val_mean']:.4f} & {d['val_ci']:.4f} & {d['spars_mean']*100:.1f}\\% & {d['s2t_mean']:.0f} \\\\\n")
        f.write("\\hline\n\\end{tabular}\n")
    # fig4: downstream lm-eval accuracy by method (only if scored)
    try:
        if LMEVAL_AVAILABLE:
            sz=BIG; summ=ANALYSIS[sz]; ms=list(summ.keys())
            arcv=[np.nanmean(summ[m].get("_arc",[float("nan")])) for m in ms]
            hsv=[np.nanmean(summ[m].get("_hsw",[float("nan")])) for m in ms]
            arcn=[np.nanmean(summ[m].get("_arcn",[float("nan")])) for m in ms]
            hswn=[np.nanmean(summ[m].get("_hswn",[float("nan")])) for m in ms]
            import numpy as _np
            if _np.any(~_np.isnan(arcv)) or _np.any(~_np.isnan(hsv)):
                x=_np.arange(len(ms)); w=0.2
                plt.figure(figsize=(8,4))
                plt.bar(x-1.5*w,arcv,w,label="ARC acc")
                plt.bar(x-0.5*w,arcn,w,label="ARC acc_norm")
                plt.bar(x+0.5*w,hsv,w,label="HSwag acc")
                plt.bar(x+1.5*w,hswn,w,label="HSwag acc_norm")
                plt.axhline(0.25,ls="--",lw=0.8,color="grey",label="chance (~0.25)")
                plt.xticks(x,ms,rotation=20); plt.ylabel("loglik accuracy")
                plt.title(f"Downstream lm-eval ({sz}, loglik mode)")
                plt.legend(fontsize=8); plt.grid(alpha=0.3,axis="y"); plt.tight_layout()
                plt.savefig(f"{FIG}/fig4_lmeval.png",dpi=300); plt.close()
                print("Saved fig4_lmeval.png")
    except Exception as _e:
        print("fig4 skipped:",str(_e)[:80])

    # fig5: H5 quality-sparsity Pareto frontier (rho0 sweep)
    try:
        import numpy as _np
        xs=[p["sparsity"]*100 for p in PARETO]; ys=[p["val_loss"] for p in PARETO]; rs=[p["rho0"] for p in PARETO]
        plt.figure(figsize=(6,4))
        plt.plot(xs,ys,"-o",lw=1.5)
        for x,y,r in zip(xs,ys,rs): plt.annotate(f"{r:.2f}",(x,y),fontsize=7,xytext=(3,3),textcoords="offset points")
        plt.scatter([RHO0_STAR["sparsity"]*100],[RHO0_STAR["val_loss"]],s=120,facecolors="none",edgecolors="red",label="rho0*")
        plt.xlabel("final learned sparsity (%)"); plt.ylabel("val loss")
        plt.title("H5: quality-sparsity Pareto frontier (TAI rho0 sweep)")
        plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(f"{FIG}/fig5_pareto.png",dpi=300); plt.close(); print("Saved fig5_pareto.png")
    except Exception as _e: print("fig5 skipped:",str(_e)[:80])
    # fig6: H6 TAI effect vs scale (per-scale mean of baseline_best - tai_warmup)
    try:
        import numpy as _np
        from collections import defaultdict as _dd
        eff_by=_dd(list)
        for lp,_s,e,_sz,_set in SCALE_ROWS: eff_by[lp].append(e)
        pts=sorted((math.exp(lp)/1e6,_np.mean(v),_np.std(v,ddof=1) if len(v)>1 else 0.0) for lp,v in eff_by.items())
        if pts:
            X=[p[0] for p in pts]; Y=[p[1] for p in pts]; E=[p[2] for p in pts]
            plt.figure(figsize=(6,4))
            plt.errorbar(X,Y,yerr=E,marker="o",capsize=4)
            plt.axhline(0,ls="--",lw=0.8,color="grey")
            plt.xscale("log"); plt.xlabel("non-embed params (M, log)"); plt.ylabel("TAI effect (nats, baseline-best - tai_warmup)")
            plt.title("H6: TAI effect across the model-scale ladder")
            plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(f"{FIG}/fig6_scale.png",dpi=300); plt.close()
            print("Saved fig6_scale.png")
    except Exception as _e: print("fig6 skipped:",str(_e)[:80])

    # =====================================================================
    # NEW CONTROL FIGURES (revision): placebo/SNR, 2x2 init ablation, two-slope.
    # Each is independently try/except-guarded so a partial run never breaks the
    # cell; each reads ONLY recompute artifacts produced by earlier cells.
    # =====================================================================

    # fig7: H3 STRONG FALSIFICATION — real vs label-shuffled placebo + RNG SNR.
    # Left panel: per-seed Spearman(rho0, sparsity) distribution, real vs placebo,
    # with the sign-test verdict. Right panel: the RNG signal/noise bar (across-
    # rho0 range vs within-rho0 across-seed SD). A reviewer sees in one glance that
    # the causal axis dominates and the placebo is flat.
    try:
        if "H3_NEGCTRL" in dir() and H3_NEGCTRL is not None:
            import numpy as _np
            fig,ax=plt.subplots(1,2,figsize=(10,4))
            # left: real vs placebo per-seed Spearman (reconstruct real from SWEEP)
            from collections import defaultdict as _dd7
            from scipy import stats as _st7
            _bys=_dd7(list)
            for _r in SWEEP: _bys[_r["seed"]].append(_r)
            real_rho=[]
            for _sd,_rows in _bys.items():
                _rows=sorted(_rows,key=lambda r:r["rho0"]); _x=[r["rho0"] for r in _rows]; _y=[r["final_sparsity"] for r in _rows]
                if len(set(_x))>2:
                    _rr,_=_st7.spearmanr(_x,_y)
                    if not _np.isnan(_rr): real_rho.append(_rr)
            placebo_mean=H3_NEGCTRL.get("placebo_mean_spearman",float("nan"))
            data=[real_rho,[placebo_mean]*max(len(real_rho),1)]
            _bxdata=[real_rho if real_rho else [0.0],[placebo_mean] if not _np.isnan(placebo_mean) else [0.0]]
            _bxlab=["TAI (real)","placebo\n(label-shuffled)"]
            try:
                bp=ax[0].boxplot(_bxdata,tick_labels=_bxlab,patch_artist=True,widths=0.5)
            except TypeError:
                bp=ax[0].boxplot(_bxdata,labels=_bxlab,patch_artist=True,widths=0.5)
            for patch,col in zip(bp.get("boxes",[]),["#2c7fb8","#bdbdbd"]): patch.set_facecolor(col)
            ax[0].axhline(0,ls="--",lw=0.8,color="grey")
            _w=H3_NEGCTRL.get("sign_test_wins"); _n=H3_NEGCTRL.get("sign_test_n"); _p=H3_NEGCTRL.get("sign_test_p")
            _sub=(f"sign-test: real>placebo in {_w}/{_n} seeds, p={_p:.3g}"
                  if _w is not None and _n else "sign-test: n/a")
            ax[0].set_ylabel("per-seed Spearman(rho0, final sparsity)")
            ax[0].set_title("H3 falsification: real vs placebo\n"+_sub,fontsize=9)
            ax[0].grid(alpha=0.3,axis="y")
            # right: RNG signal/noise bar
            sig=H3_NEGCTRL.get("across_rho0_range",float("nan"))*100
            noi=H3_NEGCTRL.get("within_rho0_seed_sd",float("nan"))*100
            snr=H3_NEGCTRL.get("snr",float("nan"))
            ax[1].bar(["signal\n(across-rho0 range)","noise\n(within-rho0 SD)"],[sig,noi],
                      color=["#2c7fb8","#bdbdbd"])
            ax[1].set_ylabel("final sparsity (% points)")
            ax[1].set_title(f"H3 negative control: SNR = {snr:.1f}x",fontsize=9)
            ax[1].grid(alpha=0.3,axis="y")
            plt.tight_layout(); plt.savefig(f"{FIG}/fig7_h3_falsification.png",dpi=300); plt.close()
            print("Saved fig7_h3_falsification.png")
        else:
            print("fig7 skipped: H3_NEGCTRL not present (sweep/negctrl not run).")
    except Exception as _e: print("fig7 skipped:",str(_e)[:80])

    # fig8: (#3) 2x2 INIT x PRECISION ablation. Grouped bars: the INIT effect
    # (std-init minus TAI-init val loss) in fp16 vs in ternary, with paired-t p.
    # The story: ~0 in fp16 (no threshold to control) but >0 in ternary => the
    # gain is operating-point control, not a generically better initializer.
    try:
        if "FP16_INIT_ABL" in dir() and FP16_INIT_ABL is not None:
            import numpy as _np
            fpe=FP16_INIT_ABL.get("fp16_init_effect"); tne=FP16_INIT_ABL.get("ternary_init_effect")
            labels=["FP16","Ternary"]
            vals=[ (fpe or {}).get("dloss",_np.nan), (tne or {}).get("dloss",_np.nan) ]
            ps  =[ (fpe or {}).get("p",_np.nan),     (tne or {}).get("p",_np.nan) ]
            plt.figure(figsize=(6,4))
            bars=plt.bar(labels,vals,color=["#bdbdbd","#2c7fb8"],width=0.55)
            plt.axhline(0,ls="--",lw=0.8,color="grey")
            plt.ylabel("init effect: loss(std-init) - loss(TAI-init)  [nats]")
            plt.title("(#3) Is the gain init-distribution or threshold control?",fontsize=10)
            for b,v,p in zip(bars,vals,ps):
                if not _np.isnan(v):
                    plt.annotate(f"d={v:+.3f}\np={p:.3g}",(b.get_x()+b.get_width()/2,v),
                                 ha="center",va="bottom" if v>=0 else "top",fontsize=8,
                                 xytext=(0,4 if v>=0 else -4),textcoords="offset points")
            plt.figtext(0.5,-0.02,"~0 in FP16 + positive in Ternary => benefit is ternary operating-point control",
                        ha="center",fontsize=8,style="italic")
            plt.grid(alpha=0.3,axis="y"); plt.tight_layout()
            plt.savefig(f"{FIG}/fig8_init_ablation.png",dpi=300,bbox_inches="tight"); plt.close()
            print("Saved fig8_init_ablation.png")
        else:
            print("fig8 skipped: FP16_INIT_ABL not present (ablation not run).")
    except Exception as _e: print("fig8 skipped:",str(_e)[:80])

    # fig9: (#2) TWO-SLOPE comparison. Effect-vs-log(params) for the MAIN ladder
    # (width+depth co-vary) overlaid with the DEPTH-CONTROLLED width-only ladder.
    # Both fitted lines + slope CIs; if the width-only slope matches sign, the H6
    # trend is not a depth artefact. Reuses SCALING_LAW + SCALING_LAW_WIDTHCTRL.
    try:
        if "SCALING_LAW" in dir() and SCALING_LAW.get("val_loss"):
            import numpy as _np
            main=SCALING_LAW["val_loss"]; wc=SCALING_LAW.get("val_loss_widthctrl")
            plt.figure(figsize=(6.5,4))
            xs=_np.linspace(main["x_min"],main["x_max"],50)
            plt.plot(xs/_np.log(10),main["slope"]*xs+main["intercept"],"-",color="#2c7fb8",lw=2,
                     label=(f"main ladder (depth+width)\nslope={main['slope']:+.3f} "
                            f"CI[{main['slope_ci'][0]:+.3f},{main['slope_ci'][1]:+.3f}]"))
            plt.fill_between(xs/_np.log(10),
                             main["slope_ci"][0]*xs+main["intercept"],
                             main["slope_ci"][1]*xs+main["intercept"],color="#2c7fb8",alpha=0.12)
            if wc:
                xw=_np.linspace(wc["x_min"],wc["x_max"],50)
                plt.plot(xw/_np.log(10),wc["slope"]*xw+wc["intercept"],"--",color="#d95f0e",lw=2,
                         label=(f"width-only (depth FIXED)\nslope={wc['slope']:+.3f} "
                                f"CI[{wc['slope_ci'][0]:+.3f},{wc['slope_ci'][1]:+.3f}]"))
                plt.fill_between(xw/_np.log(10),
                                 wc["slope_ci"][0]*xw+wc["intercept"],
                                 wc["slope_ci"][1]*xw+wc["intercept"],color="#d95f0e",alpha=0.12)
            plt.axhline(0,ls=":",lw=0.8,color="grey")
            plt.xlabel("log10 non-embed params"); plt.ylabel("TAI val-loss effect (nats, +=TAI better)")
            plt.title("(#2) Scaling slope: depth-confounded vs depth-controlled",fontsize=10)
            plt.legend(fontsize=7,loc="best"); plt.grid(alpha=0.3); plt.tight_layout()
            plt.savefig(f"{FIG}/fig9_two_slope.png",dpi=300); plt.close()
            print("Saved fig9_two_slope.png")
        else:
            print("fig9 skipped: SCALING_LAW not present.")
    except Exception as _e: print("fig9 skipped:",str(_e)[:80])

    # ---- fig10: (R6) MECHANISM COMPARISON — the paper's decisive figure --------
    # Two panels answering the question a reviewer asks first: is the contribution
    # the OPERATING POINT, or TAI's INITIALIZER?
    #   left  : final sparsity vs rho0 for BOTH arms. If both track rho0, control
    #           of the operating point is a property of the threshold, not of TAI.
    #   right : paired quality difference (TAI - gamma_offset) per rho0 with a CI
    #           band. A band straddling zero is the honest headline: matched.
    # This figure can embarrass the method, which is exactly why it belongs in the
    # paper rather than in a rebuttal.
    try:
        import numpy as _np
        if "SWEEP_ARM_ROWS" in dir() and SWEEP_ARM_ROWS.get(("dim384","gamma_offset")):
            _tai=SWEEP_BY_SCALE.get("dim384",[])
            _go =SWEEP_ARM_ROWS[("dim384","gamma_offset")]
            _grid=sorted(set(r["rho0"] for r in _tai))
            fig,axes=plt.subplots(1,2,figsize=(11,4))
            # --- left: sparsity control by both mechanisms ---
            for rows,lab,col,mk in ((_tai,"TAI (shape init)","#2166ac","o"),
                                    (_go,"gamma_offset (scale threshold)","#b2182b","s")):
                m=[_np.mean([r["final_sparsity"] for r in rows if r["rho0"]==g])*100 for g in _grid]
                sd=[_np.std([r["final_sparsity"] for r in rows if r["rho0"]==g],ddof=1)*100
                    if len([r for r in rows if r["rho0"]==g])>1 else 0.0 for g in _grid]
                axes[0].errorbar(_grid,m,yerr=sd,marker=mk,capsize=4,color=col,label=lab,lw=1.6)
            axes[0].axhline(31.0,ls=":",lw=1.0,color="grey")
            axes[0].text(_grid[0],31.6,"locked baseline (~31%)",fontsize=7,color="grey")
            axes[0].set_xlabel(r"target $\rho_0$"); axes[0].set_ylabel("final learned sparsity (%)")
            axes[0].set_title("(R6) Both mechanisms control the operating point?",fontsize=10)
            axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
            # --- right: paired quality difference at matched rho0 ---
            _ti={(r["rho0"],r["seed"]):r for r in _tai}
            _gi={(r["rho0"],r["seed"]):r for r in _go}
            means=[];los=[];his=[]
            _rng=_np.random.default_rng(0)
            for g in _grid:
                ks=[k for k in _ti if k[0]==g and k in _gi]
                d=_np.array([_ti[k]["val_loss"]-_gi[k]["val_loss"] for k in ks],float)
                if len(d)>=2:
                    bs=[_rng.choice(d,len(d),replace=True).mean() for _ in range(2000)]
                    means.append(d.mean()); los.append(_np.percentile(bs,2.5)); his.append(_np.percentile(bs,97.5))
                else:
                    means.append(_np.nan); los.append(_np.nan); his.append(_np.nan)
            means=_np.array(means); los=_np.array(los); his=_np.array(his)
            axes[1].plot(_grid,means,marker="o",color="#4d4d4d",lw=1.6)
            axes[1].fill_between(_grid,los,his,alpha=0.18,color="#4d4d4d",label="95% bootstrap CI")
            axes[1].axhline(0,ls="--",lw=1.2,color="#b2182b")
            axes[1].set_xlabel(r"target $\rho_0$")
            axes[1].set_ylabel("val loss:  TAI $-$ gamma_offset  (nats)")
            axes[1].set_title("Negative = TAI better; CI over 0 = mechanisms matched",fontsize=10)
            axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
            plt.tight_layout(); plt.savefig(f"{FIG}/fig10_mechanism.png",dpi=300); plt.close()
            print("Saved fig10_mechanism.png")
        else:
            print("fig10 skipped: no gamma_offset sweep rows.")
    except Exception as _e: print("fig10 skipped:",str(_e)[:80])

    # results_controls.tex: one compact LaTeX table summarising all three new
    # controls, ready to drop into the paper next to the main results table.
    try:
        import numpy as _np
        with open(f"{RES}/results_controls.tex","w") as f:
            f.write("\\begin{tabular}{lll}\n\\hline\nControl & Quantity & Value \\\\\n\\hline\n")
            if "H3_NEGCTRL" in dir() and H3_NEGCTRL:
                f.write(f"H3 placebo & mean per-seed Spearman (real sweep) & see Fig.~7 \\\\\n")
                f.write(f"H3 placebo & placebo mean Spearman (expected $\\approx$0) & {H3_NEGCTRL.get('placebo_mean_spearman',float('nan')):.3f} \\\\\n")
                _w=H3_NEGCTRL.get('sign_test_wins'); _n=H3_NEGCTRL.get('sign_test_n'); _p=H3_NEGCTRL.get('sign_test_p')
                if _n: f.write(f"H3 sign-test & real$>$placebo seeds & {_w}/{_n} ($p={_p:.3g}$) \\\\\n")
                f.write(f"H3 RNG SNR & across-$\\rho_0$ / within-$\\rho_0$ & {H3_NEGCTRL.get('snr',float('nan')):.1f}$\\times$ \\\\\n")
            if "FP16_INIT_ABL" in dir() and FP16_INIT_ABL:
                fpe=FP16_INIT_ABL.get("fp16_init_effect") or {}; tne=FP16_INIT_ABL.get("ternary_init_effect") or {}
                f.write(f"Init$\\times$prec & FP16 init effect (nats) & {fpe.get('dloss',float('nan')):+.4f} ($p={fpe.get('p',float('nan')):.3g}$) \\\\\n")
                f.write(f"Init$\\times$prec & Ternary init effect (nats) & {tne.get('dloss',float('nan')):+.4f} ($p={tne.get('p',float('nan')):.3g}$) \\\\\n")
            if "SCALING_LAW" in dir() and SCALING_LAW.get("val_loss"):
                m=SCALING_LAW["val_loss"]
                f.write(f"Scaling slope & main ladder (depth+width) & {m['slope']:+.4f} CI[{m['slope_ci'][0]:+.3f},{m['slope_ci'][1]:+.3f}] \\\\\n")
                wc=SCALING_LAW.get("val_loss_widthctrl")
                if wc: f.write(f"Scaling slope & width-only (depth fixed) & {wc['slope']:+.4f} CI[{wc['slope_ci'][0]:+.3f},{wc['slope_ci'][1]:+.3f}] \\\\\n")
            f.write("\\hline\n\\end{tabular}\n")
        print("Saved results_controls.tex")
    except Exception as _e: print("results_controls.tex skipped:",str(_e)[:80])

    notify("PHASE DONE: figures+tables","persisted to Drive")
    print("Saved to Drive:",os.listdir(FIG),"+ results_main.tex + results_controls.tex")


In [ ]:
# === CELL 15: MANIFEST + SCALE-UP + FINAL EMAIL ============================
if not _require("RESULTS", "ANALYSIS"):
    pass  # prerequisites missing -> skip this phase (see message above)
else:
    import json,hashlib,sys,torch,os
    def _lr_boundary_cells():
        """(R33) Cells whose LR* is still on a probed-grid boundary. A missing or
        unreadable summary must not take the manifest down, so every lookup is
        guarded; an unreadable cell is reported rather than silently dropped."""
        out=[]
        for k in (LRSTAR or {}):
            try: rec=load_run(run_id("lrcal",k[0],k[1],0)) if have_run(run_id("lrcal",k[0],k[1],0)) else None
            except Exception: rec=None
            if rec is None: out.append(f"{k[0]}/{k[1]}:unknown")
            elif not rec.get("lr_interior",True): out.append(f"{k[0]}/{k[1]}")
        return out
    manifest={"method":"Threshold-Aware Initialization (TAI) + gamma-warmup",
      # ====================================================================
      # (R36) CONTRIBUTION CLAIM — FROZEN MID-RUN, BEFORE ANY HYPOTHESIS TEST.
      # --------------------------------------------------------------------
      # Written while the main grid was at dim640 of 11 scales, with roughly 90
      # GPU-hours still to run and NOT ONE statistic from Cell 13 computed. It is
      # recorded here, dated, for the same reason the hypotheses were pre-
      # registered: a claim written after seeing every number is unfalsifiable,
      # and a reviewer has no way to tell the difference.
      #
      # WHY IT CHANGED FROM THE ORIGINAL FRAMING. The pre-registration expected
      # TAI to win. The calibration and the completed confirmatory cells say
      # otherwise, consistently across five scales at each arm's own optimal LR:
      #   tai - gamma_offset : +0.02 .. +0.15  (TAI WORSE; gap GROWS with scale)
      #   tai - standard     : +0.06 .. +0.13  (TAI WORSE)
      #   tai_warmup - standard : -0.13 .. -0.17 (WARMUP BETTER, stable)
      # So the effect is real but it is not the effect that was hypothesised: the
      # gain does not come from where the operating point starts, it comes from
      # moving the threshold gradually. The mechanism-matched control settles the
      # attribution -- gamma_offset reaches the same operating point via threshold
      # scaling and beats the init-distribution route at every scale, so the
      # active ingredient is the SCHEDULE, not the initial weight distribution.
      #
      # The claim is stated so that it survives either direction of the remaining
      # evidence: if the warmup gap widens or narrows, the sentence still holds;
      # what would falsify it is the gap crossing zero or gamma_offset losing to
      # tai. Both remain possible and both are reportable.
      "contribution_claim":(
          "We show that the initial active ratio rho0 in ternary pre-training is "
          "locked near 0.69 for every zero-mean initialiser, independently of sigma, "
          "and make it tunable. In a pre-registered comparison over 1.4M-352M "
          "parameters with 8 arms and 15 seeds we find that MOVING THE INITIAL "
          "OPERATING POINT DEGRADES QUALITY, while approaching the same point "
          "GRADUALLY over training improves it consistently. A mechanism-matched "
          "control (TWN-style constant threshold scaling) beats the "
          "init-distribution variant at every scale, locating the effect in the "
          "TIMING of the threshold rather than in the initial weight distribution."),
      "claim_frozen_at":{"when":"main grid at dim640; ~90 GPU-hours remaining",
                         "statistics_computed":False,
                         "evidence_basis":"LR calibration (88/88 cells) + completed "
                                          "confirmatory cells dim192-dim512, seed means only",
                         "note":"No Cell 13 hypothesis test had been run when this was written."},
      # (R36) Prior art this claim must be read against. gamma_offset is NOT an
      # invention of this work: constant threshold scaling at Delta = c*E|W| is
      # TWN (Li & Liu 2016), and the paper names it as such rather than presenting
      # it as a control of our own design. Gradual threshold increase is
      # Hyperspherical LATQ (Liu et al. 2022, t: 0.3 -> 0.7) in the CNN setting.
      # Ternary Mamba (arXiv 2606.18114, June 2026) independently reports that a
      # single asymmetric threshold moved sparsity 25.7% -> 20.9% and cost +0.12
      # PPL -- the same direction as our tai result, in a different architecture
      # and a QAT-from-pretrained regime. We cite it as independent confirmation.
      # What is new here is the three-way separation -- move it at init, move it by
      # threshold, move it on a schedule -- in from-scratch LLM pre-training, with
      # a placebo arm and 15 seeds.
      "claim_prior_art":["TWN (Li & Liu 2016) = the gamma_offset arm",
                         "TTQ (Zhu et al. ICLR 2017) = fixed sparsity ratio as a hyperparameter",
                         "Hyperspherical LATQ (Liu et al. 2022) = gradual threshold, CNN regime",
                         "Sparse TNN (2017) = threshold as an explicit sparsity control",
                         "BitNet b1.58 (2024) = absmean, Delta=gamma/2 -> rho0~0.69",
                         "Ternary Mamba (2606.18114, 2026) = independent confirmation, SSM/QAT"],
      "design":{"sizes":list(SIZES.keys()),"methods":[m for m,_ in METHODS],"seeds":SEEDS,
                "confirmatory_scales":RUN_CFG.get("confirmatory_scales"),
                "exploratory_scales":RUN_CFG.get("exploratory_scales"),
                "confirmatory_scale_primary":RUN_CFG.get("confirmatory_scale"),
                "primary_baseline":RUN_CFG.get("primary_baseline"),
                "secondary_baselines":RUN_CFG.get("secondary_baselines"),
                "primary_test_family":RUN_CFG.get("primary_family"),
                "tost_margin_frac":RUN_CFG.get("tost_margin_frac"),
                "h6_primary_readout":"per-scale sign-consistency (binomial); slope is secondary",
                "h3_negative_control":H3_NEGCTRL if "H3_NEGCTRL" in dir() else None,
                "token_budget":TOKEN_BUDGET,"steps_per_run":STEPS,"rho0_grid":RUN_CFG["rho0_grid"],"tost_margin":RUN_CFG["tost_margin"],"plateau_tol":RUN_CFG["plateau_tol"],
                "target_loss_H1":TARGET_LOSS,"bonferroni_alpha":RUN_CFG["bonferroni_alpha"]},
      "env":{"python":sys.version.split()[0],"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0)},
      "data":DATA_META_INFO,
      "used_synthetic_fallback":bool(USED_SYNTHETIC),
      "lmeval_available":bool(LMEVAL_AVAILABLE),
      "pareto":PARETO if "PARETO" in dir() else None,
      "rho0_star":RHO0_STAR if "RHO0_STAR" in dir() else None,
      "scale_model":SCALE_REG if "SCALE_REG" in dir() else None,
      "efficiency":EFF if "EFF" in dir() else None,
      # (R3) NOTE: plateau_ok now means "val-trace slope indistinguishable from zero
      # in both directions", NOT the old noise-dominated level test. Counts are not
      # comparable with any pre-revision manifest.
      "plateau_verified_runs":sum(1 for r in RESULTS if r.get("plateau_ok")),
      "plateau_gate_definition":("slope test with 95% CI on held-out val trace over the last "
                                 "plateau_frac of probes; rejects descending (undertrained) and "
                                 "ascending (unstable) tails; requires >= plateau_min_tail probes"),
      "runs_with_val_trace":sum(1 for r in RESULTS if r.get("val_trace")),
      "cross_architecture_v2":ARCH_SUMMARY if "ARCH_SUMMARY" in dir() else None,
      "external_validity_sign":EXT_VALIDITY if "EXT_VALIDITY" in dir() else None,
      "rho0_star_feedback":{"rho0_star":RHO0_STAR["rho0"] if "RHO0_STAR" in dir() else None,
                            "tai_star_runs":sum(1 for r in RESULTS if r.get("method")=="tai_star")},
      "sparsity_schedule_baseline":"sparse_sched (gradual-pruning, recipe=sched)",
      "h3_strong_falsification":{k:H3_NEGCTRL.get(k) for k in
            ("placebo_mean_spearman","sign_test_wins","sign_test_n","sign_test_p")} if "H3_NEGCTRL" in dir() else None,
      # ---- (R0-R7) REVISION RECORD. Every entry below exists because a specific
      #      defect was found and fixed; the manifest states what changed so a
      #      reader can tell this run apart from any pre-revision artefact.
      "revisions":{
        "R20_shadowed_lsuv_definitions":("cell 8 defined init_xavier/init_kaiming/init_lsuv and "
                            "lsuv_calibrate TWICE; python keeps the LAST binding, so the version "
                            "that ran was the older pre-R13 one -- R13 was silently undone inside "
                            "its own cell. Consequences: (a) LSUV_DEGENERATE was NEVER written (all "
                            "three assignments live in the first definition), so this manifest field "
                            "reported nulls under prose claiming the value was measured, and the "
                            "LSUV-is-degenerate finding was unmeasurable; (b) the preflight check "
                            "'LSUV_DEGENERATE.is_noop == true' could never pass; (c) tol reverted "
                            "0.01 -> 0.10, exactly the loosening R13 documented tightening. "
                            "Duplicate block removed. Found by grepping for duplicate top-level "
                            "definitions -- invisible to linear reading."),
        "R25_compile_enabled_on_measurement":("torch.compile turned ON after direct measurement on "
                            "the target A100, not on expectation: 2.03x at dim512, 1.71x at dim384, "
                            "with an eager-vs-compiled val_loss delta of 0.051 against a same-code "
                            "run-to-run noise floor of 0.029 -- the first benchmark had flagged 0.058 "
                            "as disqualifying, but that comparison HAD NO CONTROL, and flash attention "
                            "is already non-deterministic. Measuring the noise floor reversed the "
                            "decision. Compile also lowered peak VRAM 10.09 -> 8.88 GB. The preflight "
                            "A/B re-verifies on the real corpus before the grid starts. Also tested "
                            "and REJECTED: casting normalisation weights to fp32 to recover the fused "
                            "rms_norm kernel (the profile showed aten::copy_ at 16% of CUDA time and a "
                            "dtype-mismatch warning every step). It was SLOWER -- 0.88x eager, 0.97x "
                            "compiled, and +1.7 GB -- because the extra fp32<->bf16 conversions cost "
                            "more than the unfused kernel. Under compile the issue is moot: inductor "
                            "fuses the norm regardless. Hypothesis tested, disproved, not applied."),
        "R27_embedding_never_initialised":("Found in Phase 0. nn.Embedding defaults to "
                            "normal_(0,1) and is TIED to the output head, so initial logits had "
                            "std ~ sqrt(dim) and the loss at step 0 was 121.6 / 211.4 / 351.9 at "
                            "dim192 / dim384 / dim768, against a uniform-prediction baseline of "
                            "log(50257)=10.83. apply_init walks BitLinear layers only; nothing ever "
                            "touched the embedding, so all eight arms inherited the PyTorch default. "
                            "This initially looked like divergence -- losses of 50-120 that grew with "
                            "model size -- but it was the STARTING point, not a blow-up: every run "
                            "was descending correctly from an absurd origin, and the apparent "
                            "scaling with model size was just sqrt(dim). Why it matters here more "
                            "than in an ordinary training script: (a) a large slice of the FIXED 20M "
                            "token budget goes to shrinking the embedding norm before any language "
                            "modelling starts, and that budget was chosen to reach a plateau; (b) the "
                            "waste grows as sqrt(dim), biasing H6 and the scaling-law fit in a "
                            "direction unrelated to rho0; (c) this is a paper about INITIALISATION, "
                            "and a large init-driven transient common to all arms dilutes the effect "
                            "under test -- it could mask a real TAI advantage as easily as fabricate "
                            "one. Set to the standard GPT-2 std=0.02, giving initial loss 10.88/10.92/"
                            "11.00 -- at the baseline and, importantly, scale-independent. A bug fix "
                            "rather than a tuning choice: the prior state was 'uninitialised'. "
                            "Identical across arms and touches no BitLinear, so the ternary mechanism "
                            "under test is unaffected."),
        "R37_h3_narrowed_to_retention":("Recorded from the dim384 sweep BEFORE the dim768 sweep, "
                            "the placebo arm, or any Cell 13 statistic had run, so the revised "
                            "wording is not a description of a finished result. Sweeping rho0 from "
                            "0.40 to 0.70 changes INITIAL sparsity by thirty points (0.60 -> 0.30) "
                            "and leaves TWO points at the end (0.3240 -> 0.3025). A linear fit gives "
                            "final = 0.2800 + 0.0714 * initial with R^2 = 0.978: the retention "
                            "coefficient is 0.071, so roughly 93% of the intervention is undone by "
                            "training, and the implied fixed point 0.3015 is essentially where the "
                            "`standard` arm lands unaided (~0.313). H3 AS PRE-REGISTERED -- that the "
                            "initial operating point causally SHAPES final sparsity -- is therefore "
                            "NOT supported in its strong form. What the data do support is narrower: "
                            "a small, strictly ordered, extremely reliable residue (rho0 0.40 vs "
                            "0.70 gives +0.0216, p=8.5e-15, d=44.6, against a within-seed sparsity SD "
                            "of 0.0004) sitting on top of an attractor near 0.30 that training "
                            "reaches from wherever it starts. This reframes the paper rather than "
                            "weakening it: the locked ~0.69 value is not merely a default every "
                            "zero-mean initialiser happens to share, it is where the optimisation "
                            "GOES -- which explains H1, since quality improves monotonically as rho0 "
                            "moves back toward the locked point (5.2519 -> 5.1901), exactly why `tai` "
                            "loses to `standard` while `tai_warmup`, which approaches the same point "
                            "gradually, wins. This is a REPORTING change, not an analysis change: the "
                            "sweep, the placebo arm and every test are untouched, and retention is "
                            "computed for every swept (scale, arm) including tai_negctrl, whose "
                            "retention slope remains the falsification instrument -- a placebo slope "
                            "comparable to the real one would mean even the narrowed effect is "
                            "artefact."),
        "R44_manifest_depended_on_a_helper_from_another_cell":("R42's json_safe() lives in Cell 10 "
                            "with the run store, and Cell 15 called it by name. That works in a "
                            "linear Run All and fails the moment a cell is skipped, re-ordered, or "
                            "the analysis is re-run in a fresh kernel from the run store -- which is "
                            "the normal way to regenerate artifacts without a GPU. The manifest is "
                            "the last output of a multi-day experiment; losing it to a missing "
                            "four-line helper is a failure mode worth removing outright, so Cell 15 "
                            "now falls back to a local definition. Found by the offline dry-run "
                            "harness, not by a session."),
        "R45_h2_companion_token_equal_test":("The pre-registered H2 is an equivalence claim about "
                            "CONVERGED quality, tested by TOST over plateau-verified paired seeds. "
                            "With 0 of 760 runs reaching plateau it was skipped at every scale, so "
                            "H2 -- one of the four primary hypotheses -- went entirely unreported. "
                            "The pre-registered verdict remains UNTESTABLE and is printed and stored "
                            "as such; it is not overwritten. Alongside it we now run a clearly "
                            "labelled COMPANION test of the weaker claim the data supports: "
                            "equivalence at a fixed 20M-token budget over all paired seeds. Three "
                            "reasons this is not a post-hoc rescue. (1) The re-statement is forced "
                            "by a measurement -- the budget does not reach plateau, confirmed "
                            "independently by the 2x-budget check -- not by an outcome. (2) It does "
                            "not favour the method: it rejects equivalence for BOTH arms, reporting "
                            "tai as WORSE and tai_warmup as BETTER by roughly ten to thirty times "
                            "the margin at every scale. A substitution that makes the eponymous arm "
                            "fail is not the shape p-hacking takes. (3) Both results go into the "
                            "manifest under separate keys (h2_prereg_status, "
                            "h2_token_equal_companion), so which test was pre-registered and which "
                            "was added is never in doubt."),
        "R46_offline_dryrun_harness":("Process change, prompted by six consecutive crashes (R38-R43) "
                            "in the analysis chain, each costing a Colab session to discover. The "
                            "common cause was structural: those code paths only become reachable "
                            "once the full grid exists, so no smoke test could reach them. Now that "
                            "the run store is complete, dryrun.py executes cells 12 through 16 "
                            "against the REAL 3016-run store on CPU in seconds, with the GPU, Drive "
                            "and network layers stubbed and the cell source otherwise unmodified. It "
                            "immediately found R44 and would have caught R38-R43. Every future edit "
                            "to the analysis chain is validated end-to-end before it reaches a GPU "
                            "session."),
        "R43_h6_slope_singular_and_the_obvious_repair_was_wrong":("The pre-registered H6 estimator, "
                            "mixedlm(eff ~ logp, groups=seed), failed on the completed grid with "
                            "'Random effects covariance is singular' and fell through to a "
                            "descriptive OLS that printed a slope WITH NO INTERVAL. A point estimate "
                            "without uncertainty is not a result, so H6 was in effect unreported. "
                            "Cause: the seed-level random intercept has almost nothing to estimate -- "
                            "between-seed SD of the effect is 0.017 against a between-scale SD of "
                            "0.046 -- so the covariance collapses to the boundary. The tempting "
                            "repair, OLS with SEs clustered on seed, is worse than the failure "
                            "because it answers the wrong question: H6 asks whether the effect varies "
                            "WITH SCALE, and the unit of that question is the scale, of which there "
                            "are 11. Clustering on seed treats the scales as independent replicates "
                            "of one line and ignores scale-level residual structure, which here is as "
                            "large as the seed noise (scale-level residual SD 0.042 vs within-scale "
                            "seed SD 0.030). The consequence is not cosmetic -- all three estimators "
                            "return the SAME slope (-0.0126) and opposite conclusions: clustered on "
                            "seed CI [-0.0181,-0.0071] excludes zero, clustered on scale "
                            "[-0.0236,-0.0016] excludes zero, and regression on the 11 scale means "
                            "[-0.0291,+0.0037] SPANS zero. Picking the narrowest interval would "
                            "manufacture a scale-dependence finding out of an analysis choice. The "
                            "conservative scale-level regression is therefore PRIMARY, the other two "
                            "are reported beside it as sensitivity analyses, all of them go into the "
                            "manifest, and the code prints an explicit warning when they disagree "
                            "about whether the CI covers zero. Operative reading for the paper: the "
                            "direction of the tai_warmup effect is consistent across all 11 scales "
                            "(that is the PRIMARY H6 sign test, which is unaffected), but the "
                            "magnitude is NOT resolved as scale-varying."),
        "R42_manifest_could_not_serialise_tuple_keys":("The manifest write -- the LAST artifact "
                            "produced, after the entire compute budget has been spent -- died with "
                            "\"keys must be str, int, float, bool or None, not tuple\". LRSTAR and "
                            "LR_CURVE, added to the manifest by R33, are keyed by the tuple "
                            "(size, method), and json.dumps rejects non-scalar KEYS outright; the "
                            "`default=str` fallback already in place does not help, because it is "
                            "consulted for unknown VALUES only. Every phase had completed and the "
                            "provenance record for the whole experiment was discarded at the final "
                            "line. Fixed at source (both fields flattened to 'size/method') and, "
                            "because auditing forty-odd manifest fields by hand is a check that rots "
                            "the moment someone adds a dict, with a shared json_safe() helper that "
                            "recursively normalises keys, applied to the manifest and to the "
                            "efficiency dump. A second failure surfaced while testing the first: "
                            "json.dumps(sort_keys=True), needed for a stable hash, cannot order a "
                            "dict mixing int and str keys either -- so keys are coerced to STRINGS "
                            "rather than merely to json-legal types, which makes the ordering total "
                            "and the hash reproducible. Verified against the real run store: 88 "
                            "tuple-keyed calibration cells serialise cleanly and hash stably."),
        "R41_module_level_names_shadowed_import_aliases":("Cell 12 assigned the count of "
                            "architecture-x-scale cells to the name `nn`, which is the module-level "
                            "alias for torch.nn established in Cell 5. Notebook cells share ONE "
                            "namespace, so from that line on torch.nn WAS AN INTEGER, and the next "
                            "phase to construct a model -- the width-control ladder -- died inside "
                            "LM.__init__ with \"'int' object has no attribute 'Embedding'\". The "
                            "traceback pointed at the model, hundreds of lines from the cause, and "
                            "the same crash would have hit the cross-architecture and ablation "
                            "phases. An AST scan of all 60 import aliases found two more of the same "
                            "kind: `_t` (the `import time as _t` alias) was rebound both by the "
                            "quantiser self-test in Cell 5 and by a t-statistic in Cell 12, and `_m` "
                            "(`import math as _m`) was rebound by a demo model in Cell 8. None had "
                            "surfaced because the colliding cells happened to run before the alias "
                            "was first needed -- pure luck of ordering, not correctness. All four "
                            "renamed (_n_cells, _tq, _tstat, _mdl), and the check is now permanent: "
                            "verify_patch.py fails if any module-level assignment or for-target "
                            "rebinds an import alias anywhere in the notebook."),
        "R40_statistics_iterated_over_unrun_scales":("The primary statistics cell looped over SIZES, "
                            "the full 12-point ladder, rather than over the scales that were actually "
                            "run. SIZES includes dim2048 (783M), which is opt-in and absent from "
                            "SCALES_TO_RUN, so for that scale rows was empty, the per-method summary "
                            "stayed empty, and the primary-baseline fallback called min() on an empty "
                            "sequence: ValueError, raised AFTER all eleven completed scales had "
                            "already been analysed and printed, discarding the entire cell's work. "
                            "The loop now runs over scales with at least one record and announces "
                            "the skip instead of passing over it silently, and the fallback itself "
                            "guards against an empty candidate set so a baseline can never be chosen "
                            "from nothing. Verified against the real store with dim2048 present in "
                            "SIZES: eleven scales analysed, one skipped, no exception. This is the "
                            "third failure in the same family as R38 and R39 -- code paths that only "
                            "execute once the full grid exists, and therefore could not be reached "
                            "until the grid finished."),
        "R39_diverged_runs_declared_only_on_the_dead_path":("DIVERGED_RUNS was initialised on a line "
                            "that lives inside the ARM_FULL_RUN=False branch -- the safe-mode path "
                            "where nothing trains. On any real run the name never came into "
                            "existence, so R26's collector and R38's invariant assert both referenced "
                            "an undefined global and the cell died with NameError AFTER the whole run "
                            "store had been reloaded. The bug was latent from R26 and only surfaced "
                            "when R38's assert reached the name on a path R26's own branch never "
                            "took. All three accumulators are now declared unconditionally at module "
                            "level, guarded so that re-running the cell inside a live session does "
                            "not wipe a list an earlier pass filled. Second defect fixed in the same "
                            "place: the collection loop reset RESULTS but not DIVERGED_RUNS or "
                            "PLATEAU_STATS, so a second pass over the store in one session would "
                            "append duplicates and trip R38's invariant with a confusing off-by-N. "
                            "All three now reset together. Verified end-to-end against the real "
                            "store on a fresh namespace: 760 records collected, assert passes, and a "
                            "second pass leaves the count unchanged."),
        "R38_results_collection_lost_99_percent_of_the_grid":("A one-level indentation error, "
                            "introduced by the R26 divergence filter, put the collection block in the "
                            "`method` loop instead of the `seed` loop. After the seed loop finished, "
                            "only the LAST seed's record was examined, so exactly ONE run per "
                            "(scale, method) entered RESULTS instead of all 5 or 15. Verified against "
                            "the real store: 760 completed runs would have yielded 88 records. Every "
                            "confirmatory test consumes RESULTS, so H1/H2/H4 would have been computed "
                            "at n=1 per cell -- paired tests would have collapsed or thrown, and any "
                            "number that survived would have been meaningless. Entirely silent: 88 is "
                            "a plausible-looking count and nothing asserted the expected size. Fixed, "
                            "and an assert now checks the invariant "
                            "len(RESULTS)+len(DIVERGED_RUNS)==sum over scales of methods*seeds, so a "
                            "future edit that breaks the collection fails loudly instead of quietly "
                            "producing statistics over a hundredth of the data. This is the same "
                            "failure mode as R15 and R16 -- code that runs, produces plausible "
                            "output, and is wrong -- and it was introduced by a fix, not by the "
                            "original design, which is exactly why the invariant is now asserted "
                            "rather than trusted."),
        "R37_h3_tested_the_wrong_sign_and_the_wrong_quantity":("Two problems, found when the first "
                            "rho0 sweep cells completed and BEFORE the second sweep scale had run. "
                            "(1) WRONG SIGN. rho0 is the initial ACTIVE ratio, so a higher rho0 means "
                            "FEWER zeros and therefore LOWER sparsity: the relationship is "
                            "necessarily decreasing. The pre-registered code tested whether the "
                            "per-seed Spearman was centered ABOVE zero and counted seeds that were "
                            "monotone NON-DECREASING in rho0 -- both the opposite expectation. "
                            "Measured on dim384: per-seed Spearman is -1.000 on all 15 seeds and the "
                            "old monotonicity counter reads 0.00. This was not benign: the old "
                            "one-sample t-test returns t=-inf, p=0.0, so a reader scanning for a "
                            "small p-value would have recorded H3 as strongly SUPPORTED while the "
                            "monotonicity number beside it said 0.00 -- two headline figures in "
                            "direct contradiction, with the significant-looking one pointing the "
                            "wrong way. The test now checks the mechanism's actual prediction and "
                            "states the expected direction in the printed output. "
                            "(2) WRONG QUANTITY. A Spearman of -1 says the ORDER is preserved and "
                            "nothing about MAGNITUDE, which is the entire claim. On dim384 a 30-point "
                            "swing in initial sparsity (rho0 0.4->0.7, i.e. 0.60->0.30) leaves a "
                            "2.15-point spread in final sparsity (0.3240->0.3025): roughly 7% of the "
                            "intervention survives, and all four operating points collapse toward "
                            "~0.31 -- the same value the `standard` arm reaches unaided. So the "
                            "locked point is not merely a default, it is an ATTRACTOR. H3 is "
                            "therefore NOT supported in its strong pre-registered form. A retention "
                            "fraction (final swing / initial swing) is now computed as a first-class "
                            "number, printed, and stored in the manifest as h3_retention, with "
                            "h3_operative_wording derived from it so the paper cannot lean on the "
                            "correlation alone. The fig5 title, which asserted 'initial operating "
                            "point causally shapes final sparsity', now describes what is plotted and "
                            "reports the retention instead of pre-judging it. Note this also explains "
                            "H1: quality improves monotonically as rho0 approaches the locked value, "
                            "which is why the tai arm loses to standard -- moving away from the "
                            "attractor costs quality, and that is the coherent reading of all three "
                            "results together."),
        "R36_contribution_claim_frozen_mid_run":("The contribution claim was rewritten and FROZEN "
                            "while the main grid was at dim640, with ~90 GPU-hours still to run and "
                            "not a single Cell 13 statistic computed. Recorded here, dated, for the "
                            "same reason the hypotheses were pre-registered: a claim written after "
                            "seeing every number is unfalsifiable and a reviewer cannot tell the "
                            "difference. It CHANGED from the pre-registered framing, which expected "
                            "TAI to win. Across five scales, at each arm's own optimal LR, the "
                            "completed cells give tai - gamma_offset = +0.02..+0.15 (TAI worse, gap "
                            "growing with scale), tai - standard = +0.06..+0.13 (TAI worse), and "
                            "tai_warmup - standard = -0.13..-0.17 (warmup better, stable). The effect "
                            "is real but it is not the hypothesised effect: the gain comes from "
                            "moving the threshold gradually, not from where the operating point "
                            "starts. The mechanism-matched control settles the attribution, since "
                            "gamma_offset reaches the same operating point by threshold scaling and "
                            "beats the init-distribution route at every scale -- so the active "
                            "ingredient is the SCHEDULE, not the initial weight distribution. The "
                            "claim is phrased to survive either direction of the remaining evidence; "
                            "what would falsify it is the warmup gap crossing zero or gamma_offset "
                            "losing to tai, and both remain possible. Prior art is named in "
                            "claim_prior_art rather than left for a reviewer to find: gamma_offset is "
                            "TWN (Li & Liu 2016) and is called that in the paper, gradual threshold "
                            "increase is Hyperspherical LATQ (Liu et al. 2022) in the CNN setting, "
                            "and Ternary Mamba (arXiv 2606.18114, June 2026) independently reports "
                            "that a single asymmetric threshold moved sparsity 25.7% -> 20.9% at a "
                            "cost of +0.12 PPL -- the same direction as our tai result in a different "
                            "architecture and regime, cited as confirmation rather than competition. "
                            "What is new is the three-way separation -- move the operating point at "
                            "init, by threshold, or on a schedule -- in from-scratch LLM "
                            "pre-training, with a placebo arm and 15 seeds."),
        "R35_h2_wording_token_equal_not_converged":("Recorded mid-run, from measurement, BEFORE any "
                            "hypothesis test was computed. The pre-registration describes 20M tokens "
                            "as a 'plateau-verified budget' and phrases H2 in terms of converged "
                            "quality. The first 502 main-grid runs returned plateau_ok on ZERO of "
                            "them: at dim384 the validation trace still falls ~0.21 nats across the "
                            "final 40% of probes, and the picture is uniform across scales (-0.20 to "
                            "-0.27). The plateau gate is not broken -- it is correctly reporting that "
                            "the models are still learning when the budget ends. Nothing about the "
                            "comparison is invalidated: a token-equal comparison at a fixed budget is "
                            "a legitimate protocol and every arm is treated identically. What is "
                            "invalidated is the WORDING. Every claim must read 'at a fixed 20M-token "
                            "budget'; none may say 'at convergence' or 'converged quality'. H2's TOST "
                            "is defined over plateau-verified pairs and is therefore reported as "
                            "SKIPPED rather than computed over an empty set -- the statistics cell "
                            "already guards n<2, so this is a reporting change, not a code change. "
                            "The budget itself is deliberately NOT moved: it is pre-registered, and "
                            "extending it now to manufacture a plateau would be precisely the "
                            "post-hoc adjustment this harness exists to prevent. Cell 11c's 2x-budget "
                            "check remains the pre-registered instrument for asking whether the "
                            "ranking is budget-dependent. Added: a PLATEAU_STATS counter that makes "
                            "the situation visible in the log and in the manifest "
                            "(plateau_status, h2_operative_wording) rather than leaving it to be "
                            "discovered by reading 500 run files."),
        "R33_lr_grid_widened_and_adaptive":("Session-1 calibration found the optimum ON A GRID "
                            "BOUNDARY in 23 of the 32 confirmatory (scale, method) cells: every arm "
                            "at dim256 and dim384 chose the top point 3e-3 with the curve still "
                            "descending, and every arm at dim1280/dim1536 chose the bottom point "
                            "1e-3. A boundary optimum alone would be a tolerable limitation; what is "
                            "not is that the boundary SLOPES differ by arm. At dim384 the 2e-3 to "
                            "3e-3 gain is +0.026 for gamma_offset and +0.046 for tai, a 1.8x "
                            "difference, and the sign of that asymmetry flips at dim256. Since "
                            "gamma_offset is the mechanism-matched control that decides whether "
                            "TAI's contribution is the initialisation distribution or merely "
                            "threshold control, a ranking sensitive to where the grid happened to "
                            "stop would put the paper's central claim on an artifact -- and no "
                            "post-hoc argument recovers it. Grid widened at both ends to "
                            "[5e-4, 1e-3, 2e-3, 3e-3, 5e-3], plus adaptive extension: an argmin on "
                            "an end point triggers one probe further out (x2 or /2), up to twice, so "
                            "'the optimum lies outside the grid' becomes a measured fact rather than "
                            "an untestable caveat. Each summary record now stores lr_interior and the "
                            "full probed grid, so any cell still pinned at a boundary is visible in "
                            "the manifest instead of silent. Cost is modest because the run store "
                            "keys on (scale, method, seed, lr): all 747 completed calibration runs "
                            "are reused and only the new points are trained (~498 short runs). Stale "
                            "LR* SUMMARY records are deleted automatically -- without that the cached "
                            "three-point curve would short-circuit the whole change and the widening "
                            "would silently do nothing. Also added: _validate_lr_proxy, which re-runs "
                            "one calibration curve at FULL length at a single scale and seed and "
                            "compares argmin and Spearman rank against the 300-step proxy. R22 chose "
                            "300 steps to save ~32% of compute, but with 120 warmup steps that is "
                            "~180 post-warmup steps of evidence applied to 1221-step runs, and high "
                            "LRs flatter themselves early -- the same direction as the boundary "
                            "problem, so the two confounds compound. The check is a few runs and "
                            "converts an assumption into evidence the paper can cite; if it "
                            "disagrees, lr_cal_steps must rise before the main grid is trusted."),
        "R34_compile_only_for_long_runs":("Session-1 timings showed torch.compile was a net LOSS on "
                            "calibration runs. At dim1280 a 300-step run took 285 s: ~71 s training, "
                            "~28 s evaluation, ~190 s model construction plus compilation. The "
                            "decisive evidence that compilation is paid PER RUN rather than once is "
                            "that the nine runs within a cell took 285, 285, 285... to the second, "
                            "with no warm-up effect -- each run builds a fresh LM, so dynamo's guards "
                            "miss and inductor re-lowers the graph, and the on-disk cache does not "
                            "rescue a fresh module instance. The controlled 2.0-2.5x benchmark was "
                            "not wrong, it measured the steady-state step, which a 300-step run "
                            "barely reaches. Compile is now conditional on run length "
                            "(compile_min_steps=600): short calibration runs go eager, full "
                            "1221-step runs stay compiled. Numerics are unaffected -- the earlier "
                            "noise-floor test showed eager and compiled differ by less than "
                            "run-to-run non-determinism -- so this is purely a throughput decision, "
                            "now taken per run rather than globally. This also explains why session 1 "
                            "ran ~2.4x slower than projected: the projection was built from "
                            "steady-state timings and never modelled a per-run fixed cost."),
        "R32_orthogonal_init_has_no_bfloat16_kernel":("Crashed the first real grid 9 minutes in. "
                            "init_lsuv calls torch.nn.init.orthogonal_, which runs a QR factorisation, "
                            "and the LAPACK/cuSOLVER routine behind it (geqrf) has no bfloat16 kernel "
                            "on CPU or CUDA. Models are built with .to(DEVICE, DTYPE) with "
                            "DTYPE=bfloat16, so the run died with NotImplementedError: 'geqrf_cuda' "
                            "not implemented for 'BFloat16' the moment the calibration loop reached "
                            "the first lsuv arm. Only lsuv was affected -- every other init uses "
                            "elementwise ops bfloat16 supports -- which is why fp16_control and "
                            "standard finished first and the failure appeared to come out of nowhere. "
                            "It survived the CPU checks because those exercised the model in float32; "
                            "the dtype, not the logic, was the variable. Fixed by computing the QR on "
                            "an fp32 scratch tensor and copying back: columns stay orthonormal through "
                            "the round-trip (max |QtQ - I| ~ 1e-3, bf16 rounding rather than a broken "
                            "factorisation) and the resulting std matches the 1/sqrt(fan_in) target to "
                            "five decimals. The algorithm is unchanged; only the dtype the "
                            "factorisation runs in. Verified end-to-end on a bf16 model: all six init "
                            "functions now run, and lsuv_calibrate completes with LSUV_DEGENERATE "
                            "populated (24 rescale ops, is_noop False) -- the first time the R13/R20 "
                            "degeneracy measurement has actually produced a value."),
        "R31_console_tee_and_export_layer":("Two publication artifacts existed only as console "
                            "output or not at all. (a) The statistics cell prints every H1-H8 verdict "
                            "-- supported or not, p, Cohen's d, CI, Holm-corrected -- to stdout and "
                            "nowhere else, across a grid spanning 3-4 sessions; a disconnect or a "
                            "runtime restart would erase the single most important artifact for the "
                            "paper. stdout is now mirrored to logs/console_YYYYMMDD.log on Drive, "
                            "which duplicates no logic: the statistics cell remains the one source of "
                            "truth and we simply keep what it says. (b) Added Cell 16: a tidy "
                            "results_all_runs.csv (the data-availability artifact -- no reviewer wants "
                            "2800 JSON files), analysis_summary.csv, draft figure captions, and a "
                            "README describing every artifact. It reads the RUN STORE rather than "
                            "in-memory state, so it can be re-run in a fresh session at any time "
                            "without repeating a GPU-second. Note on the captions: several plot titles "
                            "assert a conclusion (e.g. 'causally shapes final sparsity'); the drafted "
                            "captions describe what is shown and leave the inference to the text."),
        "R30_session_budget_removed":("The 22 h soft cutoff inside a 24 h Colab Pro+ session was "
                            "discarding ~2 h per session -- about 8 h, or 10% of the whole experiment "
                            "-- to guard against a failure the checkpoint layer already handles. "
                            "save_run writes to a temp path and os.replace()s it, which is atomic on "
                            "POSIX, so a session terminated mid-write cannot leave a truncated or "
                            "half-parsed record; the store is either the old file or the new one. The "
                            "genuine cost of an abrupt kill is therefore the single run in flight: "
                            "roughly two minutes at the small scales and about seven at dim1536, "
                            "which have_run() re-runs on the next Run All. Spending two hours to "
                            "avoid losing seven minutes is the wrong trade. session_budget_hours=None "
                            "now means 'run until the platform stops us'; budget_exceeded() returns "
                            "False and every clean-stop path simply never fires. Setting it back to a "
                            "number restores the soft cutoff unchanged, and the projection lines "
                            "report against 24 h when it is None."),
        "R29_cost_model_is_affine_not_proportional":("R28 fixed WHAT was timed; this fixes HOW it "
                            "was extrapolated. Even with a correct sec/step at one scale, pricing the "
                            "rest in proportion to FLOPs assumes constant throughput, and throughput "
                            "is not constant: measured MFU runs from ~4% at dim192 to ~18% at dim1536 "
                            "because small models are overhead-bound. Measured fixed overhead is "
                            "59.8 ms/step -- 89% of the entire step at dim192, 17% at dim1536. The "
                            "real dim1536/dim192 cost ratio is ~9x against a FLOP ratio of ~39x, so "
                            "extrapolating from the smallest scale over-priced the ladder ~2.5x "
                            "(202 h projected against 55-75 h measured). Preflight now times three "
                            "scales in steady state and fits sec/step = a + b*FLOPs_per_token, where "
                            "the intercept IS the fixed overhead, so both regimes are represented "
                            "rather than only one. The same steady-state timer replaced the compile "
                            "A/B's contaminated measurement, which had reported 1.06x; it now reports "
                            "1.29x at dim192, consistent with the 2.01x and 2.48x measured directly "
                            "at dim512 and dim1024. Note for the paper: this overhead structure is "
                            "measured evidence and belongs in the efficiency discussion, which is "
                            "otherwise entirely theoretical."),
        "R28_preflight_projection_measured_the_wrong_thing":("The Phase 0 projection reported 6459 "
                            "GPU-hours / 294 sessions for a grid that direct measurement puts at "
                            "73-87 hours / 4 sessions. Cause: sec_per_step was (total wall time of a "
                            "30-step run)/30, an interval that also contains model construction, "
                            "torch.compile warm-up paid once, and -- dominating -- the evaluation "
                            "machinery. eval_every is a FRACTION of run length, so a 30-step run "
                            "probes every 2 steps: roughly 260 evaluation forwards bolted onto 30 "
                            "training steps, all of which amortise away in a real 1221-step run. "
                            "Measured 2.066 s/step where the true steady-state figure at that scale "
                            "is 0.084. An error of this size does not merely mislead, it would have "
                            "caused a perfectly feasible experiment to be abandoned. Replaced with a "
                            "dedicated post-warm-up timing window containing no evaluation; the raw "
                            "smoke figure is still reported alongside it for comparison. The same "
                            "contamination explains why the preflight compile A/B showed only 1.08x "
                            "where a controlled benchmark showed 2.03x, and why its loss delta was "
                            "meaningless: both arms were sliding down the steep early descent from "
                            "the uninitialised-embedding starting point."),
        "R26_divergence_guard":("A benchmark on random tokens produced final losses of 48-122 where "
                            "the uniform-prediction baseline is log(50257)=10.83, rising monotonically "
                            "with model size -- i.e. divergence, and worse at scale. That run used no "
                            "warmup on unlearnable data so it does not predict the real corpus, but it "
                            "exposed a genuine hole: THERE WAS NO DIVERGENCE CHECK ANYWHERE. plateau_ok "
                            "asks whether the val trace stopped descending, and an exploded run has "
                            "also stopped descending, so it passes; val_ppl=exp(min(val,20)) then caps "
                            "the damage into a finite, innocent-looking number. Because every claim is "
                            "a seed-paired comparison, a few silent divergences in one arm would make "
                            "the paired test measure optimisation stability rather than the rho0 "
                            "mechanism -- manufacturing a large, significant, spurious effect for "
                            "H1/H2/H5. Diverged runs are now flagged, kept on disk (so have_run does "
                            "not re-run them), excluded from RESULTS, and counted in this manifest. "
                            "STILL OPEN, to be settled in Phase 0 on real data: lr_grid=[1e-3,2e-3,3e-3] "
                            "is aggressive for a 352M ternary model, and if all three diverge at the "
                            "top scales the calibration would silently select the least-bad explosion."),
        "R24_max_seq_len_never_rebound":("FATAL, and it would have stopped the grid on run #1. "
                            "Cfg carries TWO placeholder fields, vocab_size=8192 and "
                            "max_seq_len=256, both meant to be rebound once the real tokenizer is "
                            "known. Cell 9 rebound only vocab_size. RUN_CFG['seq_len'] is 512, and "
                            "LM.__init__ builds the RoPE cos/sin buffers from c.max_seq_len, so the "
                            "tables covered 256 positions while every batch carried 512 -- the first "
                            "forward pass of every run raised 'The size of tensor a (512) must match "
                            "the size of tensor b (256)'. Not a bias, not a slowdown: nothing could "
                            "train at all. The existing schema/geometry guard reported PASSED for all "
                            "28 configs because it only validated Cfg fields against one another and "
                            "never against RUN_CFG -- the two halves were each self-consistent and "
                            "mutually contradictory. Fixed by rebinding max_seq_len alongside "
                            "vocab_size, plus an assert that every config satisfies "
                            "max_seq_len >= RUN_CFG['seq_len']. Found by a standalone benchmark that "
                            "instantiated the real model at the real sequence length on CPU; the "
                            "notebook's own preflight would also have hit it, but only after the "
                            "corpus download, and the failure would have looked like a model bug "
                            "rather than a config-binding bug."),
        "R23_compile_enablement":("PRE-RUN, follows R22. GAMMA_SCALE['val'] was read inside "
                            "BitLinear.forward as a PYTHON FLOAT; dynamo specialises on python "
                            "scalar values, so the two arms that move gamma every step by design "
                            "(tai_warmup, sparse_sched) would have recompiled once per step and run "
                            "SLOWER than eager -- torch.compile was therefore unusable. The float is "
                            "now mirrored into a persistent 0-dim device tensor mutated in place, so "
                            "the graph survives a changing threshold. The float remains the source of "
                            "truth for every guard and log; all 17 write sites (including the three "
                            "ablation cells, which run their own training loops) go through "
                            "set_gamma(), and assert_quant_state_clean() cross-checks float against "
                            "mirror at every run boundary -- a future edit that bypasses set_gamma "
                            "fails loudly instead of silently ternarising at a stale threshold. "
                            "Quantiser output verified BITWISE identical across 36 "
                            "(shape, gamma, recipe) combinations. While writing that test, the "
                            "mirror check itself was found to use an absolute 1e-9 tolerance against "
                            "an fp32 tensor, which would have falsely aborted every sparse_sched run "
                            "(its target 1.6 is not exact in fp32); changed to relative 1e-6. "
                            "torch.compile remains OFF by default: preflight now runs an "
                            "eager-vs-compiled A/B at the smallest scale and reports measured speedup "
                            "AND loss parity, so the flag is flipped on evidence, not on hope. Also "
                            "added: fused AdamW (these models have many small parameters, so the "
                            "optimizer step was launch-bound) and a kernel profile in preflight, so "
                            "the NEXT optimisation is chosen from measurement rather than a model."),
        "R22_throughput_pass":("PRE-RUN performance revision, applied BEFORE any GPU result existed, "
                            "so no hypothesis, arm, or wording was chosen with knowledge of an outcome. "
                            "Nothing that affects optimisation dynamics was touched: batch_size, "
                            "seq_len, token budget, LR schedule, seed list, method set and all "
                            "pre-registered hypotheses are unchanged. Changes: (1) FAST_MODE trades "
                            "BIT-level reproducibility for flash-attention + cudnn autotuning; "
                            "seed-level comparability (same init, same data order per seed) is "
                            "preserved, which is what every claim here actually rests on -- see "
                            "reproducibility_class below; (2) chunked cross-entropy: algebraically "
                            "identical, ~4x lower peak VRAM, removes the OOM risk that threatened the "
                            "R19 large scales; (3) vectorised get_batch -- verified to draw "
                            "BIT-IDENTICAL token windows, so pre-patch runs stay comparable; "
                            "(4) lm-eval scored in padded batches instead of ~2400 batch-1 forwards, "
                            "and on the first lmeval_max_seeds seeds of each cell (same seeds for "
                            "every method, so the paired test is unaffected); (5) LR calibration runs "
                            "lr_cal_steps instead of the full budget -- its only output is an argmin "
                            "over 3 LRs; (6) wallclock_equal_h1 defaults off (secondary H1 reading, "
                            "re-ran the whole confirmatory grid); (7) the preflight wall-clock "
                            "projection used n_params() as its cost proxy, which excludes the "
                            "dim x vocab head -- dominant at small scales -- inflating the projection "
                            "by roughly an order of magnitude; replaced with a per-token FLOP model."),
        "R0_cfg_dataclass":("n_layers/n_heads/ffn_hidden were swallowed by an end-of-line "
                            "comment and were never dataclass fields -> every Cfg(...) call "
                            "raised TypeError. Fixed + schema/geometry asserts added."),
        "R0c_gqa_kv_heads":("n_kv_heads was n_heads//2, not a divisor for odd head counts "
                            "(e.g. dim320: 5%2!=0), which would have malformed the whole "
                            "decoder_v2 ladder behind H7. Now the largest proper divisor."),
        "R1_dead_fixed_point":("init_tai advertised an 8-step fixed-point search whose output "
                               "was bit-identical with the loop removed (vals recomputed each "
                               "iteration; dormant mass always below threshold; final std-norm "
                               "makes am/dm scale irrelevant). Replaced by the closed-form "
                               "construction + explicit realised-rho0 verification."),
        "R2_tai_hyperparams":("dm/jitter were hard-coded magic numbers with no sensitivity "
                              "analysis. Now explicit (TAI_HP), swept in Cell 13c-b, and the "
                              "separation bound is asserted rather than assumed."),
        "R3_plateau_gate":("the gate compared mean(tail) to final on the SINGLE-BATCH TRAIN "
                           "loss: dominated by batch noise (perfectly flat traces passed only "
                           "63-100% of the time depending on sigma), and a level test that "
                           "cosine LR decay satisfies mechanically. Replaced by a slope test "
                           "with CI on the held-out VAL trace, rejecting both still-descending "
                           "and ascending tails."),
        "R5_h4_data_variance":("H4 was measured at one val fold; the old cell called this "
                               "unfixable without persisting weights. The model is in memory at "
                               "the end of train_run, so all K folds are now scored there "
                               "(val_loss_folds) and H4's Levene runs on pooled seed x fold."),
        "R6_mechanism_control":("TAI was the only arm able to move rho0 at all, making H3 nearly "
                                "unfalsifiable. gamma_offset reaches the same rho0 via a constant "
                                "threshold scale c=gamma_scale_for_rho0(rho0) with a STANDARD "
                                "init, giving a mechanism-matched competitor for H3/H5."),
        "R13_lsuv_not_lsuv":("init_lsuv was orthogonal-init + a static rescale, not LSUV. The grid "
                             "sold {standard,lsuv} as two locked-rho0 FAMILIES (measured: 0.6900 vs "
                             "0.6906 -- one family) and took the per-seed BEST of them as a baseline, "
                             "i.e. max-selection over one family. Implementing LSUV properly showed it "
                             "is DEGENERATE here: BitLinear pre-normalises, so output variance at init "
                             "is 1.0004 and calibration rescales nothing. Measured (LSUV_DEGENERATE), "
                             "reported, and the H7 baseline rebuilt on bitnet158 + gamma_offset."),
        "R19_extended_scaling_ladder":("the original ladder topped out at 117M non-embed params, so "
                            "H6's scaling claim spanned <1 order of magnitude on an n=9 slope. Added "
                            "dim1280 (214M), dim1536 (352M) as EXPLORATORY points and dim2048 (783M) as "
                            "opt-in, extending the confirmatory-to-largest span past 1.5 orders of "
                            "magnitude (2.16, or 2.50 with the opt-in point). At a single-A100 budget "
                            "these cannot be trained compute-optimally, so they are token-equal and their "
                            "role is H6 SIGN-consistency, not an absolute-loss comparison -- stated "
                            "explicitly rather than presented as convergence. Verified: 352M model builds, "
                            "forwards, and takes a correct TAI init (realised rho0=0.7504) on CPU. Also "
                            "fixed the wall-clock projection (counted every declared config, incl. the "
                            "opt-in 783M, inflating the estimate) and a TARGET_LOSS print KeyError on "
                            "unrun scales."),
        "R18_undefined_vars_crashed_stats":("cell 18 -- the PRIMARY statistics cell computing H1-H4 -- "
                            "referenced two names that were never defined: `cp` (in the per-method summary "
                            "dict) and `cpm` (in the summary print). Both were remnants of a 'copy "
                            "accuracy' metric that was never implemented. Either one raises NameError on "
                            "the FIRST method of the FIRST scale, so the entire confirmatory analysis "
                            "produced nothing -- H1, H2, H3, H4 all crashed before printing. Found by "
                            "executing the cell on synthetic RESULTS; removed both. This bug was invisible "
                            "to every static read because the names looked like plausible locals."),
        "R17_placebo_inverted":("train_run routed init by method.startswith('tai'), which also matches "
                            "'tai_negctrl' -- so H3's PRE-REGISTERED FALSIFICATION arm silently received "
                            "the real, rho0-honouring TAI init, and apply_init's tai_negctrl branch was "
                            "unreachable dead code. The placebo therefore tracked rho0 as strongly as the "
                            "true arm (Spearman ~+1) against a frozen prediction of 'FLAT (~0)', and Cell "
                            "15 would have reported that under h3_strong_falsification: the paper's most "
                            "rigorous instrument would have concluded the primary contribution is an "
                            "artefact even if the real effect were sound. A broken placebo does not weaken "
                            "a claim, it inverts it. Routing is now explicit and order-independent, with a "
                            "per-run assertion that the placebo's realised rho0 stays at 0.69 regardless "
                            "of label (verified: flat 30.9% across labels 0.4-0.9 while the real arm moves "
                            "59.7% -> 9.8%)."),
        "R15_holdout_leak":("the split promised 'K interleaved val folds' but built K CONTIGUOUS "
                            "slices, and derived TRAIN per fold while only fold 0's TRAIN was ever "
                            "trained on. R5 then scored folds 1..K-1 -- which sit INSIDE that TRAIN "
                            "(measured: 100% overlap). H4's 'data-sampling variance' would have been "
                            "four train-loss readings mislabelled as held-out. Rebuilt: ONE genuinely "
                            "interleaved held-out region (every stride-th chunk across the corpus), "
                            "removed from training once, then split into K unseen sub-folds sharing a "
                            "single TRAIN."),
        "R16_sparsity_measured_wrong":("BitLinear.sparsity() hard-coded gamma_scale=1 and ignored "
                            "QUANT_MODE/GAMMA_SCALE, so it reported a counterfactual rather than the "
                            "forward pass that ran. gamma_offset (whose entire mechanism IS a shifted "
                            "gamma) measured ~31% at every rho0 -- flat -- which would have made H3's "
                            "mechanism control, H5's second frontier and fig10 'prove' that gamma_offset "
                            "cannot control sparsity when only the ruler was wrong (smoke-run: 31.0% "
                            "before vs 25.1% after, matching TAI's 24.9%). It also reported 31.2% for "
                            "fp16_control, which quantises nothing, and could never return 0 -- silently "
                            "disarming R14's leak guard. Now mirrors forward exactly; the canonical-"
                            "threshold reading survives as sparsity_at_default_gamma() for Cell 13b."),
        "R14_global_state_leak":("train_run set QUANT_MODE/QUANT_RECIPE globally and restored them ~64 "
                                 "lines later with no try/finally. SystemExit at the session boundary is "
                                 "this notebook's DESIGNED control flow, so an fp16_control run hitting "
                                 "the budget leaked full_precision=True into the next session: later "
                                 "'tai'/'bitnet158' runs would train in FULL PRECISION, look plausible, "
                                 "save under a valid run_id, and be skipped forever by have_run(). Fixed "
                                 "with a quant_state context manager (restores on every exit path), an "
                                 "entry assertion (fails loudly on a dirty state), and a post-hoc guard "
                                 "rejecting any ternary run reporting final_sparsity==0. Same leak "
                                 "closed in cells 22/24/25."),
        "R7_efficiency_framing":("the measured table was presented as 'efficiency'; with STE it "
                                 "measures the cost of SIMULATING ternary in bf16 and is expected "
                                 "to be slower than fp16. Reframed: theoretical deployment model "
                                 "is primary, simulation cost is secondary and labelled."),
      },
      # (R3) Does the 20M-token budget actually reach a plateau? If false, H2 is
      # restated as a token-equal claim rather than a convergence claim.
      "budget_sufficiency":BUDGET_CHECK if "BUDGET_CHECK" in dir() else None,
      "h2_wording_supported":(BUDGET_CHECK.get("h2_wording_supported")
                              if ("BUDGET_CHECK" in dir() and BUDGET_CHECK) else None),
      "h2_claim_as_stated":(("quality at convergence"
                             if ("BUDGET_CHECK" in dir() and BUDGET_CHECK
                                 and BUDGET_CHECK.get("h2_wording_supported"))
                             else "quality at a fixed 20M-token budget (convergence NOT established)")),
      # (R2) TAI hyper-parameter validity region + quality sensitivity
      "tai_hp_realisation":TAI_HP_REALISATION if "TAI_HP_REALISATION" in dir() else None,
      "tai_hp_quality_ablation":TAI_HP_ABL_STATS if "TAI_HP_ABL_STATS" in dir() else None,
      "tai_hp_defaults":{"dm":TAI_HP.get("dm") if "TAI_HP" in dir() else None,
                         "jitter":TAI_HP.get("jitter") if "TAI_HP" in dir() else None},
      # (R6) init-vs-threshold attribution: the decisive comparison
      "mechanism_comparison_tai_vs_gamma_offset":(MECHANISM_COMPARISON
                                                  if "MECHANISM_COMPARISON" in dir() else None),
      "gamma_offset_calibration":(GAMMA_OFFSET_C if "GAMMA_OFFSET_C" in dir() else None),
      # (R7) explicit statement of what the efficiency numbers are NOT
      # (R13) Was LSUV a real second baseline family, or a relabelled `standard`?
      # Measured, not assumed: BitLinear pre-normalises its input, so LSUV's
      # calibration provably rescales nothing (is_noop). Recorded so the paper
      # cannot claim baseline breadth the architecture does not permit.
      "lsuv_degeneracy":(LSUV_DEGENERATE if "LSUV_DEGENERATE" in dir() else None),
      # (R22) State the reproducibility CLASS rather than letting a reader assume
      # the strongest one. FAST_MODE keeps seed-level comparability (identical
      # init and identical data order per seed, verified) but not bit-level
      # reruns. Every claim in this work is a seed-aggregated statistic, so the
      # weaker guarantee is the one the claims need -- but the paper must say so.
      # (R33) LR provenance. Two things a reviewer will ask about the calibration
      # belong in the record, not in a console scrollback: whether each cell's LR*
      # ended up INTERIOR to the probed grid, and whether the 300-step proxy
      # agrees with a full-length curve. Both are stored so the paper can cite
      # them as measurements rather than assertions.
      # (R35) The single most consequential sentence in the paper is how H1/H2
      # results are worded, and it must follow the measurement rather than the
      # pre-registration's expectation. plateau_status is computed from every
      # recorded run; h2_operative_wording is derived from it, so the manifest
      # states in one place which phrasing the evidence licenses.
      # (R37) H3's operative readout. "Does final sparsity move with rho0" is
      # trivially yes; RETENTION -- the fraction of the initial intervention still
      # present at the end -- is what decides whether the initial operating point
      # is a usable design knob. The placebo arm's retention sits alongside the
      # real one so the falsification is visible in the same field.
      "rho0_retention":(RHO0_RETENTION if "RHO0_RETENTION" in dir() else None),
      "h3_operative_wording":(
          "rho0 leaves a small, strictly ordered, highly reliable RESIDUE in final "
          "sparsity on top of a strong attractor; it does not durably control it. "
          "Report the retention coefficient, not merely the sign of the effect."),
      # (R37) H3's operative reading, derived from the retention measurement
      # rather than from the pre-registered expectation.
      # (R43) H6's operative reading. The sign test is the pre-registered primary
      # and is unaffected by the slope estimator; the slope is reported with the
      # conservative scale-level interval and flagged when estimators disagree.
      "h6_slope_estimators":(SCALE_SLOPE_ALL if "SCALE_SLOPE_ALL" in dir() else None),
      "h6_operative_wording":(
          "direction consistent across scales; magnitude NOT resolved as scale-varying "
          "(primary scale-level CI spans zero; seed-clustered intervals are "
          "anticonservative for a scale-level question -- see R43)"
          if ("SCALE_SLOPE_CI" in dir() and SCALE_SLOPE_CI
              and SCALE_SLOPE_CI[0]<=0<=SCALE_SLOPE_CI[1])
          else "scale-dependent magnitude resolved by the primary scale-level estimator"),
      # (R45) H2 has TWO entries and they must never be conflated: the
      # pre-registered equivalence test (untestable here -- no run reached
      # plateau) and the token-equal companion that the data can support.
      "h2_token_equal_companion":(H2_TOKEN_EQUAL if "H2_TOKEN_EQUAL" in dir() else None),
      "h2_prereg_status":("UNTESTABLE: the pre-registered H2 is an equivalence claim about "
                          "CONVERGED quality over plateau-verified paired seeds, and no run "
                          "reached plateau at the 20M-token budget (0/760; independently "
                          "confirmed by the 2x-budget check in budget_sufficiency). Reported as "
                          "unresolved, NOT as a null result. See h2_token_equal_companion for the "
                          "weaker, testable claim and R45 for why the companion is not a post-hoc "
                          "substitution."),
      "h3_retention":(H3_RETENTION if "H3_RETENTION" in dir() else None),
      "h3_operative_wording":(
          ("rho0 causally controls final sparsity"
           if ("H3_RETENTION" in dir() and (H3_RETENTION.get("retention_fraction") or 0)>=0.5) else
          "rho0 leaves a small, perfectly ordered, highly reproducible RESIDUAL on final "
          "sparsity; the large majority of the initial intervention is erased by training, "
          "which pulls every operating point back toward an attractor near the value the "
          "`standard` arm reaches unaided. H3 is NOT supported in its strong pre-registered "
          "form and must be reported as a bounded residual effect with the retention "
          "fraction stated.")),
      "plateau_status":(plateau_summary() if "plateau_summary" in dir() else None),
      "h2_operative_wording":(
          "at convergence (plateau verified)"
          if ("PLATEAU_STATS" in dir() and PLATEAU_STATS.get("n")
              and PLATEAU_STATS["n_ok"]/max(PLATEAU_STATS["n"],1)>=0.5)
          else "at a fixed 20M-token budget (NOT converged; the val trace is still "
               "descending at budget end, so no claim may say 'converged quality')"),
      # (R42 FIX) LRSTAR and LR_CURVE are keyed by the TUPLE (size, method), and
      # json.dumps refuses tuple keys outright -- `default=` only handles unknown
      # VALUES, never keys, so the fallback did not save it. The manifest write
      # died with "keys must be str, int, float, bool or None, not tuple" AFTER
      # every phase had completed, discarding the provenance record for the whole
      # experiment. Flattened to "size/method" strings.
      "lr_star":({f"{k[0]}/{k[1]}":v for k,v in LRSTAR.items()} if "LRSTAR" in dir() else None),
      "lr_curves":({f"{k[0]}/{k[1]}":v for k,v in LR_CURVE.items()} if "LR_CURVE" in dir() else None),
      "lr_boundary_cells":(_lr_boundary_cells() if "LRSTAR" in dir() else None),
      "lr_proxy_check":(LR_PROXY_CHECK if "LR_PROXY_CHECK" in dir() else None),
      "diverged_runs":(DIVERGED_RUNS if "DIVERGED_RUNS" in dir() else None),
      "divergence_rule":("val_loss > 1.25*log(vocab) (=%.2f), non-finite, or a late "
                         "1.5x rise in the val trace. A model that learned NOTHING still "
                         "scores log(vocab); above that is divergence, not poor training."
                         % (1.25*__import__("math").log(50257))),
      "reproducibility_class":("statistical (seed-level): identical init + data order per seed; "
                               "bit-level rerun equality NOT guaranteed"
                               if (globals().get("FAST_CFG") or {}).get("fast") else
                               "bit-level: deterministic kernels, reruns are bit-identical"),
      "fast_mode":(FAST_CFG if "FAST_CFG" in dir() else None),
      "baseline_family_note":("standard and lsuv are ONE family in this architecture "
                              "(rho0 0.6900 vs 0.6906; LSUV degenerate under RMSNorm "
                              "pre-normalisation). The genuine second family is "
                              "gamma_offset, which differs at the operating point."),
      "efficiency_limitation":("no ternary kernel benchmarked; deployment numbers are an "
                               "arithmetic model, measured numbers are simulation overhead"),
      "fp16_init_ablation":FP16_INIT_ABL if "FP16_INIT_ABL" in dir() else None,
      "data_variance_H4":DATA_VARIANCE_H4 if "DATA_VARIANCE_H4" in dir() else None,
      "scaling_law_widthctrl":SCALING_LAW.get("val_loss_widthctrl") if "SCALING_LAW" in dir() else None,
      "val_split":{"n_folds":RUN_CFG.get("val_folds"),"guard_band_tokens":RUN_CFG.get("seq_len"),
                   "primary_fold":0,"protocol":"interleaved blocks + per-side guard band (leakage-free)"},
      "compute_cost":{"total_run_hours":round(sum(r.get("wall_sec",0) for r in RESULTS)/3600,2),
                      "n_runs_timed":sum(1 for r in RESULTS if r.get("wall_sec"))},
      "lmeval_metrics":["acc (token-norm)","acc_norm (byte-norm)"],
      "stats_dependency_note":("All confirmatory tests (paired t, Wilcoxon, bootstrap CI, Holm, TOST equivalence, Levene) use scipy only and run without statsmodels; statsmodels is used ONLY for the secondary H6 mixed-effects slope, which falls back to a descriptive per-scale OLS if statsmodels is absent. No confirmatory claim depends on an optional package."),
      "tokenizer":"gpt2-bpe",
      "main_results":[{k:r.get(k) for k in ("size","method","seed","val_loss","val_ppl","final_sparsity","steps_to_target","nexttoken_acc","arc_easy_acc","arc_easy_acc_norm","hellaswag_acc","hellaswag_acc_norm","plateau_ok","recipe","wall_sec")} for r in RESULTS],
      "rho0_sweep":[{k:s.get(k) for k in ("rho0","seed","init_active","final_sparsity","val_loss")} for s in SWEEP],
      "analysis":{sz:{m:{k:(round(v,4) if isinstance(v,float) else v) for k,v in d.items() if not k.startswith("_")} for m,d in summ.items()} for sz,summ in ANALYSIS.items()}}
    # ========================================================================
    # (R42) SANITISE KEYS BEFORE SERIALISING — the manifest is the last artifact
    # produced, so an exception here destroys the provenance record for a run that
    # has already cost the entire compute budget.
    # json.dumps rejects any non-(str,int,float,bool,None) KEY outright, and the
    # `default=` hook does not help: it is consulted for unknown VALUES only.
    # LRSTAR and LR_CURVE are keyed by the tuple (size, method) and are fixed at
    # source above, but several other collections here are assembled by analysis
    # cells that may key on tuples too, and auditing each one by hand is exactly
    # the kind of check that rots the next time a dict is added. Normalising once,
    # recursively, is cheap and cannot be forgotten.
    # (R44) Uses the shared json_safe() from Cell 10, but does NOT depend on it.
    # The manifest is the last artifact of a multi-day run; making it fail because
    # an earlier cell was skipped, re-ordered, or run in a fresh kernel would throw
    # away the provenance record for the whole experiment over a missing helper.
    # A local fallback costs four lines and removes that failure mode entirely.
    if "json_safe" not in dir():
        def json_safe(o):
            if isinstance(o,dict):
                return {("/".join(map(str,k)) if isinstance(k,tuple) else str(k)):json_safe(v)
                        for k,v in o.items()}
            if isinstance(o,(list,tuple)): return [json_safe(x) for x in o]
            return o
    manifest=json_safe(manifest)
    manifest["hash"]=hashlib.sha256(json.dumps(manifest,sort_keys=True,default=str).encode()).hexdigest()[:16]
    json.dump(manifest,open(os.path.join(PATHS["results"],"tai_manifest.json"),"w"),indent=2,default=str)
    open(os.path.join(PATHS["results"],"scale_up.txt"),"w").write(
    "Cluster scale-up: this artifact already spans a 9-scale ladder to >=50M non-embed and a "
    "20M-token plateau-verified budget. Next step is >=2 scales above 100M with the same "
    "token-equal protocol and the bitnet158 independent recipe, to test whether the H6 "
    "mixed-effects slope CI excludes zero at larger scale.\n")
    notify("RUN COMPLETE — all phases finished",
           f"runs {len(RESULTS)}+{len(SWEEP)} | manifest hash {manifest['hash']}\nArtifacts in {PROJECT_ROOT}")
    VALIDITY=("SYNTHETIC FALLBACK WAS USED — downstream lm-eval numbers are NOT "
              "meaningful; rerun with network access for FineWeb-EDU." if USED_SYNTHETIC
              else "Real FineWeb-EDU (GPT-2 BPE) corpus used; lm-eval %s."
                   % ("scored" if LMEVAL_AVAILABLE else "tasks unavailable"))
    manifest["validity_banner"]=VALIDITY
    json.dump(manifest,open(os.path.join(PATHS["results"],"tai_manifest.json"),"w"),indent=2,default=str)
    print("\n"+"="*70+"\nDATA VALIDITY:",VALIDITY,"\n"+"="*70)
    print("Manifest hash:",manifest["hash"],"| all artifacts in",PROJECT_ROOT)


In [ ]:
# === CELL 16: (R31) PUBLICATION EXPORT — tidy data + captions + index ========
# Everything above writes figures, two LaTeX tables and the manifest. Three
# things a reviewer or a co-author will ask for were still missing, and all
# three are cheap because they read the RUN STORE rather than in-memory state:
# this cell can therefore be re-run at any time, in a fresh session, without
# repeating a single GPU-second.
#
#   results_all_runs.csv  — one tidy row per completed run. This is the data
#                           availability artifact; per-run JSONs are the source
#                           of truth but no reviewer wants 2800 files.
#   analysis_summary.csv  — the per-(scale,method) aggregates behind the figures.
#   figure_captions.md    — draft captions. Plot titles are working labels, not
#                           captions, and several state a CONCLUSION ("causally
#                           shapes") which a caption must not assert before the
#                           text has earned it. These are neutral starting points.
#   README.md             — what each artifact is, so the results folder is
#                           self-describing six months from now.
if not _require("PATHS"):
    print("[skip] export: project paths unavailable.")
else:
    import csv,os,json,datetime
    RES=PATHS["results"]; FIG=PATHS["figures"]
    _rows=all_runs()
    _cols=["phase_hint","size","method","seed","recipe","val_loss","val_ppl",
           "final_sparsity","plateau_ok","plateau_gap","steps_to_target",
           "nexttoken_acc","diverged","gamma_offset_c","rho0",
           "arc_easy_acc","arc_easy_acc_norm","hellaswag_acc","hellaswag_acc_norm"]
    _p=os.path.join(RES,"results_all_runs.csv")
    with open(_p,"w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=_cols,extrasaction="ignore"); w.writeheader()
        for r in _rows:
            r=dict(r)
            r.setdefault("rho0",(r.get("opts") or {}).get("rho0"))
            for _k in ("steps","loss","val_steps","val_trace","val_loss_folds"): r.pop(_k,None)
            w.writerow(r)
    print(f"Saved results_all_runs.csv  ({len(_rows)} runs)")

    if "ANALYSIS" in dir() and ANALYSIS:
        _p2=os.path.join(RES,"analysis_summary.csv")
        _keys=sorted({k for sz in ANALYSIS.values() for m in sz.values()
                      for k in m if not k.startswith("_")})
        with open(_p2,"w",newline="") as f:
            w=csv.writer(f); w.writerow(["size","method"]+_keys)
            for sz,summ in ANALYSIS.items():
                for m,d in summ.items():
                    w.writerow([sz,m]+[d.get(k) for k in _keys])
        print("Saved analysis_summary.csv")

    _caps={
     "fig1_convergence.png":"Training loss against optimisation step for each initialisation "
        "strategy at the primary confirmatory scale, averaged over 15 seeds at a matched 20M-token "
        "budget. Shaded bands are bootstrap 95% CIs.",
     "fig2_final_loss.png":"Final held-out loss by initialisation strategy at each confirmatory "
        "scale. Points are per-seed values; bars show the mean with a bootstrap 95% CI.",
     "fig3_rho0_sweep.png":"Final weight sparsity as a function of the initial active ratio rho0, "
        "for the TAI arm and the rho0-labelled placebo. The placebo holds the true split fixed at "
        "0.69 while varying only the label, so any dependence it shows is measurement artifact.",
     "fig4_lmeval.png":"Zero-shot accuracy on ARC-Easy and HellaSwag (log-likelihood scoring) by "
        "initialisation strategy. Both plain and length-normalised accuracy are reported.",
     "fig5_pareto.png":"Quality against sparsity across the rho0 sweep. Each point is one operating "
        "point; the frontier shows the achievable trade-off at a fixed token budget.",
     "fig6_scale.png":"Effect size across the model-scale ladder, with the mixed-effects slope and "
        "its confidence interval. Depth-controlled points are shown separately.",
     "fig7_h3_falsification.png":"Pre-registered falsification for H3. Left: the real rho0 sweep "
        "against the placebo arm. Right: signal-to-noise ratio between them.",
     "fig8_init_ablation.png":"Decomposition of the effect into initialisation distribution versus "
        "threshold control, comparing TAI against the mechanism-matched gamma-offset arm.",
     "fig9_two_slope.png":"Scaling slope estimated with width and depth varying together, against "
        "the depth-controlled ladder in which depth is held fixed.",
     "fig10_mechanism.png":"Left: initial operating point reached by each mechanism. Right: paired "
        "difference between TAI and the gamma-offset control, with a 95% CI; an interval covering "
        "zero indicates the two mechanisms are matched.",
    }
    with open(os.path.join(RES,"figure_captions.md"),"w") as f:
        f.write("# Figure captions (draft)\n\n")
        f.write("Plot titles in the notebook are working labels. Several assert a conclusion; a\n"
                "caption should describe what is shown and let the text draw the inference.\n"
                "Rewrite freely -- these are neutral starting points.\n\n")
        for k,v in _caps.items():
            f.write(f"**{k}**\n\n{v}\n\n")
    print("Saved figure_captions.md")

    with open(os.path.join(RES,"README.md"),"w") as f:
        f.write(f"""# TAI results — generated {datetime.datetime.now():%Y-%m-%d %H:%M}

| file | what it is |
|---|---|
| `tai_manifest.json` | full provenance: config, every revision R0-R31 with its reasoning, per-cell analysis summary, diverged-run list, reproducibility class, SHA hash |
| `results_all_runs.csv` | one tidy row per completed run — the data availability artifact |
| `analysis_summary.csv` | per-(scale, method) aggregates behind the figures |
| `results_main.tex` | main results table, LaTeX |
| `results_controls.tex` | control-arm table, LaTeX |
| `figure_captions.md` | draft captions for fig1-fig10 |
| `../figures/*.png` | 300 dpi figures |
| `../runs/*.json` | raw per-run records (source of truth; every table above is derived from these) |
| `../logs/console_*.log` | mirrored console, including the H1-H8 verdicts printed by the statistics cell |

Reproducing the analysis needs no GPU: the run store is complete, so the
statistics, figure and export cells can be re-run from it at any time.
""")
    print("Saved README.md")
    print("\nExport complete ->",RES)
